# 🛰️ 공개 위성영상 장비·차량 객체분류 정확도 평가

**어서오세요, 전우님!** 이번 프로젝트에서 전우님은 가상의 **위성분석 TF**에 바이브 코딩 담당으로 합류합니다.
팀원들의 요청을 메신저로 받아, 가상 AI에게 프롬프트를 보내 위성영상 판독 데이터를 분석하는 코드와 보고 문서를 만들어요. 위성영상 판독은 소재일 뿐이고, 이번 프로젝트의 중심은 프롬프트를 쓰고 결과를 확인하는 방법이에요.
업무 요청을 프롬프트로 바꾸고, AI가 만든 코드와 문서를 실행해 확인하고, 고쳐 달라고 다시 요청하는 **바이브 코딩** 과정을 직접 해 봐요.

### ▶ 시작하는 방법
0. 이번 프로젝트는 **Colab 기본 CPU 런타임**에서 끝까지 진행할 수 있어요. 런타임 유형은 바꾸지 않아도 돼요.
1. 위쪽 메뉴에서 **런타임 → 모두 실행**을 누르세요. (단축키 `Ctrl` + `F9`)
2. "이 노트북은 Google에서 작성하지 않았습니다"라는 안내가 나오면 **무시하고 계속하기**를 누르세요.
3. 아래 **🚀 실습 준비하기** 셀의 진행 막대가 **100%가** 될 때까지 기다려 주세요.
4. 그 아래에 메신저 화면이 열리면 **[시작하기 ▶] 버튼**을 누르고, **M1부터** 차례로 진행해 주세요.

### 💡 알아 두면 좋아요
- 인터넷이 없어도 돼요. 설치하거나 내려받는 것 없이, 필요한 가상 데이터와 가상 AI가 모두 이 파일 안에 들어 있어요.
- 진행 상황은 노트북이 켜져 있는 동안만 저장돼요. 오래 자리를 비워 연결이 끊기면 **런타임 → 모두 실행**을 다시 눌러 주세요. (처음부터 다시 시작해요)
- 데이터, 팀원, 장소, 판독 상황은 모두 교육용 가상 자료예요. 실제 조직·위치·장비 정보는 쓰지 않아요.
- 이번 과정의 AI는 전우님이 고른 문장에 맞춰 준비된 응답을 돌려주는 가상 AI예요. 다만 AI가 만든 코드는 실습 환경에서 실제로 실행돼요. 같은 방법을 실제 AI 도구에 그대로 쓸 수 있어요.
- 모델 학습이나 영상 내려받기는 하지 않아요. 가상 모델의 점수를 미리 만들어 두었어요.

In [ ]:
#@title 🚀 [먼저 실행] 실습 준비하기 { display-mode: "form" }
#@markdown ▶ 왼쪽의 실행 버튼을 누르거나, 메뉴에서 **런타임 → 모두 실행**을 누르세요. 아래 진행 막대가 **100%가** 되면 준비가 끝나요.
FILL_ANSWERS = False   # 학습자용: 빈 상태로 시작해요
from IPython.display import HTML, display

_SETUP_STEPS = ["메신저 · 진행 상태 · 가이드북", "미션 활동 · 가상 AI", "첫 화면"]


def _setup_html(pct, done, msg):
    color = "#16a34a" if pct >= 100 else "#0b57d0"
    items = "".join(f"<li style='margin:3px 0;color:#111827'>{'✅' if i < done else '⏳' if i == done else '▫️'} {name}</li>"
                    for i, name in enumerate(_SETUP_STEPS))
    return (f"<div style='font-family:sans-serif;max-width:640px;border:2px solid {color};border-radius:12px;padding:12px 16px;"
            f"background:#fff;color:#111827'><div style='font-weight:bold;font-size:16px;color:#111827'>🚀 실습 준비 {pct}%</div>"
            f"<div style='height:16px;background:#e3e8ef;border-radius:8px;overflow:hidden;margin:8px 0'>"
            f"<div style='height:100%;width:{pct}%;background:{color}'></div></div>"
            f"<div style='font-size:14px;color:#111827'>{msg}</div>"
            f"<ul style='list-style:none;padding:0;margin:8px 0 0;font-size:13.5px'>{items}</ul></div>")


_setup_bar = display(HTML(_setup_html(2, 0, "준비를 시작합니다...")), display_id=True)


def _setup(pct, done, msg):
    _setup_bar.update(HTML(_setup_html(pct, done, msg)))

# ===== 프로젝트 공통: 메신저 · 진행 상태 · 산출물 · 가이드북 =====
# 로그인이 없음: 개인 정보를 받지 않고 학습자는 모두 '전우님'으로 부름 (2026-11-05 피드백)
# 미션은 처음부터 차례로 하는 것이 원칙. 뒷부분을 미리 보려는 사람을 위해 잠그지 않고 열어 둠 (2026-11-05 피드백)
import base64, html, inspect, io, json, math, os, random, re, textwrap, threading, time, traceback
from datetime import date, datetime, timedelta, timezone
import numpy as np
from IPython.display import JSON
from google.colab import output

# FILL_ANSWERS(선생님용)는 이 셀 맨 위의 체크박스로 정함: True면 모든 활동의 값을 정답으로 채워 둔 상태로 시작함
BOOT_ID = datetime.now().strftime("%H%M%S%f")   # 이 셀을 실행할 때마다 바뀜: 화면이 예전 실행에 머물러 있으면 알아채고 처음 화면으로 돌림

SPEC = {'project': {'id': 'pbl_isr_sat_a1',
             'emoji': '🛰️',
             'title': '공개 위성영상 장비·차량 객체분류 정확도 평가',
             'goal': '업무를 AI에게 말로 설명해 도구와 문서를 만드는 바이브 코딩을 익혀요. 위성영상으로 장비와 차량을 판독하고 AI의 정확도를 평가하는 일을 소재로, 프롬프트를 쓰고, AI가 만든 코드를 실행해 확인하고, 고쳐 달라고 다시 '
                     '요청해요.',
             'welcome_lead': '가상의 위성분석 TF에 바이브 코딩 담당으로 합류했어요. 팀원들의 요청을 프롬프트로 바꿔 AI와 함께 도구와 문서를 만들어 봐요. M1부터 차례로 진행해 보세요.',
             'team_name': '위성분석 TF',
             'my_role': '바이브 코딩 담당',
             'use_label': '바이브 코딩에서는',
             'intro': '팀원들의 요청을 메신저로 받아, 가상 AI에게 프롬프트를 보내 위성영상 판독 데이터를 분석하는 코드와 보고 문서를 만들어요. 위성영상 판독은 소재일 뿐이고, 이번 프로젝트의 중심은 프롬프트를 쓰고 결과를 확인하는 방법이에요.',
             'banned_phrases': ['실제 위성영상'],
             'notes': ['데이터, 팀원, 장소, 판독 상황은 모두 교육용 가상 자료예요. 실제 조직·위치·장비 정보는 쓰지 않아요.',
                       '이번 과정의 AI는 전우님이 고른 문장에 맞춰 준비된 응답을 돌려주는 가상 AI예요. 다만 AI가 만든 코드는 실습 환경에서 실제로 실행돼요. 같은 방법을 실제 AI 도구에 그대로 쓸 수 있어요.',
                       '모델 학습이나 영상 내려받기는 하지 않아요. 가상 모델의 점수를 미리 만들어 두었어요.'],
             'members': {'kim': ['김팀장', '위성분석 TF', '팀장', '#c2410c'],
                         'han': ['한지우', '위성분석 TF', '판독 담당', '#1d4ed8'],
                         'park': ['박도윤', '위성분석 TF', '판독 담당', '#b91c1c'],
                         'choi': ['최서연', '위성분석 TF', '검토 담당', '#15803d'],
                         'lee': ['이정민', '수요 부서', '요청 담당', '#7e22ce'],
                         'jung': ['정우진', 'AI 모델팀', '모델 담당', '#0f766e'],
                         'me': ['전우님', '위성분석 TF', '바이브 코딩 담당', '#334155']},
             'channels': {'notice': '# 프로젝트-공지',
                          'team': '# 판독팀',
                          'dm_kim': '✉ 김팀장',
                          'dm_lee': '✉ 이정민 (수요 부서)',
                          'dm_choi': '✉ 최서연',
                          'dm_jung': '✉ 정우진 (AI 모델팀)',
                          'dm_han': '✉ 한지우'},
             'always_channels': ['notice', 'team'],
             'welcome': [['notice', 'kim', '{call}, 위성분석 TF에 오신 것을 환영합니다 👋 이번에는 판독 업무에 쓸 작은 도구와 문서를 AI와 함께 만들어요.'],
                         ['notice', 'kim', '왼쪽 메뉴의 M1부터 차례로 진행해 주세요. 코드를 외울 필요는 없어요. AI에게 어떻게 설명할지가 중요해요.'],
                         ['team', 'han', '{call}, 판독 담당 한지우예요. 제가 데이터를 넘겨드리면 AI와 함께 표와 문서로 만들어 주세요 🙂']],
             'stages': [{'no': 1, 'name': '위성영상 이해와 판독 기초', 'subtitle': '프롬프트의 기본', 'missions': ['m1', 'm2']},
                        {'no': 2, 'name': '영상 데이터 준비: 메타데이터·진위 검증·전처리', 'subtitle': '고치고 쪼개서 요청하기', 'missions': ['m3', 'm4']},
                        {'no': 3, 'name': '탐지·분류와 기만·은폐 대응', 'subtitle': '결과를 믿기 전에 검증하기', 'missions': ['m5', 'm6']},
                        {'no': 4, 'name': '신뢰도와 사람-AI 협업 판독', 'subtitle': '표현 규칙과 안전한 활용', 'missions': ['m7', 'm8']},
                        {'no': 5, 'name': '사후검토와 협업형 보고 체계', 'subtitle': '검토하고 묶어서 보고하기', 'missions': ['m9', 'm10']}],
             'vfiles': {'m1_equipment.csv': 'equipment,length_m,width_m\ntank,6.9,3.6\napc,6.7,3.2\nspg,7.2,3.4\ntruck,8.4,2.5\n',
                        'm2_reads.csv': 'object_id,reader_a,reader_b,reader_c\n'
                                        'OBJ01,tank,tank,decoy \n'
                                        'OBJ02,tank,tank ,tank\n'
                                        'OBJ03,decoy,Decoy,decoy \n'
                                        'OBJ04,Unknown,apc,apc\n'
                                        'OBJ05,decoy,tank,tank\n'
                                        'OBJ06,tank,tank,tank \n'
                                        'OBJ07,unknown ,Unknown,unknown\n'
                                        'OBJ08,decoy,Unknown,Decoy\n'
                                        'OBJ09,apc,Apc,apc\n'
                                        'OBJ10,truck,truck,truck \n'
                                        'OBJ11,tank,truck,truck\n'
                                        'OBJ12,tank,tank,tank\n'
                                        'OBJ13,truck,truck,unknown\n'
                                        'OBJ14,Decoy,decoy,decoy\n'
                                        'OBJ15,Apc,apc,apc\n'
                                        'OBJ16,truck,truck,apc\n'
                                        'OBJ17,truck,truck ,decoy\n'
                                        'OBJ18,unknown,unknown,Unknown\n'
                                        'OBJ19,decoy,apc ,apc\n'
                                        'OBJ20,apc,apc,apc \n'
                                        'OBJ21,apc,apc,apc\n'
                                        'OBJ22,tank,tank,tank\n'
                                        'OBJ23,Decoy,unknown,Apc\n'
                                        'OBJ24,tank,tank ,tank\n',
                        'm3_scenes.csv': 'image_id,captured_utc,sun_azimuth_deg,shadow_azimuth_deg\n'
                                         'S01,2025-05-12T05:30:00Z,150,331\n'
                                         'S02,2025-05-12T06:10:00Z,165,346\n'
                                         'S03,2025-05-12T06:40:00Z,178,3\n'
                                         'S04,2025-05-12T07:00:00Z,190,262\n'
                                         'S05,2025-05-13T05:20:00Z,148,327\n'
                                         'S06,2025-05-13T06:00:00Z,160,342\n'
                                         'S07,2025-05-13T06:30:00Z,172,40\n'
                                         'S08,2025-05-13T07:15:00Z,196,18\n',
                        'm3_scenes_new.csv': 'image_id,captured_utc,sun_azimuth_deg,shadow_azimuth_deg\n'
                                             'N01,2025-05-20T05:30:00Z,152,330\n'
                                             'N02,2025-05-20T06:00:00Z,161,343\n'
                                             'N03,2025/05/20 06:30,173,355\n'
                                             'N04,2025-05-20T07:00:00Z,188,7\n'
                                             'N05,2025/05/20 07:30,201,95\n',
                        'm4_tiles.csv': 'tile_id,sensor,gsd_m,cloud_pct,off_nadir_deg,crs\n'
                                        'T01,optical,0.5,4,12,EPSG:32652\n'
                                        'T02,optical,0.5,18,28,EPSG:32652\n'
                                        'T03,optical,0.5,35,15,EPSG:32652\n'
                                        'T04,optical,1.0,2,10,EPSG:32652\n'
                                        'T05,sar,0.5,0,33,EPSG:32652\n'
                                        'T06,sar,0.5,0,22,EPSG:4326\n'
                                        'T07,optical,0.52,8,25,EPSG:32652\n'
                                        'T08,optical,0.5,27,36,EPSG:32652\n',
                        'm5_detections.csv': 'det_id,airport_id,split,cloud_band,off_nadir_band,is_true,score\n'
                                             'D001,A1,val,clear,high,0,0.153\n'
                                             'D002,A1,val,cloudy,low,1,0.774\n'
                                             'D003,A1,val,cloudy,low,0,0.49\n'
                                             'D004,A1,val,clear,low,0,0.348\n'
                                             'D005,A1,val,clear,low,0,0.096\n'
                                             'D006,A1,val,cloudy,low,1,0.73\n'
                                             'D007,A1,val,clear,low,0,0.205\n'
                                             'D008,A1,val,clear,low,0,0.376\n'
                                             'D009,A1,val,clear,low,1,0.552\n'
                                             'D010,A1,val,cloudy,low,1,0.491\n'
                                             'D011,A1,val,clear,low,0,0.301\n'
                                             'D012,A1,val,cloudy,low,0,0.215\n'
                                             'D013,A1,val,clear,low,1,0.93\n'
                                             'D014,A1,val,clear,low,0,0.202\n'
                                             'D015,A1,val,clear,low,1,0.771\n'
                                             'D016,A1,val,clear,high,1,0.849\n'
                                             'D017,A1,val,clear,low,0,0.162\n'
                                             'D018,A1,val,clear,low,0,0.225\n'
                                             'D019,A1,val,clear,low,0,0.298\n'
                                             'D020,A1,val,clear,low,1,0.893\n'
                                             'D021,A1,val,clear,low,1,0.907\n'
                                             'D022,A1,val,clear,high,0,0.442\n'
                                             'D023,A1,val,cloudy,low,0,0.611\n'
                                             'D024,A1,val,clear,low,1,0.825\n'
                                             'D025,A1,val,clear,high,1,0.687\n'
                                             'D026,A1,val,clear,low,0,0.263\n'
                                             'D027,A1,val,clear,high,1,0.856\n'
                                             'D028,A1,val,cloudy,low,1,0.332\n'
                                             'D029,A1,val,clear,low,0,0.105\n'
                                             'D030,A1,val,clear,low,1,0.821\n'
                                             'D031,A1,val,clear,low,1,0.908\n'
                                             'D032,A1,val,clear,high,1,0.799\n'
                                             'D033,A1,val,clear,low,0,0.098\n'
                                             'D034,A1,val,cloudy,high,0,0.444\n'
                                             'D035,A1,val,cloudy,low,0,0.219\n'
                                             'D036,A1,val,clear,high,1,0.805\n'
                                             'D037,A1,val,clear,low,0,0.261\n'
                                             'D038,A1,val,clear,high,0,0.097\n'
                                             'D039,A1,val,cloudy,low,1,0.494\n'
                                             'D040,A1,val,clear,low,0,0.717\n'
                                             'D041,A1,val,clear,high,1,0.768\n'
                                             'D042,A1,val,clear,high,0,0.257\n'
                                             'D043,A1,val,cloudy,low,0,0.321\n'
                                             'D044,A1,val,clear,low,0,0.335\n'
                                             'D045,A1,val,clear,low,1,0.886\n'
                                             'D046,A1,val,clear,low,1,0.947\n'
                                             'D047,A1,val,cloudy,low,0,0.072\n'
                                             'D048,A1,val,cloudy,low,0,0.583\n'
                                             'D049,A1,val,cloudy,low,1,0.647\n'
                                             'D050,A1,val,clear,high,0,0.238\n'
                                             'D051,A1,val,cloudy,low,1,0.816\n'
                                             'D052,A1,val,cloudy,high,0,0.445\n'
                                             'D053,A1,val,clear,high,1,0.325\n'
                                             'D054,A1,val,clear,high,1,0.28\n'
                                             'D055,A1,val,clear,high,0,0.149\n'
                                             'D056,A1,val,clear,high,1,0.507\n'
                                             'D057,A1,val,clear,low,0,0.223\n'
                                             'D058,A1,val,clear,high,0,0.31\n'
                                             'D059,A1,val,clear,low,1,0.686\n'
                                             'D060,A1,val,clear,low,0,0.361\n'
                                             'D061,A2,val,cloudy,low,0,0.194\n'
                                             'D062,A2,val,clear,high,1,0.763\n'
                                             'D063,A2,val,clear,high,1,0.695\n'
                                             'D064,A2,val,clear,high,0,0.283\n'
                                             'D065,A2,val,clear,low,0,0.144\n'
                                             'D066,A2,val,clear,low,1,0.576\n'
                                             'D067,A2,val,clear,low,1,0.705\n'
                                             'D068,A2,val,clear,low,0,0.64\n'
                                             'D069,A2,val,clear,low,0,0.449\n'
                                             'D070,A2,val,clear,low,0,0.065\n'
                                             'D071,A2,val,clear,low,0,0.326\n'
                                             'D072,A2,val,clear,low,1,0.795\n'
                                             'D073,A2,val,clear,low,1,0.871\n'
                                             'D074,A2,val,clear,high,0,0.401\n'
                                             'D075,A2,val,clear,high,0,0.472\n'
                                             'D076,A2,val,cloudy,low,1,0.417\n'
                                             'D077,A2,val,cloudy,low,0,0.189\n'
                                             'D078,A2,val,clear,high,1,0.656\n'
                                             'D079,A2,val,cloudy,low,1,0.835\n'
                                             'D080,A2,val,clear,high,1,0.764\n'
                                             'D081,A2,val,cloudy,high,0,0.26\n'
                                             'D082,A2,val,clear,low,1,0.584\n'
                                             'D083,A2,val,clear,high,1,0.681\n'
                                             'D084,A2,val,clear,low,1,0.737\n'
                                             'D085,A2,val,clear,low,1,0.877\n'
                                             'D086,A2,val,cloudy,low,0,0.38\n'
                                             'D087,A2,val,clear,high,1,0.742\n'
                                             'D088,A2,val,clear,high,1,0.79\n'
                                             'D089,A2,val,cloudy,high,1,0.635\n'
                                             'D090,A2,val,clear,low,1,0.75\n'
                                             'D091,A2,val,clear,low,0,0.265\n'
                                             'D092,A2,val,clear,high,1,0.485\n'
                                             'D093,A2,val,clear,high,0,0.377\n'
                                             'D094,A2,val,clear,high,0,0.146\n'
                                             'D095,A2,val,clear,low,0,0.053\n'
                                             'D096,A2,val,cloudy,low,0,0.219\n'
                                             'D097,A2,val,clear,high,0,0.653\n'
                                             'D098,A2,val,clear,low,0,0.13\n'
                                             'D099,A2,val,clear,low,1,0.953\n'
                                             'D100,A2,val,cloudy,high,0,0.171\n'
                                             'D101,A2,val,clear,low,0,0.463\n'
                                             'D102,A2,val,clear,high,1,0.847\n'
                                             'D103,A2,val,clear,low,1,0.745\n'
                                             'D104,A2,val,clear,high,0,0.193\n'
                                             'D105,A2,val,clear,low,0,0.527\n'
                                             'D106,A2,val,clear,high,0,0.266\n'
                                             'D107,A2,val,clear,low,1,0.894\n'
                                             'D108,A2,val,clear,low,0,0.193\n'
                                             'D109,A2,val,clear,low,0,0.306\n'
                                             'D110,A2,val,clear,low,0,0.154\n'
                                             'D111,A2,val,clear,high,0,0.151\n'
                                             'D112,A2,val,cloudy,low,0,0.207\n'
                                             'D113,A2,val,clear,high,0,0.445\n'
                                             'D114,A2,val,clear,low,1,0.713\n'
                                             'D115,A2,val,cloudy,low,0,0.203\n'
                                             'D116,A2,val,clear,high,0,0.173\n'
                                             'D117,A2,val,clear,high,0,0.285\n'
                                             'D118,A2,val,cloudy,low,1,0.645\n'
                                             'D119,A2,val,cloudy,high,1,0.273\n'
                                             'D120,A2,val,cloudy,low,0,0.111\n'
                                             'D121,A3,val,cloudy,high,1,0.422\n'
                                             'D122,A3,val,clear,high,1,0.586\n'
                                             'D123,A3,val,cloudy,high,1,0.613\n'
                                             'D124,A3,val,cloudy,low,0,0.468\n'
                                             'D125,A3,val,clear,low,0,0.366\n'
                                             'D126,A3,val,clear,low,1,0.86\n'
                                             'D127,A3,val,clear,low,1,0.565\n'
                                             'D128,A3,val,clear,low,0,0.436\n'
                                             'D129,A3,val,clear,low,1,0.525\n'
                                             'D130,A3,val,clear,high,1,0.549\n'
                                             'D131,A3,val,clear,low,1,0.472\n'
                                             'D132,A3,val,cloudy,low,0,0.061\n'
                                             'D133,A3,val,clear,low,1,0.74\n'
                                             'D134,A3,val,clear,low,0,0.738\n'
                                             'D135,A3,val,clear,high,0,0.475\n'
                                             'D136,A3,val,cloudy,low,0,0.321\n'
                                             'D137,A3,val,clear,low,1,0.571\n'
                                             'D138,A3,val,cloudy,high,1,0.605\n'
                                             'D139,A3,val,cloudy,low,0,0.208\n'
                                             'D140,A3,val,clear,low,1,0.776\n'
                                             'D141,A3,val,clear,low,1,0.748\n'
                                             'D142,A3,val,cloudy,high,1,0.549\n'
                                             'D143,A3,val,clear,low,1,0.968\n'
                                             'D144,A3,val,clear,low,1,0.925\n'
                                             'D145,A3,val,clear,high,1,0.855\n'
                                             'D146,A3,val,clear,low,0,0.183\n'
                                             'D147,A3,val,clear,high,1,0.53\n'
                                             'D148,A3,val,clear,high,0,0.4\n'
                                             'D149,A3,val,clear,low,0,0.029\n'
                                             'D150,A3,val,cloudy,low,0,0.419\n'
                                             'D151,A3,val,clear,low,0,0.058\n'
                                             'D152,A3,val,cloudy,low,0,0.508\n'
                                             'D153,A3,val,cloudy,high,0,0.22\n'
                                             'D154,A3,val,clear,high,1,0.513\n'
                                             'D155,A3,val,cloudy,low,1,0.747\n'
                                             'D156,A3,val,cloudy,low,1,0.6\n'
                                             'D157,A3,val,cloudy,low,0,0.491\n'
                                             'D158,A3,val,cloudy,low,0,0.257\n'
                                             'D159,A3,val,clear,low,0,0.145\n'
                                             'D160,A3,val,clear,low,0,0.228\n'
                                             'D161,A3,val,cloudy,low,0,0.297\n'
                                             'D162,A3,val,clear,low,1,0.762\n'
                                             'D163,A3,val,clear,low,0,0.812\n'
                                             'D164,A3,val,clear,low,0,0.407\n'
                                             'D165,A3,val,cloudy,high,1,0.484\n'
                                             'D166,A3,val,clear,low,1,0.598\n'
                                             'D167,A3,val,cloudy,low,0,0.34\n'
                                             'D168,A3,val,cloudy,high,0,0.362\n'
                                             'D169,A3,val,clear,low,1,0.624\n'
                                             'D170,A3,val,clear,low,0,0.298\n'
                                             'D171,A3,val,cloudy,high,0,0.408\n'
                                             'D172,A3,val,clear,low,1,0.858\n'
                                             'D173,A3,val,clear,low,1,0.936\n'
                                             'D174,A3,val,cloudy,high,1,0.75\n'
                                             'D175,A3,val,clear,high,1,0.6\n'
                                             'D176,A3,val,clear,low,0,0.129\n'
                                             'D177,A3,val,clear,low,1,0.814\n'
                                             'D178,A3,val,clear,low,1,0.729\n'
                                             'D179,A3,val,cloudy,high,1,0.667\n'
                                             'D180,A3,val,cloudy,high,1,0.652\n'
                                             'D181,B1,test,clear,high,1,0.843\n'
                                             'D182,B1,test,clear,low,0,0.12\n'
                                             'D183,B1,test,cloudy,low,0,0.183\n'
                                             'D184,B1,test,clear,low,1,0.679\n'
                                             'D185,B1,test,cloudy,low,1,0.771\n'
                                             'D186,B1,test,clear,high,1,0.828\n'
                                             'D187,B1,test,cloudy,low,0,0.251\n'
                                             'D188,B1,test,clear,high,1,0.507\n'
                                             'D189,B1,test,clear,high,0,0.286\n'
                                             'D190,B1,test,clear,high,1,0.751\n'
                                             'D191,B1,test,cloudy,low,0,0.252\n'
                                             'D192,B1,test,clear,low,0,0.763\n'
                                             'D193,B1,test,cloudy,high,0,0.412\n'
                                             'D194,B1,test,clear,high,0,0.19\n'
                                             'D195,B1,test,cloudy,high,0,0.208\n'
                                             'D196,B1,test,clear,low,0,0.232\n'
                                             'D197,B1,test,clear,low,1,0.665\n'
                                             'D198,B1,test,clear,low,0,0.242\n'
                                             'D199,B1,test,clear,low,1,0.943\n'
                                             'D200,B1,test,cloudy,low,1,0.672\n'
                                             'D201,B1,test,clear,low,0,0.072\n'
                                             'D202,B1,test,clear,low,0,0.229\n'
                                             'D203,B1,test,cloudy,low,1,0.509\n'
                                             'D204,B1,test,clear,high,1,0.564\n'
                                             'D205,B1,test,cloudy,low,1,0.146\n'
                                             'D206,B1,test,clear,low,1,0.349\n'
                                             'D207,B1,test,clear,low,1,0.707\n'
                                             'D208,B1,test,cloudy,low,1,0.553\n'
                                             'D209,B1,test,clear,low,0,0.197\n'
                                             'D210,B1,test,clear,low,0,0.141\n'
                                             'D211,B1,test,clear,low,1,0.7\n'
                                             'D212,B1,test,clear,high,1,0.595\n'
                                             'D213,B1,test,clear,high,0,0.326\n'
                                             'D214,B1,test,cloudy,low,1,0.555\n'
                                             'D215,B1,test,cloudy,low,0,0.313\n'
                                             'D216,B1,test,clear,high,0,0.366\n'
                                             'D217,B1,test,cloudy,high,0,0.383\n'
                                             'D218,B1,test,clear,low,0,0.251\n'
                                             'D219,B1,test,clear,low,1,0.944\n'
                                             'D220,B1,test,cloudy,low,1,0.597\n'
                                             'D221,B1,test,clear,low,1,0.846\n'
                                             'D222,B1,test,cloudy,high,0,0.602\n'
                                             'D223,B1,test,cloudy,high,1,0.315\n'
                                             'D224,B1,test,clear,high,1,0.624\n'
                                             'D225,B1,test,clear,low,0,0.467\n'
                                             'D226,B1,test,clear,low,0,0.282\n'
                                             'D227,B1,test,clear,low,0,0.222\n'
                                             'D228,B1,test,clear,low,1,0.403\n'
                                             'D229,B1,test,clear,low,0,0.142\n'
                                             'D230,B1,test,clear,low,1,0.734\n'
                                             'D231,B1,test,cloudy,high,0,0.269\n'
                                             'D232,B1,test,cloudy,low,1,0.506\n'
                                             'D233,B1,test,clear,low,1,0.918\n'
                                             'D234,B1,test,clear,low,1,0.844\n'
                                             'D235,B1,test,cloudy,high,0,0.271\n'
                                             'D236,B1,test,clear,low,0,0.377\n'
                                             'D237,B1,test,clear,low,0,0.338\n'
                                             'D238,B1,test,clear,low,1,0.856\n'
                                             'D239,B1,test,clear,low,1,0.541\n'
                                             'D240,B1,test,clear,low,0,0.172\n'
                                             'D241,B2,test,clear,low,1,0.543\n'
                                             'D242,B2,test,cloudy,low,0,0.332\n'
                                             'D243,B2,test,cloudy,low,0,0.37\n'
                                             'D244,B2,test,clear,low,0,0.137\n'
                                             'D245,B2,test,clear,low,0,0.329\n'
                                             'D246,B2,test,clear,low,0,0.163\n'
                                             'D247,B2,test,cloudy,high,0,0.159\n'
                                             'D248,B2,test,cloudy,low,1,0.707\n'
                                             'D249,B2,test,clear,low,0,0.278\n'
                                             'D250,B2,test,clear,high,0,0.142\n'
                                             'D251,B2,test,clear,low,0,0.488\n'
                                             'D252,B2,test,clear,high,0,0.106\n'
                                             'D253,B2,test,clear,high,0,0.429\n'
                                             'D254,B2,test,cloudy,high,0,0.069\n'
                                             'D255,B2,test,clear,high,0,0.221\n'
                                             'D256,B2,test,clear,low,1,0.67\n'
                                             'D257,B2,test,clear,high,1,0.805\n'
                                             'D258,B2,test,clear,low,1,0.718\n'
                                             'D259,B2,test,cloudy,low,1,0.716\n'
                                             'D260,B2,test,clear,high,0,0.269\n'
                                             'D261,B2,test,cloudy,low,1,0.827\n'
                                             'D262,B2,test,clear,low,0,0.359\n'
                                             'D263,B2,test,clear,high,1,0.734\n'
                                             'D264,B2,test,clear,high,1,0.639\n'
                                             'D265,B2,test,clear,high,0,0.302\n'
                                             'D266,B2,test,clear,low,0,0.206\n'
                                             'D267,B2,test,clear,high,1,0.687\n'
                                             'D268,B2,test,clear,high,0,0.169\n'
                                             'D269,B2,test,cloudy,low,1,0.674\n'
                                             'D270,B2,test,cloudy,low,0,0.185\n'
                                             'D271,B2,test,clear,high,0,0.275\n'
                                             'D272,B2,test,clear,high,0,0.265\n'
                                             'D273,B2,test,clear,low,0,0.074\n'
                                             'D274,B2,test,clear,low,1,0.783\n'
                                             'D275,B2,test,clear,low,1,0.916\n'
                                             'D276,B2,test,clear,low,1,0.924\n'
                                             'D277,B2,test,clear,low,0,0.232\n'
                                             'D278,B2,test,clear,low,1,0.625\n'
                                             'D279,B2,test,clear,low,1,0.821\n'
                                             'D280,B2,test,clear,high,0,0.24\n'
                                             'D281,B2,test,clear,low,0,0.718\n'
                                             'D282,B2,test,cloudy,low,0,0.195\n'
                                             'D283,B2,test,clear,low,0,0.487\n'
                                             'D284,B2,test,cloudy,low,0,0.046\n'
                                             'D285,B2,test,clear,low,1,0.909\n'
                                             'D286,B2,test,clear,high,1,0.761\n'
                                             'D287,B2,test,clear,low,1,0.616\n'
                                             'D288,B2,test,cloudy,high,1,0.667\n'
                                             'D289,B2,test,cloudy,low,0,0.124\n'
                                             'D290,B2,test,clear,high,0,0.476\n'
                                             'D291,B2,test,clear,high,0,0.303\n'
                                             'D292,B2,test,clear,low,0,0.228\n'
                                             'D293,B2,test,cloudy,low,0,0.165\n'
                                             'D294,B2,test,clear,low,0,0.08\n'
                                             'D295,B2,test,clear,high,1,0.473\n'
                                             'D296,B2,test,clear,low,0,0.345\n'
                                             'D297,B2,test,cloudy,high,0,0.333\n'
                                             'D298,B2,test,clear,high,0,0.101\n'
                                             'D299,B2,test,clear,low,0,0.15\n'
                                             'D300,B2,test,cloudy,low,1,0.683\n'
                                             'D301,B3,test,cloudy,high,0,0.346\n'
                                             'D302,B3,test,clear,low,0,0.551\n'
                                             'D303,B3,test,clear,low,1,0.702\n'
                                             'D304,B3,test,clear,low,0,0.332\n'
                                             'D305,B3,test,clear,low,1,0.622\n'
                                             'D306,B3,test,clear,low,0,0.05\n'
                                             'D307,B3,test,clear,low,0,0.439\n'
                                             'D308,B3,test,clear,high,0,0.398\n'
                                             'D309,B3,test,cloudy,high,0,0.666\n'
                                             'D310,B3,test,clear,low,1,0.806\n'
                                             'D311,B3,test,clear,high,1,0.534\n'
                                             'D312,B3,test,clear,low,1,0.794\n'
                                             'D313,B3,test,clear,low,1,0.928\n'
                                             'D314,B3,test,clear,low,0,0.17\n'
                                             'D315,B3,test,clear,high,1,0.766\n'
                                             'D316,B3,test,clear,low,0,0.409\n'
                                             'D317,B3,test,cloudy,high,1,0.236\n'
                                             'D318,B3,test,cloudy,high,0,0.131\n'
                                             'D319,B3,test,clear,high,1,0.647\n'
                                             'D320,B3,test,cloudy,high,1,0.732\n'
                                             'D321,B3,test,cloudy,high,0,0.408\n'
                                             'D322,B3,test,clear,low,0,0.233\n'
                                             'D323,B3,test,clear,high,1,0.756\n'
                                             'D324,B3,test,clear,low,1,0.948\n'
                                             'D325,B3,test,clear,low,1,0.756\n'
                                             'D326,B3,test,clear,low,0,0.296\n'
                                             'D327,B3,test,clear,low,1,0.511\n'
                                             'D328,B3,test,cloudy,high,0,0.299\n'
                                             'D329,B3,test,clear,low,1,0.874\n'
                                             'D330,B3,test,clear,low,1,0.725\n'
                                             'D331,B3,test,cloudy,high,0,0.155\n'
                                             'D332,B3,test,clear,low,0,0.247\n'
                                             'D333,B3,test,clear,low,0,0.454\n'
                                             'D334,B3,test,clear,high,1,0.763\n'
                                             'D335,B3,test,clear,high,0,0.15\n'
                                             'D336,B3,test,clear,low,1,0.806\n'
                                             'D337,B3,test,clear,low,0,0.13\n'
                                             'D338,B3,test,cloudy,low,1,0.351\n'
                                             'D339,B3,test,cloudy,high,0,0.045\n'
                                             'D340,B3,test,cloudy,low,1,0.667\n'
                                             'D341,B3,test,clear,low,0,0.259\n'
                                             'D342,B3,test,clear,high,0,0.219\n'
                                             'D343,B3,test,clear,low,1,0.605\n'
                                             'D344,B3,test,clear,low,0,0.054\n'
                                             'D345,B3,test,cloudy,low,1,0.627\n'
                                             'D346,B3,test,clear,low,1,0.57\n'
                                             'D347,B3,test,cloudy,high,0,0.51\n'
                                             'D348,B3,test,cloudy,low,1,0.52\n'
                                             'D349,B3,test,cloudy,low,0,0.159\n'
                                             'D350,B3,test,cloudy,high,0,0.243\n'
                                             'D351,B3,test,clear,low,0,0.285\n'
                                             'D352,B3,test,clear,low,1,0.83\n'
                                             'D353,B3,test,clear,low,0,0.321\n'
                                             'D354,B3,test,clear,low,0,0.123\n'
                                             'D355,B3,test,cloudy,low,0,0.263\n'
                                             'D356,B3,test,clear,low,0,0.469\n'
                                             'D357,B3,test,cloudy,low,1,0.708\n'
                                             'D358,B3,test,clear,low,0,0.245\n'
                                             'D359,B3,test,clear,high,0,0.27\n'
                                             'D360,B3,test,clear,low,1,0.926\n',
                        'm6_objects.csv': 'object_id,model_label,score,has_shadow,sar_db,is_decoy\n'
                                          'O01,tank,0.93,1,-7.2,0\n'
                                          'O02,tank,0.91,0,-18.5,1\n'
                                          'O03,tank,0.88,1,-6.4,0\n'
                                          'O04,tank,0.9,0,-17.1,1\n'
                                          'O05,tank,0.86,0,-19.3,1\n'
                                          'O06,tank,0.84,1,-16.8,1\n'
                                          'O07,tank,0.89,,-7.9,0\n'
                                          'O08,tank,0.82,,-18.2,1\n'
                                          'O09,tank,0.85,1,,0\n'
                                          'O10,tank,0.8,0,,1\n'
                                          'O11,tank,0.78,,,1\n'
                                          'O12,tank,0.92,1,-8.8,0\n',
                        'm7_scores.csv': 'det_id,x_km,y_km,score,is_correct\n'
                                         'D001,7.01,9.57,0.382,0\n'
                                         'D002,4.91,0.71,0.119,0\n'
                                         'D003,0.12,9.74,0.518,0\n'
                                         'D004,5.24,0.99,0.303,0\n'
                                         'D005,3.38,7.19,0.839,1\n'
                                         'D006,3.46,5.74,0.488,0\n'
                                         'D007,3.06,6.19,0.686,0\n'
                                         'D008,3.38,6.75,0.795,1\n'
                                         'D009,3.0,4.63,0.346,0\n'
                                         'D010,0.04,0.57,0.529,0\n'
                                         'D011,3.76,5.91,0.894,0\n'
                                         'D012,2.23,7.19,0.85,1\n'
                                         'D013,2.09,6.72,0.58,0\n'
                                         'D014,4.37,3.04,0.55,0\n'
                                         'D015,9.01,1.25,0.828,1\n'
                                         'D016,2.83,6.62,0.97,1\n'
                                         'D017,5.21,8.51,0.524,0\n'
                                         'D018,2.77,6.24,0.843,0\n'
                                         'D019,9.63,4.7,0.166,0\n'
                                         'D020,1.3,8.32,0.71,0\n'
                                         'D021,7.02,7.18,0.749,0\n'
                                         'D022,7.98,1.88,0.598,1\n'
                                         'D023,2.68,6.72,0.878,1\n'
                                         'D024,3.51,5.79,0.65,1\n'
                                         'D025,2.59,5.35,0.574,1\n'
                                         'D026,9.6,5.4,0.682,0\n'
                                         'D027,2.43,6.38,0.874,1\n'
                                         'D028,8.85,1.76,0.452,0\n'
                                         'D029,4.92,1.51,0.346,0\n'
                                         'D030,2.53,6.8,0.512,0\n'
                                         'D031,2.73,6.62,0.419,0\n'
                                         'D032,2.78,7.57,0.662,1\n'
                                         'D033,8.63,4.94,0.425,1\n'
                                         'D034,2.54,7.14,0.604,0\n'
                                         'D035,5.38,5.64,0.553,1\n'
                                         'D036,6.64,4.48,0.456,0\n'
                                         'D037,9.82,8.61,0.292,0\n'
                                         'D038,3.26,6.3,0.826,1\n'
                                         'D039,1.6,7.49,0.559,0\n'
                                         'D040,2.22,5.25,0.549,0\n'
                                         'D041,8.35,3.66,0.682,0\n'
                                         'D042,4.37,4.98,0.899,1\n'
                                         'D043,4.93,9.2,0.615,1\n'
                                         'D044,2.5,6.19,0.724,0\n'
                                         'D045,3.22,5.78,0.679,1\n'
                                         'D046,9.54,4.19,0.418,0\n'
                                         'D047,2.62,7.12,0.787,0\n'
                                         'D048,2.09,2.17,0.663,0\n'
                                         'D049,3.5,6.7,0.834,1\n'
                                         'D050,2.28,5.46,0.357,0\n'
                                         'D051,7.53,5.68,0.054,0\n'
                                         'D052,2.24,6.22,0.587,0\n'
                                         'D053,9.59,0.04,0.311,0\n'
                                         'D054,1.49,3.4,0.536,0\n'
                                         'D055,3.18,7.12,0.894,1\n'
                                         'D056,3.06,6.9,0.478,0\n'
                                         'D057,2.08,7.12,0.807,0\n'
                                         'D058,4.41,2.4,0.616,0\n'
                                         'D059,9.61,8.28,0.687,1\n'
                                         'D060,5.21,3.1,0.707,0\n'
                                         'D061,1.62,1.31,0.147,0\n'
                                         'D062,2.01,6.34,0.737,0\n'
                                         'D063,3.33,6.24,0.272,0\n'
                                         'D064,6.21,6.78,0.552,0\n'
                                         'D065,7.85,7.16,0.673,1\n'
                                         'D066,3.12,7.18,0.783,0\n'
                                         'D067,2.75,6.53,0.7,0\n'
                                         'D068,3.55,7.27,0.731,0\n'
                                         'D069,5.87,5.45,0.406,0\n'
                                         'D070,2.17,6.91,0.53,1\n'
                                         'D071,5.71,6.0,0.302,0\n'
                                         'D072,2.17,5.56,0.649,1\n'
                                         'D073,3.11,5.85,0.818,0\n'
                                         'D074,2.59,6.63,0.945,0\n'
                                         'D075,3.25,6.6,0.738,0\n'
                                         'D076,2.49,6.29,0.598,0\n'
                                         'D077,1.18,7.77,0.75,1\n'
                                         'D078,3.01,7.07,0.558,0\n'
                                         'D079,3.03,6.49,0.728,0\n'
                                         'D080,0.57,1.65,0.896,1\n'
                                         'D081,3.26,1.84,0.199,0\n'
                                         'D082,8.24,1.01,0.344,0\n'
                                         'D083,9.01,0.8,0.663,0\n'
                                         'D084,7.48,4.94,0.204,1\n'
                                         'D085,2.26,6.36,0.471,0\n'
                                         'D086,3.3,5.94,0.692,1\n'
                                         'D087,2.65,6.62,0.91,0\n'
                                         'D088,2.23,7.46,0.709,0\n'
                                         'D089,2.9,6.52,0.631,0\n'
                                         'D090,3.5,0.68,0.354,1\n'
                                         'D091,3.09,6.25,0.556,0\n'
                                         'D092,3.1,6.75,0.779,1\n'
                                         'D093,6.48,4.22,0.312,0\n'
                                         'D094,9.9,6.89,0.315,1\n'
                                         'D095,2.73,6.5,0.782,1\n'
                                         'D096,3.21,5.51,0.872,0\n'
                                         'D097,4.11,1.6,0.505,0\n'
                                         'D098,1.78,6.44,0.644,0\n'
                                         'D099,3.12,6.62,0.836,1\n'
                                         'D100,0.53,3.6,0.229,0\n'
                                         'D101,3.76,7.11,0.591,0\n'
                                         'D102,8.41,8.91,0.66,1\n'
                                         'D103,2.71,2.41,0.256,0\n'
                                         'D104,3.01,6.45,0.927,1\n'
                                         'D105,8.01,2.24,0.67,1\n'
                                         'D106,6.6,9.96,0.724,1\n'
                                         'D107,9.68,8.27,0.478,0\n'
                                         'D108,7.06,4.48,0.315,0\n'
                                         'D109,3.52,5.24,0.629,0\n'
                                         'D110,2.59,7.21,0.523,0\n'
                                         'D111,6.37,8.83,0.945,1\n'
                                         'D112,7.22,0.12,0.36,0\n'
                                         'D113,6.72,8.45,0.88,0\n'
                                         'D114,4.77,4.95,0.254,0\n'
                                         'D115,0.9,1.48,0.758,1\n'
                                         'D116,9.63,6.07,0.617,0\n'
                                         'D117,3.0,8.89,0.681,0\n'
                                         'D118,9.48,6.84,0.479,0\n'
                                         'D119,9.42,2.2,0.446,1\n'
                                         'D120,5.11,2.99,0.693,0\n'
                                         'D121,3.02,3.72,0.446,1\n'
                                         'D122,3.88,6.99,0.567,0\n'
                                         'D123,2.88,6.59,0.887,0\n'
                                         'D124,2.44,6.37,0.721,1\n'
                                         'D125,1.96,6.47,0.712,0\n'
                                         'D126,2.76,6.97,0.848,1\n'
                                         'D127,1.41,6.23,0.84,0\n'
                                         'D128,2.28,6.9,0.958,1\n'
                                         'D129,3.44,6.41,0.343,0\n'
                                         'D130,6.93,5.84,0.464,0\n'
                                         'D131,9.28,9.46,0.217,0\n'
                                         'D132,3.05,5.33,0.854,1\n'
                                         'D133,2.81,6.35,0.52,0\n'
                                         'D134,7.47,0.85,0.478,0\n'
                                         'D135,2.52,6.13,0.737,1\n'
                                         'D136,0.92,0.15,0.766,0\n'
                                         'D137,3.98,0.3,0.431,0\n'
                                         'D138,6.58,7.58,0.671,1\n'
                                         'D139,2.51,7.13,0.929,1\n'
                                         'D140,1.99,6.65,0.594,0\n'
                                         'D141,2.24,9.96,0.332,0\n'
                                         'D142,3.25,6.78,0.918,1\n'
                                         'D143,4.44,5.63,0.069,0\n'
                                         'D144,2.4,6.89,0.735,1\n'
                                         'D145,8.34,8.16,0.499,1\n'
                                         'D146,3.93,3.18,0.809,1\n'
                                         'D147,2.83,5.45,0.756,1\n'
                                         'D148,8.29,9.88,0.302,0\n'
                                         'D149,1.9,8.33,0.599,0\n'
                                         'D150,3.13,0.21,0.517,0\n'
                                         'D151,3.16,6.9,0.949,1\n'
                                         'D152,8.12,1.51,0.327,0\n'
                                         'D153,3.11,6.06,0.64,1\n'
                                         'D154,6.17,9.23,0.645,0\n'
                                         'D155,0.84,5.8,0.453,0\n'
                                         'D156,3.52,6.65,0.907,1\n'
                                         'D157,1.28,4.65,0.641,1\n'
                                         'D158,3.65,6.4,0.584,0\n'
                                         'D159,2.42,6.84,0.854,0\n'
                                         'D160,5.03,2.27,0.495,0\n'
                                         'D161,8.37,7.13,0.294,0\n'
                                         'D162,3.48,6.72,0.919,1\n'
                                         'D163,2.35,7.47,0.568,1\n'
                                         'D164,1.21,9.22,0.445,0\n'
                                         'D165,4.61,2.0,0.82,1\n'
                                         'D166,3.83,6.84,0.829,0\n'
                                         'D167,7.59,8.01,0.558,0\n'
                                         'D168,2.82,5.84,0.736,0\n'
                                         'D169,4.39,9.26,0.529,0\n'
                                         'D170,7.86,4.61,0.525,0\n'
                                         'D171,2.92,6.89,0.7,0\n'
                                         'D172,2.54,4.5,0.501,0\n'
                                         'D173,2.5,5.95,0.965,1\n'
                                         'D174,0.02,1.72,0.306,0\n'
                                         'D175,9.8,7.28,0.541,0\n'
                                         'D176,2.99,6.92,0.826,1\n'
                                         'D177,2.64,6.86,0.427,0\n'
                                         'D178,3.43,6.35,0.967,1\n'
                                         'D179,3.82,6.34,0.923,1\n'
                                         'D180,3.23,7.07,0.865,0\n'
                                         'D181,4.21,5.07,0.664,1\n'
                                         'D182,3.18,6.18,0.644,0\n'
                                         'D183,2.57,6.36,0.955,1\n'
                                         'D184,4.9,6.36,0.546,0\n'
                                         'D185,3.56,9.04,0.519,0\n'
                                         'D186,3.01,7.37,0.638,0\n'
                                         'D187,3.67,6.57,0.713,0\n'
                                         'D188,2.46,1.4,0.266,0\n'
                                         'D189,2.37,6.32,0.94,1\n'
                                         'D190,0.27,1.1,0.306,0\n'
                                         'D191,3.73,6.96,0.639,0\n'
                                         'D192,4.35,7.66,0.571,1\n'
                                         'D193,2.61,6.65,0.767,0\n'
                                         'D194,2.74,8.52,0.763,1\n'
                                         'D195,3.15,1.79,0.329,0\n'
                                         'D196,6.43,5.25,0.569,0\n'
                                         'D197,3.04,3.0,0.843,0\n'
                                         'D198,8.84,0.7,0.432,0\n'
                                         'D199,3.08,6.9,0.578,1\n'
                                         'D200,3.29,6.82,0.869,0\n'
                                         'D201,2.71,6.27,0.828,1\n'
                                         'D202,3.22,6.55,0.773,0\n'
                                         'D203,3.67,6.09,0.937,1\n'
                                         'D204,6.38,8.48,0.517,0\n'
                                         'D205,0.7,8.65,0.415,0\n'
                                         'D206,2.61,6.39,0.721,0\n'
                                         'D207,2.61,7.36,0.124,0\n'
                                         'D208,2.39,6.3,0.865,0\n'
                                         'D209,2.67,6.72,0.762,0\n'
                                         'D210,3.37,6.67,0.945,1\n'
                                         'D211,2.57,6.54,0.527,1\n'
                                         'D212,3.31,6.46,0.793,1\n'
                                         'D213,3.13,8.01,0.463,0\n'
                                         'D214,2.54,6.15,0.756,1\n'
                                         'D215,2.94,6.95,0.841,1\n'
                                         'D216,0.5,0.87,0.501,1\n'
                                         'D217,3.48,6.92,0.677,1\n'
                                         'D218,5.79,6.23,0.325,1\n'
                                         'D219,4.06,2.44,0.297,1\n'
                                         'D220,2.87,9.23,0.219,0\n'
                                         'D221,3.31,5.95,0.876,0\n'
                                         'D222,7.82,9.03,0.233,0\n'
                                         'D223,2.62,6.27,0.728,0\n'
                                         'D224,7.38,1.51,0.908,1\n'
                                         'D225,3.14,7.35,0.8,1\n'
                                         'D226,3.04,6.58,0.817,1\n'
                                         'D227,3.35,1.55,0.235,0\n'
                                         'D228,4.41,9.45,0.813,1\n'
                                         'D229,3.1,3.5,0.747,1\n'
                                         'D230,2.41,6.35,0.777,1\n'
                                         'D231,2.8,7.15,0.915,1\n'
                                         'D232,6.01,3.16,0.602,0\n'
                                         'D233,3.46,7.28,0.87,0\n'
                                         'D234,2.82,6.54,0.715,1\n'
                                         'D235,1.69,7.44,0.133,0\n'
                                         'D236,2.74,2.1,0.846,0\n'
                                         'D237,3.74,6.84,0.841,1\n'
                                         'D238,2.02,6.85,0.823,0\n'
                                         'D239,2.3,6.04,0.844,1\n'
                                         'D240,7.19,4.97,0.148,0\n',
                        'm8_queue.csv': 'item_id,target_class,calibrated_prob,ai_label,due_min,location_code\n'
                                        'Q01,normal,0.93,truck,120,SITE-A12\n'
                                        'Q02,hvt,0.95,launcher,45,SITE-C03\n'
                                        'Q03,normal,0.62,apc,90,SITE-A07\n'
                                        'Q04,normal,0.88,truck,180,SITE-B01\n'
                                        'Q05,hvt,0.71,radar,30,SITE-C09\n'
                                        'Q06,normal,0.55,tank,60,SITE-B04\n'
                                        'Q07,normal,0.91,apc,150,SITE-A03\n'
                                        'Q08,normal,0.74,tank,75,SITE-D02\n'
                                        'Q09,hvt,0.83,aircraft,20,SITE-C01\n'
                                        'Q10,normal,0.97,truck,200,SITE-B08\n',
                        'm9_cases.csv': 'case_id,ai_label,truth,cause\n'
                                        'C001,truck,tank,data\n'
                                        'C002,truck,truck,none\n'
                                        'C003,unknown,tank,none\n'
                                        'C004,tank,decoy,communication\n'
                                        'C005,truck,truck,none\n'
                                        'C006,apc,apc,none\n'
                                        'C007,decoy,decoy,none\n'
                                        'C008,apc,apc,none\n'
                                        'C009,truck,truck,none\n'
                                        'C010,decoy,decoy,none\n'
                                        'C011,decoy,apc,communication\n'
                                        'C012,apc,tank,process\n'
                                        'C013,decoy,apc,model\n'
                                        'C014,decoy,decoy,none\n'
                                        'C015,truck,truck,none\n'
                                        'C016,apc,truck,process\n'
                                        'C017,tank,tank,none\n'
                                        'C018,decoy,decoy,none\n'
                                        'C019,apc,apc,none\n'
                                        'C020,truck,tank,data\n'
                                        'C021,tank,tank,none\n'
                                        'C022,tank,tank,none\n'
                                        'C023,unknown,tank,none\n'
                                        'C024,decoy,decoy,none\n'
                                        'C025,apc,apc,none\n'
                                        'C026,unknown,apc,none\n'
                                        'C027,apc,apc,none\n'
                                        'C028,decoy,decoy,none\n'
                                        'C029,apc,apc,none\n'
                                        'C030,truck,tank,process\n'
                                        'C031,tank,apc,process\n'
                                        'C032,truck,decoy,communication\n'
                                        'C033,tank,tank,none\n'
                                        'C034,decoy,decoy,none\n'
                                        'C035,truck,truck,none\n'
                                        'C036,truck,truck,none\n'
                                        'C037,tank,tank,none\n'
                                        'C038,decoy,decoy,none\n'
                                        'C039,unknown,decoy,none\n'
                                        'C040,unknown,truck,none\n'
                                        'C041,tank,tank,none\n'
                                        'C042,decoy,decoy,none\n'
                                        'C043,truck,truck,none\n'
                                        'C044,apc,apc,none\n'
                                        'C045,unknown,apc,none\n'
                                        'C046,truck,truck,none\n'
                                        'C047,apc,apc,none\n'
                                        'C048,decoy,decoy,none\n'
                                        'C049,truck,truck,none\n'
                                        'C050,decoy,decoy,none\n'
                                        'C051,tank,apc,process\n'
                                        'C052,apc,tank,model\n'
                                        'C053,unknown,apc,none\n'
                                        'C054,truck,truck,none\n'
                                        'C055,tank,tank,none\n'
                                        'C056,truck,truck,none\n'
                                        'C057,tank,tank,none\n'
                                        'C058,tank,tank,none\n'
                                        'C059,apc,apc,none\n'
                                        'C060,tank,tank,none\n',
                        'm10_run.csv': 'run_id,gsd_band,cloud_band,target_type,alert,human_checked\n'
                                       'R001,1m,clear,aircraft,1,1\n'
                                       'R002,0.5m,clear,other,0,0\n'
                                       'R003,0.5m,cloudy,other,0,0\n'
                                       'R004,0.5m,clear,other,0,0\n'
                                       'R005,1m,cloudy,other,0,0\n'
                                       'R006,0.5m,clear,other,0,0\n'
                                       'R007,1m,clear,other,0,0\n'
                                       'R008,1m,clear,other,0,0\n'
                                       'R009,1m,cloudy,aircraft,1,1\n'
                                       'R010,0.5m,cloudy,aircraft,1,1\n'
                                       'R011,1m,clear,other,0,0\n'
                                       'R012,1m,cloudy,other,1,0\n'
                                       'R013,2m,clear,other,0,0\n'
                                       'R014,0.5m,clear,other,0,0\n'
                                       'R015,1m,cloudy,other,0,0\n'
                                       'R016,0.5m,clear,aircraft,1,0\n'
                                       'R017,1m,clear,aircraft,1,1\n'
                                       'R018,1m,clear,other,0,0\n'
                                       'R019,1m,cloudy,other,0,0\n'
                                       'R020,0.5m,clear,aircraft,1,1\n'
                                       'R021,1m,clear,aircraft,1,0\n'
                                       'R022,0.5m,clear,aircraft,1,1\n'
                                       'R023,1m,clear,other,0,0\n'
                                       'R024,1m,clear,aircraft,0,0\n'
                                       'R025,1m,clear,other,0,0\n'
                                       'R026,0.5m,clear,other,0,0\n'
                                       'R027,1m,cloudy,other,0,0\n'
                                       'R028,1m,clear,other,0,0\n'
                                       'R029,0.5m,clear,other,0,0\n'
                                       'R030,0.5m,clear,other,0,0\n'
                                       'R031,0.5m,cloudy,aircraft,1,1\n'
                                       'R032,1m,clear,other,0,0\n'
                                       'R033,1m,cloudy,other,0,0\n'
                                       'R034,0.5m,clear,aircraft,1,0\n'
                                       'R035,1m,clear,aircraft,1,1\n'
                                       'R036,1m,clear,aircraft,1,1\n'
                                       'R037,1m,clear,aircraft,1,0\n'
                                       'R038,1m,clear,aircraft,1,0\n'
                                       'R039,0.5m,clear,other,0,0\n'
                                       'R040,0.5m,cloudy,other,0,0\n'
                                       'R041,1m,clear,other,0,0\n'
                                       'R042,0.5m,cloudy,aircraft,1,1\n'
                                       'R043,1m,clear,other,1,0\n'
                                       'R044,0.5m,clear,aircraft,1,0\n'
                                       'R045,0.5m,clear,other,0,0\n'
                                       'R046,1m,clear,aircraft,0,0\n'
                                       'R047,1m,cloudy,other,0,0\n'
                                       'R048,0.5m,clear,aircraft,1,1\n'
                                       'R049,0.5m,clear,other,0,0\n'
                                       'R050,0.5m,clear,other,0,0\n'
                                       'R051,0.5m,clear,other,0,0\n'
                                       'R052,0.5m,clear,aircraft,1,1\n'
                                       'R053,1m,cloudy,other,0,0\n'
                                       'R054,0.5m,clear,aircraft,1,1\n'
                                       'R055,1m,cloudy,other,0,0\n'
                                       'R056,1m,clear,other,0,0\n'
                                       'R057,0.5m,clear,other,0,0\n'
                                       'R058,1m,cloudy,other,0,0\n'
                                       'R059,0.5m,clear,other,0,0\n'
                                       'R060,0.5m,clear,other,0,0\n'
                                       'R061,1m,cloudy,other,0,0\n'
                                       'R062,2m,clear,other,0,0\n'
                                       'R063,0.5m,cloudy,other,0,0\n'
                                       'R064,0.5m,cloudy,aircraft,1,0\n'
                                       'R065,1m,clear,other,1,1\n'
                                       'R066,0.5m,clear,aircraft,1,1\n'
                                       'R067,1m,clear,aircraft,1,1\n'
                                       'R068,2m,clear,other,0,0\n'
                                       'R069,0.5m,clear,aircraft,1,1\n'
                                       'R070,0.5m,clear,aircraft,1,0\n'
                                       'R071,0.5m,clear,aircraft,1,1\n'
                                       'R072,0.5m,clear,aircraft,1,1\n'
                                       'R073,0.5m,clear,aircraft,1,1\n'
                                       'R074,0.5m,clear,aircraft,1,1\n'
                                       'R075,2m,cloudy,aircraft,0,0\n'
                                       'R076,1m,cloudy,other,0,0\n'
                                       'R077,1m,clear,aircraft,1,0\n'
                                       'R078,0.5m,cloudy,other,0,0\n'
                                       'R079,1m,clear,aircraft,1,0\n'
                                       'R080,0.5m,cloudy,aircraft,1,1\n'
                                       'R081,0.5m,clear,other,1,0\n'
                                       'R082,1m,clear,aircraft,1,0\n'
                                       'R083,0.5m,clear,aircraft,1,1\n'
                                       'R084,0.5m,clear,other,0,0\n'
                                       'R085,0.5m,clear,other,0,0\n'
                                       'R086,0.5m,clear,other,0,0\n'
                                       'R087,0.5m,clear,other,0,0\n'
                                       'R088,0.5m,clear,other,0,0\n'
                                       'R089,0.5m,cloudy,other,0,0\n'
                                       'R090,1m,clear,other,0,0\n'
                                       'R091,1m,cloudy,other,0,0\n'
                                       'R092,0.5m,clear,other,0,0\n'
                                       'R093,0.5m,clear,aircraft,1,1\n'
                                       'R094,0.5m,cloudy,other,0,0\n'
                                       'R095,2m,clear,other,1,1\n'
                                       'R096,0.5m,clear,other,0,0\n'
                                       'R097,1m,clear,other,0,0\n'
                                       'R098,1m,clear,other,1,0\n'
                                       'R099,1m,clear,other,0,0\n'
                                       'R100,1m,cloudy,other,0,0\n'
                                       'R101,0.5m,clear,other,0,0\n'
                                       'R102,1m,clear,other,0,0\n'
                                       'R103,0.5m,cloudy,other,0,0\n'
                                       'R104,1m,clear,aircraft,1,0\n'
                                       'R105,2m,clear,other,0,0\n'
                                       'R106,0.5m,clear,aircraft,1,0\n'
                                       'R107,1m,cloudy,other,1,1\n'
                                       'R108,0.5m,clear,other,0,0\n'
                                       'R109,1m,clear,other,0,0\n'
                                       'R110,1m,cloudy,other,0,0\n'
                                       'R111,0.5m,clear,other,0,0\n'
                                       'R112,1m,clear,other,0,0\n'
                                       'R113,2m,clear,aircraft,1,0\n'
                                       'R114,0.5m,cloudy,aircraft,0,0\n'
                                       'R115,1m,clear,other,0,0\n'
                                       'R116,0.5m,clear,other,0,0\n'
                                       'R117,0.5m,clear,other,0,0\n'
                                       'R118,0.5m,cloudy,aircraft,1,1\n'
                                       'R119,0.5m,clear,aircraft,1,1\n'
                                       'R120,1m,cloudy,other,0,0\n'
                                       'R121,1m,cloudy,other,1,1\n'
                                       'R122,0.5m,clear,aircraft,1,0\n'
                                       'R123,0.5m,cloudy,aircraft,1,1\n'
                                       'R124,2m,clear,other,0,0\n'
                                       'R125,0.5m,cloudy,other,0,0\n'
                                       'R126,0.5m,clear,other,0,0\n'
                                       'R127,0.5m,clear,aircraft,1,1\n'
                                       'R128,0.5m,clear,aircraft,1,0\n'
                                       'R129,0.5m,cloudy,other,0,0\n'
                                       'R130,0.5m,cloudy,other,0,0\n'
                                       'R131,0.5m,clear,other,0,0\n'
                                       'R132,1m,cloudy,aircraft,1,0\n'
                                       'R133,0.5m,cloudy,other,0,0\n'
                                       'R134,1m,clear,other,0,0\n'
                                       'R135,0.5m,clear,other,0,0\n'
                                       'R136,1m,clear,aircraft,0,0\n'
                                       'R137,1m,cloudy,aircraft,1,0\n'
                                       'R138,1m,cloudy,aircraft,1,1\n'
                                       'R139,0.5m,cloudy,other,0,0\n'
                                       'R140,1m,clear,other,0,0\n'
                                       'R141,2m,clear,other,1,1\n'
                                       'R142,0.5m,clear,aircraft,1,1\n'
                                       'R143,0.5m,clear,aircraft,1,0\n'
                                       'R144,0.5m,clear,aircraft,1,0\n'
                                       'R145,1m,cloudy,other,0,0\n'
                                       'R146,0.5m,clear,other,0,0\n'
                                       'R147,0.5m,clear,aircraft,1,1\n'
                                       'R148,0.5m,clear,aircraft,0,0\n'
                                       'R149,0.5m,clear,aircraft,1,1\n'
                                       'R150,2m,clear,other,0,0\n'
                                       'R151,1m,clear,aircraft,1,1\n'
                                       'R152,0.5m,clear,aircraft,1,1\n'
                                       'R153,0.5m,clear,aircraft,1,0\n'
                                       'R154,2m,clear,other,0,0\n'
                                       'R155,1m,clear,other,0,0\n'
                                       'R156,0.5m,cloudy,aircraft,1,0\n'
                                       'R157,1m,clear,aircraft,1,1\n'
                                       'R158,0.5m,clear,aircraft,1,1\n'
                                       'R159,0.5m,clear,aircraft,1,1\n'
                                       'R160,2m,cloudy,other,0,0\n'
                                       'R161,0.5m,clear,aircraft,1,1\n'
                                       'R162,0.5m,clear,other,0,0\n'
                                       'R163,1m,clear,other,0,0\n'
                                       'R164,0.5m,clear,other,0,0\n'
                                       'R165,0.5m,clear,aircraft,1,0\n'
                                       'R166,0.5m,clear,aircraft,1,1\n'
                                       'R167,0.5m,clear,aircraft,1,0\n'
                                       'R168,0.5m,clear,aircraft,1,0\n'
                                       'R169,1m,clear,other,0,0\n'
                                       'R170,2m,cloudy,other,0,0\n'
                                       'R171,0.5m,cloudy,other,0,0\n'
                                       'R172,0.5m,clear,other,0,0\n'
                                       'R173,0.5m,cloudy,aircraft,1,0\n'
                                       'R174,1m,clear,aircraft,1,1\n'
                                       'R175,1m,cloudy,aircraft,1,0\n'
                                       'R176,0.5m,clear,aircraft,1,1\n'
                                       'R177,0.5m,clear,aircraft,1,0\n'
                                       'R178,0.5m,clear,other,0,0\n'
                                       'R179,0.5m,clear,aircraft,1,1\n'
                                       'R180,0.5m,clear,other,0,0\n'
                                       'R181,1m,clear,aircraft,1,0\n'
                                       'R182,1m,clear,aircraft,1,1\n'
                                       'R183,0.5m,clear,aircraft,1,1\n'
                                       'R184,0.5m,clear,other,0,0\n'
                                       'R185,1m,clear,aircraft,1,0\n'
                                       'R186,0.5m,clear,other,0,0\n'
                                       'R187,0.5m,clear,aircraft,0,0\n'
                                       'R188,1m,clear,aircraft,1,0\n'
                                       'R189,1m,cloudy,other,0,0\n'
                                       'R190,1m,clear,other,0,0\n'
                                       'R191,0.5m,clear,other,0,0\n'
                                       'R192,0.5m,clear,aircraft,1,0\n'
                                       'R193,0.5m,clear,other,0,0\n'
                                       'R194,1m,clear,aircraft,1,1\n'
                                       'R195,0.5m,cloudy,aircraft,1,1\n'
                                       'R196,0.5m,clear,aircraft,0,0\n'
                                       'R197,1m,cloudy,other,0,0\n'
                                       'R198,0.5m,clear,aircraft,1,0\n'
                                       'R199,1m,clear,other,0,0\n'
                                       'R200,1m,cloudy,aircraft,0,0\n'},
             'figures': {'f_loop': '<svg viewBox="0 0 320 156" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><ellipse cx="160.0" cy="70" rx="112" '
                                   'ry="48" fill="none" stroke="#e5e7eb" stroke-width="6"/><path d="M198.6,24.9 L248.1,40.3" stroke="#2563eb" '
                                   'stroke-width="2" marker-end="url(#tfa)" fill="none"/><path d="M271.9,71.8 L253.0,96.7" stroke="#2563eb" '
                                   'stroke-width="2" marker-end="url(#tfa)" fill="none"/><path d="M190.6,116.2 L129.4,116.2" stroke="#2563eb" '
                                   'stroke-width="2" marker-end="url(#tfa)" fill="none"/><path d="M67.0,96.7 L48.1,71.8" stroke="#2563eb" '
                                   'stroke-width="2" marker-end="url(#tfa)" fill="none"/><path d="M71.9,40.3 L121.4,24.9" stroke="#2563eb" '
                                   'stroke-width="2" marker-end="url(#tfa)" fill="none"/><rect x="116.0" y="10.0" width="88.0" height="24.0" rx="7" '
                                   'fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="25.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">요청</text><rect x="222.5" y="43.2" width="88.0" height="24.0" rx="7" fill="#dbeafe" '
                                   'stroke="#94a3b8"/><text x="266.5" y="58.7" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">생성</text><rect x="181.8" y="96.8" width="88.0" height="24.0" rx="7" fill="#dbeafe" '
                                   'stroke="#94a3b8"/><text x="225.8" y="112.3" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">실행</text><rect x="50.2" y="96.8" width="88.0" height="24.0" rx="7" fill="#dbeafe" '
                                   'stroke="#94a3b8"/><text x="94.2" y="112.3" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">확인</text><rect x="9.5" y="43.2" width="88.0" height="24.0" rx="7" fill="#dbeafe" '
                                   'stroke="#94a3b8"/><text x="53.5" y="58.7" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">수정 요청</text><text x="160.0" y="74.0" font-size="10.5" text-anchor="middle" fill="#475569" '
                                   'font-weight="bold">바이브 코딩</text><text x="160.0" y="151.0" font-size="9" text-anchor="middle" fill="#475569">확인하고 '
                                   '다시 요청하는 일이 핵심이에요</text></svg>',
                         'f_four': '<svg viewBox="0 0 320 120" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="154.0" '
                                   'height="18" fill="#e5e7eb" stroke="#fff"/><text x="83.0" y="20.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">칸</text><rect x="160.0" y="8" width="154.0" height="18" fill="#e5e7eb" '
                                   'stroke="#fff"/><text x="237.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">무엇을 '
                                   '적나</text><rect x="6.0" y="26" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="83.0" y="38.5" '
                                   'font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">맥락</text><rect x="160.0" y="26" '
                                   'width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="38.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">왜 필요한지</text><rect x="6.0" y="44" width="154.0" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="83.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">입력</text><rect x="160.0" y="44" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="237.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827">파일·열·값</text><rect x="6.0" y="62" '
                                   'width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="83.0" y="74.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">할 일</text><rect x="160.0" y="62" width="154.0" '
                                   'height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="74.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827">규칙을 숫자로</text><rect x="6.0" y="80" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="83.0" y="92.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">출력 형식</text><rect '
                                   'x="160.0" y="80" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="92.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">받고 싶은 모양</text><text x="160.0" y="115.0" font-size="9" text-anchor="middle" '
                                   'fill="#475569">네 칸을 채우면 AI가 추측할 일이 줄어요</text></svg>',
                         'f_gsd': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                  'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                  'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="102.7" '
                                  'height="18" fill="#e5e7eb" stroke="#fff"/><text x="57.3" y="20.5" font-size="9" text-anchor="middle" '
                                  'fill="#111827" font-weight="bold">GSD</text><rect x="108.7" y="8" width="102.7" height="18" fill="#e5e7eb" '
                                  'stroke="#fff"/><text x="160.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">전차 '
                                  '크기(약 5m)</text><rect x="211.3" y="8" width="102.7" height="18" fill="#e5e7eb" stroke="#fff"/><text x="262.7" '
                                  'y="20.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">보이는 정도</text><rect x="6.0" y="26" '
                                  'width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text x="57.3" y="38.5" font-size="9" '
                                  'text-anchor="middle" fill="#111827" font-weight="bold">0.3m</text><rect x="108.7" y="26" width="102.7" '
                                  'height="18" fill="#fff" stroke="#e5e7eb"/><text x="160.0" y="38.5" font-size="9" text-anchor="middle" '
                                  'fill="#111827">약 17픽셀</text><rect x="211.3" y="26" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                  'x="262.7" y="38.5" font-size="9" text-anchor="middle" fill="#111827">식별</text><rect x="6.0" y="44" width="102.7" '
                                  'height="18" fill="#fff" stroke="#e5e7eb"/><text x="57.3" y="56.5" font-size="9" text-anchor="middle" '
                                  'fill="#111827" font-weight="bold">1m</text><rect x="108.7" y="44" width="102.7" height="18" fill="#fff" '
                                  'stroke="#e5e7eb"/><text x="160.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827">약 5픽셀</text><rect '
                                  'x="211.3" y="44" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text x="262.7" y="56.5" font-size="9" '
                                  'text-anchor="middle" fill="#111827">탐지</text><rect x="6.0" y="62" width="102.7" height="18" fill="#fff" '
                                  'stroke="#e5e7eb"/><text x="57.3" y="74.5" font-size="9" text-anchor="middle" fill="#111827" '
                                  'font-weight="bold">5m</text><rect x="108.7" y="62" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                  'x="160.0" y="74.5" font-size="9" text-anchor="middle" fill="#111827">약 1픽셀</text><rect x="211.3" y="62" '
                                  'width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text x="262.7" y="74.5" font-size="9" '
                                  'text-anchor="middle" fill="#111827">어려움</text><text x="160.0" y="97.0" font-size="9" text-anchor="middle" '
                                  'fill="#475569">숫자가 작을수록 자세히 보여요</text></svg>',
                         'f_dri': '<svg viewBox="0 0 320 96" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                  'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                  'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="18" width="99.3" '
                                  'height="30" rx="5" fill="rgb(37,99,235)" fill-opacity="0.35"/><text x="57.7" y="37.0" font-size="9.5" '
                                  'text-anchor="middle" fill="#111827">탐지</text><text x="57.7" y="62.0" font-size="8" text-anchor="middle" '
                                  'fill="#475569">1.5픽셀~</text><rect x="109.3" y="18" width="99.3" height="30" rx="5" fill="rgb(128,68,136)" '
                                  'fill-opacity="0.35"/><text x="159.0" y="37.0" font-size="9.5" text-anchor="middle" fill="#111827">인식</text><text '
                                  'x="159.0" y="62.0" font-size="8" text-anchor="middle" fill="#475569">6픽셀~</text><rect x="210.7" y="18" '
                                  'width="99.3" height="30" rx="5" fill="rgb(220,38,38)" fill-opacity="0.35"/><text x="260.3" y="37.0" '
                                  'font-size="9.5" text-anchor="middle" fill="#111827">식별</text><text x="260.3" y="62.0" font-size="8" '
                                  'text-anchor="middle" fill="#475569">12픽셀~</text><text x="160.0" y="91.0" font-size="9" text-anchor="middle" '
                                  'fill="#475569">크게 보일수록 더 자세히 알 수 있어요</text></svg>',
                         'f_agree': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="77.0" '
                                    'height="18" fill="#e5e7eb" stroke="#fff"/><text x="44.5" y="20.5" font-size="9" text-anchor="middle" '
                                    'fill="#111827" font-weight="bold">물체</text><rect x="83.0" y="8" width="77.0" height="18" fill="#e5e7eb" '
                                    'stroke="#fff"/><text x="121.5" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">A</text><rect x="160.0" y="8" width="77.0" height="18" fill="#e5e7eb" stroke="#fff"/><text '
                                    'x="198.5" y="20.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">B</text><rect x="237.0" '
                                    'y="8" width="77.0" height="18" fill="#e5e7eb" stroke="#fff"/><text x="275.5" y="20.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">같음</text><rect x="6.0" y="26" width="77.0" height="18" '
                                    'fill="#fff" stroke="#e5e7eb"/><text x="44.5" y="38.5" font-size="9" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">OBJ01</text><rect x="83.0" y="26" width="77.0" height="18" fill="#fff" '
                                    'stroke="#e5e7eb"/><text x="121.5" y="38.5" font-size="9" text-anchor="middle" fill="#111827">tank</text><rect '
                                    'x="160.0" y="26" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="198.5" y="38.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827">tank</text><rect x="237.0" y="26" width="77.0" height="18" fill="#fff" '
                                    'stroke="#e5e7eb"/><text x="275.5" y="38.5" font-size="9" text-anchor="middle" fill="#111827">O</text><rect '
                                    'x="6.0" y="44" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="44.5" y="56.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">OBJ02</text><rect x="83.0" y="44" width="77.0" '
                                    'height="18" fill="#fff" stroke="#e5e7eb"/><text x="121.5" y="56.5" font-size="9" text-anchor="middle" '
                                    'fill="#111827">apc</text><rect x="160.0" y="44" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                    'x="198.5" y="56.5" font-size="9" text-anchor="middle" fill="#111827">tank</text><rect x="237.0" y="44" '
                                    'width="77.0" height="18" fill="#fef3c7" stroke="#e5e7eb"/><text x="275.5" y="56.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">X</text><rect x="6.0" y="62" width="77.0" height="18" '
                                    'fill="#fff" stroke="#e5e7eb"/><text x="44.5" y="74.5" font-size="9" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">OBJ03</text><rect x="83.0" y="62" width="77.0" height="18" fill="#fff" '
                                    'stroke="#e5e7eb"/><text x="121.5" y="74.5" font-size="9" text-anchor="middle" fill="#111827">truck</text><rect '
                                    'x="160.0" y="62" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="198.5" y="74.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827">truck</text><rect x="237.0" y="62" width="77.0" height="18" fill="#fff" '
                                    'stroke="#e5e7eb"/><text x="275.5" y="74.5" font-size="9" text-anchor="middle" fill="#111827">O</text><text '
                                    'x="160.0" y="97.0" font-size="9" text-anchor="middle" fill="#475569">3개 중 2개가 같으면 일치율 67%</text></svg>',
                         'f_clean': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="92.0" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="54.0" y="34.0" font-size="10" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">Tank</text><text x="54.0" y="47.0" font-size="8.5" '
                                    'text-anchor="middle" fill="#475569">대문자</text><path d="M101.0,37.0 L113.0,37.0" stroke="#475569" '
                                    'stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="114.0" y="14.0" width="92.0" height="46.0" '
                                    'rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="34.0" font-size="10" text-anchor="middle" '
                                    'fill="#111827" font-weight="bold">tank </text><text x="160.0" y="47.0" font-size="8.5" text-anchor="middle" '
                                    'fill="#475569">빈칸</text><path d="M207.0,37.0 L219.0,37.0" stroke="#475569" stroke-width="1.6" '
                                    'marker-end="url(#tfa)" fill="none"/><rect x="220.0" y="14.0" width="92.0" height="46.0" rx="7" fill="#dbeafe" '
                                    'stroke="#94a3b8"/><text x="266.0" y="34.0" font-size="10" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">tank</text><text x="266.0" y="47.0" font-size="8.5" text-anchor="middle" fill="#475569">같은 '
                                    '값</text><text x="160.0" y="87.0" font-size="9" text-anchor="middle" fill="#475569">제각각인 값을 하나로 맞춰요</text></svg>',
                         'f_meta': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="154.0" '
                                   'height="18" fill="#e5e7eb" stroke="#fff"/><text x="83.0" y="20.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">항목</text><rect x="160.0" y="8" width="154.0" height="18" fill="#e5e7eb" '
                                   'stroke="#fff"/><text x="237.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">예</text><rect x="6.0" y="26" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="83.0" y="38.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">촬영 시각</text><rect '
                                   'x="160.0" y="26" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="38.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">UTC로 기록</text><rect x="6.0" y="44" width="154.0" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="83.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">GSD</text><rect x="160.0" y="44" width="154.0" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="237.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827">0.5m</text><rect '
                                   'x="6.0" y="62" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="83.0" y="74.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">태양 방위각</text><rect x="160.0" y="62" width="154.0" '
                                   'height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="74.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827">150도</text><text x="160.0" y="97.0" font-size="9" text-anchor="middle" fill="#475569">언제, 어디서, 어떤 '
                                   '조건인지 적어요</text></svg>',
                         'f_shadow': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                     'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                     'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="92.0" '
                                     'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="54.0" y="34.0" font-size="9.5" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">태양 방위각</text><text x="54.0" y="47.0" font-size="8.5" '
                                     'text-anchor="middle" fill="#475569">150도</text><path d="M101.0,37.0 L113.0,37.0" stroke="#475569" '
                                     'stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="114.0" y="14.0" width="92.0" height="46.0" '
                                     'rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="34.0" font-size="9.5" text-anchor="middle" '
                                     'fill="#111827" font-weight="bold">+ 180도</text><text x="160.0" y="47.0" font-size="8.5" text-anchor="middle" '
                                     'fill="#475569">반대쪽</text><path d="M207.0,37.0 L219.0,37.0" stroke="#475569" stroke-width="1.6" '
                                     'marker-end="url(#tfa)" fill="none"/><rect x="220.0" y="14.0" width="92.0" height="46.0" rx="7" fill="#dbeafe" '
                                     'stroke="#94a3b8"/><text x="266.0" y="34.0" font-size="9.5" text-anchor="middle" fill="#111827" '
                                     'font-weight="bold">예상 그림자</text><text x="266.0" y="47.0" font-size="8.5" text-anchor="middle" '
                                     'fill="#475569">330도</text><text x="160.0" y="87.0" font-size="9" text-anchor="middle" fill="#475569">0도를 넘는 '
                                     '경우도 함께 생각해요</text></svg>',
                         'f_debug': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="92.0" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="54.0" y="40.5" font-size="9.5" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">오류 메시지</text><path d="M101.0,37.0 L113.0,37.0" '
                                    'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="114.0" y="14.0" width="92.0" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="40.5" font-size="9.5" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">원하는 동작</text><path d="M207.0,37.0 L219.0,37.0" '
                                    'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="220.0" y="14.0" width="92.0" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="266.0" y="40.5" font-size="10" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">고칠 범위</text><text x="160.0" y="87.0" font-size="9" '
                                    'text-anchor="middle" fill="#475569">이 세 가지를 담아 다시 요청해요</text></svg>',
                         'f_pre': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                  'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                  'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="92.0" '
                                  'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="54.0" y="40.5" font-size="10" text-anchor="middle" '
                                  'fill="#111827" font-weight="bold">정사보정</text><path d="M101.0,37.0 L113.0,37.0" stroke="#475569" '
                                  'stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="114.0" y="14.0" width="92.0" height="46.0" rx="7" '
                                  'fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="40.5" font-size="9.5" text-anchor="middle" fill="#111827" '
                                  'font-weight="bold">GSD 표준화</text><path d="M207.0,37.0 L219.0,37.0" stroke="#475569" stroke-width="1.6" '
                                  'marker-end="url(#tfa)" fill="none"/><rect x="220.0" y="14.0" width="92.0" height="46.0" rx="7" fill="#dbeafe" '
                                  'stroke="#94a3b8"/><text x="266.0" y="40.5" font-size="9.5" text-anchor="middle" fill="#111827" '
                                  'font-weight="bold">구름 마스킹</text><text x="160.0" y="87.0" font-size="9" text-anchor="middle" fill="#475569">같은 '
                                  '규격으로 맞춰요</text></svg>',
                         'f_contract': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                       'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                       'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="154.0" '
                                       'height="18" fill="#e5e7eb" stroke="#fff"/><text x="83.0" y="20.5" font-size="9" text-anchor="middle" '
                                       'fill="#111827" font-weight="bold">항목</text><rect x="160.0" y="8" width="154.0" height="18" fill="#e5e7eb" '
                                       'stroke="#fff"/><text x="237.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                       'font-weight="bold">기준</text><rect x="6.0" y="26" width="154.0" height="18" fill="#fff" '
                                       'stroke="#e5e7eb"/><text x="83.0" y="38.5" font-size="9" text-anchor="middle" fill="#111827" '
                                       'font-weight="bold">GSD</text><rect x="160.0" y="26" width="154.0" height="18" fill="#fff" '
                                       'stroke="#e5e7eb"/><text x="237.0" y="38.5" font-size="9" text-anchor="middle" fill="#111827">0.5m ± '
                                       '0.05</text><rect x="6.0" y="44" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="83.0" '
                                       'y="56.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">구름</text><rect x="160.0" '
                                       'y="44" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="56.5" font-size="9" '
                                       'text-anchor="middle" fill="#111827">20% 이하</text><rect x="6.0" y="62" width="154.0" height="18" fill="#fff" '
                                       'stroke="#e5e7eb"/><text x="83.0" y="74.5" font-size="9" text-anchor="middle" fill="#111827" '
                                       'font-weight="bold">오프나디르</text><rect x="160.0" y="62" width="154.0" height="18" fill="#fff" '
                                       'stroke="#e5e7eb"/><text x="237.0" y="74.5" font-size="9" text-anchor="middle" fill="#111827">30도 '
                                       '이하</text><text x="160.0" y="97.0" font-size="9" text-anchor="middle" fill="#475569">기준을 숫자로 '
                                       '약속해요</text></svg>',
                         'f_param': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="65.5" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="40.8" y="40.5" font-size="10" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">작은 함수</text><path d="M74.5,37.0 L86.5,37.0" '
                                    'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="87.5" y="14.0" width="65.5" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="120.2" y="40.5" font-size="9.5" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">예시로 확인</text><path d="M154.0,37.0 L166.0,37.0" '
                                    'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="167.0" y="14.0" width="65.5" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="199.8" y="40.5" font-size="10" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">전체 적용</text><path d="M233.5,37.0 L245.5,37.0" '
                                    'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="246.5" y="14.0" width="65.5" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="279.2" y="40.5" font-size="10" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">비교표</text><text x="160.0" y="87.0" font-size="9" '
                                    'text-anchor="middle" fill="#475569">작게 만들고 확인한 뒤 넓혀요</text></svg>',
                         'f_verify': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                     'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                     'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="65.5" '
                                     'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="40.8" y="40.5" font-size="10" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">AI 답변</text><path d="M74.5,37.0 L86.5,37.0" '
                                     'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="87.5" y="14.0" width="65.5" '
                                     'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="120.2" y="40.5" font-size="10" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">근거 수치</text><path d="M154.0,37.0 L166.0,37.0" '
                                     'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="167.0" y="14.0" width="65.5" '
                                     'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="199.8" y="40.5" font-size="10" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">작은 예시</text><path d="M233.5,37.0 L245.5,37.0" '
                                     'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="246.5" y="14.0" width="65.5" '
                                     'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="279.2" y="40.5" font-size="10" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">판단</text><text x="160.0" y="87.0" font-size="9" '
                                     'text-anchor="middle" fill="#475569">답변만 믿지 말고 근거를 함께 받아요</text></svg>',
                         'f_leak': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="145.0" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="80.5" y="34.0" font-size="9.5" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">검증 데이터</text><text x="80.5" y="47.0" font-size="8.5" '
                                   'text-anchor="middle" fill="#475569">임계값 고르기</text><path d="M154.0,37.0 L166.0,37.0" stroke="#475569" '
                                   'stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="167.0" y="14.0" width="145.0" height="46.0" '
                                   'rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="239.5" y="34.0" font-size="9.5" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">테스트 데이터</text><text x="239.5" y="47.0" font-size="8.5" text-anchor="middle" '
                                   'fill="#475569">한 번 확인</text><text x="160.0" y="87.0" font-size="9" text-anchor="middle" fill="#475569">고를 때 쓴 '
                                   '데이터로 성능을 재면 좋게 보여요</text></svg>',
                         'f_halluc': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                     'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                     'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="92.0" '
                                     'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="54.0" y="40.5" font-size="9.5" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">자료에 없음</text><path d="M101.0,37.0 L113.0,37.0" '
                                     'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="114.0" y="14.0" width="92.0" '
                                     'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="34.0" font-size="9.5" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">AI가 지어냄</text><text x="160.0" y="47.0" font-size="8.5" '
                                     'text-anchor="middle" fill="#475569">그럴듯함</text><path d="M207.0,37.0 L219.0,37.0" stroke="#475569" '
                                     'stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="220.0" y="14.0" width="92.0" height="46.0" '
                                     'rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="266.0" y="34.0" font-size="10" text-anchor="middle" '
                                     'fill="#111827" font-weight="bold">사실 확인</text><text x="266.0" y="47.0" font-size="8.5" text-anchor="middle" '
                                     'fill="#475569">자료와 대조</text><text x="160.0" y="87.0" font-size="9" text-anchor="middle" fill="#475569">없는 내용은 '
                                     '‘자료 없음’으로 쓰게 해요</text></svg>',
                         'f_card': '<svg viewBox="0 0 320 148" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><polygon points="20,118 90,104 '
                                   '180,104 110,118" fill="#dbeafe" fill-opacity="0.55" stroke="#dbeafe"/><text x="192.0" y="115.0" font-size="9.5" '
                                   'text-anchor="start" fill="#111827" font-weight="bold">사람이 판독할 조건</text><polygon points="20,92 90,78 180,78 '
                                   '110,92" fill="#e0e7ff" fill-opacity="0.55" stroke="#e0e7ff"/><text x="192.0" y="89.0" font-size="9.5" '
                                   'text-anchor="start" fill="#111827" font-weight="bold">조심할 조건</text><polygon points="20,66 90,52 180,52 110,66" '
                                   'fill="#c7d2fe" fill-opacity="0.55" stroke="#c7d2fe"/><text x="192.0" y="63.0" font-size="9.5" '
                                   'text-anchor="start" fill="#111827" font-weight="bold">조건별 성능</text><polygon points="20,40 90,26 180,26 110,40" '
                                   'fill="#bfdbfe" fill-opacity="0.55" stroke="#bfdbfe"/><text x="192.0" y="37.0" font-size="9.5" '
                                   'text-anchor="start" fill="#111827" font-weight="bold">결론 한 줄</text><text x="160.0" y="143.0" font-size="9" '
                                   'text-anchor="middle" fill="#475569">언제 믿고 언제 조심할지 적어요</text></svg>',
                         'f_decoy': '<svg viewBox="0 0 320 128" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><text x="200.0" y="14.0" '
                                    'font-size="9.5" text-anchor="middle" fill="#475569" font-weight="bold">단서 →</text><text x="28.0" y="69.0" '
                                    'font-size="9.5" text-anchor="middle" fill="#475569" font-weight="bold">실제 ↓</text><text x="152.0" y="26.0" '
                                    'font-size="9.5" text-anchor="middle" fill="#111827" font-weight="bold">그림자·반사 있음</text><text x="248.0" y="26.0" '
                                    'font-size="9.5" text-anchor="middle" fill="#111827" font-weight="bold">없거나 약함</text><text x="98.0" y="51.0" '
                                    'font-size="9.5" text-anchor="end" fill="#111827" font-weight="bold">장비</text><rect x="104" y="30" width="96" '
                                    'height="36" fill="#dcfce7" stroke="#fff" stroke-width="2"/><text x="152.0" y="52.0" font-size="11" '
                                    'text-anchor="middle" fill="#16a34a" font-weight="bold">통과</text><rect x="200" y="30" width="96" height="36" '
                                    'fill="#fee2e2" stroke="#fff" stroke-width="2"/><text x="248.0" y="52.0" font-size="11" text-anchor="middle" '
                                    'fill="#dc2626" font-weight="bold">재확인</text><text x="98.0" y="87.0" font-size="9.5" text-anchor="end" '
                                    'fill="#111827" font-weight="bold">기만체</text><rect x="104" y="66" width="96" height="36" fill="#fee2e2" '
                                    'stroke="#fff" stroke-width="2"/><text x="152.0" y="88.0" font-size="11" text-anchor="middle" fill="#dc2626" '
                                    'font-weight="bold">드묾</text><rect x="200" y="66" width="96" height="36" fill="#dcfce7" stroke="#fff" '
                                    'stroke-width="2"/><text x="248.0" y="88.0" font-size="11" text-anchor="middle" fill="#16a34a" '
                                    'font-weight="bold">의심</text></svg>',
                         'f_edge': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="77.0" '
                                   'height="18" fill="#e5e7eb" stroke="#fff"/><text x="44.5" y="20.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">경우</text><rect x="83.0" y="8" width="77.0" height="18" fill="#e5e7eb" '
                                   'stroke="#fff"/><text x="121.5" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">그림자</text><rect x="160.0" y="8" width="77.0" height="18" fill="#e5e7eb" stroke="#fff"/><text '
                                   'x="198.5" y="20.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">SAR</text><rect '
                                   'x="237.0" y="8" width="77.0" height="18" fill="#e5e7eb" stroke="#fff"/><text x="275.5" y="20.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">판정</text><rect x="6.0" y="26" width="77.0" height="18" '
                                   'fill="#fff" stroke="#e5e7eb"/><text x="44.5" y="38.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">보통</text><rect x="83.0" y="26" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="121.5" y="38.5" font-size="9" text-anchor="middle" fill="#111827">있음</text><rect x="160.0" y="26" '
                                   'width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="198.5" y="38.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">강함</text><rect x="237.0" y="26" width="77.0" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="275.5" y="38.5" font-size="9" text-anchor="middle" fill="#111827">장비</text><rect '
                                   'x="6.0" y="44" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="44.5" y="56.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">기만</text><rect x="83.0" y="44" width="77.0" height="18" '
                                   'fill="#fff" stroke="#e5e7eb"/><text x="121.5" y="56.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827">없음</text><rect x="160.0" y="44" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="198.5" y="56.5" font-size="9" text-anchor="middle" fill="#111827">약함</text><rect x="237.0" y="44" '
                                   'width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="275.5" y="56.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">의심</text><rect x="6.0" y="62" width="77.0" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="44.5" y="74.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">구름</text><rect x="83.0" y="62" width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="121.5" y="74.5" font-size="9" text-anchor="middle" fill="#111827">빈칸</text><rect x="160.0" y="62" '
                                   'width="77.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="198.5" y="74.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">빈칸</text><rect x="237.0" y="62" width="77.0" height="18" fill="#fef3c7" '
                                   'stroke="#e5e7eb"/><text x="275.5" y="74.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">자료 없음</text><text x="160.0" y="97.0" font-size="9" text-anchor="middle" fill="#475569">빈칸을 '
                                   '어떻게 할지 미리 알려 주세요</text></svg>',
                         'f_count': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="154.0" '
                                    'height="18" fill="#e5e7eb" stroke="#fff"/><text x="83.0" y="20.5" font-size="9" text-anchor="middle" '
                                    'fill="#111827" font-weight="bold">구분</text><rect x="160.0" y="8" width="154.0" height="18" fill="#e5e7eb" '
                                    'stroke="#fff"/><text x="237.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">개수</text><rect x="6.0" y="26" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                    'x="83.0" y="38.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">모델이 센 수</text><rect '
                                    'x="160.0" y="26" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="38.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827">12</text><rect x="6.0" y="44" width="154.0" height="18" fill="#fff" '
                                    'stroke="#e5e7eb"/><text x="83.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">정답: 진짜 장비</text><rect x="160.0" y="44" width="154.0" height="18" fill="#fff" '
                                    'stroke="#e5e7eb"/><text x="237.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827">5</text><rect '
                                    'x="6.0" y="62" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="83.0" y="74.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">개수 오차</text><rect x="160.0" y="62" width="154.0" '
                                    'height="18" fill="#fef3c7" stroke="#e5e7eb"/><text x="237.0" y="74.5" font-size="9" text-anchor="middle" '
                                    'fill="#111827" font-weight="bold">7</text><text x="160.0" y="97.0" font-size="9" text-anchor="middle" '
                                    'fill="#475569">개수만 세면 규모를 잘못 볼 수 있어요</text></svg>',
                         'f_rel': '<svg viewBox="0 0 320 150" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                  'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                  'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="34" y="12" width="274" '
                                  'height="108" fill="#fff" stroke="#94a3b8"/><line x1="34.0" y1="120.0" x2="308.0" y2="12.0" stroke="#94a3b8" '
                                  'stroke-dasharray="4 3"/><text x="304.0" y="24.0" font-size="8.5" text-anchor="end" '
                                  'fill="#475569">완벽</text><polyline points="61.4,115.7 116.2,107.0 171.0,90.8 225.8,66.0 280.6,31.4" fill="none" '
                                  'stroke="#2563eb" stroke-width="2"/><circle cx="61.4" cy="115.7" r="2.5" fill="#2563eb"/><circle cx="116.2" '
                                  'cy="107.0" r="2.5" fill="#2563eb"/><circle cx="171.0" cy="90.8" r="2.5" fill="#2563eb"/><circle cx="225.8" '
                                  'cy="66.0" r="2.5" fill="#2563eb"/><circle cx="280.6" cy="31.4" r="2.5" fill="#2563eb"/><text x="171.0" y="142.0" '
                                  'font-size="9" text-anchor="middle" fill="#475569">mean score</text><text x="10" y="66.0" font-size="9" '
                                  'text-anchor="middle" fill="#475569" transform="rotate(-90 10 66.0)">actual accuracy</text><text x="40.0" y="24.0" '
                                  'font-size="8.5" text-anchor="start" fill="#475569">대각선 아래면 점수가 부풀어 있어요</text></svg>',
                         'f_scale': '<svg viewBox="0 0 320 96" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="18" width="74.0" '
                                    'height="30" rx="5" fill="rgb(37,99,235)" fill-opacity="0.35"/><text x="45.0" y="37.0" font-size="9.5" '
                                    'text-anchor="middle" fill="#111827">낮음</text><text x="45.0" y="62.0" font-size="8" text-anchor="middle" '
                                    'fill="#475569">20~45%</text><rect x="84.0" y="18" width="74.0" height="30" rx="5" fill="rgb(98,78,169)" '
                                    'fill-opacity="0.35"/><text x="121.0" y="37.0" font-size="9.5" text-anchor="middle" '
                                    'fill="#111827">반반</text><text x="121.0" y="62.0" font-size="8" text-anchor="middle" '
                                    'fill="#475569">45~55%</text><rect x="160.0" y="18" width="74.0" height="30" rx="5" fill="rgb(159,58,103)" '
                                    'fill-opacity="0.35"/><text x="197.0" y="37.0" font-size="9.5" text-anchor="middle" '
                                    'fill="#111827">높음</text><text x="197.0" y="62.0" font-size="8" text-anchor="middle" '
                                    'fill="#475569">55~80%</text><rect x="236.0" y="18" width="74.0" height="30" rx="5" fill="rgb(220,38,38)" '
                                    'fill-opacity="0.35"/><text x="273.0" y="37.0" font-size="8.5" text-anchor="middle" fill="#111827">매우 '
                                    '높음</text><text x="273.0" y="62.0" font-size="8" text-anchor="middle" fill="#475569">80~95%</text><text '
                                    'x="160.0" y="91.0" font-size="9" text-anchor="middle" fill="#475569">낱말과 확률을 약속해 둬요 (일부)</text></svg>',
                         'f_conf2': '<svg viewBox="0 0 320 84" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="102.7" '
                                    'height="18" fill="#e5e7eb" stroke="#fff"/><text x="57.3" y="20.5" font-size="9" text-anchor="middle" '
                                    'fill="#111827" font-weight="bold">구분</text><rect x="108.7" y="8" width="102.7" height="18" fill="#e5e7eb" '
                                    'stroke="#fff"/><text x="160.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">무엇을 말하나</text><rect x="211.3" y="8" width="102.7" height="18" fill="#e5e7eb" '
                                    'stroke="#fff"/><text x="262.7" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">예</text><rect x="6.0" y="26" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                    'x="57.3" y="38.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">가능성</text><rect '
                                    'x="108.7" y="26" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text x="160.0" y="38.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827">얼마나 그럴듯한가</text><rect x="211.3" y="26" width="102.7" height="18" '
                                    'fill="#fff" stroke="#e5e7eb"/><text x="262.7" y="38.5" font-size="9" text-anchor="middle" '
                                    'fill="#111827">높음</text><rect x="6.0" y="44" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                    'x="57.3" y="56.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">확신도</text><rect '
                                    'x="108.7" y="44" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text x="160.0" y="56.5" font-size="9" '
                                    'text-anchor="middle" fill="#111827">근거가 얼마나 튼튼한가</text><rect x="211.3" y="44" width="102.7" height="18" '
                                    'fill="#fff" stroke="#e5e7eb"/><text x="262.7" y="56.5" font-size="9" text-anchor="middle" '
                                    'fill="#111827">낮음</text><text x="160.0" y="79.0" font-size="9" text-anchor="middle" fill="#475569">두 가지를 나눠 '
                                    '써요</text></svg>',
                         'f_hitl': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="92.0" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="54.0" y="40.5" font-size="10" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">AI 1차</text><path d="M101.0,37.0 L113.0,37.0" '
                                   'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="114.0" y="14.0" width="92.0" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="40.5" font-size="10" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">사람 판독</text><path d="M207.0,37.0 L219.0,37.0" '
                                   'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="220.0" y="14.0" width="92.0" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="266.0" y="40.5" font-size="10" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">승인</text><text x="160.0" y="87.0" font-size="9" '
                                   'text-anchor="middle" fill="#475569">확인되지 않은 내용은 미확인으로 남겨요</text></svg>',
                         'f_secret': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                     'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                     'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="154.0" '
                                     'height="18" fill="#e5e7eb" stroke="#fff"/><text x="83.0" y="20.5" font-size="9" text-anchor="middle" '
                                     'fill="#111827" font-weight="bold">넣어도 돼요</text><rect x="160.0" y="8" width="154.0" height="18" fill="#e5e7eb" '
                                     'stroke="#fff"/><text x="237.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                     'font-weight="bold">넣으면 안 돼요</text><rect x="6.0" y="26" width="154.0" height="18" fill="#fff" '
                                     'stroke="#e5e7eb"/><text x="83.0" y="38.5" font-size="9" text-anchor="middle" fill="#111827" '
                                     'font-weight="bold">열 이름</text><rect x="160.0" y="26" width="154.0" height="18" fill="#fff" '
                                     'stroke="#e5e7eb"/><text x="237.0" y="38.5" font-size="9" text-anchor="middle" fill="#111827">실제 위치</text><rect '
                                     'x="6.0" y="44" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="83.0" y="56.5" font-size="9" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">가상 샘플</text><rect x="160.0" y="44" width="154.0" '
                                     'height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="56.5" font-size="9" text-anchor="middle" '
                                     'fill="#111827">실제 인물</text><rect x="6.0" y="62" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                     'x="83.0" y="74.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">오류 메시지</text><rect '
                                     'x="160.0" y="62" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="74.5" '
                                     'font-size="9" text-anchor="middle" fill="#111827">장비 정보</text><text x="160.0" y="97.0" font-size="9" '
                                     'text-anchor="middle" fill="#475569">공개 AI에 넣기 전에 나눠 봐요</text></svg>',
                         'f_review': '<svg viewBox="0 0 320 156" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                     'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                     'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><ellipse cx="160.0" cy="70" '
                                     'rx="112" ry="48" fill="none" stroke="#e5e7eb" stroke-width="6"/><path d="M207.7,26.6 L261.3,49.6" '
                                     'stroke="#2563eb" stroke-width="2" marker-end="url(#tfa)" fill="none"/><path d="M261.3,90.4 L207.7,113.4" '
                                     'stroke="#2563eb" stroke-width="2" marker-end="url(#tfa)" fill="none"/><path d="M112.3,113.4 L58.7,90.4" '
                                     'stroke="#2563eb" stroke-width="2" marker-end="url(#tfa)" fill="none"/><path d="M58.7,49.6 L112.3,26.6" '
                                     'stroke="#2563eb" stroke-width="2" marker-end="url(#tfa)" fill="none"/><rect x="116.0" y="10.0" width="88.0" '
                                     'height="24.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="25.5" font-size="9" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">설명</text><rect x="228.0" y="58.0" width="88.0" '
                                     'height="24.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="272.0" y="73.5" font-size="9" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">테스트</text><rect x="116.0" y="106.0" width="88.0" '
                                     'height="24.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="121.5" font-size="9" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">수정</text><rect x="4.0" y="58.0" width="88.0" '
                                     'height="24.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="48.0" y="73.5" font-size="9" '
                                     'text-anchor="middle" fill="#111827" font-weight="bold">비교</text><text x="160.0" y="74.0" font-size="10.5" '
                                     'text-anchor="middle" fill="#475569" font-weight="bold">검토</text><text x="160.0" y="151.0" font-size="9" '
                                     'text-anchor="middle" fill="#475569">고치기 전에 먼저 설명하게 해요</text></svg>',
                         'f_scope': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                    'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                    'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="92.0" '
                                    'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="54.0" y="34.0" font-size="10" '
                                    'text-anchor="middle" fill="#111827" font-weight="bold">고칠 곳</text><text x="54.0" y="47.0" font-size="8.5" '
                                    'text-anchor="middle" fill="#475569">이 함수만</text><path d="M101.0,37.0 L113.0,37.0" stroke="#475569" '
                                    'stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="114.0" y="14.0" width="92.0" height="46.0" '
                                    'rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="160.0" y="34.0" font-size="10" text-anchor="middle" '
                                    'fill="#111827" font-weight="bold">그대로</text><text x="160.0" y="47.0" font-size="8.5" text-anchor="middle" '
                                    'fill="#475569">원본 파일</text><path d="M207.0,37.0 L219.0,37.0" stroke="#475569" stroke-width="1.6" '
                                    'marker-end="url(#tfa)" fill="none"/><rect x="220.0" y="14.0" width="92.0" height="46.0" rx="7" fill="#dbeafe" '
                                    'stroke="#94a3b8"/><text x="266.0" y="34.0" font-size="10" text-anchor="middle" fill="#111827" '
                                    'font-weight="bold">그대로</text><text x="266.0" y="47.0" font-size="8.5" text-anchor="middle" fill="#475569">다른 '
                                    '코드</text><text x="160.0" y="87.0" font-size="9" text-anchor="middle" fill="#475569">고칠 범위를 말해 주세요</text></svg>',
                         'f_tool': '<svg viewBox="0 0 320 92" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="8.0" y="14.0" width="65.5" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="40.8" y="40.5" font-size="10" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">설정값</text><path d="M74.5,37.0 L86.5,37.0" '
                                   'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="87.5" y="14.0" width="65.5" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="120.2" y="40.5" font-size="10" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">계산 함수</text><path d="M154.0,37.0 L166.0,37.0" '
                                   'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="167.0" y="14.0" width="65.5" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="199.8" y="40.5" font-size="10" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">표·그림</text><path d="M233.5,37.0 L245.5,37.0" '
                                   'stroke="#475569" stroke-width="1.6" marker-end="url(#tfa)" fill="none"/><rect x="246.5" y="14.0" width="65.5" '
                                   'height="46.0" rx="7" fill="#dbeafe" stroke="#94a3b8"/><text x="279.2" y="40.5" font-size="10" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">보고서</text><text x="160.0" y="87.0" font-size="9" '
                                   'text-anchor="middle" fill="#475569">한 번에 다시 돌릴 수 있게 묶어요</text></svg>',
                         'f_lint': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="154.0" '
                                   'height="18" fill="#e5e7eb" stroke="#fff"/><text x="83.0" y="20.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">항목</text><rect x="160.0" y="8" width="154.0" height="18" fill="#e5e7eb" '
                                   'stroke="#fff"/><text x="237.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">있음?</text><rect x="6.0" y="26" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="83.0" y="38.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">표본 수</text><rect '
                                   'x="160.0" y="26" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text x="237.0" y="38.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">O</text><rect x="6.0" y="44" width="154.0" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="83.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">한계</text><rect x="160.0" y="44" width="154.0" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="237.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827">O</text><rect x="6.0" y="62" width="154.0" '
                                   'height="18" fill="#fff" stroke="#e5e7eb"/><text x="83.0" y="74.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">AI 산출물 표기</text><rect x="160.0" y="62" width="154.0" height="18" '
                                   'fill="#fef3c7" stroke="#e5e7eb"/><text x="237.0" y="74.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">X</text><text x="160.0" y="97.0" font-size="9" text-anchor="middle" fill="#475569">빠진 항목을 코드로 '
                                   '찾아요</text></svg>',
                         'f_cond': '<svg viewBox="0 0 320 102" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" '
                                   'fill="#111827"><defs><marker id="tfa" viewBox="0 0 10 10" refX="8" refY="5" markerWidth="6" markerHeight="6" '
                                   'orient="auto"><path d="M0,0 L10,5 L0,10 z" fill="#475569"/></marker></defs><rect x="6.0" y="8" width="102.7" '
                                   'height="18" fill="#e5e7eb" stroke="#fff"/><text x="57.3" y="20.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">GSD</text><rect x="108.7" y="8" width="102.7" height="18" fill="#e5e7eb" '
                                   'stroke="#fff"/><text x="160.0" y="20.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">재현율</text><rect x="211.3" y="8" width="102.7" height="18" fill="#e5e7eb" stroke="#fff"/><text '
                                   'x="262.7" y="20.5" font-size="9" text-anchor="middle" fill="#111827" font-weight="bold">표본</text><rect x="6.0" '
                                   'y="26" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text x="57.3" y="38.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">0.5m</text><rect x="108.7" y="26" width="102.7" '
                                   'height="18" fill="#fff" stroke="#e5e7eb"/><text x="160.0" y="38.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827">높음</text><rect x="211.3" y="26" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="262.7" y="38.5" font-size="9" text-anchor="middle" fill="#111827">많음</text><rect x="6.0" y="44" width="102.7" '
                                   'height="18" fill="#fff" stroke="#e5e7eb"/><text x="57.3" y="56.5" font-size="9" text-anchor="middle" '
                                   'fill="#111827" font-weight="bold">1m</text><rect x="108.7" y="44" width="102.7" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="160.0" y="56.5" font-size="9" text-anchor="middle" fill="#111827">중간</text><rect '
                                   'x="211.3" y="44" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text x="262.7" y="56.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827">많음</text><rect x="6.0" y="62" width="102.7" height="18" fill="#fff" '
                                   'stroke="#e5e7eb"/><text x="57.3" y="74.5" font-size="9" text-anchor="middle" fill="#111827" '
                                   'font-weight="bold">2m</text><rect x="108.7" y="62" width="102.7" height="18" fill="#fff" stroke="#e5e7eb"/><text '
                                   'x="160.0" y="74.5" font-size="9" text-anchor="middle" fill="#111827">낮음</text><rect x="211.3" y="62" '
                                   'width="102.7" height="18" fill="#fef3c7" stroke="#e5e7eb"/><text x="262.7" y="74.5" font-size="9" '
                                   'text-anchor="middle" fill="#111827" font-weight="bold">적음</text><text x="160.0" y="97.0" font-size="9" '
                                   'text-anchor="middle" fill="#475569">수치에는 조건과 표본 수를 함께 적어요</text></svg>'},
             'workspace': '🛰️ 위성분석 TF',
             'call_default': '전우님',
             'stay_jobs': {}},
 'competencies': [{'id': 'D1',
                   'group': '디지털·AI 활용 역량',
                   'name': '프롬프트 구조화',
                   'criteria': '업무 요청을 맥락·입력·할 일·출력 형식으로 나눠 AI에게 분명하게 요청할 수 있다.',
                   'ncs': '디지털능력 (인공지능(AI)활용능력), 의사소통능력 (문서소통능력)'},
                  {'id': 'D2',
                   'group': '디지털·AI 활용 역량',
                   'name': '데이터 설명',
                   'criteria': '파일·열·값의 모양을 AI에게 정확히 알려 주어 추측 없이 동작하는 코드를 받을 수 있다.',
                   'ncs': '디지털능력 (디지털활용능력)'},
                  {'id': 'D3',
                   'group': '디지털·AI 활용 역량',
                   'name': '수정 요청(디버깅)',
                   'criteria': '오류 메시지와 기대 동작을 담아 수정을 요청하고, 고칠 범위를 정할 수 있다.',
                   'ncs': '문제해결능력 (문제분석능력), 디지털능력 (디지털활용능력)'},
                  {'id': 'D4',
                   'group': '디지털·AI 활용 역량',
                   'name': '나누어 만들기',
                   'criteria': '큰 일을 작은 단계로 쪼개 요청하고, 조건을 바꿔 돌릴 수 있는 함수로 만들어 달라고 할 수 있다.',
                   'ncs': '문제해결능력 (대안발굴능력), 디지털능력 (디지털활용능력)'},
                  {'id': 'D5',
                   'group': '디지털·AI 활용 역량',
                   'name': '결과 검증',
                   'criteria': 'AI의 결과를 근거 수치·경계 사례·작은 테스트로 확인하고, 믿어도 되는지 판단할 수 있다.',
                   'ncs': '수리능력 (통계활용능력), 문제해결능력 (문제분석능력)'},
                  {'id': 'D6',
                   'group': '디지털·AI 활용 역량',
                   'name': '안전한 AI 활용',
                   'criteria': 'AI에 넣으면 안 되는 정보를 구분하고, 사람이 확인할 일을 정할 수 있다.',
                   'ncs': '디지털능력 (디지털책임의식)'},
                  {'id': 'J1',
                   'group': '직무전문 역량',
                   'name': 'AI로 문서 만들기',
                   'criteria': '독자·양식·근거를 지정해 AI가 업무 문서 초안을 만들게 하고, 사실을 확인해 고칠 수 있다.',
                   'ncs': '의사소통능력 (문서소통능력)'},
                  {'id': 'C1',
                   'group': '협업·소통 역량',
                   'name': '결과로 조율하기',
                   'criteria': 'AI로 만든 비교 결과를 근거로 팀원의 다른 의견을 조율해 기준을 합의할 수 있다.',
                   'ncs': '대인관계능력 (협업능력·갈등관리능력)'}],
 'missions': [{'id': 'm1',
               'no': 1,
               'title': '광학·SAR 영상의 원리와 장비 판독 단서 이해',
               'short': '프롬프트 네 칸',
               'kind': '개인',
               'competencies': ['D1'],
               'outline': 'AI에게 일을 시킬 때 꼭 넣어야 할 네 칸(맥락·입력·할 일·출력 형식)을 익히고, 장비가 GSD에 따라 몇 픽셀로 보이는지 계산하는 표를 만들어요.',
               'brief': {'subtitle': 'GSD별 판독 가능 수준표 만들기: 프롬프트의 네 칸',
                         'time': '약 25분',
                         'goal': '프롬프트의 네 칸을 채워 AI에게 요청하고, GSD별 판독 가능 수준표를 코드와 문서로 받아요.',
                         'why': '“코드 짜 줘”와 무엇이 다를까요? 위성영상은 픽셀 한 칸의 크기(GSD)에 따라 같은 장비도 전혀 다르게 보여요. 이런 계산을 AI에게 맡길 때, 무엇을 알려 주느냐에 따라 결과가 크게 달라져요.',
                         'case': {'title': '해외 사례: 구름과 밤에도 찍는 SAR, 그래도 보이지 않는 것',
                                  'source': 'Kyiv Independent, Janes 보도 (2025.6)',
                                  'points': ['2025년 6월 우크라이나의 ‘거미줄 작전’ 직후, 공개 분석가들은 상용 SAR 위성 Umbra의 영상으로 러시아 벨라야 비행장의 피해를 확인했어요. SAR는 전파로 찍기 때문에 '
                                             '구름이나 밤의 영향을 덜 받아요.',
                                             '다른 비행장 영상에서는 큰 피해가 보이지 않았지만, 해상도가 낮아 작은 피해는 가려졌을 수 있다는 평가도 함께 나왔어요.',
                                             '어떤 조건에서 무엇이 보이는지 아는 것이 판독의 출발이에요. 이번 미션에서는 그 계산을 AI에게 정확히 설명하는 연습을 해요.']},
                         'team': '한지우(판독 담당)가 표를 부탁해요. 전우님이 AI에게 프롬프트를 보내 표와 문서를 받아요.',
                         'terms': [{'t': '바이브 코딩',
                                    'en': 'vibe coding',
                                    'fig': 'f_loop',
                                    'd': '만들고 싶은 것을 말로 설명해 AI가 코드를 쓰게 하고, 실행 결과를 보며 고쳐 가는 방식이에요.',
                                    'easy': '코드를 직접 쓰는 대신, AI에게 일을 설명하고 결과를 확인하는 역할을 맡는 거예요.',
                                    'use': '요청 → 생성 → 실행 → 확인 → 수정 요청을 반복해요.'},
                                   {'t': '프롬프트',
                                    'en': 'prompt',
                                    'fig': 'f_four',
                                    'd': 'AI에게 보내는 요청 글이에요.',
                                    'easy': '새로 온 동료에게 주는 업무 지시서라고 생각하면 돼요.',
                                    'use': '맥락·입력·할 일·출력 형식의 네 칸을 채우면 결과가 안정돼요.'},
                                   {'t': 'GSD (지상표본거리)',
                                    'en': 'Ground Sample Distance',
                                    'fig': 'f_gsd',
                                    'd': '영상의 픽셀 한 칸이 땅에서 차지하는 길이예요.',
                                    'easy': 'GSD 0.5m면 픽셀 한 칸이 50cm예요. 숫자가 작을수록 자세히 보여요.',
                                    'use': '프롬프트에 어떤 GSD로 계산할지 숫자로 적어요.'},
                                   {'t': '탐지·인식·식별',
                                    'en': 'Detection·Recognition·Identification',
                                    'fig': 'f_dri',
                                    'd': '무언가 있음을 아는 단계(탐지), 어떤 부류인지 아는 단계(인식), 구체적으로 무엇인지 아는 단계(식별)예요.',
                                    'easy': '‘뭔가 있다’ → ‘궤도 차량이다’ → ‘전차다’ 순서예요.',
                                    'use': '물체가 몇 픽셀로 보이는지로 어림해요. 이번 실습은 1.5·6·12픽셀 기준을 써요.'}],
                         'steps': ['네 칸 익히기: 프롬프트의 네 칸', 'AI에게 시키기: 프롬프트를 만들어 AI에게 보내기', '내 말로 쓰기: 같은 일을 내 말로 요청하기'],
                         'glossary': [],
                         'outputs': ['M1 GSD별 판독 가능 수준표']},
               'start_messages': [['dm_han', 'han', '{call}, 장비가 GSD별로 몇 픽셀로 보이는지 표가 필요해요. 직접 계산하지 말고 AI에게 시켜 보세요. 파일은 m1_equipment.csv예요.']],
               'finish_messages': [['dm_han', 'han', '{call}, 표 잘 받았어요! 프롬프트도 다음에 그대로 쓸 수 있겠어요 👍']],
               'steps': [{'short': '네 칸 익히기',
                          'title': '프롬프트의 네 칸',
                          'lead': 'AI는 전우님 업무도, 파일도 몰라요. **네 칸**을 채워 알려 주면 돼요.',
                          'blocks': [{'id': 's1_info',
                                      'type': 'info',
                                      'title': '좋은 프롬프트의 네 칸',
                                      'icon': '🧩',
                                      'text': ['같은 일을 시켜도 무엇을 알려 주느냐에 따라 결과가 달라져요. 처음에는 아래 네 칸만 기억하세요.'],
                                      'table': {'head': ['칸', '무엇을 적나', '예'],
                                                'rows': [['맥락', '왜 필요한지, 어떤 일인지', '장비가 GSD별로 몇 픽셀로 보이는지 확인하려고 해'],
                                                         ['입력', '파일 이름, 열 이름, 값의 뜻', 'm1_equipment.csv, 열은 equipment·length_m·width_m'],
                                                         ['할 일', '계산 규칙을 숫자로', '픽셀 수 = 대표 크기 ÷ GSD'],
                                                         ['출력 형식', '받고 싶은 모양', '장비는 행, GSD는 열인 표']]},
                                      'figure': 'f_four'},
                                     {'id': 's1_q',
                                      'type': 'quiz',
                                      'title': '어떤 프롬프트가 가장 좋을까요?',
                                      'text': '같은 일을 시키는 세 프롬프트예요. 다시 묻는 일이 가장 적을 것은 무엇일까요?',
                                      'options': ['위성영상 분석하는 코드 좀 짜 줘.',
                                                  '장비가 잘 보이는지 알려 줘. 잘 부탁해.',
                                                  '장비가 GSD별로 몇 픽셀로 보이는지 확인하려고 해. m1_equipment.csv의 크기로 GSD마다 픽셀 수를 계산해 표로 보여 줘.'],
                                      'answer': 2,
                                      'explain': '맥락, 입력, 할 일, 출력 형식이 모두 들어 있어요. 정중한 말보다 구체적인 정보가 결과를 바꿔요.',
                                      'hints': ['네 칸 가운데 몇 칸이 채워졌는지 세어 보세요.', '파일 이름과 받고 싶은 모양까지 적힌 것을 찾아보세요.']}]},
                         {'short': 'AI에게 시키기',
                          'title': '프롬프트를 만들어 AI에게 보내기',
                          'lead': '칸마다 문장을 고르면 프롬프트가 만들어져요. 일부러 **아쉬운 문장**도 보내 보세요. 무엇이 달라지는지 보는 것이 이번 활동의 핵심이에요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': 'GSD별 판독 가능 수준표 만들기',
                                      'ask': ['han', '장비 크기 파일을 드릴게요. GSD 0.3m~10m에서 각각 몇 픽셀로 보이는지 표로 받고 싶어요.'],
                                      'file': 'm1_equipment.csv',
                                      'slots': [{'id': 'ctx',
                                                 'label': '맥락',
                                                 'why': '왜 필요한지 알려요.',
                                                 'options': [{'text': '위성영상 판독 업무를 돕고 있어. 장비가 GSD별로 화면에서 몇 픽셀로 보이는지 확인하려고 해.', 'good': True},
                                                             {'text': '코드 좀 짜 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '데이터를 읽어 요약 통계를 만들었어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m1_equipment.csv')\n"
                                                                  'print(df.describe())',
                                                          'problem': '평균, 최댓값 같은 요약만 나왔어요. 왜 필요한지 몰라서 AI가 가장 흔한 일을 했어요.',
                                                          'hint': '어떤 업무이고 무엇을 확인하려는지 한 문장으로 알려 주세요.'}},
                                                {'id': 'input',
                                                 'label': '입력',
                                                 'why': '파일과 열을 알려요.',
                                                 'options': [{'text': '데이터는 m1_equipment.csv이고, 열은 equipment(장비 종류), length_m(길이, 미터), width_m(폭, '
                                                                      '미터)이야.',
                                                              'good': True},
                                                             {'text': '장비 데이터 파일이 있어.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '장비 파일을 읽는 코드를 만들었어요.',
                                                          'expect': 'error',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/vehicles.csv')\n"
                                                                  "print(df['size'] / df['resolution'])",
                                                          'problem': 'AI가 파일 이름을 몰라 그럴듯한 이름을 지어냈고, 그래서 파일을 찾지 못했어요.',
                                                          'hint': '파일 이름과 열 이름을 그대로 적어 주세요. AI는 전우님 컴퓨터의 파일을 볼 수 없어요.'}},
                                                {'id': 'task',
                                                 'label': '할 일',
                                                 'why': '계산 규칙을 숫자로 적어요.',
                                                 'options': [{'text': '대표 크기 = √(length_m × width_m), 픽셀 수 = 대표 크기 ÷ GSD로 GSD 0.3, 0.5, 1, 2, 5, '
                                                                      '10m마다 계산하고, 1.5픽셀 이상은 탐지, 6픽셀 이상은 인식, 12픽셀 이상은 식별로 표시해 줘.',
                                                              'good': True},
                                                             {'text': 'GSD별로 잘 보이는지 계산해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '길이를 GSD로 나눠 얼마나 잘 보이는지 계산했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m1_equipment.csv')\n"
                                                                  'for g in [0.3, 0.5, 1, 2, 5, 10]:\n'
                                                                  "    df[f'{g}m'] = (df['length_m'] / g).round(1)\n"
                                                                  'display(df)',
                                                          'problem': '오류는 없지만 길이만 나눠 픽셀 수가 부풀었고, 탐지·인식·식별 표시도 없어요. 계산식을 몰라서 AI가 추측한 거예요. **오류가 없다고 '
                                                                     '맞는 결과는 아니에요.**',
                                                          'hint': '계산식과 기준값을 숫자로 적어 주세요.'}},
                                                {'id': 'out',
                                                 'label': '출력 형식',
                                                 'why': '받고 싶은 모양을 알려요.',
                                                 'options': [{'text': '장비는 행, GSD는 열인 표로 보여 주고, 같은 표를 넣은 HTML 보고서를 report 변수에 담아 줘. 전차 크기 물체가 GSD별로 '
                                                                      '어떻게 보이는지 그림도 그려 줘.',
                                                              'good': True},
                                                             {'text': '결과를 보여 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '계산 결과를 출력했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m1_equipment.csv')\n"
                                                                  'for _, r in df.iterrows():\n'
                                                                  '    for g in [0.3, 0.5, 1, 2, 5, 10]:\n'
                                                                  "        print(r['equipment'], g, round((r['length_m'] * r['width_m']) ** 0.5 / g, "
                                                                  '1))',
                                                          'problem': '숫자는 맞지만 한 줄씩 길게 나와 비교하기 어렵고, 그대로 문서에 넣을 수도 없어요.',
                                                          'hint': '행과 열을 어떻게 놓을지, 문서와 그림도 받을지 적어 주세요.'}}],
                                      'done': {'say': '요청대로 GSD별 픽셀 수와 수준을 표로 만들고, 그림과 보고서도 만들었어요.',
                                               'code': 'import numpy as np\n'
                                                       'import pandas as pd\n'
                                                       'import matplotlib.pyplot as plt\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m1_equipment.csv')\n"
                                                       'GSDS = [0.3, 0.5, 1, 2, 5, 10]\n'
                                                       '\n'
                                                       'def level(px):\n'
                                                       '    if px >= 12:\n'
                                                       "        return '식별'\n"
                                                       '    if px >= 6:\n'
                                                       "        return '인식'\n"
                                                       '    if px >= 1.5:\n'
                                                       "        return '탐지'\n"
                                                       "    return '어려움'\n"
                                                       '\n'
                                                       'rows = []\n'
                                                       'for _, r in df.iterrows():\n'
                                                       "    size = (r['length_m'] * r['width_m']) ** 0.5\n"
                                                       "    row = {'장비': r['equipment']}\n"
                                                       '    for g in GSDS:\n'
                                                       '        px = round(size / g, 1)\n'
                                                       "        row[f'{g}m'] = f'{px}px {level(px)}'\n"
                                                       '    rows.append(row)\n'
                                                       'table = pd.DataFrame(rows)\n'
                                                       'display(table)\n'
                                                       '\n'
                                                       '# 전차 크기(6.9m x 3.6m) 물체를 GSD별로 줄여 보기 (0.1m 간격 가상 영상)\n'
                                                       'base = np.full((400, 400), 0.35)\n'
                                                       'base[182:218, 165:234] = 0.85\n'
                                                       'fig, axes = plt.subplots(1, 4, figsize=(8, 2.2))\n'
                                                       'for ax, g in zip(axes, [0.5, 1, 2, 5]):\n'
                                                       '    k = int(g / 0.1)\n'
                                                       '    n = 400 // k\n'
                                                       '    small = base[:n * k, :n * k].reshape(n, k, n, k).mean(axis=(1, 3))\n'
                                                       "    ax.imshow(small, cmap='gray', vmin=0, vmax=1, interpolation='nearest')\n"
                                                       "    ax.set_title(f'GSD {g} m')\n"
                                                       "    ax.axis('off')\n"
                                                       '\n'
                                                       "report = ('<h2>GSD별 판독 가능 수준표</h2>'\n"
                                                       "          '<p>기준: 1.5픽셀 탐지, 6픽셀 인식, 12픽셀 식별. 대표 크기는 길이와 폭의 기하평균이에요.</p>'\n"
                                                       '          + table.to_html(index=False))',
                                               'lesson': '네 칸을 모두 채우니 AI가 추측할 일이 없어졌어요. 전차도 GSD 0.5m까지는 인식할 수 있지만, 1m부터는 탐지까지만, 5m에서는 찾기도 어렵다는 것이 표와 '
                                                         '그림에서 바로 보여요.'},
                                      'on_ok': [['dm_han', 'han', '오, 표가 바로 나왔네요. {call}, 이 프롬프트 저도 좀 빌려 쓸게요!']]}]},
                         {'short': '내 말로 쓰기',
                          'title': '같은 일을 내 말로 요청하기',
                          'lead': '고른 문장을 그대로 외울 필요는 없어요. **네 칸만 채우면** 말투는 자유예요.',
                          'blocks': [{'id': 's3_q',
                                      'type': 'quiz',
                                      'title': '결과를 받은 뒤에는',
                                      'text': 'AI가 만든 코드가 오류 없이 돌았어요. 다음으로 무엇을 하면 좋을까요?',
                                      'options': ['오류가 없으니 그대로 보고한다.', '하나만 손으로 계산해 맞는지 확인한다.', '더 어려운 말로 다시 요청한다.'],
                                      'answer': 1,
                                      'explain': '전차는 √(6.9 × 3.6) ≈ 5m이니 GSD 1m면 약 5픽셀이에요. 표의 값과 같은지 한 칸만 봐도 큰 실수를 잡을 수 있어요.',
                                      'hints': ['앞 활동에서 오류 없이 이상한 숫자가 나온 경우를 떠올려 보세요.', '전부 확인할 필요는 없어요. 한 칸이면 충분해요.']},
                                     {'id': 's3_write',
                                      'type': 'write',
                                      'title': '내 프롬프트 쓰기',
                                      'prompt': '**새 요청**: 새 장비(길이 5.0m, 폭 2.0m)가 추가됐어요. 파일은 m1_equipment_new.csv이고 열은 같아요. GSD 0.3m, 1m에서의 픽셀 수를 보고 '
                                                '싶어요. 네 칸을 채워 내 말로 프롬프트를 써 보세요.',
                                      'example': '새 장비가 위성영상에서 몇 픽셀로 보이는지 확인하려고 해. 데이터는 m1_equipment_new.csv이고 열은 equipment, length_m, width_m이야. 대표 '
                                                 '크기 = √(length_m × width_m), 픽셀 수 = 대표 크기 ÷ GSD로 GSD 0.3m, 1m에서 계산해 줘. 장비는 행, GSD는 열인 표로 보여 줘.',
                                      'min': 30,
                                      'checklist': ['맥락(왜 필요한지)을 적었어요', '파일 이름과 열 이름을 적었어요', '계산식이나 기준을 숫자로 적었어요', '받고 싶은 모양을 적었어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm1_gsd_table',
                            'title': 'M1 GSD별 판독 가능 수준표',
                            'sections': [{'h': '1. AI에게 보낸 프롬프트와 결과', 'blocks': ['s2_vibe']}, {'h': '2. 내가 쓴 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm2',
               'no': 2,
               'title': '판독 요청 분석 및 판독 기준 수립',
               'short': '데이터 설명하기',
               'kind': '협업',
               'competencies': ['D2', 'C1'],
               'outline': '판독 담당 3명의 판독이 얼마나 같은지 계산하면서, 파일과 값의 모양을 AI에게 알려 주는 법을 익혀요. 결과로 판독 기준 문서도 요청해요.',
               'brief': {'subtitle': '판독 일치율 계산과 판독 기준 문서: 데이터를 설명하기',
                         'time': '약 25분',
                         'goal': '값이 지저분한 데이터를 AI에게 설명해 판독 일치율을 계산하고, 그 결과를 근거로 판독 기준 문서를 요청해요.',
                         'why': 'AI는 전우님의 데이터가 지저분하다는 걸 모를 수 있어요. 숫자는 무엇을 어떻게 세었는지에 따라 달라져요. 데이터의 모양과 세는 기준을 먼저 맞추지 않으면 AI도 사람도 다른 답을 내놓아요.',
                         'case': {'title': '해외 사례: 같은 비행장 영상, 다르게 나온 피해 대수',
                                  'source': 'Kyiv Independent, Janes 보도 (2025.6)',
                                  'points': ['2025년 6월 러시아 벨라야 비행장의 첫 공개 SAR 영상을 본 분석에서는 Tu-95MS 폭격기 최소 3대 파괴, 1대 손상으로 전해졌어요.',
                                             '이후 Janes와 우크라이나 분석가들은 6월 3일 영상 등을 바탕으로 4대가 파괴됐다고 봤어요. 영상 시점과 판정 기준에 따라 숫자가 달라진 거예요.',
                                             '판독 데이터를 AI에게 맡길 때도 같아요. 무엇을 한 건으로 세는지, 값이 어떻게 적혀 있는지 먼저 알려 줘야 해요.']},
                         'team': '한지우·박도윤(판독 담당)의 판독이 서로 달라요. 전우님이 AI로 일치율을 계산해 이야기의 근거를 만들어요.',
                         'terms': [{'t': '판독 일치율',
                                    'en': 'agreement rate',
                                    'fig': 'f_agree',
                                    'd': '두 사람이 같은 물체를 같은 값으로 판독한 비율이에요.',
                                    'easy': '24개 중 18개가 같으면 일치율 75%예요.',
                                    'use': '일치율이 낮으면 사람 탓을 하기 전에 기준이 모호한지 먼저 봐요.'},
                                   {'t': '데이터 정리',
                                    'en': 'data cleaning',
                                    'fig': 'f_clean',
                                    'd': '대소문자, 빈칸처럼 값이 제각각인 것을 하나로 맞추는 일이에요.',
                                    'easy': '‘Tank’와 ‘tank ’을 같은 값으로 맞추는 일이에요.',
                                    'use': '프롬프트에 ‘값이 지저분하니 먼저 정리해 줘’라고 적어요.'}],
                         'steps': ['데이터 보여 주기: AI에게 데이터를 설명하는 법', '일치율 계산: 일치율을 계산하고 갈린 물체 찾기', '기준 합의: 결과로 기준을 합의하고 내 말로 요청하기'],
                         'glossary': [],
                         'outputs': ['M2 판독 일치율과 판독 기준 요청']},
               'start_messages': [['team', 'han', '{call}, 같은 영상의 물체 24개를 세 명이 판독했는데 서로 많이 다른 것 같아요.'],
                                  ['team', 'park', '먼저 숫자로 봐요. 파일은 m2_reads.csv예요. 그런데 다들 값을 제각각으로 적었어요.']],
               'finish_messages': [['team', 'kim', '{call}, 일치율과 기준 요청까지 나왔네요. 다음 회의는 이걸로 시작합시다.']],
               'steps': [{'short': '데이터 보여 주기',
                          'title': 'AI에게 데이터를 설명하는 법',
                          'lead': 'AI는 파일을 볼 수 없어요. **앞부분 몇 줄과 값의 종류**를 알려 주는 것이 가장 빠른 방법이에요.',
                          'blocks': [{'id': 's1_info',
                                      'type': 'info',
                                      'title': '데이터를 설명할 때 넣을 것',
                                      'icon': '🗂',
                                      'text': ['열 이름만으로는 부족할 때가 많아요. 값이 어떻게 적혀 있는지까지 알려 주세요.'],
                                      'bullets': ['**파일 이름과 열 이름**: m2_reads.csv, object_id, reader_a, reader_b, reader_c',
                                                  '**값의 종류**: tank, apc, truck, decoy, unknown 다섯 가지',
                                                  '**주의할 점**: 대소문자와 뒤쪽 빈칸이 섞여 있음',
                                                  '**샘플 몇 줄**: 실제 값 대신 가상 샘플로 보여 줘도 돼요'],
                                      'figure': 'f_clean'},
                                     {'id': 's1_q',
                                      'type': 'quiz',
                                      'title': '먼저 알려 줄 것',
                                      'text': '값에 ‘Tank’, ‘tank ’이 섞여 있어요. 이 사실을 알려 주지 않으면 어떻게 될까요?',
                                      'options': ['AI가 스스로 알아채 항상 정리해 준다.', '같은 값을 다른 값으로 세어 일치율이 실제보다 낮게 나올 수 있다.', '파일을 읽지 못해 반드시 오류가 난다.'],
                                      'answer': 1,
                                      'explain': '오류 없이 틀린 숫자가 나와서 더 위험해요. 다음 활동에서 직접 확인해 봐요.',
                                      'hints': ['컴퓨터는 ‘Tank’와 ‘tank’를 같은 글자로 볼까요?', '오류가 나는 경우와 조용히 틀리는 경우를 나눠 생각해 보세요.']}]},
                         {'short': '일치율 계산',
                          'title': '일치율을 계산하고 갈린 물체 찾기',
                          'lead': '**값 정리** 문장을 빼고도 보내 보세요. 숫자가 어떻게 달라지는지 비교해 보면 좋아요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '판독 일치율 계산',
                                      'ask': ['park', '세 사람이 둘씩 얼마나 같게 판독했는지, 판독이 갈린 물체는 무엇인지 알고 싶어요.'],
                                      'file': 'm2_reads.csv',
                                      'slots': [{'id': 'input',
                                                 'label': '입력',
                                                 'why': '값의 모양까지 알려요.',
                                                 'options': [{'text': 'm2_reads.csv의 열은 object_id, reader_a, reader_b, reader_c이고, 값은 tank, apc, '
                                                                      'truck, decoy, unknown이야. 대소문자와 빈칸이 섞여 있으니 먼저 소문자로 바꾸고 빈칸을 지워 줘.',
                                                              'good': True},
                                                             {'text': 'm2_reads.csv에 세 사람의 판독이 있어.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '둘씩 일치율을 계산했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  'from itertools import combinations\n'
                                                                  "df = pd.read_csv('{VDIR}/m2_reads.csv')\n"
                                                                  "for a, b in combinations(['reader_a', 'reader_b', 'reader_c'], 2):\n"
                                                                  "    print(a, b, round((df[a] == df[b]).mean() * 100, 1), '%')",
                                                          'problem': '오류는 없지만 ‘Tank’와 ‘tank ’을 다른 값으로 세어 일치율이 실제보다 낮아요. 값을 정리한 결과와 비교해 보세요.',
                                                          'hint': '값이 어떻게 적혀 있는지, 무엇을 먼저 정리할지 적어 주세요.'}},
                                                {'id': 'task',
                                                 'label': '할 일',
                                                 'why': '무엇을 세는지 정해요.',
                                                 'options': [{'text': '세 사람을 둘씩 묶어 같은 값으로 판독한 비율(%)을 계산하고, 세 사람이 모두 같지 않은 물체 목록도 뽑아 줘.',
                                                              'good': True},
                                                             {'text': '일치도를 분석해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '전체 정확도를 계산했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m2_reads.csv')\n"
                                                                  "for c in ['reader_a', 'reader_b', 'reader_c']:\n"
                                                                  '    df[c] = df[c].str.strip().str.lower()\n'
                                                                  "print('accuracy:', round((df['reader_a'] == df['reader_b']).mean(), 2))",
                                                          'problem': '두 사람만 비교했고 ‘정확도’라고 표시됐어요. 어떤 비교인지 말하지 않아 AI가 추측했어요.',
                                                          'hint': '누구와 누구를 어떻게 비교할지, 어떤 목록이 필요한지 적어 주세요.'}},
                                                {'id': 'out',
                                                 'label': '출력 형식',
                                                 'why': '회의에서 볼 모양으로 받아요.',
                                                 'options': [{'text': '일치율 표와 갈린 물체 표를 보여 주고, 두 표를 넣은 HTML 문서를 report 변수에 담아 줘.', 'good': True},
                                                             {'text': '결과를 정리해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '결과를 문장으로 정리했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m2_reads.csv')\n"
                                                                  "for c in ['reader_a', 'reader_b', 'reader_c']:\n"
                                                                  '    df[c] = df[c].str.strip().str.lower()\n'
                                                                  "print('두 사람씩 비교한 결과는 대체로 양호합니다.')",
                                                          'problem': '‘대체로 양호’라는 말뿐이에요. 숫자가 없어 회의에서 근거로 쓸 수 없어요.',
                                                          'hint': '표로 받고 싶다고, 무엇을 넣을지 적어 주세요.'}}],
                                      'done': {'say': '값을 정리한 뒤 일치율 표와 갈린 물체 목록을 만들었어요.',
                                               'code': 'import pandas as pd\n'
                                                       'from itertools import combinations\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m2_reads.csv')\n"
                                                       "readers = ['reader_a', 'reader_b', 'reader_c']\n"
                                                       'for c in readers:\n'
                                                       '    df[c] = df[c].str.strip().str.lower()\n'
                                                       '\n'
                                                       'rows = []\n'
                                                       'for a, b in combinations(readers, 2):\n'
                                                       "    rows.append({'비교': f'{a} - {b}', '일치율(%)': round((df[a] == df[b]).mean() * 100, 1)})\n"
                                                       'agree = pd.DataFrame(rows)\n'
                                                       'display(agree)\n'
                                                       '\n'
                                                       "split = df[(df['reader_a'] != df['reader_b']) | (df['reader_b'] != df['reader_c'])]\n"
                                                       'display(split)\n'
                                                       '\n'
                                                       "report = ('<h2>판독 일치율</h2>' + agree.to_html(index=False)\n"
                                                       "          + f'<h3>판독이 갈린 물체 {len(split)}개</h3>' + split.to_html(index=False))",
                                               'lesson': '값의 모양을 알려 주니 조용히 틀리는 일이 사라졌어요. 데이터 설명은 프롬프트에서 가장 값어치 있는 몇 줄이에요.'},
                                      'on_ok': [['team', 'park', '{call}, 표로 보니 어느 물체에서 갈렸는지 보이네요. 이제 기준을 어떻게 고칠지 정해 봐요.']]}]},
                         {'short': '기준 합의',
                          'title': '결과로 기준을 합의하고 내 말로 요청하기',
                          'lead': 'AI는 계산을 도와줄 뿐이에요. 어떤 기준을 쓸지는 **사람이 근거를 보고 정해요.**',
                          'blocks': [{'id': 's3_decide',
                                      'type': 'decide',
                                      'title': '팀원의 제안 판단하기',
                                      'text': '일치율 결과를 본 팀원들의 제안이에요. 받아들일지 정하고 이유를 적어 주세요.',
                                      'reason_min': 8,
                                      'items': [{'id': 'a',
                                                 'who': 'han',
                                                 'text': '애매한 물체는 일단 전부 tank로 적어요. 놓치는 것보다 낫잖아요.',
                                                 'answer': 'reject',
                                                 'hint': '애매한 값을 모두 tank로 적으면 장비 수는 어떻게 될까요?',
                                                 'explain': '장비 수가 부풀어 기록을 믿을 수 없게 돼요. ‘unknown’을 두고 추가 판독으로 넘기는 편이 나아요.',
                                                 'reason': '애매한 건 unknown으로 두고 추가 판독해야 기록을 믿을 수 있어요.'},
                                                {'id': 'b',
                                                 'who': 'park',
                                                 'text': '값을 tank, apc, truck, decoy, unknown 다섯 가지 소문자로만 입력하게 규칙을 정해요.',
                                                 'answer': 'accept',
                                                 'hint': '앞에서 데이터를 정리하느라 무엇이 번거로웠는지 떠올려 보세요.',
                                                 'explain': '입력 규칙을 정하면 다음부터 정리할 일이 줄고, AI에게 설명하기도 쉬워져요.',
                                                 'reason': '입력 규칙이 있으면 값이 흔들리지 않아 계산과 설명이 쉬워져요.'}]},
                                     {'id': 's3_write',
                                      'type': 'write',
                                      'title': '기준 문서를 요청하는 프롬프트 쓰기',
                                      'prompt': '방금 합의한 내용으로 **판독 기준 문서**를 AI에게 요청해 보세요. 누가 읽는지, 어떤 항목이 들어가야 하는지, 근거 데이터를 적어 주세요.',
                                      'example': '새로 온 판독 담당이 읽을 판독 기준 문서를 만들어 줘. 항목은 값의 종류(tank, apc, truck, decoy, unknown), 값마다의 판독 기준, 입력 '
                                                 '규칙(소문자, 빈칸 없음), 애매할 때 처리 방법이야. 근거로 m2_reads.csv의 일치율 표와 갈린 물체 목록을 넣어 줘. 한 쪽 분량의 HTML로 만들어 줘.',
                                      'min': 30,
                                      'checklist': ['누가 읽는 문서인지 적었어요', '들어갈 항목을 적었어요', '근거 데이터를 적었어요', '분량이나 형식을 적었어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm2_reader_doc',
                            'title': 'M2 판독 일치율과 판독 기준 요청',
                            'sections': [{'h': '1. 일치율 계산', 'blocks': ['s2_vibe']},
                                         {'h': '2. 팀원 제안에 대한 결정', 'blocks': ['s3_decide']},
                                         {'h': '3. 판독 기준 문서를 요청하는 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm3',
               'no': 3,
               'title': '메타데이터 규격과 진위 검증',
               'short': '오류 고치기',
               'kind': '개인',
               'competencies': ['D3', 'D5'],
               'outline': '영상 메타데이터로 그림자 방향을 점검하는 코드를 만들고, 오류가 났을 때 어떻게 요청해야 빨리 고쳐지는지 익혀요.',
               'brief': {'subtitle': '영상 진위 점검표 만들기: 오류 메시지로 고쳐 달라고 하기',
                         'time': '약 25분',
                         'goal': '메타데이터로 그림자 방향을 점검하는 표를 만들고, 오류 메시지와 기대 동작을 담아 수정을 요청해 진위 점검 보고서까지 받아요.',
                         'why': '“안 돼, 다시 해 줘”는 왜 잘 안 먹힐까요? AI 코드는 빠르게 나오지만 늘 맞지는 않아요. 공급사마다 메타데이터 형식도 달라 오류가 자주 나요. 오류를 읽고 알맞게 다시 요청하는 일이 코드를 받는 '
                                '일만큼 중요해요.',
                         'case': {'title': '해외 사례: 폭격 흔적이라던 얼룩은 구름 그림자였어요',
                                  'source': 'WION 보도 (2025.5)',
                                  'points': ['2025년 5월 인도와 파키스탄의 무력 충돌 때, 인도 날리야 비행장의 활주로 주변이 검게 보이는 위성영상이 피해 증거처럼 퍼졌어요.',
                                             '인도 매체 보도에 따르면, 공개 분석가가 확인해 보니 검은 부분은 지나가는 구름의 그림자였어요.',
                                             '촬영 시각과 그림자가 맞는지 따져 보면 이런 오해를 줄일 수 있어요. 이번 미션에서는 그 점검표를 AI와 함께 만들어요.']},
                         'team': '최서연(검토 담당)이 메타데이터를 넘겨주고, 정우진(모델 담당)이 다른 공급사 파일도 점검해 달라고 부탁해요.',
                         'terms': [{'t': '메타데이터',
                                    'en': 'metadata',
                                    'fig': 'f_meta',
                                    'd': '영상이 언제, 어디서, 어떤 조건으로 찍혔는지 적은 자료예요.',
                                    'easy': '사진 뒷면에 적어 둔 촬영 메모예요.',
                                    'use': 'AI에게 열 이름과 단위, 시간대까지 알려 줘요.'},
                                   {'t': '그림자 검증',
                                    'en': 'shadow check',
                                    'fig': 'f_shadow',
                                    'd': '그림자가 그 시각의 태양 반대쪽을 향하는지 따져 영상을 점검하는 방법이에요.',
                                    'easy': '해가 남쪽이면 그림자는 북쪽으로 져야 해요.',
                                    'use': '예상 방향 = 태양 방위각 + 180도. 0도를 넘는 경우를 프롬프트에 적어요.'},
                                   {'t': '수정 프롬프트',
                                    'en': 'debugging prompt',
                                    'fig': 'f_debug',
                                    'd': '오류 메시지와 원하는 동작, 고칠 범위를 담은 후속 요청이에요.',
                                    'easy': '병원에서 증상을 정확히 말하는 것과 같아요.',
                                    'use': '오류는 요약하지 말고 그대로 붙여 넣어요.'}],
                         'steps': ['점검표 만들기: 그림자 점검표 만들기', '오류 고치기: 오류 메시지로 수정 요청하기', '내 말로 쓰기: 다른 오류를 내 말로 요청하기'],
                         'glossary': [],
                         'outputs': ['M3 영상 진위 점검표와 수정 기록']},
               'start_messages': [['dm_choi', 'choi', '{call}, SNS에 도는 영상들의 메타데이터예요. 그림자 방향이 맞는지 표로 점검해 주세요. 파일은 m3_scenes.csv예요.']],
               'finish_messages': [['dm_jung', 'jung', '{call}, 오류도 금방 고쳤네요! 공급사가 바뀌어도 이제 걱정이 덜해요.']],
               'steps': [{'short': '점검표 만들기',
                          'title': '그림자 점검표 만들기',
                          'lead': '계산식을 프롬프트에 직접 적어 보세요. **0도를 넘는 경우**처럼 헷갈리기 쉬운 점도 알려 주면 좋아요.',
                          'blocks': [{'id': 's1_vibe',
                                      'type': 'vibe',
                                      'title': '그림자 방향 점검표',
                                      'ask': ['choi', '영상마다 그림자 방향이 태양 위치와 맞는지 표로 보고 싶어요.'],
                                      'file': 'm3_scenes.csv',
                                      'slots': [{'id': 'input',
                                                 'label': '입력',
                                                 'why': '열의 뜻을 알려요.',
                                                 'options': [{'text': 'm3_scenes.csv의 열은 image_id, captured_utc(UTC 촬영 시각), sun_azimuth_deg(태양 방위각), '
                                                                      'shadow_azimuth_deg(영상에서 잰 그림자 방향)야.',
                                                              'good': True},
                                                             {'text': '영상 메타데이터 CSV가 있어.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '메타데이터로 그림자를 점검했어요.',
                                                          'expect': 'error',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m3_scenes.csv')\n"
                                                                  "print(df[df['sun_elevation'] > 30]['shadow_length'].mean())",
                                                          'problem': 'AI가 열 이름을 sun_elevation, shadow_length로 추측해서 KeyError가 났어요.',
                                                          'hint': '열 이름과 뜻을 그대로 적어 주세요.'}},
                                                {'id': 'task',
                                                 'label': '할 일',
                                                 'why': '규칙과 경계를 숫자로 적어요.',
                                                 'options': [{'text': '그림자는 태양 반대쪽이니까 예상 방향 = (태양 방위각 + 180) % 360으로 구하고, 차이는 0도를 넘는 경우까지 생각해 '
                                                                      'min(|예상 − 실제|, 360 − |예상 − 실제|)로 계산해 줘. 차이가 15도 이하면 일치, 넘으면 재검토로 표시해 줘.',
                                                              'good': True},
                                                             {'text': '그림자가 맞는지 확인해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '예상 방향과 실제 방향의 차이를 계산했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m3_scenes.csv')\n"
                                                                  "df['expected'] = (df['sun_azimuth_deg'] + 180) % 360\n"
                                                                  "df['gap'] = (df['expected'] - df['shadow_azimuth_deg']).abs()\n"
                                                                  "df['판정'] = df['gap'].apply(lambda g: '일치' if g <= 15 else '재검토')\n"
                                                                  "display(df[['image_id', 'expected', 'shadow_azimuth_deg', 'gap', '판정']])",
                                                          'problem': '오류는 없지만 S03이 재검토로 나왔어요. 358도와 3도는 실제로 5도 차이인데, 단순히 빼서 355도로 계산했어요. **조용히 틀린 '
                                                                     '대표적인 경우**예요.',
                                                          'hint': '규칙을 숫자로 적고, 0도를 넘는 경우를 어떻게 계산할지 알려 주세요.'}},
                                                {'id': 'out',
                                                 'label': '출력 형식',
                                                 'why': '표의 모양을 정해요.',
                                                 'options': [{'text': 'image_id, 한국 시각(UTC+9), 예상 방향, 실제 방향, 차이, 판정 순서의 표로 보여 줘.', 'good': True},
                                                             {'text': '보기 좋게 보여 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '보기 좋게 문장으로 정리했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m3_scenes.csv')\n"
                                                                  'for _, r in df.iterrows():\n'
                                                                  '    print(f"영상 {r[\'image_id\']}의 점검을 마쳤습니다!")',
                                                          'problem': '‘보기 좋게’는 사람마다 달라요. 결과 없이 점검했다는 말만 나왔어요.',
                                                          'hint': '열 순서까지 적어 주세요.'}}],
                                      'done': {'say': '그림자 방향 점검표를 만들었어요.',
                                               'code': 'import pandas as pd\n'
                                                       'from datetime import datetime, timedelta\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m3_scenes.csv')\n"
                                                       '\n'
                                                       'def judge(r):\n'
                                                       "    expected = (r['sun_azimuth_deg'] + 180) % 360\n"
                                                       "    gap = abs(expected - r['shadow_azimuth_deg'])\n"
                                                       '    gap = min(gap, 360 - gap)\n'
                                                       "    return expected, gap, '일치' if gap <= 15 else '재검토'\n"
                                                       '\n'
                                                       'rows = []\n'
                                                       'for _, r in df.iterrows():\n'
                                                       "    utc = datetime.strptime(r['captured_utc'], '%Y-%m-%dT%H:%M:%SZ')\n"
                                                       '    expected, gap, result = judge(r)\n'
                                                       "    rows.append({'영상': r['image_id'], '한국 시각': (utc + timedelta(hours=9)).strftime('%m-%d "
                                                       "%H:%M'), '예상 방향': expected,\n"
                                                       "                 '실제 방향': r['shadow_azimuth_deg'], '차이': gap, '판정': result})\n"
                                                       'table = pd.DataFrame(rows)\n'
                                                       'display(table)',
                                               'lesson': '계산식과 경계를 적으니 S03처럼 0도를 넘는 경우도 맞게 계산됐어요. 재검토로 나온 영상은 사람이 다시 봐요.'},
                                      'on_ok': [['dm_jung',
                                                 'jung',
                                                 '{call}, 표 좋아요. 그런데 다른 공급사 파일 m3_scenes_new.csv도 돌려 보고 싶어요. AI에게 파일만 바꿔 달라고 해 볼래요?']]}]},
                         {'short': '오류 고치기',
                          'title': '오류 메시지로 수정 요청하기',
                          'lead': '파일을 바꿨더니 **오류**가 났어요. 어떻게 다시 요청할까요?',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '오류 고쳐 달라고 하기',
                                      'ask': ['jung', '다른 공급사 파일을 넣었더니 오류가 나요. 저는 원인을 잘 모르겠어요… 부탁해요!'],
                                      'file': 'm3_scenes_new.csv',
                                      'slots': [{'id': 'what',
                                                 'label': '무슨 일이 있었는지',
                                                 'why': '오류 메시지를 그대로 붙여 넣어요.',
                                                 'options': [{'text': "m3_scenes_new.csv를 읽다가 이 오류가 났어: ValueError: time data '2025/05/20 06:30' "
                                                                      "does not match format '%Y-%m-%dT%H:%M:%SZ'",
                                                              'good': True},
                                                             {'text': '안 돼. 다시 해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '다시 만들었어요. 이번에는 날짜 형식을 바꿔 봤어요.',
                                                          'expect': 'error',
                                                          'code': 'import pandas as pd\n'
                                                                  'from datetime import datetime\n'
                                                                  "df = pd.read_csv('{VDIR}/m3_scenes_new.csv')\n"
                                                                  "for v in df['captured_utc']:\n"
                                                                  "    print(datetime.strptime(v, '%Y/%m/%d %H:%M'))",
                                                          'problem': '무엇이 안 되는지 몰라서 AI가 형식을 통째로 바꿨고, 이번에는 첫 행부터 같은 종류의 오류가 났어요.',
                                                          'hint': '오류 메시지를 그대로 알려 주세요.'}},
                                                {'id': 'fix',
                                                 'label': '어떻게 고칠지',
                                                 'why': '원하는 동작과 범위를 정해요.',
                                                 'options': [{'text': '공급사마다 날짜 형식이 달라. 두 형식을 모두 읽게 하고, 그 밖의 형식은 ‘자료 없음’으로 표시해 줘. 판정 계산은 그대로 두고, 표를 '
                                                                      '넣은 HTML 진위 점검 보고서를 report 변수에 담아 줘.',
                                                              'good': True},
                                                             {'text': '전부 새로 짜 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '처음부터 새로 짰어요. 더 간단하게 정리했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m3_scenes_new.csv')\n"
                                                                  "print(df[['image_id', 'captured_utc']].to_string(index=False))",
                                                          'problem': '오류는 사라졌지만 그림자 판정까지 함께 사라졌어요. 범위를 정하지 않으면 잘 되던 부분도 바뀌어요.',
                                                          'hint': '무엇은 그대로 두고 어디만 어떻게 바꿀지 적어 주세요.'}}],
                                      'done': {'say': '날짜를 읽는 부분만 고치고, 보고서를 추가했어요.',
                                               'code': 'import pandas as pd\n'
                                                       'from datetime import datetime, timedelta\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m3_scenes_new.csv')\n"
                                                       "FORMATS = ['%Y-%m-%dT%H:%M:%SZ', '%Y/%m/%d %H:%M']\n"
                                                       '\n'
                                                       'def read_time(v):\n'
                                                       '    for f in FORMATS:\n'
                                                       '        try:\n'
                                                       '            return datetime.strptime(v, f)\n'
                                                       '        except ValueError:\n'
                                                       '            pass\n'
                                                       '    return None\n'
                                                       '\n'
                                                       'def judge(r):\n'
                                                       "    expected = (r['sun_azimuth_deg'] + 180) % 360\n"
                                                       "    gap = abs(expected - r['shadow_azimuth_deg'])\n"
                                                       '    gap = min(gap, 360 - gap)\n'
                                                       "    return expected, gap, '일치' if gap <= 15 else '재검토'\n"
                                                       '\n'
                                                       'rows = []\n'
                                                       'for _, r in df.iterrows():\n'
                                                       "    utc = read_time(r['captured_utc'])\n"
                                                       '    expected, gap, result = judge(r)\n'
                                                       "    rows.append({'영상': r['image_id'], '한국 시각': (utc + timedelta(hours=9)).strftime('%m-%d "
                                                       "%H:%M') if utc else '자료 없음',\n"
                                                       "                 '예상 방향': expected, '실제 방향': r['shadow_azimuth_deg'], '차이': gap, '판정': "
                                                       'result})\n'
                                                       'table = pd.DataFrame(rows)\n'
                                                       'display(table)\n'
                                                       '\n'
                                                       "redo = table[table['판정'] == '재검토']['영상'].tolist()\n"
                                                       "report = ('<h2>영상 진위 점검 보고서</h2>'\n"
                                                       '          f"<p><b>결론:</b> {len(table)}건 가운데 재검토 {len(redo)}건({\', \'.join(redo)})입니다. 그림자만으로 '
                                                       '진위를 단정하지 않고 사람이 다시 봐요.</p>"\n'
                                                       '          + table.to_html(index=False))',
                                               'lesson': '오류 메시지와 원하는 동작, 고칠 범위를 함께 적으니 한 번에 고쳐졌어요. 잘 되던 판정 계산도 그대로 남았어요.'},
                                      'start': {'say': '파일을 m3_scenes_new.csv로 바꿨어요.',
                                                'expect': 'error',
                                                'code': 'import pandas as pd\n'
                                                        'from datetime import datetime, timedelta\n'
                                                        '\n'
                                                        "df = pd.read_csv('{VDIR}/m3_scenes_new.csv')\n"
                                                        '\n'
                                                        'def judge(r):\n'
                                                        "    expected = (r['sun_azimuth_deg'] + 180) % 360\n"
                                                        "    gap = abs(expected - r['shadow_azimuth_deg'])\n"
                                                        '    gap = min(gap, 360 - gap)\n'
                                                        "    return expected, gap, '일치' if gap <= 15 else '재검토'\n"
                                                        '\n'
                                                        'rows = []\n'
                                                        'for _, r in df.iterrows():\n'
                                                        "    utc = datetime.strptime(r['captured_utc'], '%Y-%m-%dT%H:%M:%SZ')\n"
                                                        '    expected, gap, result = judge(r)\n'
                                                        "    rows.append({'영상': r['image_id'], '한국 시각': (utc + timedelta(hours=9)).strftime('%m-%d "
                                                        "%H:%M'), '판정': result})\n"
                                                        'display(pd.DataFrame(rows))'}}]},
                         {'short': '내 말로 쓰기',
                          'title': '다른 오류를 내 말로 요청하기',
                          'lead': '오류를 만나면 **겁먹지 말고 그대로 붙여 넣으세요.** 코드를 다 읽지 못해도 수정을 요청할 수 있어요.',
                          'blocks': [{'id': 's3_write',
                                      'type': 'write',
                                      'title': '수정 프롬프트 쓰기',
                                      'prompt': "**새 상황**: 공급사 C의 파일을 넣었더니 `KeyError: 'sun_azimuth_deg'`가 났어요. 이 파일의 실제 열 이름은 sun_az예요. 잘 되던 판정 계산은 "
                                                '그대로 두고 싶어요. 어떻게 요청할까요?',
                                      'example': "공급사 C 파일을 넣었더니 KeyError: 'sun_azimuth_deg'가 났어. 이 파일의 실제 열 이름은 sun_az야. 파일을 읽은 뒤 sun_az를 "
                                                 'sun_azimuth_deg로 바꾸는 줄만 추가하고, 판정 계산과 나머지 코드는 그대로 둬.',
                                      'min': 30,
                                      'checklist': ['오류 메시지를 그대로 적었어요', '원인이 될 만한 사실(실제 열 이름)을 적었어요', '원하는 동작을 적었어요', '그대로 둘 부분을 적었어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm3_authenticity',
                            'title': 'M3 영상 진위 점검표와 수정 기록',
                            'sections': [{'h': '1. 그림자 방향 점검표', 'blocks': ['s1_vibe']},
                                         {'h': '2. 오류 수정과 보고서', 'blocks': ['s2_vibe']},
                                         {'h': '3. 내가 쓴 수정 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm4',
               'no': 4,
               'title': '전처리 모듈과 데이터 계약',
               'short': '쪼개서 요청하기',
               'kind': '협업',
               'competencies': ['D4', 'C1'],
               'outline': '큰 일을 한 번에 시키지 않고 작게 쪼개 요청하고, 기준을 바꿔 돌릴 수 있는 함수로 받아 데이터 계약 검사기를 만들어요.',
               'brief': {'subtitle': '데이터 계약 검사기 만들기: 쪼개서 요청하고 함수로 받기',
                         'time': '약 25분',
                         'goal': '타일 하나를 검사하는 함수부터 요청하고, 기준을 바꿔 돌려 계약별 비교표로 모델팀과 합의해요.',
                         'why': '한 번에 다 시키면 어디서 틀렸는지 찾기 어려워요. 공급사가 달라도 모델이 같은 규격을 받도록 맞추는 것이 전처리와 데이터 계약이에요. 이런 기준은 바꿔 가며 비교해야 하므로, 처음부터 바꿔 돌릴 수 있게 '
                                '요청하는 것이 좋아요.',
                         'case': {'title': '해외 사례: 위성영상 공급이 일주일 끊겼을 때',
                                  'source': 'Ukrainska Pravda, Euromaidan Press 보도 (2025.3)',
                                  'points': ['2025년 3월 초 미국 정부의 결정으로 우크라이나 이용자의 Maxar GEGD 상용 위성영상 접근이 일시 중단됐어요. 회사 측은 이 계약에만 해당하고 다른 경로는 가능하다고 '
                                             '설명했어요.',
                                             '접근은 약 일주일 뒤인 3월 12일 무렵 복구됐다고 이용자들이 전했어요. 공식 확인보다 이용자 전언이 중심인 보도예요.',
                                             '한 공급사에 묶인 도구는 공급이 바뀌면 함께 멈춰요. 그래서 규격을 코드로 검사하는 계약 검사기가 쓸모 있어요.']},
                         'team': '정우진(모델 담당)은 타일이 많이 통과되길 바라고, 한지우(판독 담당)는 품질을 지키고 싶어 해요. 전우님이 AI로 비교표를 만들어 합의를 도와요.',
                         'terms': [{'t': '전처리',
                                    'en': 'preprocessing',
                                    'fig': 'f_pre',
                                    'd': '공급사가 보낸 영상을 모델이 쓸 수 있는 표준 형태로 바꾸는 과정이에요.',
                                    'easy': '재료를 손질해 같은 크기로 맞추는 일이에요.',
                                    'use': '정사보정, GSD 표준화, 구름 마스킹 같은 단계가 있어요.'},
                                   {'t': '데이터 계약',
                                    'en': 'data contract',
                                    'fig': 'f_contract',
                                    'd': '주고받을 데이터의 형식, 품질 기준, 위반 처리를 미리 합의한 약속이에요.',
                                    'easy': '납품 규격서예요.',
                                    'use': '기준을 숫자로 적으면 AI에게 그대로 검사를 시킬 수 있어요.'},
                                   {'t': '함수로 요청하기',
                                    'en': 'parameterize',
                                    'fig': 'f_param',
                                    'd': '값을 바꿔 다시 돌릴 수 있게 코드를 함수로 만들어 달라고 요청하는 것이에요.',
                                    'easy': '계산기처럼 숫자만 바꿔 다시 누를 수 있게 하는 거예요.',
                                    'use': '‘구름 20%로 검사해 줘’ 대신 ‘max_cloud를 인자로 받게 해 줘’라고 해요.'}],
                         'steps': ['쪼개기: 큰 일을 작게 나누기', '함수로 받기: 함수로 받아 계약 비교하기', '합의하고 쓰기: 결과로 합의하고 내 말로 요청하기'],
                         'glossary': [],
                         'outputs': ['M4 데이터 계약 검사와 합의']},
               'start_messages': [['team', 'jung', '{call}, 모델에 넣을 타일이 부족해요. 기준을 좀 풀면 안 될까요?'],
                                  ['team', 'han', '저는 생각이 달라요. 품질이 낮은 타일은 판독 결과를 망쳐요. 파일 m4_tiles.csv로 비교부터 해 봐요.']],
               'finish_messages': [['team', 'kim', '{call}, 비교표 덕분에 금방 합의했네요. 기준을 바꿀 때마다 다시 돌려 볼 수 있어 좋아요.']],
               'steps': [{'short': '쪼개기',
                          'title': '큰 일을 작게 나누기',
                          'lead': '한 번에 다 시키는 대신, **확인할 수 있는 크기**로 나눠요.',
                          'blocks': [{'id': 's1_info',
                                      'type': 'info',
                                      'title': '왜 쪼개야 할까요?',
                                      'icon': '✂️',
                                      'text': ['한 번에 전부 만들어 달라고 하면 코드가 길어지고, 틀려도 어디인지 찾기 어려워요. 한 단계씩 시키고 확인하면 AI도 전우님도 덜 헤매요.'],
                                      'bullets': ['**1단계**: 타일 하나만 검사하는 함수', '**2단계**: 작은 예시로 확인', '**3단계**: 전체 타일에 적용', '**4단계**: 계약별 비교표와 문서'],
                                      'figure': 'f_param'},
                                     {'id': 's1_order',
                                      'type': 'order',
                                      'title': '요청 순서 정하기',
                                      'text': '데이터 계약 검사기를 만들려고 해요. 어떤 순서로 요청하면 좋을까요?',
                                      'items': [{'id': 'a', 'text': '데이터 모양을 알려 주고 파일을 읽게 하기', 'hint': 'AI가 파일을 먼저 알아야 해요.'},
                                                {'id': 'b', 'text': '타일 하나를 검사하는 함수 만들기', 'hint': '작은 단위부터 시작해요.'},
                                                {'id': 'c', 'text': '예시 한 행으로 함수 확인하기', 'hint': '전체에 적용하기 전에 확인해요.'},
                                                {'id': 'd', 'text': '전체 타일에 적용해 계약별 비교표 만들기', 'hint': '확인된 함수를 넓혀 쓰는 단계예요.'},
                                                {'id': 'e', 'text': '비교표를 넣은 계약 검사 문서 만들기', 'hint': '결과가 나온 뒤에 문서를 만들어요.'}],
                                      'hints': ['앞 단계의 결과가 다음 단계의 재료가 되는지 따져 보세요.'],
                                      'explain': '작은 단위를 확인하고 넓혀 가면, 틀려도 어느 단계에서 틀렸는지 바로 알 수 있어요.'}]},
                         {'short': '함수로 받기',
                          'title': '함수로 받아 계약 비교하기',
                          'lead': '기준을 **하나로 정해 주지 말고**, 바꿔 돌릴 수 있게 요청해 보세요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '데이터 계약 검사기',
                                      'ask': ['han', '엄격한 계약과 완화한 계약을 비교하고 싶어요. 앞으로 다른 기준도 볼 수 있으면 좋겠어요.'],
                                      'file': 'm4_tiles.csv',
                                      'slots': [{'id': 'split',
                                                 'label': '나눠서 요청',
                                                 'why': '작은 단위부터 시켜요.',
                                                 'options': [{'text': '먼저 타일 한 행을 받아 어긴 기준을 목록으로 돌려주는 함수 check_tile을 만들고, 첫 번째 행으로 결과를 출력해 확인해 줘.',
                                                              'good': True},
                                                             {'text': '데이터 계약 검사를 한 번에 전부 만들어 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '전체를 한 번에 만들었어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m4_tiles.csv')\n"
                                                                  "print('통과한 타일:', int((df['cloud_pct'] <= 20).sum()), '개')",
                                                          'problem': '숫자 하나만 나왔고, 구름만 보고 GSD와 오프나디르 각, 좌표계는 검사하지 않았어요. 한 번에 시키면 이런 누락을 찾기 어려워요.',
                                                          'hint': '타일 하나를 검사하는 작은 함수부터 만들어 달라고 해 보세요.'}},
                                                {'id': 'param',
                                                 'label': '바꿔 돌리기',
                                                 'why': '값을 인자로 받게 해요.',
                                                 'options': [{'text': '구름 기준 max_cloud와 오프나디르 기준 max_off는 인자로 받게 해서, 엄격(20, 30)과 완화(30, 35) 두 계약을 바꿔 '
                                                                      '돌릴 수 있게 해 줘.',
                                                              'good': True},
                                                             {'text': '구름 20%, 오프나디르 30도로 검사해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '말씀하신 기준으로 검사했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m4_tiles.csv')\n"
                                                                  "ok = df[(df['cloud_pct'] <= 20) & (df['off_nadir_deg'] <= 30) & ((df['gsd_m'] - "
                                                                  "0.5).abs() <= 0.05) & (df['crs'] == 'EPSG:32652')]\n"
                                                                  "display(ok[['tile_id', 'cloud_pct', 'off_nadir_deg']])",
                                                          'problem': '엄격한 계약 결과만 나왔어요. 다른 기준을 보려면 또 요청해야 하고, 숫자가 코드 안에 박혀 있어요.',
                                                          'hint': '기준의 숫자를 코드에 박지 말고 인자로 받게 해 달라고 요청하세요.'}},
                                                {'id': 'out',
                                                 'label': '출력 형식',
                                                 'why': '비교할 내용을 정해요.',
                                                 'options': [{'text': '계약마다 통과·재처리 타일 수와 타일별 어긴 기준을 표로 보여 주고, 막대그림과 HTML 계약 검사 보고서를 report 변수에 담아 줘. '
                                                                      'GSD 기준은 0.5m, 허용 오차는 0.05m, 좌표계 기준은 EPSG:32652야.',
                                                              'good': True},
                                                             {'text': '계약별로 결과를 보여 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '계약별 통과 수를 정리했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m4_tiles.csv')\n"
                                                                  "for name, c, o in [('strict', 20, 30), ('relaxed', 30, 35)]:\n"
                                                                  "    print(name, int(((df['cloud_pct'] <= c) & (df['off_nadir_deg'] <= o)).sum()))",
                                                          'problem': '어느 타일이 무엇을 어겼는지 몰라 재처리를 맡길 수 없어요. GSD와 좌표계 기준도 빠졌어요.',
                                                          'hint': '무엇을 비교하고 싶은지, 기준값을 적어 주세요.'}}],
                                      'done': {'say': '함수를 먼저 확인하고, 계약별 비교표와 보고서를 만들었어요.',
                                               'code': 'import pandas as pd\n'
                                                       'import matplotlib.pyplot as plt\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m4_tiles.csv')\n"
                                                       '\n'
                                                       "def check_tile(r, max_cloud, max_off, gsd=0.5, gsd_tol=0.05, crs='EPSG:32652'):\n"
                                                       '    # 어긴 기준을 목록으로 돌려줌 (비어 있으면 통과)\n'
                                                       '    bad = []\n'
                                                       "    if r['cloud_pct'] > max_cloud:\n"
                                                       "        bad.append('구름')\n"
                                                       "    if r['off_nadir_deg'] > max_off:\n"
                                                       "        bad.append('오프나디르')\n"
                                                       "    if abs(r['gsd_m'] - gsd) > gsd_tol:\n"
                                                       "        bad.append('GSD')\n"
                                                       "    if r['crs'] != crs:\n"
                                                       "        bad.append('좌표계')\n"
                                                       '    return bad\n'
                                                       '\n'
                                                       "print('확인(첫 행, 엄격):', check_tile(df.iloc[0], 20, 30))\n"
                                                       '\n'
                                                       "CONTRACTS = {'엄격': (20, 30), '완화': (30, 35)}\n"
                                                       "summary, detail = [], df[['tile_id']].copy()\n"
                                                       'for name, (c, o) in CONTRACTS.items():\n'
                                                       '    res = [check_tile(r, c, o) for _, r in df.iterrows()]\n'
                                                       "    detail[name] = [', '.join(x) if x else '통과' for x in res]\n"
                                                       '    ok = sum(1 for x in res if not x)\n'
                                                       "    summary.append({'계약': name, '통과': ok, '재처리': len(res) - ok})\n"
                                                       'summary = pd.DataFrame(summary)\n'
                                                       'display(summary)\n'
                                                       'display(detail)\n'
                                                       '\n'
                                                       'plt.figure(figsize=(4, 2.5))\n'
                                                       "plt.bar(['strict', 'relaxed'], summary['통과'])\n"
                                                       "plt.ylabel('passed tiles')\n"
                                                       '\n'
                                                       "report = ('<h2>데이터 계약 검사 보고서</h2><p>가상 타일 ' + str(len(df)) + '개 기준</p>' + "
                                                       'summary.to_html(index=False) + detail.to_html(index=False))',
                                               'lesson': '함수로 받으니 숫자만 바꿔 계약을 계속 비교할 수 있어요. 첫 행으로 먼저 확인해서 믿고 넓혀 쓸 수 있었어요.'},
                                      'on_ok': [['team', 'jung', '{call}, 완화 계약이면 한 장 더 통과하네요. 생각보다 차이가 작아요.'],
                                                ['team', 'han', '표를 보고 정해 봐요. 무엇을 어겼는지도 보이니까요.']]}]},
                         {'short': '합의하고 쓰기',
                          'title': '결과로 합의하고 내 말로 요청하기',
                          'lead': '표를 근거로 팀원의 제안을 판단하세요. 정답보다 **근거를 대는 것**이 중요해요.',
                          'blocks': [{'id': 's3_decide',
                                      'type': 'decide',
                                      'title': '계약 제안 판단하기',
                                      'text': '비교표를 근거로 제안을 판단해 주세요.',
                                      'reason_min': 8,
                                      'items': [{'id': 'a',
                                                 'who': 'jung',
                                                 'text': '구름 낀 타일은 생성형 AI로 구름 아래를 그려 넣어 통과시켜요. 타일 수가 늘잖아요.',
                                                 'answer': 'reject',
                                                 'hint': '관측하지 않은 픽셀이 생기면 판독 결과는 믿을 수 있을까요?',
                                                 'explain': '관측하지 않은 픽셀을 만들어 넣으면 판독과 평가를 믿을 수 없어요. 구름은 마스킹하고 기록으로 남겨요.',
                                                 'reason': '관측하지 않은 픽셀을 만들면 판독을 믿을 수 없어 마스킹하고 기록해요.'},
                                                {'id': 'b',
                                                 'who': 'han',
                                                 'text': '기본은 엄격 계약으로 하고, 완화 계약으로 통과한 타일은 결과에 표시해 구분해요.',
                                                 'answer': 'accept',
                                                 'hint': '함수로 받아 두었으니 무엇이 쉬워졌나요?',
                                                 'explain': '표시만 남기면 두 사람의 요구를 함께 살릴 수 있어요. 기준을 바꿔 돌리는 비용도 작아요.',
                                                 'reason': '표시를 남기면 품질과 타일 수를 함께 챙기고 기준도 쉽게 바꿔 시험해요.'}]},
                                     {'id': 's3_write',
                                      'type': 'write',
                                      'title': '쪼개서 요청하는 프롬프트 쓰기',
                                      'prompt': '**새 요청**: SAR 타일에만 보정 여부(calibrated 열)를 추가로 검사하고 싶어요. 한 번에 시키지 말고, 첫 번째 요청만 써 보세요.',
                                      'example': 'm4_tiles.csv에 calibrated 열(1 또는 0)을 추가했어. 먼저 check_tile 함수에 sensor가 sar이고 calibrated가 0이면 ‘보정 없음’을 '
                                                 '추가하는 부분만 고쳐 줘. sar 행 하나와 optical 행 하나로 결과를 출력해 확인해 줘. 나머지 코드는 그대로 두고, 전체 적용은 내가 확인한 다음에 요청할게.',
                                      'min': 30,
                                      'checklist': ['한 번에 하나만 시켰어요', '확인할 방법을 함께 적었어요', '그대로 둘 부분을 적었어요', '다음 단계는 뒤로 미뤘어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm4_contract_doc',
                            'title': 'M4 데이터 계약 검사와 합의',
                            'sections': [{'h': '1. 계약별 비교표', 'blocks': ['s2_vibe']},
                                         {'h': '2. 합의 결정', 'blocks': ['s3_decide']},
                                         {'h': '3. 내가 쓴 첫 요청', 'blocks': ['s3_write']}]}]},
              {'id': 'm5',
               'no': 5,
               'title': 'Baseline 탐지·대분류와 조건별 정직한 평가',
               'short': '결과 검증하기',
               'kind': '개인',
               'competencies': ['D5', 'J1'],
               'outline': '가상 모델의 탐지 점수로 임계값을 고르고 조건별 성능표와 모델 카드를 만들면서, AI의 자신 있는 답을 근거로 확인하는 법을 익혀요.',
               'brief': {'subtitle': '조건별 성능표와 모델 카드 만들기: AI의 답을 검증하기',
                         'time': '약 30분',
                         'goal': 'AI에게 근거 수치까지 함께 요청해 임계값을 고르고, 조건별 성능표와 모델 카드를 받아 믿어도 되는 결과인지 판단해요.',
                         'why': 'AI가 “최적값은 0.42입니다”라고 하면 믿어도 될까요? AI 답변은 맞든 틀리든 자신 있게 들려요. 모델 성능도 어떤 조건에서 나온 숫자인지 모르면 판단할 수 없어요. 근거를 함께 요청하면 사람이 판단할 '
                                '수 있어요.',
                         'case': {'title': '해외 사례: 점수만이 아니라 시험 방법까지 보는 모델 인증',
                                  'source': '미국 국가지리정보국(NGA) 발표, DefenseScoop 보도 (2024.9)',
                                  'points': ['미국 국가지리정보국(NGA)은 2024년 9월 지리공간정보 AI 모델 인증 시범 사업(AGAIM)을 발표했어요.',
                                             '이 제도는 모델의 개발과 시험 절차가 얼마나 체계적이고 튼튼한지를 평가해요. 국장은 NGA 시스템에 쓰는 모델은 인증을 받아야 한다고 설명했어요.',
                                             '성능 숫자 하나보다 그 숫자를 어떻게 얻었는지가 중요하다는 뜻이에요. AI에게 근거를 함께 요청하는 이유도 같아요.']},
                         'team': '정우진(모델 담당)이 가상 모델의 점수를 넘겨주고, 이정민(요청 담당)이 어디까지 믿어도 되는지 궁금해해요.',
                         'terms': [{'t': '가상 모델',
                                    'en': 'virtual model',
                                    'fig': 'f_verify',
                                    'd': '실습을 위해 점수를 미리 만들어 둔 모델이에요. 학습이나 내려받기 없이 입력과 출력만 다뤄요.',
                                    'easy': '모델 속을 만들지 않고, 나온 점수를 어떻게 쓸지에 집중해요.',
                                    'use': '실제 업무에서도 모델 점수를 어떻게 검증할지가 먼저예요.'},
                                   {'t': '데이터 누수',
                                    'en': 'data leakage',
                                    'fig': 'f_leak',
                                    'd': '평가에 쓸 데이터가 고르는 단계에 섞여 성능이 실제보다 좋아 보이는 현상이에요.',
                                    'easy': '시험 문제를 미리 보고 시험을 치는 것과 같아요.',
                                    'use': '‘검증 데이터로 고르고 테스트 데이터로 한 번만 확인해 줘’라고 요청해요. 같은 공항이 양쪽에 섞이지 않게 해요.'},
                                   {'t': '환각',
                                    'en': 'hallucination',
                                    'fig': 'f_halluc',
                                    'd': 'AI가 근거 없는 내용을 사실처럼 말하는 현상이에요.',
                                    'easy': '모르는데 아는 척 자신 있게 말하는 거예요.',
                                    'use': '숫자와 함께 근거를 요청하고 자료와 대조해요.'},
                                   {'t': '모델 카드',
                                    'en': 'model card',
                                    'fig': 'f_card',
                                    'd': '모델의 사용 범위, 성능, 한계를 정리한 문서예요.',
                                    'easy': '약의 설명서처럼 언제 쓰고 언제 쓰면 안 되는지 적어요.',
                                    'use': '조건별 성능과 쓰면 안 되는 조건을 함께 요청해요.'}],
                         'steps': ['근거 요청하기: 답만 받지 말고 근거도 받기', '성능표와 모델 카드: 검증까지 요청해 성능표와 모델 카드 받기', '내 말로 쓰기: 검증을 요청하는 프롬프트 쓰기'],
                         'glossary': [],
                         'outputs': ['M5 조건별 성능표와 모델 카드']},
               'start_messages': [['dm_jung',
                                   'jung',
                                   '{call}, 가상 모델 결과예요. 파일은 m5_detections.csv예요. 공항 단위로 val과 test를 나눠 두었어요. 재현율 0.85 이상을 지키는 임계값을 찾아 주세요.']],
               'finish_messages': [['dm_lee', 'lee', '{call}, 모델 카드 받았어요. 어떤 조건에서 조심해야 하는지 알겠네요.']],
               'steps': [{'short': '근거 요청하기',
                          'title': '답만 받지 말고 근거도 받기',
                          'lead': 'AI에게 **답과 근거를 함께** 달라고 하면 틀린 답을 알아채기 쉬워져요.',
                          'blocks': [{'id': 's1_info',
                                      'type': 'info',
                                      'title': '믿기 전에 받을 것',
                                      'icon': '🔍',
                                      'text': ['AI가 자신 있게 말해도 사실과 다를 수 있어요. 아래 세 가지를 프롬프트에 넣으면 사람이 확인할 수 있어요.'],
                                      'bullets': ['**계산에 쓴 수치**: TP, FN, FP처럼 손으로 다시 셀 수 있는 숫자',
                                                  '**어떤 데이터로 계산했는지**: 고를 때 쓴 데이터와 확인에 쓴 데이터',
                                                  '**한계**: 표본 수, 믿기 어려운 조건'],
                                      'figure': 'f_leak'},
                                     {'id': 's1_q',
                                      'type': 'quiz',
                                      'title': '왜 공항 단위로 나눌까요?',
                                      'text': '같은 공항의 영상을 학습·검증·테스트에 섞지 않고 공항 단위로 나눴어요. 이유는 무엇일까요?',
                                      'options': ['파일을 읽기 쉬워져서', '같은 공항의 비슷한 영상이 섞이면 성능이 실제보다 좋아 보일 수 있어서', '공항마다 파일 이름이 달라서'],
                                      'answer': 1,
                                      'explain': '비슷한 장면이 양쪽에 있으면 처음 보는 공항에서의 성능을 알 수 없어요. 이것이 데이터 누수예요.',
                                      'hints': ['처음 보는 공항에서 잘 동작하는지 알고 싶다면 무엇을 피해야 할까요?', '시험 문제를 미리 본 것과 비슷한 상황을 떠올려 보세요.']}]},
                         {'short': '성능표와 모델 카드',
                          'title': '검증까지 요청해 성능표와 모델 카드 받기',
                          'lead': '**고를 때 쓴 데이터**와 **확인에 쓴 데이터**를 나눠 요청해 보세요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '조건별 성능표와 모델 카드',
                                      'ask': ['jung', '임계값을 고르고, 조건별 성능과 모델 카드까지 받고 싶어요.'],
                                      'file': 'm5_detections.csv',
                                      'slots': [{'id': 'input',
                                                 'label': '입력',
                                                 'why': '데이터를 나눈 방법을 알려요.',
                                                 'options': [{'text': 'm5_detections.csv의 열은 det_id, airport_id, split(val 또는 test, 공항 단위로 나뉨), '
                                                                      'cloud_band, off_nadir_band, is_true(맞는 탐지면 1), score(가상 모델 점수)야.',
                                                              'good': True},
                                                             {'text': '모델 점수 파일이 있어.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '점수 파일로 계산했어요.',
                                                          'expect': 'error',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m5_detections.csv')\n"
                                                                  "print(df.groupby('label')['confidence'].mean())",
                                                          'problem': '열 이름을 추측해 KeyError가 났어요.',
                                                          'hint': '열 이름과 값의 뜻을 적어 주세요.'}},
                                                {'id': 'task',
                                                 'label': '할 일',
                                                 'why': '어떤 데이터로 고를지 정해요.',
                                                 'options': [{'text': 'split이 val인 데이터에서 재현율 0.85 이상인 임계값 가운데 가장 높은 값을 고르고, 그 임계값을 test 데이터에서 한 번만 '
                                                                      '확인해 줘.',
                                                              'good': True},
                                                             {'text': '데이터 전체에서 가장 좋은 임계값을 찾아 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '전체 데이터에서 가장 좋은 임계값을 찾았어요. 성능이 아주 좋아요!',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m5_detections.csv')\n"
                                                                  'best = None\n'
                                                                  'for t in [x / 100 for x in range(5, 95)]:\n'
                                                                  "    a = df['score'] >= t\n"
                                                                  "    tp = int((a & (df['is_true'] == 1)).sum())\n"
                                                                  "    fn = int((~a & (df['is_true'] == 1)).sum())\n"
                                                                  '    if tp / (tp + fn) >= 0.85:\n'
                                                                  '        best = t\n'
                                                                  "print('임계값', best, '(전체 데이터 사용)')",
                                                          'problem': '확인에 쓸 데이터까지 고르는 데 써 버렸어요. 시험 문제를 미리 본 것과 같아서 새 데이터에서는 이 성능이 나오지 않을 수 있어요.',
                                                          'hint': '고를 때 쓸 데이터와 확인에 쓸 데이터를 나눠 적어 주세요.'}},
                                                {'id': 'verify',
                                                 'label': '검증',
                                                 'why': '다시 셀 수 있는 숫자를 받아요.',
                                                 'options': [{'text': 'val, test의 TP, FN, FP, 재현율, 정밀도를 표로 보여 주고, test는 cloud_band와 off_nadir_band '
                                                                      '조건별로도 나눠 표본 수와 함께 보여 줘.',
                                                              'good': True},
                                                             {'text': '가장 좋은 임계값만 알려 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '분석을 마쳤어요. 가장 좋은 값은 0.42예요!',
                                                          'open_code': True,
                                                          'code': "print('최적 임계값: 0.42')\nprint('성능이 매우 우수합니다!')",
                                                          'problem': '코드를 보면 계산이 하나도 없어요. 숫자를 그냥 출력했을 뿐이에요. 근거를 요구하지 않으면 이런 답도 그럴듯하게 들려요. **이런 것이 '
                                                                     '환각이에요.**',
                                                          'hint': '손으로 다시 셀 수 있는 숫자를 함께 달라고 하세요.'}},
                                                {'id': 'out',
                                                 'label': '문서',
                                                 'why': '누가 읽을지 정해요.',
                                                 'options': [{'text': '요청 담당이 읽을 모델 카드를 HTML로 report 변수에 담아 줘. 순서는 결론 한 줄, 근거 표, 조건별 성능, 조심할 조건, 사람이 '
                                                                      '판독할 조건이야.',
                                                              'good': True},
                                                             {'text': '모델 카드도 만들어 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '모델 카드를 만들었어요.',
                                                          'code': "report = '<h2>모델 카드</h2><p>본 모델은 우수한 성능을 보였습니다. 모든 조건에서 사용을 권장합니다.</p>'\n"
                                                                  "print('문서를 만들었어요.')",
                                                          'problem': '누가 읽는지, 무엇이 들어가야 하는지 몰라서 근거 없는 문서가 나왔어요. ‘모든 조건’이라는 말도 확인되지 않았어요.',
                                                          'hint': '독자와 문서의 순서를 적어 주세요.'}}],
                                      'done': {'say': '검증 데이터로 고르고 테스트 데이터로 확인했어요. 조건별 표와 모델 카드도 만들었어요.',
                                               'code': 'import pandas as pd\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m5_detections.csv')\n"
                                                       '\n'
                                                       'def counts(d, t):\n'
                                                       "    a = d['score'] >= t\n"
                                                       "    tp = int((a & (d['is_true'] == 1)).sum())\n"
                                                       "    fn = int((~a & (d['is_true'] == 1)).sum())\n"
                                                       "    fp = int((a & (d['is_true'] == 0)).sum())\n"
                                                       "    return {'TP': tp, 'FN': fn, 'FP': fp, '재현율': round(tp / (tp + fn), 3) if tp + fn else "
                                                       'None,\n'
                                                       "            '정밀도': round(tp / (tp + fp), 3) if tp + fp else None, '표본 수': len(d)}\n"
                                                       '\n'
                                                       "val, test = df[df['split'] == 'val'], df[df['split'] == 'test']\n"
                                                       "cands = [t / 100 for t in range(5, 95) if counts(val, t / 100)['재현율'] >= 0.85]\n"
                                                       'best = max(cands)  # 재현율 0.85를 지키는 가장 높은 임계값\n'
                                                       '\n'
                                                       "table = pd.DataFrame([{'데이터': 'val (고르는 데 씀)', **counts(val, best)}, {'데이터': 'test (한 번만 "
                                                       "확인)', **counts(test, best)}])\n"
                                                       'display(table)\n'
                                                       "print('고른 임계값:', best)\n"
                                                       '\n'
                                                       'rows = []\n'
                                                       "for col in ['cloud_band', 'off_nadir_band']:\n"
                                                       '    for v, d in test.groupby(col):\n'
                                                       "        rows.append({'조건': f'{col} = {v}', **counts(d, best)})\n"
                                                       'cond = pd.DataFrame(rows)\n'
                                                       'display(cond)\n'
                                                       '\n'
                                                       "worst = cond.sort_values('재현율').iloc[0]['조건']\n"
                                                       "report = (f'<h2>모델 카드</h2><p><b>결론:</b> 제안하는 임계값은 {best}입니다. 가상 모델과 가상 데이터 {len(df)}건 "
                                                       "기준이에요.</p>'\n"
                                                       "          '<h3>근거</h3>' + table.to_html(index=False)\n"
                                                       "          + '<h3>조건별 성능 (test)</h3>' + cond.to_html(index=False)\n"
                                                       "          + f'<h3>조심할 조건</h3><p>{worst}에서 재현율이 가장 낮아요.</p>'\n"
                                                       "          '<h3>사람이 판독할 조건</h3><p>조심할 조건에 해당하거나 점수가 낮은 탐지는 사람이 판독해요.</p>')",
                                               'lesson': '다시 셀 수 있는 숫자를 함께 받으니 믿어도 되는지 전우님이 판단할 수 있어요. 조건별로 나누니 평균에 가려졌던 약한 조건도 보여요.'},
                                      'on_ok': [['dm_jung', 'jung', '{call}, TP·FN이 같이 있으니 제가 바로 검산할 수 있네요. 좋습니다!']]}]},
                         {'short': '내 말로 쓰기',
                          'title': '검증을 요청하는 프롬프트 쓰기',
                          'lead': '**새 데이터가 와도** 같은 방식으로 요청할 수 있게 내 말로 써 봐요.',
                          'blocks': [{'id': 's3_write',
                                      'type': 'write',
                                      'title': '검증 프롬프트 쓰기',
                                      'prompt': '**새 요청**: 이번에는 요구 조건이 ‘정밀도 0.8 이상’이에요. 파일 m5_new.csv의 열은 같아요. 답을 믿을 수 있도록 근거까지 요청해 보세요.',
                                      'example': 'm5_new.csv의 열은 det_id, airport_id, split(val 또는 test), cloud_band, off_nadir_band, is_true, '
                                                 'score야. val 데이터에서 정밀도 0.8 이상을 지키면서 재현율이 가장 높은 임계값을 골라 줘. 그 값을 test에서 한 번만 확인하고, 양쪽의 TP, FN, FP와 '
                                                 '재현율, 정밀도를 표로 보여 줘. 조건별 성능과 표본 수, 한계도 적어 줘.',
                                      'min': 30,
                                      'checklist': ['고를 데이터와 확인할 데이터를 나눴어요', '다시 셀 수 있는 숫자를 요청했어요', '요구 조건을 숫자로 적었어요', '조건별 성능과 한계까지 적어 달라고 했어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm5_model_card',
                            'title': 'M5 조건별 성능표와 모델 카드',
                            'sections': [{'h': '1. AI에게 보낸 프롬프트와 결과', 'blocks': ['s2_vibe']}, {'h': '2. 내가 쓴 검증 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm6',
               'no': 6,
               'title': '레드팀·블루팀 기만 대응 훈련',
               'short': '규칙을 숫자로',
               'kind': '협업',
               'competencies': ['D1', 'D5', 'C1'],
               'outline': '‘진짜 장비만 골라 줘’ 같은 말을 숫자 규칙으로 바꾸고, 단서가 빈칸인 경우까지 알려 주어 기만체 판별 표를 만들어요.',
               'brief': {'subtitle': '기만체 판별 규칙 만들기: 모호한 말을 숫자와 예외로',
                         'time': '약 25분',
                         'goal': '모호한 요구를 숫자 규칙과 예외로 바꿔 요청하고, 모델이 센 수와 규칙으로 가린 수를 비교하는 표를 만들어요.',
                         'why': '‘진짜만 골라 줘’라는 말을 AI는 어떻게 알아들을까요? 장비 개수만 세는 모델은 기만체까지 세어 규모를 크게 잘못 볼 수 있어요. 그래서 판별 규칙이 필요한데, 규칙이 모호하면 AI가 멋대로 해석해요.',
                         'case': {'title': '해외 사례: 실제 장비 5대에 기만체는 최대 30대',
                                  'source': 'The Times를 인용한 The Defense Post, Militarnyi 등 보도 (2025.5)',
                                  'points': ['2025년 5월 영국 일간지 The Times를 인용한 보도에 따르면, 영국은 우크라이나에 실제 차량 5대를 보낼 때 기만체를 최대 30대까지 함께 보낼 수 있어요.',
                                             '드론과 위성으로 풍선형 기만체를 가려낼 수 있게 되자, 실제 장비를 본떠 인쇄한 조립식 기만체를 쓴다고 전해졌어요. 우크라이나 국방차관은 러시아도 기만체를 쓴다고 말했어요.',
                                             '개수만 세면 규모를 크게 잘못 볼 수 있어요. 이번 미션은 교차 확인 규칙을 AI에게 정확히 설명하는 연습이에요.']},
                         'team': '박도윤(판독 담당)이 레드팀, 최서연(검토 담당)이 블루팀을 맡아요. 전우님이 AI로 판별 표를 만들어요.',
                         'terms': [{'t': '기만체',
                                    'en': 'decoy',
                                    'fig': 'f_decoy',
                                    'd': '진짜 장비처럼 보이게 만든 가짜 표적이에요.',
                                    'easy': '위에서 보면 전차 같지만 그림자나 전파 반사가 달라요.',
                                    'use': '그림자, SAR 후방산란처럼 다른 단서로 교차 확인해요.'},
                                   {'t': '경계 사례',
                                    'en': 'edge case',
                                    'fig': 'f_edge',
                                    'd': '빈칸처럼 평소 규칙이 맞지 않는 드문 경우예요.',
                                    'easy': '계산이 이상해지는 특이한 경우예요.',
                                    'use': '프롬프트에 ‘빈칸이면 이렇게 해 줘’라고 미리 적어요.'},
                                   {'t': '개수 오차',
                                    'en': 'count error',
                                    'fig': 'f_count',
                                    'd': '보고한 개수와 실제 개수의 차이예요.',
                                    'easy': '12대라고 했는데 실제는 5대라면 오차가 7대예요.',
                                    'use': '모델이 센 수와 규칙으로 가린 수를 표로 비교해 달라고 요청해요.'}],
                         'steps': ['숫자로 바꾸기: 모호한 말을 숫자로', '규칙 코드 받기: 숫자 규칙과 예외를 담아 요청하기', '합의하고 쓰기: 제안을 판단하고 내 말로 요청하기'],
                         'glossary': [],
                         'outputs': ['M6 기만체 판별 표']},
               'start_messages': [['team', 'choi', '{call}, 모델이 전부 전차라는데 정말 그럴까요? 진짜만 골라 달라고 AI에게 부탁해 볼까요? 파일은 m6_objects.csv예요.']],
               'finish_messages': [['team', 'park', '{call}, 다음 라운드에는 제가 레드팀으로 이 규칙을 속여 볼게요 😄']],
               'steps': [{'short': '숫자로 바꾸기',
                          'title': '모호한 말을 숫자로',
                          'lead': '‘진짜처럼 보이는 것’은 사람마다 달라요. **숫자로 바꾸면** AI와 팀이 같은 뜻으로 알아들어요.',
                          'blocks': [{'id': 's1_match',
                                      'type': 'match',
                                      'title': '모호한 말을 숫자 규칙과 짝짓기',
                                      'text': '왼쪽 말을 프롬프트에 쓸 수 있게 바꾼 문장을 골라 주세요.',
                                      'left_label': '모호한 말',
                                      'right_label': '프롬프트에 쓸 문장',
                                      'pairs': [{'left': '그림자가 없는 것', 'right': 'has_shadow가 0', 'hint': '그림자 열의 값이 무엇일 때일까요?'},
                                                {'left': '전파 반사가 약한 것', 'right': 'sar_db가 -15 미만', 'hint': 'SAR 값이 낮을 때예요.'},
                                                {'left': '믿을 만한 탐지', 'right': 'score가 0.6 이상', 'hint': '모델 점수와 관련된 조건이에요.'},
                                                {'left': '단서를 알 수 없는 것', 'right': 'has_shadow 또는 sar_db가 빈칸', 'hint': '자료가 없는 경우를 찾아보세요.'}],
                                      'hints': ['오른쪽 문장은 모두 열 이름과 숫자로 되어 있어요. 왼쪽 말과 뜻이 같은 것을 찾아보세요.'],
                                      'explain': '숫자로 바꾸면 AI가 해석할 여지가 줄고, 나중에 값만 바꿔 조정할 수도 있어요.'}]},
                         {'short': '규칙 코드 받기',
                          'title': '숫자 규칙과 예외를 담아 요청하기',
                          'lead': '**예외 문장을 빼고도** 보내 보세요. 단서가 없는 O11이 어떻게 판정되는지 눈여겨보세요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '기만체 판별 표',
                                      'ask': ['choi', '모델이 12개 모두 전차라고 해요. 교차 확인으로 다시 봐 주세요.'],
                                      'file': 'm6_objects.csv',
                                      'slots': [{'id': 'rule',
                                                 'label': '규칙',
                                                 'why': '숫자로 정해요.',
                                                 'options': [{'text': 'm6_objects.csv의 열은 object_id, model_label, score, has_shadow, sar_db, '
                                                                      'is_decoy(정답)야. has_shadow가 0이거나 sar_db가 -15 미만이면 기만 의심, 둘 다 아니면 장비로 판정해 줘.',
                                                              'good': True},
                                                             {'text': '진짜 장비만 잘 골라 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '점수가 높은 물체를 진짜 장비로 골랐어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m6_objects.csv')\n"
                                                                  "real = df[df['score'] >= 0.8]\n"
                                                                  "print('진짜 장비:', len(real), '대')",
                                                          'problem': '‘진짜’의 뜻을 몰라 모델 점수로만 골랐어요. 기만체도 점수가 높아서 거의 다 장비로 세어졌어요.',
                                                          'hint': '무엇을 기만체로 볼지 열 이름과 숫자로 적어 주세요.'}},
                                                {'id': 'edge',
                                                 'label': '예외',
                                                 'why': '단서가 빈칸인 경우를 알려요.',
                                                 'options': [{'text': 'has_shadow나 sar_db가 빈칸이면 판정하지 말고 ‘자료 없음, 사람 확인’으로 표시해 줘.', 'good': True},
                                                             {'text': '빈칸은 알아서 처리해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '그림자와 SAR 규칙으로 판정했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m6_objects.csv')\n"
                                                                  "suspect = (df['has_shadow'] == 0) | (df['sar_db'] < -15)\n"
                                                                  "df['판정'] = suspect.map({True: '기만 의심', False: '장비'})\n"
                                                                  "display(df[['object_id', 'has_shadow', 'sar_db', '판정']])",
                                                          'problem': '오류는 없지만 O11은 단서가 하나도 없는데 장비로 판정됐어요. 빈칸을 ‘기만이 아님’으로 처리했기 때문이에요. 정답을 보면 O11은 '
                                                                     '기만체예요. **조용히 틀린 대표적인 경우**예요.',
                                                          'hint': '빈칸일 때 어떻게 할지 적어 주세요.'}},
                                                {'id': 'out',
                                                 'label': '출력 형식',
                                                 'why': '비교할 모양으로 받아요.',
                                                 'options': [{'text': 'object_id, 판정, 이유 표와, 모델이 센 수·판정별 수·정답 수를 비교한 표를 보여 주고, HTML 보고서를 report 변수에 '
                                                                      '담아 줘.',
                                                              'good': True},
                                                             {'text': '결과를 정리해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '판정 결과예요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m6_objects.csv')\n"
                                                                  "print('확인이 필요한 물체가 있습니다. 주의 바랍니다.')",
                                                          'problem': '어느 물체가 왜 의심되는지, 개수가 얼마나 차이 나는지 알 수 없어요.',
                                                          'hint': '어떤 표가 필요한지 적어 주세요.'}}],
                                      'done': {'say': '규칙과 예외를 반영해 판별 표와 보고서를 만들었어요.',
                                               'code': 'import pandas as pd\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m6_objects.csv')\n"
                                                       'SAR_LIMIT = -15  # 값만 바꾸면 돼요\n'
                                                       '\n'
                                                       'def judge(r):\n'
                                                       "    if pd.isna(r['has_shadow']) or pd.isna(r['sar_db']):\n"
                                                       "        return '자료 없음', '단서가 빈칸이라 사람 확인'\n"
                                                       "    if r['has_shadow'] == 0:\n"
                                                       "        return '기만 의심', '그림자 없음'\n"
                                                       "    if r['sar_db'] < SAR_LIMIT:\n"
                                                       "        return '기만 의심', 'SAR 반사 약함'\n"
                                                       "    return '장비', '두 단서 모두 통과'\n"
                                                       '\n'
                                                       "df[['판정', '이유']] = df.apply(lambda r: pd.Series(judge(r)), axis=1)\n"
                                                       "table = df[['object_id', 'score', 'has_shadow', 'sar_db', '판정', '이유']]\n"
                                                       'display(table)\n'
                                                       '\n'
                                                       "count = pd.DataFrame([{'구분': '모델이 전차로 센 수', '개수': len(df)},\n"
                                                       "                      {'구분': '규칙: 장비', '개수': int((df['판정'] == '장비').sum())},\n"
                                                       "                      {'구분': '규칙: 기만 의심', '개수': int((df['판정'] == '기만 의심').sum())},\n"
                                                       "                      {'구분': '규칙: 자료 없음', '개수': int((df['판정'] == '자료 없음').sum())},\n"
                                                       "                      {'구분': '정답: 진짜 장비', '개수': int((df['is_decoy'] == 0).sum())}])\n"
                                                       'display(count)\n'
                                                       '\n'
                                                       "report = ('<h2>기만체 판별 결과</h2><p>가상 물체 ' + str(len(df)) + '개 기준. 자료 없음은 사람이 판독해요.</p>'\n"
                                                       '          + count.to_html(index=False) + table.to_html(index=False))',
                                               'lesson': '규칙을 숫자로, 예외를 문장으로 적으니 표를 믿을 수 있어요. 모델은 12개를 모두 전차로 셌지만 실제 장비는 5개예요. 자료 없음은 사람이 봐야 할 일로 '
                                                         '남아요.'},
                                      'on_ok': [['team', 'park', '{call}, 표가 나왔네요. 레드팀 입장에서 제안할 게 있어요.']]}]},
                         {'short': '합의하고 쓰기',
                          'title': '제안을 판단하고 내 말로 요청하기',
                          'lead': '기준값을 바꾸는 일은 **사람이 정하고**, AI에게는 바꾼 값으로 다시 돌려 달라고 하면 돼요.',
                          'blocks': [{'id': 's3_decide',
                                      'type': 'decide',
                                      'title': '규칙 조정 제안 판단하기',
                                      'text': '판별 표를 본 팀원들의 제안이에요.',
                                      'reason_min': 8,
                                      'items': [{'id': 'a',
                                                 'who': 'han',
                                                 'text': '자료 없음은 번거로우니 전부 장비로 세어 보고해요.',
                                                 'answer': 'reject',
                                                 'hint': 'O11을 떠올려 보세요.',
                                                 'explain': '자료가 없는데 장비로 세면 개수가 부풀어요. 사람이 판독하거나 다른 시점 영상을 찾아야 해요.',
                                                 'reason': '자료가 없는데 장비로 세면 개수가 부풀어 사람이 판독해야 해요.'},
                                                {'id': 'b',
                                                 'who': 'park',
                                                 'text': '레드팀이 그림자까지 칠한 기만체를 낼 수 있으니, SAR 기준을 -12로 바꾸면 표가 어떻게 달라지는지 돌려 보고 정해요.',
                                                 'answer': 'accept',
                                                 'hint': '기준값을 코드 어디에 두었는지 떠올려 보세요.',
                                                 'explain': '값만 바꿔 다시 돌릴 수 있으니 비용이 작아요. 결과를 보고 사람이 정하면 돼요.',
                                                 'reason': '값만 바꿔 다시 돌려 볼 수 있으니 결과를 보고 사람이 정하면 돼요.'}]},
                                     {'id': 's3_write',
                                      'type': 'write',
                                      'title': '예외를 담은 프롬프트 쓰기',
                                      'prompt': '**새 요청**: 두 시점을 비교한 changed 열(움직였으면 1, 아니면 0, 빈칸 있음)이 추가됐어요. 며칠 동안 전혀 움직이지 않은 물체도 기만 의심으로 보고 싶어요. '
                                                '판별 표를 다시 요청해 보세요.',
                                      'example': 'm6_objects_new.csv에 changed 열을 추가했어. 움직였으면 1, 아니면 0이고 빈칸도 있어. 기존 규칙은 그대로 두고, changed가 0이면 ‘움직임 없음’ '
                                                 '이유를 추가해 기만 의심으로 판정해 줘. changed가 빈칸이면 그 조건은 건너뛰고 지우지 마. 몇 행이 그렇게 처리됐는지 개수도 출력해 줘.',
                                      'min': 30,
                                      'checklist': ['규칙을 숫자로 적었어요', '빈칸을 어떻게 할지 적었어요', '기존 규칙을 그대로 두게 했어요', '처리 결과를 셀 수 있게 했어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm6_decoy_table',
                            'title': 'M6 기만체 판별 표',
                            'sections': [{'h': '1. AI에게 보낸 프롬프트와 결과', 'blocks': ['s2_vibe']},
                                         {'h': '2. 규칙 조정 결정', 'blocks': ['s3_decide']},
                                         {'h': '3. 내가 쓴 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm7',
               'no': 7,
               'title': '신뢰도 보정과 식별 확률 지도',
               'short': '표현 규칙',
               'kind': '개인',
               'competencies': ['D5', 'J1'],
               'outline': '모델 점수와 실제 정답률을 비교하는 표와 식별 확률 지도를 만들고, 보고 문장의 표현 규칙까지 정해 AI에게 요청해요.',
               'brief': {'subtitle': '점수표와 보고 문장 만들기: 표현 규칙까지 정해 요청하기',
                         'time': '약 25분',
                         'goal': '점수 구간별 실제 정답률을 표와 그림으로 확인하고, 가능성 표현과 분석 확신도를 나눠 쓴 보고 문장을 AI에게 요청해요.',
                         'why': '모델 점수 0.8은 ‘80% 맞다’는 뜻일까요? 점수를 그대로 확률처럼 쓰면 확신이 부풀어요. AI에게 문서를 시킬 때도 표현 규칙을 정해 주지 않으면 단정하는 문장이 나오기 쉬워요.',
                         'case': {'title': '해외 사례: 보이는 것과 확신할 수 있는 것은 달라요',
                                  'source': 'Reuters 보도 (2025.6)',
                                  'points': ['2025년 6월 미국의 이란 포르도 핵시설 타격 뒤 미국 대통령은 ‘완전히 파괴했다’고 말했어요.',
                                             '상용 위성영상에는 산등성이의 구멍 여섯 개 등 심각한 피해 흔적이 보였지만, 전문가들은 깊은 지하 시설의 피해는 영상으로 평가할 수 없다고 했어요. 국제원자력기구 사무총장도 아직 '
                                             '누구도 지하 피해를 온전히 평가하지 못했다고 밝혔어요.',
                                             '무엇을 얼마나 확신하는지 근거와 함께 나눠 말하지 않으면 보고가 단정이 돼요.']},
                         'team': '이정민(요청 담당)이 보고 문장을 받아요. 전우님이 AI로 표와 지도, 문장을 만들고 확인해요.',
                         'terms': [{'t': '신뢰도 곡선',
                                    'en': 'reliability curve',
                                    'fig': 'f_rel',
                                    'd': '모델 점수와 실제 정답률을 비교한 그림이에요.',
                                    'easy': '대각선보다 아래에 있으면 모델이 자신을 과대평가한 거예요.',
                                    'use': '점수 구간별 표와 곡선을 함께 요청해요.'},
                                   {'t': '가능성 표현',
                                    'en': 'words of estimative probability',
                                    'fig': 'f_scale',
                                    'd': '확률을 정해진 낱말로 나타내는 규칙이에요. 이번 실습은 미국 정보기관 지침(ICD 203)의 7단계를 써요.',
                                    'easy': '‘높음’이 몇 %쯤인지 약속해 두는 거예요.',
                                    'use': '표를 프롬프트에 그대로 적어 AI가 다른 뜻으로 쓰지 않게 해요.'},
                                   {'t': '분석 확신도',
                                    'en': 'analytic confidence',
                                    'fig': 'f_conf2',
                                    'd': '근거의 양과 질을 보고 판단을 얼마나 믿는지 나타낸 말이에요. 가능성 표현과 따로 써요.',
                                    'easy': '‘가능성은 높은데 자료가 적어 확신도는 낮음’처럼 말할 수 있어요.',
                                    'use': '프롬프트에 두 가지를 나눠 쓰라고 적어요.'}],
                         'steps': ['표현 규칙 익히기: 점수와 확률은 달라요', '점수표와 지도: 점수표와 지도, 보고 문장 요청하기', '내 말로 쓰기: 표현 규칙을 담은 프롬프트 쓰기'],
                         'glossary': [],
                         'outputs': ['M7 점수 신뢰도와 보고 문장']},
               'start_messages': [['dm_lee', 'lee', '{call}, 이 구역에 항공기가 몇 대쯤 있는지 보고가 필요해요. 점수 0.8이면 80% 확실하다고 적어도 되나요? 파일은 m7_scores.csv예요.']],
               'finish_messages': [['dm_lee', 'lee', '{call}, 보고 문장이 훨씬 믿음직하네요. 어디까지 확신할 수 있는지 바로 보여요.']],
               'steps': [{'short': '표현 규칙 익히기',
                          'title': '점수와 확률은 달라요',
                          'lead': '문서를 시키기 전에 **표현 규칙**을 정해 두면 AI가 단정하지 않아요.',
                          'blocks': [{'id': 's1_info',
                                      'type': 'info',
                                      'title': '가능성 표현 7단계 (ICD 203)',
                                      'icon': '📏',
                                      'text': ['모델 점수를 그대로 확률이라고 쓰지 않아요. 실제 정답률을 본 뒤 정해진 낱말로 나타내고, 분석 확신도는 근거의 양과 질로 따로 적어요.'],
                                      'table': {'head': ['표현', '확률'],
                                                'rows': [['거의 없음', '1~5%'],
                                                         ['매우 낮음', '5~20%'],
                                                         ['낮음', '20~45%'],
                                                         ['반반', '45~55%'],
                                                         ['높음', '55~80%'],
                                                         ['매우 높음', '80~95%'],
                                                         ['거의 확실', '95~99%']]},
                                      'figure': 'f_scale'},
                                     {'id': 's1_q',
                                      'type': 'quiz',
                                      'title': '어떤 문장이 좋을까요?',
                                      'text': '수요 부서에 보낼 문장이에요. 어느 쪽이 좋을까요?',
                                      'options': ['이 물체는 80% 확실히 항공기입니다.',
                                                  '이 물체가 항공기일 가능성은 높음(실제 정답률 기준)이고, 자료가 영상 한 장뿐이라 분석 확신도는 낮음입니다.',
                                                  '이 물체는 틀림없이 항공기로 보입니다.'],
                                      'answer': 1,
                                      'explain': '가능성과 확신도를 나눠 말하고, 무엇을 근거로 했는지도 드러나요.',
                                      'hints': ['가능성과 확신도를 나눠 쓴 문장은 무엇일까요?', '점수를 그대로 확률이라고 쓴 문장은 피해 보세요.']}]},
                         {'short': '점수표와 지도',
                          'title': '점수표와 지도, 보고 문장 요청하기',
                          'lead': '**표현 규칙 문장을 빼고** 보내 보세요. AI가 어떤 문장을 쓰는지 직접 보세요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '점수표와 식별 확률 지도',
                                      'ask': ['lee', '점수를 믿어도 되는지, 항공기가 어디에 모여 있는지 보고 싶어요.'],
                                      'file': 'm7_scores.csv',
                                      'slots': [{'id': 'input',
                                                 'label': '입력',
                                                 'why': '열의 뜻을 알려요.',
                                                 'options': [{'text': 'm7_scores.csv의 열은 det_id, x_km, y_km(위치), score(가상 모델 점수), is_correct(맞는 탐지면 '
                                                                      '1)야.',
                                                              'good': True},
                                                             {'text': '점수 파일이 있어.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '점수 파일을 읽었어요.',
                                                          'expect': 'error',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m7_scores.csv')\n"
                                                                  "print(df.groupby('confidence')['label'].mean())",
                                                          'problem': '열 이름을 추측해 KeyError가 났어요.',
                                                          'hint': '열 이름과 값의 뜻을 적어 주세요.'}},
                                                {'id': 'task',
                                                 'label': '할 일',
                                                 'why': '계산 방법을 정해요.',
                                                 'options': [{'text': 'score를 0.1 간격 구간으로 나눠 구간마다 평균 점수, 실제 정답률(is_correct 평균), 표본 수를 표로 만들고, 표본 수로 '
                                                                      '가중한 평균 차이(ECE)도 계산해 줘.',
                                                              'good': True},
                                                             {'text': '모델이 정확한지 알려 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '전체 정확도를 계산했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m7_scores.csv')\n"
                                                                  "print('accuracy:', round(df['is_correct'].mean(), 3))",
                                                          'problem': '평균 하나만 나왔어요. 점수 0.8대에서 실제로 얼마나 맞는지는 알 수 없어요.',
                                                          'hint': '구간을 어떻게 나눌지, 무엇을 비교할지 적어 주세요.'}},
                                                {'id': 'expr',
                                                 'label': '표현 규칙',
                                                 'why': '낱말의 뜻을 정해 줘요.',
                                                 'options': [{'text': '구간별 실제 정답률을 ICD 203 7단계(거의 없음 1~5%, 매우 낮음 5~20%, 낮음 20~45%, 반반 45~55%, 높음 '
                                                                      '55~80%, 매우 높음 80~95%, 거의 확실 95~99%)로 바꿔 열로 넣어 줘. 모델 점수를 그대로 확률이라고 쓰지 마.',
                                                              'good': True},
                                                             {'text': '점수를 퍼센트로 바꿔 ‘80% 확실’처럼 적어 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '점수를 퍼센트로 바꿔 적었어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m7_scores.csv')\n"
                                                                  "df['보고 문장'] = df['score'].apply(lambda s: f'{s * 100:.0f}% 확실히 항공기입니다')\n"
                                                                  "display(df[['det_id', 'score', '보고 문장']].head(5))",
                                                          'problem': '점수를 확률처럼 적어 확신이 부풀었어요. 점수 구간별 실제 정답률을 보면 이 숫자보다 낮아요.',
                                                          'hint': '낱말과 확률의 대응표를 적고, 점수를 확률로 쓰지 말라고 적어 주세요.'}},
                                                {'id': 'out',
                                                 'label': '문서',
                                                 'why': '독자와 사실 범위를 정해요.',
                                                 'options': [{'text': '신뢰도 곡선과 x_km, y_km 위치를 점수로 칠한 식별 확률 지도를 그려 주고, HTML 보고서를 report 변수에 담아 줘. 보고 '
                                                                      '문장은 가능성 표현과 분석 확신도를 나눠 쓰고, 파일에 없는 내용은 쓰지 마.',
                                                              'good': True},
                                                             {'text': '상황이 잘 드러나게 생생하게 보고서를 써 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '상황이 잘 드러나게 생생하게 작성했어요.',
                                                          'code': "report = ('<h2>판독 보고</h2><p><b>이 구역에 항공기 전력이 대규모로 집결해 있으며 공격이 임박한 것으로 확인됐습니다.</b> "
                                                                  "'\n"
                                                                  "          '모델 성능은 매우 우수합니다.</p>')\n"
                                                                  "print('보고서를 만들었어요.')",
                                                          'problem': '파일에 없는 내용이 들어갔어요. ‘집결’, ‘공격 임박’은 어디에도 없어요. **이런 것이 환각이에요.**',
                                                          'hint': '파일에 있는 내용만 쓰게 하고, 두 가지 표현을 나눠 달라고 적어 주세요.'}}],
                                      'done': {'say': '구간별 실제 정답률과 표현, 그림, 보고서를 만들었어요.',
                                               'code': 'import pandas as pd\n'
                                                       'import matplotlib.pyplot as plt\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m7_scores.csv')\n"
                                                       "WORDS = [(5, '거의 없음'), (20, '매우 낮음'), (45, '낮음'), (55, '반반'), (80, '높음'), (95, '매우 높음'), "
                                                       "(100, '거의 확실')]\n"
                                                       '\n'
                                                       'def word(p):\n'
                                                       '    for limit, w in WORDS:\n'
                                                       '        if p * 100 <= limit:\n'
                                                       '            return w\n'
                                                       '\n'
                                                       "df['bin'] = (df['score'] * 10).astype(int).clip(0, 9) / 10\n"
                                                       "groups = df.groupby('bin').agg(평균점수=('score', 'mean'), 실제정답률=('is_correct', 'mean'), "
                                                       "표본수=('score', 'size')).reset_index()\n"
                                                       "groups['가능성 표현'] = groups['실제정답률'].apply(word)\n"
                                                       "ece = ((groups['평균점수'] - groups['실제정답률']).abs() * groups['표본수']).sum() / len(df)\n"
                                                       'table = groups.round(3)\n'
                                                       'display(table)\n'
                                                       "print('ECE:', round(ece, 3))\n"
                                                       '\n'
                                                       'fig, axes = plt.subplots(1, 2, figsize=(8, 3.2))\n'
                                                       "axes[0].plot([0, 1], [0, 1], linestyle='--', color='gray')\n"
                                                       "axes[0].plot(groups['평균점수'], groups['실제정답률'], marker='o')\n"
                                                       "axes[0].set_xlabel('mean score')\n"
                                                       "axes[0].set_ylabel('actual accuracy')\n"
                                                       "axes[0].set_title('reliability curve')\n"
                                                       "map_ = axes[1].scatter(df['x_km'], df['y_km'], c=df['score'], cmap='viridis', s=16)\n"
                                                       "fig.colorbar(map_, ax=axes[1], label='score')\n"
                                                       "axes[1].set_xlabel('x (km)')\n"
                                                       "axes[1].set_ylabel('y (km)')\n"
                                                       "axes[1].set_title('identification map')\n"
                                                       '\n'
                                                       "high = groups[groups['bin'] == 0.8].iloc[0]\n"
                                                       'report = (f\'<h2>점수 신뢰도와 보고 문장</h2><p><b>결론:</b> 점수 0.8대 탐지의 실제 정답률은 {high["실제정답률"]:.0%}로, '
                                                       '가능성 표현은 ‘{high["가능성 표현"]}’입니다. \'\n'
                                                       "          f'가상 데이터 {len(df)}건, 영상 한 장 기준이라 분석 확신도는 낮음입니다.</p>' + table.to_html(index=False)\n"
                                                       "          + f'<p>ECE {ece:.3f}. 지도에서 점수가 높은 탐지가 모인 구역은 사람이 먼저 판독해요. 의도와 앞으로의 움직임은 자료 "
                                                       "없음.</p>')",
                                               'lesson': '구간표를 보면 점수가 실제 정답률보다 높게 나오는 것이 보여요. 표현 규칙을 정해 주니 AI도 단정하지 않아요.'},
                                      'on_ok': [['dm_lee', 'lee', '{call}, 점수 0.8이 곧 80%는 아니었네요. 앞으로는 표현 규칙대로 받을게요.']]}]},
                         {'short': '내 말로 쓰기',
                          'title': '표현 규칙을 담은 프롬프트 쓰기',
                          'lead': '독자가 바뀌어도 쓸 수 있게 **표현 규칙과 사실 범위**를 내 말로 적어 보세요.',
                          'blocks': [{'id': 's3_write',
                                      'type': 'write',
                                      'title': '문서 프롬프트 쓰기',
                                      'prompt': '**새 요청**: 같은 결과로 팀장이 1분 안에 읽을 요약을 만들어 달라고 요청해 보세요. 가능성과 분석 확신도를 나눠 쓰고, 없는 내용을 지어내지 않게 해 주세요.',
                                      'example': '팀장이 1분 안에 읽을 요약을 만들어 줘. m7_scores.csv의 구간별 실제 정답률을 근거로, 결론 한 줄, 구간표, 다음 조치 순서로 써 줘. 가능성은 ICD 203 '
                                                 '7단계 낱말로, 분석 확신도는 근거의 양과 질로 나눠 적어 줘. 모델 점수를 확률이라고 쓰지 말고, 파일에 없는 내용은 ‘자료 없음’으로 적어 줘.',
                                      'min': 30,
                                      'checklist': ['누가 얼마나 빨리 읽는지 적었어요', '표현 규칙을 적었어요', '가능성과 확신도를 나누게 했어요', '없는 내용을 지어내지 않게 했어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm7_reliability_doc',
                            'title': 'M7 점수 신뢰도와 보고 문장',
                            'sections': [{'h': '1. AI에게 보낸 프롬프트와 결과', 'blocks': ['s2_vibe']}, {'h': '2. 내가 쓴 문서 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm8',
               'no': 8,
               'title': '고가치 표적 판독 승인(HITL)과 판독 대기열 설계',
               'short': '안전하게 쓰기',
               'kind': '협업',
               'competencies': ['D6', 'J1'],
               'outline': 'AI에 넣어도 되는 정보와 안 되는 정보를 나누고, 사람이 판독할 대상을 빠짐없이 모은 대기열과 승인 기록 양식을 요청해요.',
               'brief': {'subtitle': '판독 대기열 만들기: AI에 넣으면 안 되는 정보 가리기',
                         'time': '약 25분',
                         'goal': '민감한 값을 빼고 데이터 모양만 설명해 판독 대기열을 만드는 코드를 받고, 사람이 볼 대상을 빠뜨리지 않게 해요.',
                         'why': '편하다고 다 붙여 넣어도 될까요? 공개된 AI에 넣는 순간 조직 밖으로 나간 것이나 마찬가지일 수 있어요. 또 자동화할수록 무엇을 기계가 했고 무엇을 사람이 확인했는지 남겨야 해요.',
                         'case': {'title': '해외 사례: 사람의 손을 거치지 않았다고 표시하는 보고서',
                                  'source': 'Breaking Defense 보도 (2025.6)',
                                  'points': ['2025년 6월 미국 국가지리정보국(NGA) 국장은 AI가 만든 지리공간정보 산출물에 ‘machine-generated GEOINT’ 표시를 붙인다고 밝혔어요.',
                                             '이 표시에는 사람이 제작에 관여하지 않았다는 내용이 담겨요. 국장은 받는 사람이 정보를 판단하고 추가 질문을 할지 정하려면 이 사실을 알아야 한다고 설명했어요.',
                                             '이번 미션에서도 자동 배포와 사람 판독을 나누고, 그 구분을 문서에 남겨요.']},
                         'team': '한지우·박도윤(판독 담당)이 판독하고, 최서연(검토 담당)이 승인해요. 이정민(요청 담당)의 요청도 판단해야 해요.',
                         'terms': [{'t': 'HITL (사람 참여 판독)',
                                    'en': 'human in the loop',
                                    'fig': 'f_hitl',
                                    'd': 'AI 결과를 사람이 확인한 뒤에 쓰도록 절차에 넣는 것이에요.',
                                    'easy': 'AI는 초안, 결정은 사람이에요.',
                                    'use': '무엇을 사람이 볼지 숫자 규칙으로 정해 프롬프트에 적어요.'},
                                   {'t': '민감 정보',
                                    'en': 'sensitive data',
                                    'fig': 'f_secret',
                                    'd': '조직 밖으로 나가면 안 되는 정보예요. 실제 위치, 인물, 장비 정보가 대표적이에요.',
                                    'easy': '동네 게시판에 붙이면 안 되는 내용이라고 생각하면 돼요.',
                                    'use': 'AI에는 값 대신 열 이름과 가상 샘플만 보여 줘요.'},
                                   {'t': '데이터 모양만 보여 주기',
                                    'en': 'schema only',
                                    'fig': 'f_clean',
                                    'd': '실제 값 없이 파일 이름, 열 이름, 값의 종류만 AI에게 알려 주는 방법이에요.',
                                    'easy': '내용은 가리고 양식만 보여 주는 거예요.',
                                    'use': '코드는 받고, 실제 데이터는 내 환경에서만 돌려요.'}],
                         'steps': ['나눠 보기: AI에 넣어도 되는 것과 안 되는 것', '대기열 만들기: 민감한 값을 빼고 대기열 요청하기', '내 말로 쓰기: 요청을 판단하고 내 말로 쓰기'],
                         'glossary': [],
                         'outputs': ['M8 판독 대기열과 승인 기록']},
               'start_messages': [['team', 'han', '{call}, 판독할 게 많아요. 대기열을 AI로 빨리 만들 수 있을까요? 파일 전체를 그대로 붙여 넣으면 되죠?'],
                                  ['team', 'kim', '잠깐만요. 파일에 위치 코드가 들어 있어요. 무엇을 넣어도 되는지 먼저 나눠 봅시다.']],
               'finish_messages': [['team', 'choi', '{call}, 대기열 받았어요. 승인 칸까지 있어 바로 쓸 수 있겠어요.']],
               'steps': [{'short': '나눠 보기',
                          'title': 'AI에 넣어도 되는 것과 안 되는 것',
                          'lead': '바이브 코딩에서는 **값을 보여 주지 않아도** 코드를 받을 수 있어요. 데이터의 모양만 알려 주면 돼요.',
                          'blocks': [{'id': 's1_cls',
                                      'type': 'classify',
                                      'title': '공개 AI에 넣어도 될까요?',
                                      'text': '조직이 승인한 환경이 아니라 일반 공개 AI라고 생각하고 나눠 보세요.',
                                      'primary_label': '판단',
                                      'item_label': '내용',
                                      'contrib': False,
                                      'categories': [{'id': 'ok', 'name': '넣어도 돼요'}, {'id': 'no', 'name': '넣으면 안 돼요'}],
                                      'items': [{'id': 'a',
                                                 'title': '열 이름: item_id, target_class, due_min',
                                                 'accept': ['ok'],
                                                 'explain': '열 이름만으로는 실제 상황을 알 수 없어요. 코드를 받는 데 꼭 필요한 정보예요.',
                                                 'hints': ['실제 값이 들어 있나요?', '이 정보만으로 위치나 사람을 알 수 있나요?']},
                                                {'id': 'b',
                                                 'title': '판독 대기열 원본 전체',
                                                 'accept': ['no'],
                                                 'explain': '실제 위치와 표적 정보가 그대로 나가요. 원본은 내 환경에서만 돌려요.',
                                                 'hints': ['원본에는 무엇이 들어 있을까요?', '조직 밖으로 나가도 되는 내용인가요?']},
                                                {'id': 'c',
                                                 'title': '직접 지어낸 가상 샘플 2줄',
                                                 'accept': ['ok'],
                                                 'explain': '값의 모양을 보여 주는 가장 안전한 방법이에요.',
                                                 'hints': ['실제로 있었던 값인가요?', '지어낸 값이면 무엇이 문제일까요?']},
                                                {'id': 'd',
                                                 'title': '실제 위치 코드',
                                                 'accept': ['no'],
                                                 'explain': '위치 정보는 대표적인 민감 정보예요. 코드를 받는 데도 필요 없어요.',
                                                 'hints': ['위치를 알려 주는 정보인가요?', '코드를 만드는 데 꼭 필요한가요?']}]}]},
                         {'short': '대기열 만들기',
                          'title': '민감한 값을 빼고 대기열 요청하기',
                          'lead': '**파일 전체를 붙여 넣지 않아도** 돼요. 칸마다 문장을 골라 보내 보세요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '판독 대기열 만들기',
                                      'ask': ['choi', '사람이 볼 대상이 빠지면 안 돼요. 위치 코드는 문서에 나오면 안 되고요.'],
                                      'file': 'm8_queue.csv',
                                      'slots': [{'id': 'safe',
                                                 'label': '넣을 정보',
                                                 'why': '값 대신 모양만 알려요.',
                                                 'options': [{'text': '실제 데이터는 내 환경에서 돌릴 거라 값은 보여 주지 않을게. 파일은 m8_queue.csv이고 열은 item_id, '
                                                                      'target_class(hvt·normal), calibrated_prob, ai_label, due_min, location_code야. '
                                                                      'location_code는 결과에 절대 넣지 마.',
                                                              'good': True},
                                                             {'text': '대기열 파일 전체를 붙여 넣을게. 알아서 정리해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '붙여 넣어 주신 파일로 대기열을 만들었어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m8_queue.csv')\n"
                                                                  "display(df[df['target_class'] == 'hvt'])",
                                                          'problem': '원본을 그대로 넘겼고, 결과에 location_code까지 나왔어요. 실제 업무였다면 민감 정보가 밖으로 나간 거예요.',
                                                          'hint': '값은 보여 주지 말고 열 이름과 값의 종류만 알려 주세요. 빠질 열도 적어 주세요.'}},
                                                {'id': 'rule',
                                                 'label': '구분 규칙',
                                                 'why': '사람이 볼 기준을 숫자로 정해요.',
                                                 'options': [{'text': 'target_class가 hvt이거나 calibrated_prob가 0.8 미만이면 사람 판독, 나머지는 자동 배포로 나누고, 자동 '
                                                                      '배포에는 ‘machine-generated’ 표시를 붙여 줘.',
                                                              'good': True},
                                                             {'text': '중요한 것만 사람이 보게 해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '중요한 표적을 사람 판독으로 나눴어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m8_queue.csv').drop(columns='location_code')\n"
                                                                  "df['구분'] = df['target_class'].map({'hvt': '사람 판독', 'normal': '자동 배포'})\n"
                                                                  "display(df[['item_id', 'target_class', 'calibrated_prob', '구분']])",
                                                          'problem': '‘중요한 것’을 hvt로만 해석했어요. 확률이 0.55인 Q06도 자동 배포로 나갔어요. 확신이 낮은 결과가 사람을 거치지 않게 된 거예요.',
                                                          'hint': '무엇을 사람이 볼지 열 이름과 숫자로 적어 주세요.'}},
                                                {'id': 'all',
                                                 'label': '빠짐없이',
                                                 'why': '요약하지 말아야 할 것을 정해요.',
                                                 'options': [{'text': '사람 판독 대상은 하나도 빼지 말고 전부 넣고, hvt 먼저, 그 안에서는 due_min이 짧은 순서로 정렬하고 개수도 출력해 줘.',
                                                              'good': True},
                                                             {'text': '급한 것만 추려 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '급한 것을 추렸어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m8_queue.csv').drop(columns='location_code')\n"
                                                                  "need = df[(df['target_class'] == 'hvt') | (df['calibrated_prob'] < 0.8)]\n"
                                                                  "display(need.sort_values('due_min').head(3))",
                                                          'problem': '사람이 볼 대상은 6건인데 3건만 나왔어요. 무엇이 급한지 AI는 몰라요. 대기열에서 빠진 건은 그대로 놓칠 수 있어요.',
                                                          'hint': '빼면 안 되는 범위와 개수를 확인할 방법을 적어 주세요.'}},
                                                {'id': 'form',
                                                 'label': '양식',
                                                 'why': '받는 사람이 할 일이 보이게 해요.',
                                                 'options': [{'text': 'HTML 문서를 report 변수에 담아 줘. 순서는 제목, 사람 판독 개수, 대기열 표, 자동 배포 표, 승인 기록 칸(판독자 A, '
                                                                      '판독자 B, 승인자)이야.',
                                                              'good': True},
                                                             {'text': '보고서 형식으로 만들어 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '보고서 형식으로 만들었어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m8_queue.csv')\n"
                                                                  "report = '<h2>판독 현황</h2><p>금일 판독 대상은 다음과 같습니다.</p><p>' + ' / "
                                                                  "'.join(df['item_id']) + '</p>'\n"
                                                                  "print('문서를 만들었어요.')",
                                                          'problem': '번호만 이어져 무엇을 사람이 보는지 보이지 않고, 승인 기록을 남길 곳도 없어요.',
                                                          'hint': '항목과 순서, 표로 받을지 적어 주세요.'}}],
                                      'done': {'say': '사람 판독 대상을 빠짐없이 모으고, location_code는 뺐어요. 개수도 함께 확인할 수 있어요.',
                                               'code': 'import pandas as pd\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m8_queue.csv').drop(columns='location_code')\n"
                                                       'PROB_LIMIT = 0.8\n'
                                                       '\n'
                                                       "human = (df['target_class'] == 'hvt') | (df['calibrated_prob'] < PROB_LIMIT)\n"
                                                       'queue = df[human].copy()\n'
                                                       "queue['순위'] = queue['target_class'].map({'hvt': 0, 'normal': 1})\n"
                                                       "queue = queue.sort_values(['순위', 'due_min']).drop(columns='순위')\n"
                                                       'auto = df[~human].copy()\n'
                                                       "auto['표시'] = 'machine-generated'\n"
                                                       "print('사람 판독:', len(queue), '건 / 자동 배포:', len(auto), '건 / 전체:', len(df), '건')\n"
                                                       'display(queue)\n'
                                                       'display(auto)\n'
                                                       '\n'
                                                       "report = (f'<h2>판독 대기열</h2><p><b>사람 판독 {len(queue)}건</b>, 자동 배포 {len(auto)}건입니다.</p>'\n"
                                                       "          + queue.to_html(index=False) + '<h3>자동 배포</h3>' + auto.to_html(index=False)\n"
                                                       '          + \'<h3>승인 기록</h3><table border="1"><tr><th>판독자 A</th><th>판독자 '
                                                       "B</th><th>승인자</th></tr><tr><td>&nbsp;</td><td>&nbsp;</td><td>&nbsp;</td></tr></table>')",
                                               'lesson': '값을 보여 주지 않아도 코드를 받을 수 있었고, 데이터는 내 환경 밖으로 나가지 않았어요. 개수를 함께 출력해 빠짐이 없는지도 확인할 수 있어요.'},
                                      'on_ok': [['team', 'kim', '{call}, 값을 넘기지 않고도 대기열이 나오네요. 이 방법을 팀 규칙으로 삼읍시다.']]}]},
                         {'short': '내 말로 쓰기',
                          'title': '요청을 판단하고 내 말로 쓰기',
                          'lead': '밖에서 온 요청도 같아요. **무엇을 줘도 되는지** 먼저 생각하세요.',
                          'blocks': [{'id': 's3_decide',
                                      'type': 'decide',
                                      'title': '요청 판단하기',
                                      'text': '다른 곳에서 온 요청과 팀 제안이에요. 어떻게 할까요?',
                                      'reason_min': 8,
                                      'items': [{'id': 'a',
                                                 'who': 'lee',
                                                 'text': '대기열 원본 전체를 메일로 보내 주세요. 저희 쪽 AI로 분석해 볼게요.',
                                                 'answer': 'reject',
                                                 'hint': '원본에는 무엇이 들어 있었나요?',
                                                 'explain': '원본에는 위치 코드 같은 민감 정보가 있어요. 필요한 항목만 절차에 따라 나눠요.',
                                                 'reason': '원본에 민감 정보가 있어 필요한 항목만 승인 절차를 거쳐 공유해야 해요.'},
                                                {'id': 'b',
                                                 'who': 'choi',
                                                 'text': 'hvt는 판독자 두 명이 따로 본 뒤 제가 승인하고, 기록을 남겨요.',
                                                 'answer': 'accept',
                                                 'hint': '틀렸을 때 영향이 큰 대상에는 어떤 절차가 필요할까요?',
                                                 'explain': '중요한 대상일수록 교차 판독과 승인 기록이 필요해요. 나중에 사후검토에도 쓸 수 있어요.',
                                                 'reason': '중요한 대상은 교차 판독과 승인 기록을 남겨야 나중에 확인할 수 있어요.'}]},
                                     {'id': 's3_write',
                                      'type': 'write',
                                      'title': '안전한 프롬프트 쓰기',
                                      'prompt': '**새 요청**: 장비 점검 기록(파일 m8_equip.csv)에서 고장 난 장비 목록을 뽑는 코드가 필요해요. 파일에 담당자 이름과 설치 위치가 있어요. 값을 넘기지 않고 '
                                                '요청해 보세요.',
                                      'example': '값은 보여 주지 않을게. m8_equip.csv의 열은 equip_id, status(ok·fail), checked_at, owner, site_code야. 코드는 내 '
                                                 '환경에서 돌릴 거야. status가 fail인 행을 하나도 빼지 말고 equip_id, checked_at만 표로 보여 줘. owner와 site_code는 결과에 넣지 말고, '
                                                 '몇 건인지도 출력해 줘.',
                                      'min': 30,
                                      'checklist': ['실제 값을 넣지 않았어요', '열 이름과 값의 종류만 적었어요', '결과에서 뺄 열을 적었어요', '빠짐없는지 셀 수 있게 했어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm8_queue_doc',
                            'title': 'M8 판독 대기열과 승인 기록',
                            'sections': [{'h': '1. AI에 넣어도 되는 정보', 'blocks': ['s1_cls']},
                                         {'h': '2. AI에게 보낸 프롬프트와 대기열', 'blocks': ['s2_vibe']},
                                         {'h': '3. 요청 판단', 'blocks': ['s3_decide']},
                                         {'h': '4. 내가 쓴 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm9',
               'no': 9,
               'title': '팀 AAR과 재보정',
               'short': '리뷰와 테스트',
               'kind': '협업',
               'competencies': ['D3', 'D5'],
               'outline': '오판독을 집계하는 코드의 숨은 실수를 AI에게 먼저 설명하게 하고, 테스트로 확인하고, 고칠 범위를 정해 수정해요.',
               'brief': {'subtitle': '오판독 집계 코드 고치기: AI에게 리뷰와 테스트를 시키기',
                         'time': '약 25분',
                         'goal': 'AI에게 코드를 먼저 설명하게 하고, 손으로 계산할 수 있는 테스트를 받고, 범위를 정해 고친 뒤 사후검토 문서를 만들어요.',
                         'why': '“정리까지 해 드렸어요”가 왜 위험할까요? AI에게 범위를 넓게 주면 요청하지 않은 일까지 할 수 있어요. 숫자가 엇갈릴 때도 누구 탓인지보다 무엇을 어떻게 세었는지를 먼저 따져야 해요.',
                         'case': {'title': '해외 사례: 41대와 10여 대, 왜 갈렸을까요',
                                  'source': 'Kyiv Post, Moscow Times 등 거미줄 작전 보도 (2025.6)',
                                  'points': ['2025년 6월 ‘거미줄 작전’ 뒤 우크라이나 보안국은 러시아 항공기 41대를 타격했다고 발표했어요.',
                                             '공개 위성영상을 본 분석들은 파괴된 항공기를 10~13대 안팎으로 셌어요. 미국 당국자를 인용한 보도(재인용)는 최대 20대 타격, 약 10대 파괴로 전했어요.',
                                             '‘타격’과 ‘파괴’, 센서와 시점이 서로 달랐어요. 집계 코드도 마찬가지예요. 고치기 전에 무엇을 세는지부터 확인해요.']},
                         'team': '김팀장이 사후검토를 이끌고, 한지우(판독 담당)와 정우진(모델 담당)이 함께 봐요. 전우님이 기록과 코드 수정을 맡아요.',
                         'terms': [{'t': '사후검토 (AAR)',
                                    'en': 'After Action Review',
                                    'fig': 'f_review',
                                    'd': '일을 마친 뒤 계획, 실제, 차이의 원인, 유지하고 고칠 점을 함께 돌아보는 활동이에요.',
                                    'easy': '누구 탓인지가 아니라 무엇 때문인지를 찾는 시간이에요.',
                                    'use': '원인은 데이터, 모델, 절차, 소통으로 나눠 봐요.'},
                                   {'t': '테스트',
                                    'en': 'test',
                                    'fig': 'f_verify',
                                    'd': '답을 아는 작은 예시로 코드가 맞는지 확인하는 코드예요.',
                                    'easy': '계산기를 2+2로 확인해 보는 것과 같아요.',
                                    'use': '손으로 계산할 수 있는 예시를 달라고 요청해요.'},
                                   {'t': '수정 범위',
                                    'en': 'scope',
                                    'fig': 'f_scope',
                                    'd': 'AI가 바꿔도 되는 곳과 건드리면 안 되는 곳의 경계예요.',
                                    'easy': '‘이 방만 고치고 다른 방은 들어가지 마세요’라는 말이에요.',
                                    'use': '원본 데이터는 건드리지 말라고 꼭 적어요.'}],
                         'steps': ['먼저 설명 듣기: 고치기 전에 설명을 듣는 이유', '리뷰·테스트·수정: 리뷰와 테스트를 시키고 범위를 정해 고치기', '내 말로 쓰기: 조치를 정하고 내 말로 요청하기'],
                         'glossary': [],
                         'outputs': ['M9 사후검토 문서와 수정 기록']},
               'start_messages': [['team', 'kim', '{call}, 이번 판독으로 사후검토를 해요. 그런데 오판독률이 생각보다 높게 나와요. 집계 코드가 이상한 것 같아요.'],
                                  ['team', 'jung', 'AI에게 바로 고치라고 하지 말고, 먼저 설명부터 들어 보면 어때요? 파일은 m9_cases.csv예요.']],
               'finish_messages': [['team', 'kim', '{call}, 고친 숫자로 검토했어요. 이번 검토는 누구 탓이 아니라 무엇을 고칠지로 끝났네요. 수고했어요!']],
               'steps': [{'short': '먼저 설명 듣기',
                          'title': '고치기 전에 설명을 듣는 이유',
                          'lead': 'AI 코드는 **고치기 전에 먼저 설명**하게 하면 엉뚱한 수정을 줄일 수 있어요.',
                          'blocks': [{'id': 's1_info',
                                      'type': 'info',
                                      'title': 'AI 코드를 고칠 때 순서',
                                      'icon': '🩺',
                                      'text': ['AI는 빠르게 고치지만, 무엇을 왜 바꿨는지 묻지 않으면 알기 어려워요.'],
                                      'bullets': ['**설명**: 지금 코드가 무엇을 하는지, 어디가 의심스러운지',
                                                  '**테스트**: 답을 아는 작은 예시',
                                                  '**범위**: 고칠 곳과 건드리면 안 되는 곳',
                                                  '**비교**: 고치기 전후의 숫자'],
                                      'figure': 'f_review'},
                                     {'id': 's1_q',
                                      'type': 'quiz',
                                      'title': '어떤 요청이 가장 안전할까요?',
                                      'text': '숫자가 이상해 보이는 코드가 있어요.',
                                      'options': ['코드 전체를 네가 알아서 싹 고쳐 줘.',
                                                  '고치지 말고 먼저 이 함수가 무엇을 계산하는지 설명하고, 의심되는 곳을 알려 줘.',
                                                  '데이터가 이상한 것 같으니 이상한 행은 지워 줘.'],
                                      'answer': 1,
                                      'explain': '설명을 먼저 들으면 전우님이 판단할 수 있어요. 원인을 모른 채 고치거나 데이터를 지우면 더 큰 문제가 생길 수 있어요.',
                                      'hints': ['사람이 판단할 기회를 주는 요청은 무엇일까요?', '되돌리기 어려운 요청은 피해야 해요.']}]},
                         {'short': '리뷰·테스트·수정',
                          'title': '리뷰와 테스트를 시키고 범위를 정해 고치기',
                          'lead': '**범위 문장을 빼고** 보내 보세요. AI가 기록을 어떻게 ‘정리’하는지 직접 보세요.',
                          'blocks': [{'id': 's2_vibe',
                                      'type': 'vibe',
                                      'title': '집계 코드 고치기',
                                      'ask': ['jung', '오판독률은 AI가 판정한 것(unknown 제외) 가운데 틀린 비율이어야 해요. 지금 숫자가 맞는지 봐 주세요.'],
                                      'file': 'm9_cases.csv',
                                      'slots': [{'id': 'review',
                                                 'label': '먼저 설명',
                                                 'why': '고치기 전에 진단을 받아요.',
                                                 'options': [{'text': '고치기 전에 misread_rate가 무엇을 나누고 있는지 설명하고, 내가 원하는 ‘unknown을 뺀 판정 가운데 틀린 비율’과 무엇이 '
                                                                      '다른지 알려 줘.',
                                                              'good': True},
                                                             {'text': '이 코드 고쳐 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '고쳤습니다!',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m9_cases.csv')\n"
                                                                  'def misread_rate(d):\n'
                                                                  "    j = d[d['ai_label'] != 'unknown']\n"
                                                                  "    return round((j['ai_label'] != j['truth']).mean(), 3)\n"
                                                                  "print('오판독률:', misread_rate(df))",
                                                          'problem': '숫자는 바뀌었지만 무엇이 왜 바뀌었는지 설명이 없어요. 맞는지 판단할 방법도 없고요.',
                                                          'hint': '무엇이 문제였는지 먼저 설명하게 해 주세요.'}},
                                                {'id': 'test',
                                                 'label': '테스트',
                                                 'why': '답을 아는 예시를 받아요.',
                                                 'options': [{'text': '손으로 계산할 수 있는 작은 예시(unknown을 뺀 판정 4건 중 틀린 것 1건이면 0.25)로 확인하는 assert 테스트를 함께 '
                                                                      '만들어 줘.',
                                                              'good': True},
                                                             {'text': '잘 동작하는지 봐 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '문제가 무엇인지 설명하고 고쳤어요. 잘 동작할 거예요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m9_cases.csv')\n"
                                                                  'def misread_rate(d):\n'
                                                                  "    j = d[d['ai_label'] != 'unknown']\n"
                                                                  "    return round((j['ai_label'] != j['truth']).mean(), 3)\n"
                                                                  "print('설명: 지금 코드는 unknown까지 틀린 것으로 세고 있어서, unknown을 빼도록 바꿨어요.')\n"
                                                                  "print('오판독률:', misread_rate(df))",
                                                          'problem': '설명은 좋아요. 그런데 ‘잘 동작할 거예요’는 AI의 말일 뿐, 확인한 것이 아니에요.',
                                                          'hint': '답을 아는 작은 예시로 확인하게 해 주세요.'}},
                                                {'id': 'scope',
                                                 'label': '수정 범위',
                                                 'why': '건드리면 안 되는 곳을 정해요.',
                                                 'options': [{'text': '고치는 건 misread_rate 함수만 해. m9_cases.csv 원본과 행은 지우거나 바꾸지 마. 전후 숫자와 cause별 건수를 '
                                                                      '표로 보여 주고, HTML 사후검토 문서를 report 변수에 담아 줘.',
                                                              'good': True},
                                                             {'text': '더 좋아 보이게 데이터도 정리해 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '함수를 고치고, 보기 좋게 데이터도 정리했어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m9_cases.csv')\n"
                                                                  "clean = df[df['cause'] == 'none']\n"
                                                                  "print('정리 전 행 수:', len(df))\n"
                                                                  "print('정리 후 행 수:', len(clean))\n"
                                                                  "j = clean[clean['ai_label'] != 'unknown']\n"
                                                                  "print('오판독률:', round((j['ai_label'] != j['truth']).mean(), 3))",
                                                          'problem': '요청하지 않은 ‘정리’로 원인이 적힌 기록을 지웠어요. 그래서 오판독률이 0으로 나오고, 사후검토에 꼭 필요한 기록이 사라졌어요.',
                                                          'hint': '고칠 곳과 건드리면 안 되는 곳을 분명히 적어 주세요.'}}],
                                      'done': {'say': '설명, 테스트, 수정, 전후 비교 순서로 진행했어요. 원본은 건드리지 않았어요.',
                                               'code': 'import pandas as pd\n'
                                                       '\n'
                                                       "df = pd.read_csv('{VDIR}/m9_cases.csv')\n"
                                                       '\n'
                                                       'def old_rate(d):\n'
                                                       "    return round((d['ai_label'] != d['truth']).sum() / len(d), 3)\n"
                                                       '\n'
                                                       'def misread_rate(d):\n'
                                                       "    judged = d[d['ai_label'] != 'unknown']\n"
                                                       '    if len(judged) == 0:\n'
                                                       '        return None\n'
                                                       "    return round((judged['ai_label'] != judged['truth']).sum() / len(judged), 3)\n"
                                                       '\n'
                                                       "print('설명: 지금 코드는 판정을 보류한 unknown까지 틀린 것으로 세고, 전체 기록 수로 나눠요. 원하는 값은 unknown을 뺀 판정 가운데 틀린 "
                                                       "비율이에요.')\n"
                                                       '\n'
                                                       '# 테스트: unknown을 뺀 판정 4건 중 틀린 것 1건이면 0.25\n'
                                                       "sample = pd.DataFrame({'ai_label': ['tank', 'apc', 'truck', 'tank', 'unknown'], 'truth': "
                                                       "['tank', 'apc', 'truck', 'apc', 'tank']})\n"
                                                       'assert misread_rate(sample) == 0.25\n'
                                                       "print('테스트 통과: 판정 4건 중 틀린 것 1건 -> 0.25')\n"
                                                       '\n'
                                                       "compare = pd.DataFrame([{'구분': '고치기 전', '오판독률': old_rate(df)}, {'구분': '고친 후', '오판독률': "
                                                       'misread_rate(df)}])\n'
                                                       'display(compare)\n'
                                                       "cause = df[df['cause'] != "
                                                       "'none'].groupby('cause').size().reset_index(name='건수').sort_values('건수', ascending=False)\n"
                                                       'display(cause)\n'
                                                       '\n'
                                                       "report = ('<h2>사후검토 문서</h2><p>기록 ' + str(len(df)) + '건, 원본 변경 없음</p><h3>오판독률 전후</h3>' + "
                                                       'compare.to_html(index=False)\n'
                                                       "          + '<h3>원인별 건수</h3>' + cause.to_html(index=False)\n"
                                                       "          + '<h3>다음 조치</h3><p>건수가 가장 많은 원인부터 담당자와 함께 살펴봐요. 사람 탓이 아니라 무엇을 고칠지로 정리해요.</p>')",
                                               'lesson': '설명, 테스트, 범위, 전후 비교를 함께 요청하니 고친 내용을 믿을 수 있어요. 기록도 그대로 남았어요.'},
                                      'start': {'say': '지금 쓰고 있는 코드예요. 오판독률이 너무 높아 보여요.',
                                                'code': 'import pandas as pd\n'
                                                        '\n'
                                                        "df = pd.read_csv('{VDIR}/m9_cases.csv')\n"
                                                        '\n'
                                                        'def misread_rate(d):\n'
                                                        '    # AI가 판정한 것 가운데 틀린 비율을 구하려는 함수\n'
                                                        "    wrong = (d['ai_label'] != d['truth']).sum()\n"
                                                        '    return round(wrong / len(d), 3)\n'
                                                        '\n'
                                                        "print('오판독률:', misread_rate(df))"},
                                      'on_ok': [['team', 'jung', '{call}, 테스트까지 통과했네요. 이제 이 숫자로 검토해도 되겠어요.']]}]},
                         {'short': '내 말로 쓰기',
                          'title': '조치를 정하고 내 말로 요청하기',
                          'lead': '숫자가 고쳐졌으면 이제 **무엇을 바꿀지** 팀이 정해요. AI에게는 안전하게 다음 작업을 요청해요.',
                          'blocks': [{'id': 's3_decide',
                                      'type': 'decide',
                                      'title': '사후검토 후속 제안 판단하기',
                                      'text': '고친 숫자를 본 뒤 나온 제안이에요.',
                                      'reason_min': 8,
                                      'items': [{'id': 'a',
                                                 'who': 'han',
                                                 'text': 'AI가 원본 기록을 직접 고칠 수 있게 쓰기 권한을 줘요. 그러면 일이 빨라져요.',
                                                 'answer': 'reject',
                                                 'hint': '요청하지 않은 ‘정리’에서 무슨 일이 있었나요?',
                                                 'explain': '되돌리기 어려운 일은 AI에게 맡기지 않아요. 코드는 복사본에서 돌리고 사람이 확인해요.',
                                                 'reason': '되돌리기 어려운 일이라 원본 쓰기 권한은 주지 않고 복사본으로 시험해요.'},
                                                {'id': 'b',
                                                 'who': 'kim',
                                                 'text': '건수가 가장 많은 원인 하나를 골라 기준을 바꾸고, 다음 판독 기록으로 다시 비교해요.',
                                                 'answer': 'accept',
                                                 'hint': '한 번에 하나씩 바꾸면 무엇이 좋을까요?',
                                                 'explain': '하나씩 바꿔야 무엇이 효과가 있었는지 알 수 있어요. 도구를 다시 돌리면 비교도 쉬워요.',
                                                 'reason': '하나씩 바꿔야 효과를 알 수 있고, 같은 도구로 다시 비교할 수 있어요.'}]},
                                     {'id': 's3_write',
                                      'type': 'write',
                                      'title': '리뷰 프롬프트 쓰기',
                                      'prompt': '**새 요청**: 다른 팀에서 받은 평균 판독 시간 계산 코드가 이상해 보여요. AI에게 어떻게 요청할까요? 고치기 전 설명, 테스트, 범위를 담아 보세요.',
                                      'example': '이 코드를 고치기 전에 평균 판독 시간을 어떻게 계산하는지 먼저 설명하고, 의심되는 곳을 알려 줘. 판독 시간이 10분, 20분, 30분이면 평균 20분이 나오는 assert '
                                                 '테스트도 만들어 줘. 고치는 건 계산 함수만 하고, 원본 파일과 다른 코드는 건드리지 마. 고치기 전후 결과를 표로 보여 줘.',
                                      'min': 30,
                                      'checklist': ['먼저 설명하게 했어요', '답을 아는 테스트를 요청했어요', '고칠 곳과 건드리면 안 되는 곳을 적었어요', '전후를 비교하게 했어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요'}]}],
               'outputs': [{'id': 'm9_review_doc',
                            'title': 'M9 사후검토 문서와 수정 기록',
                            'sections': [{'h': '1. AI에게 보낸 프롬프트와 결과', 'blocks': ['s2_vibe']},
                                         {'h': '2. 후속 제안 판단', 'blocks': ['s3_decide']},
                                         {'h': '3. 내가 쓴 리뷰 프롬프트', 'blocks': ['s3_write']}]}]},
              {'id': 'm10',
               'no': 10,
               'title': '협업 기반 객체 탐지 보고서 자동화 시스템과 ISR 평가 브리핑',
               'short': '묶어서 보고하기',
               'kind': '협업',
               'competencies': ['D4', 'J1'],
               'outline': '앞에서 배운 프롬프트를 모아 조건별 성능을 계산하고 보고서를 스스로 점검하는 미니 도구를 만들고, 나만의 프롬프트 틀을 정리해요.',
               'brief': {'subtitle': '보고서 자동화 도구 만들기: 묶어서 돌리고 조건까지 적기',
                         'time': '약 30분',
                         'goal': '다시 돌릴 수 있는 미니 도구를 요청하고, 수치마다 조건을 밝힌 보고서와 자동 점검, 내 프롬프트 틀을 만들어요.',
                         'why': '버튼 한 번에 보고서가 나오면 끝일까요? 자동화할수록 틀린 내용도 빠르게 퍼져요. 수치에는 데이터와 조건을 붙이고, 무엇을 AI가 했는지 표시하고, 올리기 전에 사람이 확인해야 해요.',
                         'case': {'title': '해외 사례: AI 오류로 비용 일부를 돌려준 정부 보고서',
                                  'source': 'AP 보도 (2025.10)',
                                  'points': ['2025년 10월 컨설팅 업체 Deloitte Australia는 호주 정부에 낸 보고서에 오류가 드러나 약 44만 호주달러의 계약금 일부를 돌려주기로 했어요.',
                                             '한 대학 연구자가 존재하지 않는 참고문헌과 판사 발언을 잘못 인용한 부분을 지적했어요. 수정본에는 생성형 AI를 썼다는 내용이 추가됐고, 부처는 보고서의 핵심 내용과 권고는 그대로라고 '
                                             '밝혔어요.',
                                             '보고서를 자동으로 만들수록 자료와 대조하고 AI 사용을 표시하는 일이 중요해요.']},
                         'team': '김팀장이 최종 검토를 하고, 이정민(요청 담당)이 보고서를 받아 피드백을 줘요. 최서연(검토 담당)이 사람 확인 기록을 봐요.',
                         'terms': [{'t': '미니 도구',
                                    'en': 'small tool',
                                    'fig': 'f_tool',
                                    'd': '설정값, 계산, 표와 그림, 문서까지 한 번에 돌아가는 작은 코드 묶음이에요.',
                                    'easy': '데이터만 바꾸면 다시 쓸 수 있는 계산기예요.',
                                    'use': '설정값을 맨 위에 모아 달라고 요청해요.'},
                                   {'t': '보고서 자동 점검 (린트)',
                                    'en': 'report lint',
                                    'fig': 'f_lint',
                                    'd': '보고서에 꼭 들어갈 항목이 빠졌는지 코드로 검사하는 것이에요.',
                                    'easy': '제출 전 체크리스트를 코드로 만든 거예요.',
                                    'use': '표본 수, 한계, AI 산출물 표기가 있는지 검사하게 해요.'},
                                   {'t': '조건 표시',
                                    'en': 'stating conditions',
                                    'fig': 'f_cond',
                                    'd': '수치마다 어떤 데이터, 어떤 조건, 몇 건에서 나온 값인지 함께 적는 것이에요.',
                                    'easy': '‘GSD 2m 구간은 표본 N건, 가상 데이터 기준’처럼 적는 거예요.',
                                    'use': '표본이 적은 조건은 따로 표시하게 요청해요.'},
                                   {'t': '프롬프트 모음',
                                    'en': 'prompt library',
                                    'fig': 'f_four',
                                    'd': '잘 통한 프롬프트를 틀로 모아 두고 다시 쓰는 개인 자료예요.',
                                    'easy': '자주 쓰는 업무 양식함이에요.',
                                    'use': '파일 이름과 숫자 자리를 빈칸으로 두면 다음에 바로 채워 쓸 수 있어요.'}],
                         'steps': ['도구로 묶기: 미니 도구와 보고서 요청하기', '피드백과 점검: 수요자 피드백과 보고서 점검', '프롬프트 모음: 나만의 프롬프트 틀 만들기'],
                         'glossary': [],
                         'outputs': ['M10 조건별 성능 보고서와 프롬프트 틀']},
               'start_messages': [['dm_kim',
                                   'kim',
                                   '{call}, 마지막이에요. 이정민 님께 보낼 조건별 성능 보고서를 자동으로 만들어 주세요. 파일은 m10_run.csv예요. 다음 달에도 같은 방식으로 돌릴 수 있으면 좋겠어요.']],
               'finish_messages': [['dm_lee', 'lee', '{call}, 보고 잘 받았어요. 어떤 조건에서 믿을 수 있는지 알겠네요. 다음 달에도 부탁해요.']],
               'steps': [{'short': '도구로 묶기',
                          'title': '미니 도구와 보고서 요청하기',
                          'lead': '지금까지 배운 것을 모두 써 보세요. **네 칸, 숫자 규칙, 근거 수치, 범위**가 모두 필요해요.',
                          'blocks': [{'id': 's1_vibe',
                                      'type': 'vibe',
                                      'title': '보고서 자동화 도구',
                                      'ask': ['kim', 'GSD 구간별 재현율과 오경보 비율, 보고서와 자동 점검까지 한 번에 나오게 해 주세요.'],
                                      'file': 'm10_run.csv',
                                      'slots': [{'id': 'tool',
                                                 'label': '구성',
                                                 'why': '다시 돌릴 수 있게 해요.',
                                                 'options': [{'text': 'm10_run.csv의 열은 run_id, gsd_band(0.5m·1m·2m), cloud_band, '
                                                                      'target_type(aircraft·other), alert(탐지면 1), human_checked(사람이 확인했으면 1)야. 파일 '
                                                                      '이름과 표본 기준을 맨 위 설정값으로 두고, 계산은 함수로 나눠 줘. 다음 달 파일로도 바로 돌릴 수 있게 해 줘.',
                                                              'good': True},
                                                             {'text': '성능 분석 코드 짜 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '전체 성능을 계산했어요. 재현율이 꽤 높아요!',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m10_run.csv')\n"
                                                                  "d = df[df['target_type'] == 'aircraft']\n"
                                                                  "print('재현율:', round(d['alert'].mean() * 100, 1), '%')",
                                                          'problem': '구간을 나누지 않은 숫자 하나뿐이에요. 파일 이름도 코드 속에 박혀 있어 다음 달에 다시 쓰기 어려워요.',
                                                          'hint': '무엇을 설정값으로 둘지, 어떻게 나눌지 적어 주세요.'}},
                                                {'id': 'metric',
                                                 'label': '지표',
                                                 'why': '무엇을 어떻게 나눌지 정해요.',
                                                 'options': [{'text': 'gsd_band별로 aircraft의 재현율(alert 평균), other의 오경보 비율(alert 평균), 표본 수를 표로 만들어 줘. '
                                                                      'aircraft 표본이 30건 미만이면 ‘표본 적음’으로 표시하고, 사람이 확인한 비율도 알려 줘.',
                                                              'good': True},
                                                             {'text': '조건별 성능을 보여 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '구간별 재현율을 표로 만들었어요.',
                                                          'code': 'import pandas as pd\n'
                                                                  "df = pd.read_csv('{VDIR}/m10_run.csv')\n"
                                                                  "display(df[df['target_type'] == "
                                                                  "'aircraft'].groupby('gsd_band')['alert'].mean().round(2).reset_index())",
                                                          'problem': '2m 구간의 값이 얼마나 믿을 만한지 알 수 없어요. 표본 수와 오경보 비율이 빠졌어요.',
                                                          'hint': '지표와 표본 수, 표본이 적을 때의 표시를 적어 주세요.'}},
                                                {'id': 'report',
                                                 'label': '보고서',
                                                 'why': '독자와 조건 표시, 점검을 정해요.',
                                                 'options': [{'text': '요청 담당이 읽을 HTML 보고서를 report 변수에 담아 줘. 순서는 결론 한 줄, 조건별 표, 그림, 한계, AI 산출물 표기야. '
                                                                      '수치에는 가상 데이터 몇 건 기준인지 적고, 표본 수와 AI 산출물 표기가 빠졌는지 검사하는 lint_report 함수도 만들어 줘.',
                                                              'good': True},
                                                             {'text': '정확도가 높다고 보고서 써 줘.', 'good': False},
                                                             {'text': '', 'good': False}],
                                                 'miss': {'say': '정확도가 뛰어나다는 보고서를 썼어요.',
                                                          'code': "report = '<h2>성능 보고</h2><p>본 시스템은 정확도가 매우 우수하여 모든 상황에서 신뢰할 수 있습니다.</p>'\n"
                                                                  "print('보고서를 만들었어요.')",
                                                          'problem': '조건도 숫자도 없이 ‘모든 상황’이라고 단정했어요. 원하는 결론을 프롬프트에 적으면 AI는 거기에 맞춰 줘요. GSD 2m 구간은 실제로 '
                                                                     '성능이 낮은데도요.',
                                                          'hint': '결론을 정해 주지 말고, 독자·순서·조건 표시와 점검을 요청해 주세요.'}}],
                                      'done': {'say': '설정값과 함수로 나눈 미니 도구예요. 파일 이름만 바꾸면 다음 달에도 쓸 수 있어요.',
                                               'code': 'import pandas as pd\n'
                                                       'import matplotlib.pyplot as plt\n'
                                                       '\n'
                                                       '# ===== 설정값 =====\n'
                                                       "FILE = '{VDIR}/m10_run.csv'\n"
                                                       'MIN_SAMPLES = 30\n'
                                                       "BANDS = ['0.5m', '1m', '2m']\n"
                                                       '\n'
                                                       'def perf_table(df):\n'
                                                       '    rows = []\n'
                                                       '    for band in BANDS:\n'
                                                       "        a = df[(df['gsd_band'] == band) & (df['target_type'] == 'aircraft')]\n"
                                                       "        o = df[(df['gsd_band'] == band) & (df['target_type'] == 'other')]\n"
                                                       "        rows.append({'GSD': band, '재현율(%)': round(a['alert'].mean() * 100, 1), '항공기 표본': "
                                                       'len(a),\n'
                                                       "                     '오경보 비율(%)': round(o['alert'].mean() * 100, 1), '기타 표본': len(o),\n"
                                                       "                     '비고': '표본 적음' if len(a) < MIN_SAMPLES else ''})\n"
                                                       '    return pd.DataFrame(rows)\n'
                                                       '\n'
                                                       'def make_report(t, df):\n'
                                                       "    low = t[t['비고'] != '']['GSD'].tolist()\n"
                                                       '    head = t.iloc[0]\n'
                                                       "    checked = df[df['alert'] == 1]['human_checked'].mean()\n"
                                                       '    limit = f"표본이 적은 구간: {\', \'.join(low)}. 이 구간의 수치는 참고만 해요." if low else \'표본이 적은 구간은 '
                                                       "없어요.'\n"
                                                       '    return (f"<h2>조건별 탐지 성능 보고</h2><p><b>결론:</b> GSD 0.5m 구간 재현율은 {head[\'재현율(%)\']}%입니다. '
                                                       '(가상 데이터 {len(df)}건 기준)</p>"\n'
                                                       "            + t.to_html(index=False) + f'<h3>한계</h3><p>{limit} 실제 환경에서는 다시 확인해야 해요.</p>'\n"
                                                       "            + f'<h3>AI 산출물 표기</h3><p>이 보고서는 AI가 만든 코드로 자동 생성했어요. 탐지 가운데 사람이 확인한 비율은 "
                                                       "{checked:.0%}예요.</p>')\n"
                                                       '\n'
                                                       'def lint_report(text):\n'
                                                       '    # 꼭 들어갈 말이 빠졌는지 검사\n'
                                                       "    return [w for w in ['표본', '기준', '한계', 'AI 산출물'] if w not in text]\n"
                                                       '\n'
                                                       'df = pd.read_csv(FILE)\n'
                                                       'table = perf_table(df)\n'
                                                       'display(table)\n'
                                                       '\n'
                                                       'plt.figure(figsize=(5, 3))\n'
                                                       "plt.bar(table['GSD'], table['재현율(%)'])\n"
                                                       "plt.ylabel('recall (%)')\n"
                                                       "plt.xlabel('GSD band')\n"
                                                       'plt.ylim(0, 100)\n'
                                                       '\n'
                                                       'report = make_report(table, df)\n'
                                                       "print('자동 점검 결과(빠진 항목):', lint_report(report) or '없음')",
                                               'lesson': '결론을 정해 주지 않고 조건을 요구하니, 어떤 구간을 믿을 수 있고 어떤 구간은 조심해야 하는지 드러났어요. 자동 점검이 있어 빠뜨림도 줄어요.'},
                                      'on_ok': [['dm_kim', 'kim', '{call}, 표본이 적은 구간까지 표시됐네요. 이제 이정민 님 피드백을 볼까요?']]}]},
                         {'short': '피드백과 점검',
                          'title': '수요자 피드백과 보고서 점검',
                          'lead': '보고서를 올리기 전에 **사람이 마지막으로 확인**해요.',
                          'blocks': [{'id': 's2_decide',
                                      'type': 'decide',
                                      'title': '수요자 피드백 판단하기',
                                      'text': '보고서를 받은 사람들의 요청이에요. 받아들일지 정하고 이유를 적어 주세요.',
                                      'reason_min': 8,
                                      'items': [{'id': 'a',
                                                 'who': 'lee',
                                                 'text': '‘완전히 식별함’이라고 단정해 주세요. 읽는 사람이 헷갈려요.',
                                                 'answer': 'reject',
                                                 'hint': '표의 수치가 단정을 뒷받침하나요?',
                                                 'explain': '근거 없이 단정하면 확신이 부풀어요. 쉬운 말로 쓰되 가능성과 조건은 남겨요.',
                                                 'reason': '수치가 단정을 뒷받침하지 않아 쉬운 말로 쓰되 조건을 남겨요.'},
                                                {'id': 'b',
                                                 'who': 'lee',
                                                 'text': '맨 위에 결론 한 줄을 두고, 표는 그 아래에 놓아 주세요.',
                                                 'answer': 'accept',
                                                 'hint': '내용을 바꾸는 요청인가요, 순서를 바꾸는 요청인가요?',
                                                 'explain': '사실을 해치지 않고 읽기 쉬워져요. 프롬프트에 순서만 바꿔 다시 돌리면 돼요.',
                                                 'reason': '사실은 그대로 두고 읽기 쉬워져 프롬프트의 순서만 바꾸면 돼요.'}]},
                                     {'id': 's2_multi',
                                      'type': 'multi',
                                      'title': '올리기 전에 확인할 것',
                                      'text': 'AI가 만든 보고서를 올리기 전에 꼭 확인할 것을 모두 골라 주세요.',
                                      'items': [{'text': '표의 숫자가 파일에서 다시 계산한 값과 같은지 한두 개 대조한다.',
                                                 'correct': True,
                                                 'explain': '한두 칸만 대조해도 큰 실수를 잡을 수 있어요.',
                                                 'hint': '숫자가 틀리면 어떻게 될까요?'},
                                                {'text': '수치마다 데이터와 표본 수, 조건이 적혀 있는지 본다.',
                                                 'correct': True,
                                                 'explain': '조건이 없으면 숫자가 오해되기 쉬워요.',
                                                 'hint': '거미줄 작전의 숫자가 왜 갈렸는지 떠올려 보세요.'},
                                                {'text': '문장이 매끄러우면 내용도 맞으니 건너뛴다.',
                                                 'correct': False,
                                                 'explain': '매끄러운 문장과 맞는 내용은 별개예요.',
                                                 'hint': 'AI 문장은 틀려도 매끄러워요.'},
                                                {'text': '파일에 없는 내용이 섞이지 않았는지 본다.',
                                                 'correct': True,
                                                 'explain': '환각을 잡는 가장 기본적인 점검이에요.',
                                                 'hint': '해외 사례의 보고서에서 무엇이 문제였나요?'},
                                                {'text': 'AI가 만든 문서이니 작성 책임은 AI에게 있다고 적는다.',
                                                 'correct': False,
                                                 'explain': '문서의 책임은 사람에게 있어요. AI는 도구예요.',
                                                 'hint': '보고를 올리는 사람은 누구인가요?'}]}]},
                         {'short': '프롬프트 모음',
                          'title': '나만의 프롬프트 틀 만들기',
                          'lead': '이번 프로젝트의 **가장 큰 산출물**이에요. 다음 업무에서 바로 쓸 나만의 틀을 남겨요.',
                          'blocks': [{'id': 's3_write',
                                      'type': 'write',
                                      'title': '내 프롬프트 틀 쓰기',
                                      'prompt': '앞으로 자주 쓸 **프롬프트 틀**을 써 보세요. 파일 이름이나 숫자처럼 바뀌는 자리는 [대괄호]로 비워 두세요. 네 칸에 더해 검증, 범위, 민감 정보 내용도 넣어 보세요.',
                                      'example': '[업무 맥락]을 위해 [무엇]을 확인하려고 해. 값은 보여 주지 않을게. 파일은 [파일 이름]이고 열은 [열 이름과 값의 종류]야. [계산 규칙과 기준값]으로 계산해 줘. '
                                                 '[예외]는 [처리 방법]으로 해 줘. 결과는 [표의 모양]으로 보여 주고, 다시 셀 수 있는 숫자와 표본 수를 함께 적어 줘. 먼저 [작은 예시]로 확인한 뒤 전체에 적용해 '
                                                 '줘. 고치는 건 [범위]만 하고, 원본 파일은 건드리지 마. 문서는 [독자]가 읽을 [순서]로 만들고 없는 내용은 ‘자료 없음’으로 적어 줘. AI 산출물이라는 표시도 넣어 '
                                                 '줘.',
                                      'min': 40,
                                      'checklist': ['맥락, 입력, 할 일, 출력 형식이 있어요',
                                                    '숫자 규칙과 예외 자리가 있어요',
                                                    '근거 수치와 검증을 요청했어요',
                                                    '수정 범위와 민감 정보 내용이 있어요',
                                                    '다음 업무에 바로 쓸 수 있어요'],
                                      'model_label': '📝 모범 프롬프트',
                                      'placeholder': '꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요',
                                      'explain': '잘 통한 프롬프트는 저장해 두세요. 쓸수록 나에게 맞는 틀이 돼요.'}]}],
               'outputs': [{'id': 'm10_final_report',
                            'title': 'M10 조건별 성능 보고서와 프롬프트 틀',
                            'sections': [{'h': '1. 보고서 자동화 도구', 'blocks': ['s1_vibe']},
                                         {'h': '2. 수요자 피드백 판단', 'blocks': ['s2_decide']},
                                         {'h': '3. 보고서 점검', 'blocks': ['s2_multi']},
                                         {'h': '4. 나만의 프롬프트 틀', 'blocks': ['s3_write']}]}]}]}   # 프로젝트·미션 명세 (만드는 도구가 넣음). 정답·힌트도 여기에 있어 화면(브라우저)으로는 보내지 않음
_P = SPEC["project"]
PROJECT = {"title": _P["title"], "goal": _P["goal"], "stages": _P["stages"]}
TEAM_NAME = _P["team_name"]            # 학습자가 합류하는 가상 조직 이름 (예: 드론분석 TF)
MY_ROLE = _P["my_role"]                # 학습자의 역할 (예: 신입 분석관)
MISSION_INFO = {m["id"]: (m["no"], m["title"], m["short"], m["kind"], m["outline"]) for m in SPEC["missions"]}   # (번호, 제목, 짧은 이름, 형태, 소개)
IMPLEMENTED = [m["id"] for m in SPEC["missions"] if m.get("steps")]

# ---------------------------------------------------------------
# 학습자 호칭: 모두 '전우님' (개인 정보를 묻지 않음. 군 체계를 추정할 정보도 남지 않음)
# ---------------------------------------------------------------
CALL_NAME = _P.get("call_default") or "전우님"


def call_name():
    """학습자를 부르는 말."""
    return CALL_NAME


def me_label():
    return CALL_NAME


MEMBERS = {k: tuple(v) for k, v in _P["members"].items()}   # 아이디: (이름, 소속, 역할, 아바타 색)
CHANNELS = dict(_P["channels"])
ALWAYS_CHANNELS = tuple(_P["always_channels"])

# ---------------------------------------------------------------
# 미션 등록부 (미션 파일들이 채움)
# ---------------------------------------------------------------
BRIEFINGS = {}      # 미션 소개
VIEWS = {}          # 화면에 보낼 미션별 상태
RESETTERS = {}      # 미션별 상태 초기화
STARTERS = {}       # 미션 시작 때 오는 메시지
CALLBACKS = {}      # 화면 버튼 → 파이썬 함수
CODE_TOPICS = {}    # '본 과정의 코드 보기'에 보여 줄 함수

# ---------------------------------------------------------------
# 진행 상태
# ---------------------------------------------------------------
S = {}


def _reset_state():
    S.clear()
    S.update(msgs=[], minute=0, files=[],
             progress={m: {"started": False, "step": 0, "done": False, "issued": False} for m in MISSION_INFO})
    for reset in RESETTERS.values():
        reset()
    _welcome()
    if FILL_ANSWERS:                 # 선생님용: 빈 값 없이 모든 활동을 정답으로 채우고 산출물까지 만들어 둠
        _prefill_all()


def _welcome():
    for ch, who, text in _P["welcome"]:
        post(ch, who, text)


def post(ch, who, text):
    """메신저에 메시지 남기기. {call}은 학습자를 부르는 말(전우님)로 바뀝니다."""
    t = 9 * 60 + S["minute"]
    S["minute"] += 2
    S["msgs"].append({"id": len(S["msgs"]), "ch": ch, "who": who, "time": f"{t // 60:02d}:{t % 60:02d}", "text": text})


def _fmt(text):
    return text.replace("{call}", call_name())


def _advance(mid, step):
    p = S["progress"][mid]
    p["step"] = max(p["step"], step)


def _status(mid):
    """잠금 없음: 준비된 미션은 언제나 열려 있음. 필수 활동을 다 하면 done."""
    if mid not in IMPLEMENTED:
        return "soon"
    return "done" if S["progress"][mid]["done"] else "open"


def overall_progress():
    """전체 진행률: 미션마다 끝낸 필수 활동의 비율을 평균."""
    units = sum(_mission_fraction(mid) for mid in MISSION_INFO if mid in BRIEFINGS)
    done = sum(S["progress"][m]["done"] for m in MISSION_INFO)
    return {"pct": round(100 * units / len(MISSION_INFO)), "done": done, "total": len(MISSION_INFO)}


def _add_file(fid, title, mission, doc, blanks=0):
    S["files"] = [f for f in S["files"] if f["id"] != fid]
    S["files"].append({"id": fid, "name": f"{fid}.html", "title": title, "mission": mission, "html": doc, "blanks": blanks})


DOC_CSS = ("body{font-family:sans-serif;max-width:860px;margin:24px auto;color:#111827;background:#fff;line-height:1.6;padding:0 16px}"
           "h1{font-size:22px;border-bottom:2px solid #111827;padding-bottom:6px}h2{font-size:16px;margin-top:22px}"
           "table{border-collapse:collapse;margin:8px 0}td,th{border:1px solid #c8ced6;padding:5px 10px;font-size:13px;text-align:left;vertical-align:top;color:#111827}"
           "th{background:#eef2f7}.meta{color:#111827;font-size:13px;background:#f3f6fa;padding:6px 10px;border-radius:6px}"
           ".note{color:#111827;font-size:13px}.ok{color:#146c2e;font-weight:bold}.bad{color:#b3261e;font-weight:bold}"
           ".cards{display:grid;grid-template-columns:repeat(auto-fill,minmax(240px,1fr));gap:12px}"
           ".card{border:1px solid #c8ced6;border-radius:8px;padding:10px}.card img{width:48%;border-radius:4px}"
           ".imgs img{max-width:100%;border:1px solid #c8ced6;border-radius:6px}.chg{background:#fff4ce}"
           "pre{background:#f3f6fa;padding:8px 10px;border-radius:6px;white-space:pre-wrap}"
           ".blank{display:inline-block;min-width:180px;height:1.1em;border-bottom:1px solid #6b7280}"
           ".blankbox{min-height:54px;border:1px dashed #6b7280;border-radius:6px;margin:6px 0}"
           "td.empty{min-width:120px;height:1.6em}img{max-width:100%}")


def _doc(title, body):
    return (f"<!doctype html><html><head><meta charset='utf-8'><title>{html.escape(title)}</title><style>{DOC_CSS}</style></head>"
            f"<body><h1>{html.escape(title)}</h1><p class='meta'>프로젝트: {html.escape(PROJECT['title'])}<br>"
            f"{html.escape(TEAM_NAME)} · 작성일: {date.today():%Y-%m-%d}</p>{body}</body></html>")


# ---------------------------------------------------------------
# 본 과정의 코드 보기
# ---------------------------------------------------------------
def code_topic(key, title, desc, funcs, marks=()):
    """화면의 한 단계에서 쓰이는 함수 목록을 등록. marks에 든 글자가 있는 줄은 강조 표시됩니다."""
    CODE_TOPICS[key] = {"title": title, "desc": desc, "funcs": list(funcs), "marks": list(marks)}


def call_str(name, *args, **kw):
    """함수 호출을 코드 한 줄로 적기 (예: simulate('m1', level=0.5))."""
    parts = [repr(a) for a in args] + [f"{k}={v!r}" for k, v in kw.items()]
    return f"{name}({', '.join(parts)})"


def app_code(topic):
    t = CODE_TOPICS.get(topic)
    if not t:
        return JSON({"ok": False, "blocks": []})
    blocks = []
    for name in t["funcs"]:
        fn = globals().get(name)
        try:
            src = textwrap.dedent(inspect.getsource(fn)).rstrip()
            doc = (inspect.getdoc(fn) or "").split("\n")[0]
        except Exception:
            src, doc = f"# {name}: 코드를 불러오지 못했어요.", ""
        lines = src.splitlines()
        hl = [i for i, line in enumerate(lines) if any(m in line for m in t["marks"])]
        blocks.append({"name": name, "doc": doc, "code": src, "hl": hl})
    return JSON({"ok": True, "topic": topic, "title": t["title"], "desc": t["desc"], "blocks": blocks})


# ---------------------------------------------------------------
# 화면에 보낼 상태
# ---------------------------------------------------------------
def _mission_meta(mid):
    no, title, short, kind, outline = MISSION_INFO[mid]
    stage = next(s["no"] for s in PROJECT["stages"] if mid in s["missions"])
    meta = {"id": mid, "no": no, "stage": stage, "title": title, "short": short, "kind": kind,
            "outline": outline, "status": _status(mid), "progress": S["progress"][mid]}
    if mid in BRIEFINGS and mid in IMPLEMENTED:
        meta["brief"] = BRIEFINGS[mid]
    return meta


def app_state():
    members = {}
    for k, (n, o, r, c) in MEMBERS.items():
        members[k] = {"name": n, "org": o, "role": r, "color": c, "ext": k != "me" and o != TEAM_NAME}
    used = {m["ch"] for m in S["msgs"]}
    return {
        "call": call_name(),
        "project": {"title": PROJECT["title"], "goal": PROJECT["goal"],
                    "stages": [{"no": s["no"], "name": s["name"], "subtitle": s.get("subtitle", ""), "missions": s["missions"]} for s in PROJECT["stages"]]},
        "missions": {mid: _mission_meta(mid) for mid in MISSION_INFO},
        "views": {mid: fn() for mid, fn in VIEWS.items()},
        "members": members,
        "channels": {k: v for k, v in CHANNELS.items() if k in ALWAYS_CHANNELS or k in used},
        "msgs": [{**m, "text": m["text"] if m["who"] == "me" else _fmt(m["text"])} for m in S["msgs"]],
        "files": [{k: f.get(k) for k in ("id", "name", "title", "mission", "blanks")} for f in S["files"]],
        "overall": overall_progress(), "boot": BOOT_ID,
        "teacher": bool(FILL_ANSWERS),        # 선생님용: 값이 모두 채워진 상태 (화면에 표시만 함)
        "assets": assets_status(),            # 참고자료(사진)가 필요한 프로젝트면 폴더와 빠진 파일을 알려 줌
    }


def _resp(**kw):
    return JSON({"ok": True, **kw, "app": app_state()})


def app_get():
    return JSON({"ok": True, "app": app_state()})


def app_start(mid):
    if _status(mid) == "soon":
        return _resp(ok=False, msg="아직 준비 중인 미션이에요.")
    _ensure_started(mid)
    return _resp(ok=True)


def _ensure_started(mid):
    """미션을 처음 열 때 한 번: 시작 메시지를 보냄."""
    p = S["progress"][mid]
    if not p["started"]:
        p["started"] = True
        _advance(mid, 1)
        if mid in STARTERS:
            STARTERS[mid]()


def app_send(ch, text):
    text = (text or "").strip()
    if text:
        post(ch, "me", text)
    return _resp(ok=True)


def app_file(fid):
    f = next((f for f in S["files"] if f["id"] == fid), None)
    return JSON({"ok": bool(f), "html": f["html"] if f else "", "name": f["name"] if f else ""})


def app_reset():
    _reset_state()
    return _resp(ok=True, msg="처음 상태로 돌아갔어요.")


CALLBACKS.update({"app.state": app_get, "app.start": app_start, "app.send": app_send,
                  "app.file": app_file, "app.reset": app_reset, "app.code": app_code})
code_topic("app.chat", "메신저는 이렇게 동작해요", "팀원 메시지는 미션 진행에 맞춰 파이썬 함수 post()가 남깁니다. "
           "화면의 버튼을 누르면 google.colab의 invokeFunction이 파이썬 함수를 부르고, 결과(JSON)로 화면을 다시 그려요.",
           ["post", "app_send"], marks=["S[\"msgs\"].append", "post(ch"])
code_topic("app.mission", "미션 시작은 이렇게 동작해요", "[미션 시작]을 누르면 app_start()가 진행 상태를 바꾸고, "
           "미션마다 등록된 시작 메시지(STARTERS)를 보냅니다.", ["app_start", "_ensure_started"], marks=["STARTERS"])
code_topic("app.files", "산출물은 이렇게 만들어져요", "산출물을 받으면 입력한 내용으로 HTML 문서를 만들어 파일 목록에 저장합니다. "
           "하지 않은 활동은 빈칸으로 남아요.", ["_doc", "_add_file"], marks=["S[\"files\"]"])


# ---------------------------------------------------------------
# 뒤에서 도는 작업 (학습 · 계산 · 내려받기): 화면이 1.5초마다 app.job으로 진행을 물어봄
# 오래 걸리는 작업 중 다른 화면을 열면 Colab 런타임이 끊기는 일이 있어서, 명세의 stay_jobs에 적은 작업은 화면 이동을 잠금
# ---------------------------------------------------------------
JOBS = {}


def start_job(name, fn, *args):
    """작업을 뒤에서(스레드) 실행. 이미 같은 작업이 돌고 있으면 False."""
    if JOBS.get(name, {}).get("state") == "running":
        return False
    job = {"state": "running", "pct": 1, "msg": "시작하는 중...", "t0": time.time(), "error": None}
    JOBS[name] = job

    def run():
        try:
            fn(job, *args)
            job.update(state="done", pct=100)
        except Exception as e:
            job.update(state="error", error=f"{type(e).__name__}: {e}", trace=traceback.format_exc()[-2500:])
    threading.Thread(target=run, daemon=True).start()
    return True


def _say(job, pct, msg):
    job["pct"], job["msg"] = max(job.get("pct", 0), int(pct)), msg


def app_job(name):
    """화면이 2초마다 묻는 작업 진행 상황. 끝나면 새 화면 상태를 함께 보냄."""
    j = JOBS.get(name)
    if not j:
        return JSON({"ok": True, "state": "none"})
    out = {"ok": True, "state": j["state"], "pct": j["pct"], "msg": j["msg"], "sec": int(time.time() - j["t0"]), "error": j.get("error"),
           "data": j.get("data")}
    if j["state"] != "running":
        out["app"] = app_state()
    return JSON(out)


# ---------------------------------------------------------------
# 그림 도우미: 화면에 보낼 영상을 데이터 주소로
# ---------------------------------------------------------------
def to_data_url(img, fmt="JPEG"):
    """PIL 영상 → 데이터 주소 (JPEG는 RGB로 바꿔 저장)."""
    buf = io.BytesIO()
    (img.convert("RGB") if fmt.upper() == "JPEG" else img).save(buf, format=fmt, quality=90)
    return f"data:image/{fmt.lower()};base64," + base64.b64encode(buf.getvalue()).decode()


def _img_url(img, width=None):
    """화면에 보낼 영상: 폭을 맞춰(픽셀이 보이게 NEAREST) JPEG 데이터 주소로."""
    if width and img.size[0] != width:
        from PIL import Image
        img = img.resize((width, max(1, round(width * img.size[1] / img.size[0]))), Image.NEAREST)
    return to_data_url(img)


CALLBACKS["app.job"] = app_job
# ===== 참고자료(사진): 인터넷에서 받지 않음. 제작자가 따로 준 사진을 학습자가 폴더에 넣고, 블록의 "asset:번호"로 씀 =====
# 명세의 project.assets: {"folder"(사진을 넣을 폴더 이름), "items"(번호·파일·제목·저작자·라이선스)}
# 정답은 여기에 없음. 사진은 노트북에 담지 않고 빌드 도구가 <코드>_images 폴더·zip으로 따로 내줌.
import shutil, zipfile

ASSET_CFG = _P.get("assets") or {}
ASSET_ITEMS = {it["id"]: it for it in ASSET_CFG.get("items", [])}
ASSET_STATE = {"ok": not ASSET_ITEMS, "msg": ""}
_ASSET_URL_CACHE = {}


def asset_folder():
    """사진을 넣는 폴더: Colab은 /content/<folder>, 점검 도구는 PBL_ASSET_LOCAL(자료 폴더)이나 PBL_ASSET_ROOT(임시 폴더)."""
    local = os.environ.get("PBL_ASSET_LOCAL")
    if local:
        return local
    root = os.environ.get("PBL_ASSET_ROOT") or ("/content" if os.path.isdir("/content") else os.getcwd())
    return os.path.join(root, ASSET_CFG.get("folder") or "pbl_photos")


def _asset_missing():
    folder = asset_folder()
    return [it["file"] for it in ASSET_ITEMS.values() if not os.path.exists(os.path.join(folder, it["file"]))]


def _extract_zips(folder):
    """폴더에 넣은 zip을 풀어 사진을 폴더 바로 아래로 모음 (zip 안에 폴더가 한 겹 있어도 됨)."""
    for name in os.listdir(folder):
        if not name.lower().endswith(".zip"):
            continue
        tmp = os.path.join(folder, "_unzip_tmp")
        shutil.rmtree(tmp, ignore_errors=True)
        with zipfile.ZipFile(os.path.join(folder, name)) as z:
            z.extractall(tmp)
        for dirpath, _, files in os.walk(tmp):
            for f in files:
                shutil.move(os.path.join(dirpath, f), os.path.join(folder, f))
        shutil.rmtree(tmp, ignore_errors=True)


def assets_prepare():
    """사진 확인: 폴더를 만들어 두고, 넣어 둔 zip이 있으면 풀고, 빠진 사진을 ASSET_STATE에 남김. 다시 불러도 안전함."""
    if not ASSET_ITEMS:
        return True
    folder = asset_folder()
    os.makedirs(folder, exist_ok=True)
    try:
        _extract_zips(folder)
    except Exception as e:
        ASSET_STATE.update(ok=False, msg=f"zip을 풀지 못했어요({type(e).__name__}). 사진 파일을 그대로 넣어 주세요.")
        return False
    missing = _asset_missing()
    _ASSET_URL_CACHE.clear()
    if missing:
        ASSET_STATE.update(ok=False, msg=f"참고자료(사진) {len(missing)}개가 아직 없어요. 왼쪽 📁 파일 패널에서 {folder} 폴더에 넣어 주세요.")
        return False
    ASSET_STATE.update(ok=True, msg="참고자료(사진)를 확인했어요.")
    return True


def assets_status():
    """화면에 보낼 사진 상태: 필요 없는 프로젝트면 None."""
    if not ASSET_ITEMS:
        return None
    return {"ok": ASSET_STATE["ok"], "folder": asset_folder(), "missing": _asset_missing(), "total": len(ASSET_ITEMS), "msg": ASSET_STATE["msg"]}


def app_assets_check():
    """화면의 [다시 확인] 버튼: 사진을 넣은 뒤 다시 찾음."""
    ok = assets_prepare()
    return _resp(ok=ok, msg=ASSET_STATE["msg"])


def asset_path(aid):
    """참고자료 파일 경로 (없으면 None). 직접 만든 활동에서 원본 파일을 읽을 때 씀 (예: 출처 정보 확인)."""
    it = ASSET_ITEMS.get(str(aid))
    if not it:
        return None
    p = os.path.join(asset_folder(), it["file"])
    return p if os.path.exists(p) else None


def asset_data_url(aid, width=960):
    """화면에 보낼 이미지: 폭을 맞춘 JPEG 데이터 주소 (한 번 만들면 캐시). PIL이 없으면 파일을 그대로 보냄."""
    key = (str(aid), int(width))
    if key not in _ASSET_URL_CACHE:
        path = asset_path(aid)
        try:
            from PIL import Image
            im = Image.open(path).convert("RGB")
            if im.size[0] > width:
                im = im.resize((width, max(1, round(width * im.size[1] / im.size[0]))), Image.LANCZOS)
            _ASSET_URL_CACHE[key] = to_data_url(im)
        except ImportError:
            mime = "png" if path.lower().endswith(".png") else "jpeg"
            with open(path, "rb") as f:
                _ASSET_URL_CACHE[key] = f"data:image/{mime};base64," + base64.b64encode(f.read()).decode()
    return _ASSET_URL_CACHE[key]


def app_asset(aid, width=960):
    """화면의 이미지 요청 ("asset:번호"). 화면 상태 전체는 보내지 않음 (가볍게)."""
    it = ASSET_ITEMS.get(str(aid))
    if not it:
        return JSON({"ok": False, "msg": f"없는 참고자료예요: {aid}"})
    if not asset_path(aid):
        return JSON({"ok": False, "missing": True, "file": it["file"], "folder": asset_folder(),
                     "msg": f"사진을 찾지 못했어요: {os.path.join(asset_folder(), it['file'])}"})
    try:
        return JSON({"ok": True, "url": asset_data_url(aid, max(240, min(int(width), 2400))), "caption": ""})
    except Exception as e:
        return JSON({"ok": False, "msg": f"이미지를 열지 못했어요: {type(e).__name__}"})


def _asset_refs(obj, out=None):
    """명세 안에서 "asset:번호"를 모두 찾음 (산출물의 출처 목록용)."""
    out = [] if out is None else out
    if isinstance(obj, str) and obj.startswith("asset:"):
        if obj[6:] not in out:
            out.append(obj[6:])
    elif isinstance(obj, (list, tuple)):
        for x in obj:
            _asset_refs(x, out)
    elif isinstance(obj, dict):
        for x in obj.values():
            _asset_refs(x, out)
    return out


def asset_credits_html(ids):
    """산출물 문서에 붙는 '사용한 자료와 출처' 표 (주소 없이 이름만: 답이 새지 않게, 링크를 달지 않게)."""
    rows = []
    for aid in ids:
        it = ASSET_ITEMS.get(aid)
        if not it:
            continue
        rows.append(f"<tr><td>{html.escape(aid)}</td><td>{html.escape(it.get('title', ''))}</td><td>{html.escape(it.get('credit', ''))}</td>"
                    f"<td>{html.escape(it.get('license', ''))}</td><td>{html.escape(it.get('changes', ''))}</td></tr>")
    if not rows:
        return ""
    return ("<h2>사용한 자료와 출처</h2><table><tr><th>번호</th><th>자료</th><th>저작자·제작</th><th>라이선스</th><th>변경</th></tr>"
            + "".join(rows) + "</table>")


CALLBACKS.update({"app.asset": app_asset, "app.assets_check": app_assets_check})
_setup(30, 1, "메신저 · 진행 상태 · 가이드북 준비 완료")

# ===== 미션 단계 엔진: 명세(SPEC)의 블록으로 미션을 진행하고, 채점·힌트·산출물을 만듦 =====
# 블록 종류: info(읽기) · quiz(확인 질문) · multi(여러 개 고르기) · match(짝짓기) · order(순서 맞추기)
#            classify(사례 분류) · write(글쓰기) · decide(받아들이기/거절 결정) · vibe(바이브 코딩 실습) · custom(직접 만든 활동)
# 원칙 (시행착오에서 정한 것)
#  - 정답·풀이·힌트는 파이썬에만 둠. 화면에는 학습자가 맞힌 뒤에만 풀이를 보냄
#  - 틀리면 힌트를 한 단계씩 자세하게. 맞혀도 화면이 저절로 넘어가지 않음 (학습자가 [다음 단계]를 누름)
#  - 처음부터 차례로 하는 것이 원칙이지만, 뒷부분을 미리 보려는 사람을 위해 잠그지 않음 (2026-11-05 피드백)
#  - 산출물은 언제든 받을 수 있음: 안 한 활동이 있으면 먼저 알려 주고 물어본 뒤, 그래도 받으면 그 부분은 빈칸으로 만듦
#  - 선생님용(FILL_ANSWERS)은 모든 활동의 값이 채워진 상태로 시작함
#  - 글쓰기(write)는 정답을 판정하지 않음: 최소 글자 수만 보고 제출을 받은 뒤 모범답안(example)을 보여 줌
CUSTOM_VIEWS = {}     # 미션 아이디 → 화면에 더 보낼 상태를 돌려주는 함수 (직접 만든 활동용)
CUSTOM_RESET = {}     # 미션 아이디 → 처음 상태를 만드는 함수
CUSTOM_DOCS = {}      # (미션, 블록) → 산출물 문서에 넣을 HTML을 돌려주는 함수
AUTOPLAY = {}         # (미션, 블록) → 점검 도구가 이 활동을 끝까지 해 보는 함수 (직접 만든 활동용)
DECIDE_OPTS = [["accept", "받아들임"], ["reject", "거절하고 대안 제시"]]
MSPEC = {m["id"]: m for m in SPEC["missions"]}
BSPEC = {}            # (미션, 블록) → (단계 번호, 블록 명세)
for _m in SPEC["missions"]:
    for _si, _st in enumerate(_m.get("steps", []), 1):
        for _b in _st.get("blocks", []):
            BSPEC[(_m["id"], _b["id"])] = (_si, _b)


def _required(b):
    """필수 블록인가: 읽기(info)는 confirm이 있을 때만, 나머지는 required가 False가 아니면 필수."""
    if b["type"] == "info":
        return bool(b.get("confirm"))
    return b.get("required", True)


def _bst(mid, bid):
    return S[mid]["blk"].setdefault(bid, {"ok": False, "tries": 0, "value": None, "item_tries": {}})


def _step_blocks(mid, si):
    st = MSPEC[mid]["steps"][si - 1]
    return st.get("blocks", [])


def _step_done(mid, si):
    """단계 완료: 필수 블록을 모두 마침 (블록 없이 직접 만든 화면(page)은 그 코드가 step_ok로 알림)."""
    st = MSPEC[mid]["steps"][si - 1]
    if st.get("page"):
        return bool(S[mid].get("pages", {}).get(si))
    return all(_bst(mid, b["id"])["ok"] for b in _step_blocks(mid, si) if _required(b))


def _todo_list(mid):
    """아직 안 한 필수 활동: [{"step": 단계 번호, "short": 단계 이름, "blocks": [활동 이름…]}]."""
    out = []
    for si, st in enumerate(MSPEC[mid]["steps"], 1):
        if st.get("page"):
            if not _step_done(mid, si):
                out.append({"step": si, "short": st["short"], "blocks": [st.get("title") or st["short"]]})
            continue
        left = [b.get("title") or BLOCK_NAMES.get(b["type"], "활동") for b in st.get("blocks", []) if _required(b) and not _bst(mid, b["id"])["ok"]]
        if left:
            out.append({"step": si, "short": st["short"], "blocks": left})
    return out


def _mission_fraction(mid):
    """끝낸 필수 활동의 비율 (0~1)."""
    done = total = 0
    for si, st in enumerate(MSPEC[mid]["steps"], 1):
        if st.get("page"):
            total += 1
            done += _step_done(mid, si)
            continue
        for b in st.get("blocks", []):
            if _required(b):
                total += 1
                done += _bst(mid, b["id"])["ok"]
    return done / total if total else 0.0


def _update_done(mid):
    """필수 활동을 다 하면 미션 완료 표시."""
    S["progress"][mid]["done"] = not _todo_list(mid)


def step_ok(mid, si):
    """직접 만든 화면(page) 단계를 마쳤다고 알림."""
    S[mid].setdefault("pages", {})[si] = True
    _advance(mid, si + 1)
    _update_done(mid)


def block_ok(mid, bid, value=None):
    """블록을 마쳤다고 기록: 처음 마쳤을 때 팀원 메시지(on_ok)를 보내고, 진행 상태를 고침."""
    si, b = BSPEC[(mid, bid)]
    st = _bst(mid, bid)
    first = not st["ok"]
    st["ok"] = True
    if value is not None:
        st["value"] = value
    if first:
        for ch, who, text in b.get("on_ok", []):
            post(ch, who, text)
    if _step_done(mid, si):
        _advance(mid, si + 1)
    _update_done(mid)


WRONG_MSG = "❌ 현재 선택을 다시 고민해 보세요."      # 틀린 선택 안내: 화면에서 빨간색으로 보임 (2026-11-05 피드백 "주황색 말고 빨간색으로")


def _hint(spec, tries, key="hints"):
    hs = spec.get(key) or ["다시 생각해 보세요."]
    k = max(1, min(tries, len(hs)))
    return f"{WRONG_MSG} 힌트 {k}/{len(hs)}: {hs[k - 1]}"


# ---- 블록마다 채점: (맞음?, 저장할 값, 화면에 보낼 것) ----
def _chk_info(mid, b, st, payload):
    return True, True, {"text": "✅ 확인했어요."}


def _chk_quiz(mid, b, st, payload):
    try:
        pick = int(payload)
    except (TypeError, ValueError):
        pick = -1
    if pick == b["answer"]:
        return True, {"pick": pick}, {"text": "✅ 맞아요. " + b["explain"]}
    st["tries"] += 1
    return False, None, {"text": _hint(b, st["tries"])}


def _chk_multi(mid, b, st, payload):
    picked = {int(i) for i in (payload or []) if str(i).lstrip("-").isdigit()}
    res = []
    for i, it in enumerate(b["items"]):
        ok = (i in picked) == bool(it["correct"])
        res.append({"i": i, "ok": ok, "text": it.get("explain", "") if ok else "❌ " + it.get("hint", "다시 생각해 보세요.")})
    wrong = sum(not r["ok"] for r in res)
    if wrong:
        st["tries"] += 1
        return False, None, {"items": res, "text": f"{WRONG_MSG} 다시 볼 항목이 {wrong}개 있어요."}
    return True, {"picked": sorted(picked)}, {"items": res, "text": "✅ 모두 맞게 골랐어요."}


def _chk_match(mid, b, st, payload):
    payload = payload or {}
    res = {p["left"]: str(payload.get(p["left"])) == str(p["right"]) for p in b["pairs"]}
    wrong = [p for p in b["pairs"] if not res[p["left"]]]
    if not wrong:
        return True, {p["left"]: p["right"] for p in b["pairs"]}, {"result": res, "hints": [], "text": "✅ 모두 맞게 짝지었어요."}
    st["tries"] += 1
    n, gen = st["tries"], b.get("hints") or ["틀린 칸을 다시 살펴보세요."]
    hints = [gen[0]]
    if n >= 2:
        hints += [f"‘{p['left']}’: {p['hint']}" for p in wrong if p.get("hint")]
    if n >= 3:
        hints += gen[1:]
    return False, None, {"result": res, "hints": hints, "text": f"{WRONG_MSG} 빨간 칸 {len(wrong)}개가 맞지 않아요."}


def _chk_order(mid, b, st, payload):
    right = [it["id"] for it in b["items"]]
    order = [o for o in (payload or []) if o in right]
    if order == right:
        return True, {"order": order}, {"text": "✅ 순서가 맞아요.", "wrong": None}
    if len(order) != len(right):
        return False, None, {"text": "모든 항목을 한 번씩 놓아 주세요.", "wrong": None}
    st["tries"] += 1
    n = st["tries"]
    i = next(k for k, (a, c) in enumerate(zip(order, right)) if a != c)
    it = b["items"][i]
    text = f"❌ 현재 순서를 다시 고민해 보세요. {i + 1}번째 자리가 맞지 않아요. " + (b.get("hints") or ["앞 단계의 결과가 다음 단계의 재료가 되는지 따져 보세요."])[0]
    if n >= 2 and it.get("hint"):
        text += f" 힌트: {it['hint']}"
    if n >= 3:
        text += f" ({i + 1}번째 자리: ‘{it['text']}’)"
    return False, None, {"text": text, "wrong": i}


def _chk_classify(mid, b, st, payload):
    payload = payload or {}
    it = next((x for x in b["items"] if x["id"] == payload.get("item")), None)
    if not it:
        return False, None, {"text": "항목을 찾지 못했어요.", "item": None}
    names = {c["id"]: c["name"] for c in b["categories"]}
    val = dict(st["value"] or {})
    prim = payload.get("primary")
    contrib = [c for c in (payload.get("contrib") or []) if c in names and c != prim]
    if prim in it["accept"]:
        val[it["id"]] = {"primary": prim, "contrib": contrib}
        miss = [names[c] for c in it.get("contrib", []) if c not in contrib and c != prim]
        text = "✅ " + it["explain"] + (f" (함께 생각해 볼 것: {', '.join(miss)})" if miss else "")
        return all(x["id"] in val for x in b["items"]), val, {"text": text, "item": it["id"], "item_ok": True}
    t = st["item_tries"][it["id"]] = st["item_tries"].get(it["id"], 0) + 1
    return False, val, {"text": _hint(it, t), "item": it["id"], "item_ok": False}


def _chk_write(mid, b, st, payload):
    """글쓰기: 정답을 판정하지 않음. 최소 글자 수를 채우면 제출을 받고 모범답안(example)을 보여 줌.
    (학습자가 뜻은 맞게 써도 규칙(정규식)에는 틀릴 수 있어서, 맞고 틀림을 따지지 않기로 함: 2026-10-19 피드백)"""
    text = (payload or "").strip()
    n = b.get("min", 10)
    if len(text) < n:
        return False, None, {"text": f"✏ {n}자 이상 써 주세요. (지금 {len(text)}자)"}
    return True, text, {"text": "✅ 제출했어요.", "model": b.get("example", ""), "explain": b.get("explain", ""), "checklist": b.get("checklist", [])}


def _chk_decide(mid, b, st, payload):
    payload = payload or {}
    ch, rs = payload.get("choices") or {}, payload.get("reasons") or {}
    opts = [o[0] for o in b.get("options", DECIDE_OPTS)]
    if any(ch.get(it["id"]) not in opts for it in b["items"]):
        return False, None, {"text": "항목마다 하나를 골라 주세요.", "items": []}
    rmin = b.get("reason_min", 10)
    short = [it["id"] for it in b["items"] if len((rs.get(it["id"]) or "").strip()) < rmin]
    if short:
        return False, None, {"text": f"이유를 {rmin}자 이상 적어 주세요. (남은 항목 {len(short)}개)", "items": []}
    wrong = {it["id"] for it in b["items"] if ch[it["id"]] != it["answer"]}
    items = [{"id": it["id"], "ok": it["id"] not in wrong,
              "text": it.get("explain", "") if it["id"] not in wrong else "❌ " + it.get("hint", "다시 생각해 보세요.")} for it in b["items"]]
    if wrong:
        st["tries"] += 1
        return False, None, {"text": f"{WRONG_MSG} 다시 볼 항목이 {len(wrong)}개 있어요.", "items": items}
    return True, {it["id"]: {"choice": ch[it["id"]], "reason": rs[it["id"]].strip()} for it in b["items"]}, {"text": "✅ 모두 알맞게 결정했어요.", "items": items}


CHECKERS = {"info": _chk_info, "quiz": _chk_quiz, "multi": _chk_multi, "match": _chk_match, "order": _chk_order,
            "classify": _chk_classify, "write": _chk_write, "decide": _chk_decide}


def blk_check(mid, bid, payload=None):
    """화면의 [확인] 버튼: 블록 하나를 채점. correct=True면 그 블록을 마친 것 (화면은 저절로 넘어가지 않음)."""
    if (mid, bid) not in BSPEC:
        return _resp(ok=False, msg="없는 활동이에요.")
    si, b = BSPEC[(mid, bid)]
    _ensure_started(mid)                                            # 잠그지 않으므로 미리 열어 본 단계에서도 동작함
    fn = CHECKERS.get(b["type"])
    if not fn:
        return _resp(ok=False, msg="이 활동은 화면의 안내대로 진행해요.")
    st = _bst(mid, bid)
    ok, value, extra = fn(mid, b, st, payload)
    if value is not None:
        st["value"] = value
    if ok:
        block_ok(mid, bid)
    return _resp(ok=True, correct=bool(ok), **extra)


# ---- 산출물 문서 ----
def _esc_br(t):
    return html.escape(str(t)).replace("\n", "<br>")


def _info_html(b):
    out = "".join(f"<p>{_md(p)}</p>" for p in b.get("text", []))
    if b.get("bullets"):
        out += "<ul>" + "".join(f"<li>{_md(x)}</li>" for x in b["bullets"]) + "</ul>"
    if b.get("table"):
        t = b["table"]
        out += "<table><tr>" + "".join(f"<th>{_md(c)}</th>" for c in t["head"]) + "</tr>" + "".join(
            "<tr>" + "".join(f"<td>{_md(c)}</td>" for c in row) + "</tr>" for row in t["rows"]) + "</table>"
    if b.get("sources"):
        out += "<p class='note'>출처: " + " · ".join(f"<a href='{html.escape(u)}'>{html.escape(n)}</a>" for n, u in b["sources"]) + "</p>"
    return out


def _md(t):
    """명세 글에서 **굵게**만 지원 (나머지는 그대로 이스케이프)."""
    return re.sub(r"\*\*(.+?)\*\*", r"<b>\1</b>", html.escape(str(t)))


BLANK = "<span class='blank'></span>"
BLANKBOX = "<div class='blankbox'></div>"
BLOCK_NAMES = {"quiz": "확인 질문", "multi": "여러 개 고르기", "match": "짝짓기", "order": "순서 맞추기", "classify": "분류하기",
               "write": "글쓰기", "decide": "결정하기", "vibe": "바이브 코딩 실습", "custom": "활동", "info": "읽기"}


def _block_html(mid, b):
    """산출물에 들어가는 블록 하나. 하지 않은 활동은 답·풀이 없이 빈칸으로 (정답이 산출물로 새지 않게)."""
    st, t = _bst(mid, b["id"]), b["type"]
    v = st["value"]
    done = st["ok"]
    head = f"<h3>{html.escape(b.get('title') or '')}</h3>" if b.get("title") else ""
    if t == "info":
        return head + _info_html(b)
    if t == "quiz":
        pick = (v or {}).get("pick")
        if not done or pick is None:
            return head + f"<p><b>Q.</b> {_md(b['text'])}</p><p>고른 답: {BLANK}</p>"
        return (head + f"<p><b>Q.</b> {_md(b['text'])}</p><p>고른 답: {_md(b['options'][pick])}</p>"
                f"<p class='note'>풀이: {_md(b['explain'])}</p>")
    if t == "multi":
        if not done:
            return head + f"<p>{_md(b['text'])}</p><p>고른 항목: {BLANK}</p>"
        return head + f"<p>{_md(b['text'])}</p><ul>" + "".join(
            f"<li>{'✅ 해당' if it['correct'] else '⬜ 해당 없음'} · {_md(it['text'])}<br><span class='note'>{_md(it.get('explain', ''))}</span></li>" for it in b["items"]) + "</ul>"
    if t == "match":
        return head + "<table>" + "".join(f"<tr><th>{_md(p['left'])}</th>" + (f"<td>{_md(p.get('right_label', p['right']))}</td>" if done else "<td class='empty'></td>") + "</tr>" for p in b["pairs"]) + "</table>"
    if t == "order":
        if not done:
            return head + "<ol>" + "".join(f"<li>{BLANK}</li>" for _ in b["items"]) + "</ol>"
        return head + "<ol>" + "".join(f"<li><b>{_md(it['text'])}</b>{' (' + _md(it['desc']) + ')' if it.get('desc') else ''}</li>" for it in b["items"]) + "</ol>"
    if t == "classify":
        names = {c["id"]: c["name"] for c in b["categories"]}
        rows = ""
        for it in b["items"]:
            z = (v or {}).get(it["id"])
            if z:
                rows += (f"<tr><td>{_md(it['title'])}</td><td><b>{names.get(z.get('primary'), '')}</b></td>"
                         + ("" if b.get("contrib") is False else f"<td>{', '.join(names[c] for c in z.get('contrib', []))}</td>")
                         + f"<td>{_md(it['explain'])}</td></tr>")
            else:
                rows += (f"<tr><td>{_md(it['title'])}</td><td class='empty'></td>" + ("" if b.get("contrib") is False else "<td class='empty'></td>") + "<td class='empty'></td></tr>")
        return head + (f"<table><tr><th>{_md(b.get('item_label', '사례'))}</th><th>{_md(b.get('primary_label', '직접 원인'))}</th>"
                       + ("" if b.get("contrib") is False else "<th>함께 작용한 것</th>") + f"<th>풀이</th></tr>{rows}</table>")
    if t == "write":
        return head + (f"<p class='note'>{_md(b['prompt'])}</p>" if b.get("prompt") else "") + (f"<p>{_esc_br(v)}</p>" if done and v else BLANKBOX)
    if t == "decide":
        opts = dict(b.get("options", DECIDE_OPTS))
        rows = ""
        for it in b["items"]:
            z = (v or {}).get(it["id"]) if done else None
            rows += f"<tr><td>{_md(it.get('who', ''))}</td><td>{_md(it['text'])}</td>" + (
                f"<td><b>{opts.get(z.get('choice'), '')}</b></td><td>{_esc_br(z.get('reason', ''))}</td>" if z else "<td class='empty'></td><td class='empty'></td>") + "</tr>"
        return head + f"<table><tr><th>누가</th><th>요청</th><th>결정</th><th>이유 · 대안</th></tr>{rows}</table>"
    if t == "vibe":
        return head + _vibe_html(mid, b, st)
    if not done:
        return head + BLANKBOX
    fn = CUSTOM_DOCS.get((mid, b["id"]))
    if fn:
        return head + fn(st)
    return head + f"<pre>{html.escape(json.dumps(v, ensure_ascii=False, indent=1))}</pre>"


def _mission_docs(mid, blanks=0):
    """산출물 문서: 명세의 outputs대로 (없으면 미션 전체를 문서 하나로). blanks는 비어 있는 활동 수."""
    m = MSPEC[mid]
    outs = m.get("outputs") or [{"id": f"{mid}_report", "title": f"M{m['no']} {m['short']} 결과",
                                 "sections": [{"h": f"{si}. {st['short']}", "blocks": [b["id"] for b in st.get("blocks", [])]}
                                              for si, st in enumerate(m["steps"], 1)]}]
    credits = asset_credits_html(_asset_refs(m)) if ASSET_ITEMS and not blanks else ""      # 이 미션에 쓴 사진의 출처 (미션을 마친 뒤 공개)
    for k, o in enumerate(outs):
        body = f"<p class='meta'>{_md(o['intro'])}</p>" if o.get("intro") else ""
        for sec in o.get("sections", []):
            body += f"<h2>{html.escape(sec['h'])}</h2>" if sec.get("h") else ""
            body += "".join(f"<p>{_md(p)}</p>" for p in sec.get("text", []))
            for bid in sec.get("blocks", []):
                body += _block_html(mid, BSPEC[(mid, bid)][1])
        if k == len(outs) - 1:
            body += credits
        _add_file(o["id"], o["title"], mid, _doc(o["title"], body), blanks)


def blk_todo(mid):
    """[산출물] 화면: 아직 안 한 활동 목록."""
    return JSON({"ok": True, "todo": _todo_list(mid)})


def blk_finish(mid, force=False):
    """[산출물 받기]: 안 한 활동이 있으면 먼저 알려 주고 물어봄(need_confirm). 그래도 받으면(force) 안 한 부분은 빈칸으로 만듦."""
    m = MSPEC[mid]
    _ensure_started(mid)
    todo = _todo_list(mid)
    if todo and not force:
        return _resp(ok=False, need_confirm=True, todo=todo)
    n_blank = sum(len(t["blocks"]) for t in todo)
    _mission_docs(mid, n_blank)
    p = S["progress"][mid]
    first = not p["issued"] and not todo
    if not todo:
        p["issued"] = True
    p["done"] = not todo
    if first:
        for ch, who, text in m.get("finish_messages", []):
            post(ch, who, text)
    msg = (f"M{m['no']} 산출물이 📁 파일에 저장됐어요." if not todo
           else f"M{m['no']} 산출물을 저장했어요. 하지 않은 활동 {n_blank}개는 빈칸이에요. 마저 하고 다시 받으면 채워져요.")
    return _resp(ok=True, msg=msg, blanks=n_blank)


def _prefill_block(mid, b):
    """선생님용: 블록 하나를 정답으로 채움."""
    t, st = b["type"], _bst(mid, b["id"])
    if t == "info":
        if b.get("confirm"):
            st["value"] = True
            block_ok(mid, b["id"])
        return
    if t == "quiz":
        st["value"] = {"pick": b["answer"]}
    elif t == "multi":
        st["value"] = {"picked": [i for i, it in enumerate(b["items"]) if it["correct"]]}
    elif t == "match":
        st["value"] = {p["left"]: str(p["right"]) for p in b["pairs"]}
    elif t == "order":
        st["value"] = {"order": [it["id"] for it in b["items"]]}
    elif t == "classify":
        st["value"] = {it["id"]: {"primary": it["accept"][0], "contrib": list(it.get("contrib", []))} for it in b["items"]}
    elif t == "write":
        st["value"] = b.get("example", "")
    elif t == "decide":
        st["value"] = {it["id"]: {"choice": it["answer"], "reason": it.get("reason") or it.get("explain") or ""} for it in b["items"]}
    elif t == "vibe":
        _vibe_prefill(mid, b)
    elif t == "custom":
        fn = AUTOPLAY.get((mid, b["id"]))
        if fn:
            fn()
        return
    block_ok(mid, b["id"])


def _prefill_all():
    """선생님용: 모든 미션의 모든 활동(심화 포함)을 채우고 산출물까지 만들어 둠. 빈 값이 없는 상태."""
    for mid in IMPLEMENTED:
        _ensure_started(mid)
        for si, st in enumerate(MSPEC[mid]["steps"], 1):
            if st.get("page") and (mid, f"page{si}") in AUTOPLAY:
                AUTOPLAY[(mid, f"page{si}")]()
            for b in st.get("blocks", []):
                _prefill_block(mid, b)
        blk_finish(mid, True)


# ---- 미션 등록: 소개 · 처음 상태 · 시작 메시지 · 화면 상태 ----
def _pub_state(mid, b):
    """화면에 보낼 블록 상태: 맞힌 뒤에만 풀이를 함께 보냄 (정답은 보내지 않음)."""
    st = _bst(mid, b["id"])
    out = {"ok": st["ok"], "tries": st["tries"], "value": st["value"]}
    if st["ok"]:
        if b["type"] == "quiz":
            out["explain"] = b["explain"]
        elif b["type"] == "multi":
            out["explains"] = [it.get("explain", "") for it in b["items"]]
        elif b["type"] == "decide":
            out["explains"] = {it["id"]: it.get("explain", "") for it in b["items"]}
        elif b["type"] in ("write", "match", "order") and b.get("explain"):
            out["explain"] = b["explain"]
        if b["type"] == "write":
            out["model"] = b.get("example", "")      # 모범답안: 제출한 뒤에만 보냄
    if b["type"] == "classify":
        out["explains"] = {it["id"]: it["explain"] for it in b["items"] if it["id"] in (st["value"] or {})}
    if b["type"] == "write" and st["ok"]:
        out["checklist"] = b.get("checklist", [])
    return out


def _mission_reset(mid):
    S[mid] = {"blk": {}, "pages": {}, "vibe": {}}
    if mid in CUSTOM_RESET:
        CUSTOM_RESET[mid]()


def _mission_view(mid):
    m = MSPEC[mid]
    out = {"blk": {b["id"]: _pub_state(mid, b) for st in m["steps"] for b in st.get("blocks", [])},
           "complete": [_step_done(mid, si) for si in range(1, len(m["steps"]) + 1)]}
    if mid in CUSTOM_VIEWS:
        out["custom"] = CUSTOM_VIEWS[mid]()
    return out


def _mission_start(mid):
    for ch, who, text in MSPEC[mid].get("start_messages", []):
        post(ch, who, text)


for _m in SPEC["missions"]:
    if not _m.get("steps"):
        continue
    BRIEFINGS[_m["id"]] = dict(_m["brief"], steps_short=[st["short"] for st in _m["steps"]],
                               comps=[{"name": c["name"], "criteria": c.get("criteria", "")} for c in SPEC.get("competencies", [])
                                      if c.get("id") in _m.get("competencies", [])])
    RESETTERS[_m["id"]] = (lambda mid: lambda: _mission_reset(mid))(_m["id"])
    VIEWS[_m["id"]] = (lambda mid: lambda: _mission_view(mid))(_m["id"])
    STARTERS[_m["id"]] = (lambda mid: lambda: _mission_start(mid))(_m["id"])

CALLBACKS.update({"blk.check": blk_check, "blk.finish": blk_finish, "blk.todo": blk_todo})
code_topic("app.blocks", "미션 활동은 이렇게 채점해요", "[확인]을 누르면 blk_check()가 명세의 정답과 비교해요. 틀리면 힌트를 한 단계씩 자세하게 보내고, "
           "맞히면 풀이를 보낸 뒤 단계를 다 마쳤는지 확인해요. 글쓰기는 맞고 틀림을 따지지 않고, 제출하면 모범답안을 보내요. 정답은 파이썬에만 있고 화면으로는 보내지 않아요.",
           ["blk_check", "_chk_quiz", "_chk_match", "_chk_write", "block_ok"], marks=["_hint(", "block_ok(", "\"model\"", "_advance("])
code_topic("app.docs", "산출물 문서는 이렇게 만들어요", "산출물을 받으면 명세의 outputs 순서대로, 학습자가 답한 내용을 모아 HTML 문서를 만들어요. 하지 않은 활동은 빈칸이에요.",
           ["blk_finish", "_mission_docs", "_block_html"], marks=["_add_file(", "_doc(", "finish_messages"])
# ===== 바이브 코딩 실습: 가상 데이터 파일 · 가상 AI (인터넷·외부 AI 없이 동작) =====
# 가상 데이터: 명세의 project.vfiles {파일 이름: 글}을 첫 실행 때 폴더에 저장함. 내려받는 것이 없음
# 가상 AI: 학습자가 고른 프롬프트 문장에 따라 명세에 적힌 응답을 돌려줌. 응답의 코드는 실제로 실행해 결과·오류를 그대로 보여 줌
#  - 정답(어느 문장이 좋은지)·응답은 파이썬에만 둠. 화면에는 보낸 뒤의 응답만 감
#  - 시도 기록이 남아 산출물의 '프롬프트 개선 기록'이 됨
import contextlib

VFILES = _P.get("vfiles") or {}
_VIBE_CACHE = {}


def vdata_folder():
    """가상 데이터 폴더: Colab은 /content/pbl_data, 점검 도구는 PBL_VDATA_ROOT(임시 폴더)."""
    root = os.environ.get("PBL_VDATA_ROOT") or ("/content" if os.path.isdir("/content") else os.getcwd())
    return os.path.join(root, _P.get("vfolder") or "pbl_data")


def vdata_prepare():
    """가상 데이터 파일을 폴더에 저장 (코드를 복사해 새 셀에서 돌려도 같은 파일을 읽게)."""
    if not VFILES:
        return
    folder = vdata_folder()
    os.makedirs(folder, exist_ok=True)
    for name, text in VFILES.items():
        with open(os.path.join(folder, name), "w", encoding="utf-8") as f:
            f.write(text)


def _vibe_st(mid, bid):
    return S[mid].setdefault("vibe", {}).setdefault(bid, {"picks": {}, "attempts": [], "last": None})


def _vibe_prompt(b, picks):
    """고른 문장을 이어 붙인 프롬프트."""
    lines = []
    for sl in b["slots"]:
        i = picks.get(sl["id"])
        if i is not None and 0 <= int(i) < len(sl["options"]) and sl["options"][int(i)].get("text"):
            lines.append(sl["options"][int(i)]["text"])
    return "\n".join(lines)


def _vibe_code(code):
    return (code or "").replace("{VDIR}", vdata_folder())


def _vibe_run(code):
    """AI가 만든 코드를 실제로 실행: 출력·표·그림·오류·문서(report 변수)를 모아 돌려줌. 같은 코드는 다시 돌리지 않음."""
    code = _vibe_code(code)
    if not code.strip():
        return {}
    if code in _VIBE_CACHE:
        return _VIBE_CACHE[code]
    tables, figs = [], []

    def display(obj=None, *a, **k):               # Colab의 display와 같은 이름: 복사해 새 셀에서 돌려도 그대로 동작함
        if hasattr(obj, "to_html"):
            tables.append(obj.to_html(index=False, border=0))
        else:
            print(obj)

    env = {"__name__": "__vibe__", "display": display}
    out, err = io.StringIO(), None
    cwd = os.getcwd()
    plt, prev = None, None
    if "matplotlib" in code:
        try:
            import matplotlib.pyplot as plt
            prev = plt.get_backend()
            plt.switch_backend("Agg")
        except Exception:
            plt = None
    try:
        os.makedirs(vdata_folder(), exist_ok=True)
        os.chdir(vdata_folder())
        with contextlib.redirect_stdout(out):
            exec(compile(code, "AI가 만든 코드", "exec"), env)
    except Exception as e:
        tb = traceback.extract_tb(e.__traceback__)
        frame = next((f for f in reversed(tb) if f.filename == "AI가 만든 코드"), None)
        err = "Traceback (most recent call last):\n"
        if frame:
            src = code.splitlines()[frame.lineno - 1].strip()
            err += f'  File "AI가 만든 코드", line {frame.lineno}\n    {src}\n'
        err += f"{type(e).__name__}: {e}"
    finally:
        os.chdir(cwd)
        if plt is not None:
            for n in plt.get_fignums():
                buf = io.BytesIO()
                plt.figure(n).savefig(buf, format="png", dpi=110, bbox_inches="tight")
                figs.append("data:image/png;base64," + base64.b64encode(buf.getvalue()).decode())
            plt.close("all")
            try:
                plt.switch_backend(prev)
            except Exception:
                pass
    report = env.get("report") if isinstance(env.get("report"), str) else None
    res = {"stdout": out.getvalue()[-6000:], "tables": tables, "figs": figs, "report": report, "error": err}
    _VIBE_CACHE[code] = res
    return res


def _vibe_reply(rep, ok, slot_label=None, slot_id=None):
    """명세의 응답 + 실제 실행 결과 → 화면에 보낼 것."""
    out = {k: rep.get(k, "") for k in ("say", "problem", "hint", "lesson")}
    out.update(code=_vibe_code(rep.get("code", "")), ok=ok, slot=slot_label, slot_id=slot_id, open_code=bool(rep.get("open_code")))
    out.update(_vibe_run(rep.get("code", "")))
    return out


def _chk_vibe(mid, b, st, payload):
    """프롬프트 보내기: 고른 문장을 앞 칸부터 보고, 부족한 칸이 있으면 그 칸의 응답을, 모두 좋으면 done 응답을 돌려줌."""
    picks = {k: int(v) for k, v in ((payload or {}).get("picks") or {}).items() if str(v).lstrip("-").isdigit()}
    if any(sl["id"] not in picks for sl in b["slots"]):
        return False, None, {"text": "칸마다 문장을 하나씩 골라 주세요."}
    v = _vibe_st(mid, b["id"])
    v["picks"] = picks
    miss = None
    for sl in b["slots"]:
        opt = sl["options"][picks[sl["id"]]] if 0 <= picks[sl["id"]] < len(sl["options"]) else {}
        if not opt.get("good"):
            miss = (sl, opt)
            break
    prompt = _vibe_prompt(b, picks)
    if miss:
        sl, opt = miss
        rep = _vibe_reply(opt.get("miss") or sl["miss"], False, sl["label"], sl["id"])
        st["tries"] += 1
    else:
        rep = _vibe_reply(b["done"], True)
    v["last"] = rep
    v["attempts"].append({"no": len(v["attempts"]) + 1, "prompt": prompt, "ok": rep["ok"], "slot": rep["slot"],
                          "result": re.sub(r"\*\*", "", rep["problem"]) or "요청대로 동작함"})
    resp = {"reply": rep, "attempts": [{k: a[k] for k in ("no", "ok", "slot", "result")} for a in v["attempts"]], "text": ""}
    if rep["ok"]:
        return True, {"picks": picks, "prompt": prompt}, resp
    return False, None, resp


def app_vibe(mid, bid):
    """화면이 블록을 다시 그릴 때 마지막 응답을 받아 감 (상태 전체에 넣으면 그림 때문에 무거워져서 따로 보냄)."""
    v = _vibe_st(mid, bid)
    b = BSPEC[(mid, bid)][1]
    start = _vibe_reply(b["start"], False) if b.get("start") else None      # 지금까지의 코드와 결과: 블록 맨 위에 보여 줌
    return JSON({"ok": True, "reply": v["last"], "picks": v["picks"], "start": start,
                 "attempts": [{k: a[k] for k in ("no", "ok", "slot", "result")} for a in v["attempts"]]})


def _vibe_prefill(mid, b):
    """선생님용: 좋은 문장을 고른 상태 + done 응답."""
    picks = {sl["id"]: next(i for i, o in enumerate(sl["options"]) if o.get("good")) for sl in b["slots"]}
    st = _bst(mid, b["id"])
    ok, value, _ = _chk_vibe(mid, b, st, {"picks": picks})
    st["value"] = value
    return ok


def _vibe_html(mid, b, st):
    """산출물 문서: 최종 프롬프트 · 개선 기록 · 실행 결과 · AI가 만든 문서. 하지 않았으면 빈칸."""
    v = _vibe_st(mid, b["id"])
    if not st["ok"]:
        out = "<h4>최종 프롬프트</h4><div class='blankbox'></div><h4>실행 결과</h4><div class='blankbox'></div>"
    else:
        rep = v["last"] if (v["last"] or {}).get("ok") else _vibe_reply(b["done"], True)
        out = f"<h4>최종 프롬프트</h4><pre>{html.escape((st['value'] or {}).get('prompt', ''))}</pre>"
        if rep.get("stdout"):
            out += f"<h4>실행 결과</h4><pre>{html.escape(rep['stdout'])}</pre>"
        elif rep.get("tables") or rep.get("figs"):
            out += "<h4>실행 결과</h4>"
        out += "".join(rep.get("tables") or [])
        out += "".join(f"<p><img src='{u}'></p>" for u in rep.get("figs") or [])
        if rep.get("report"):
            out += f"<h4>AI가 만든 문서</h4><div class='card'>{rep['report']}</div>"
        if rep.get("code"):
            out += f"<details><summary>AI가 만든 코드 보기</summary><pre>{html.escape(rep['code'])}</pre></details>"
    if v["attempts"]:
        rows = "".join(f"<tr><td>{a['no']}</td><td>{'성공' if a['ok'] else html.escape(a['slot'] or '')}</td><td>{html.escape(a['result'])}</td></tr>" for a in v["attempts"])
        out += f"<h4>프롬프트 개선 기록</h4><table><tr><th>시도</th><th>부족했던 칸</th><th>결과</th></tr>{rows}</table>"
    return out


CHECKERS["vibe"] = _chk_vibe
CALLBACKS["app.vibe"] = app_vibe
vdata_prepare()
_setup(60, 2, "미션 활동 · 가상 AI 준비 완료")

# ===== 시작 준비: 화면 버튼과 파이썬 함수 연결 =====
_reset_state()
for _name, _fn in CALLBACKS.items():
    output.register_callback(_name, _fn)

_setup(100, 3, "✅ 준비 완료! 아래 <b>💬 위성분석 TF 열기</b> 셀을 실행하세요. (‘모두 실행’을 눌렀다면 곧 자동으로 열려요)")

## 💬 위성분석 TF 메신저
위쪽 **☰ 버튼**으로 왼쪽 메뉴를 접고 펼 수 있고, **화면 폭 · 높이** 버튼이나 오른쪽 아래 모서리를 끌어 크기를 바꿀 수 있어요.
오른쪽 **📘 본 과정의 가이드북**은 **[닫기 ✕] 버튼**으로 접고, 오른쪽 가장자리의 탭으로 다시 열 수 있어요.

In [ ]:
#@title 💬 위성분석 TF 열기 { display-mode: "form" }
#@markdown 위의 **🚀 [먼저 실행] 실습 준비하기** 셀의 진행 막대가 100%가 된 뒤 이 셀을 실행하면, 아래에 메신저 화면이 열립니다.
from IPython.display import HTML, display

WORKSPACE_NAME = "🛰️ 위성분석 TF"

APP_HTML = r"""
<style>
  .ws { --ink:#111827; --line:#c3cad5; --soft:#f2f5f9; --blue:#0b57d0; --blue-bg:#e8f0fe; --green:#146c2e; --green-bg:#e6f4ea;
        --red:#b3261e; --red-bg:#fce8e6; --amber:#8a4b00; --amber-bg:#fff4ce; --violet:#3f0e40;
        color-scheme: light; font-family: 'Noto Sans KR', 'Malgun Gothic', 'Apple SD Gothic Neo', sans-serif;
        border: 1px solid var(--line); border-radius: 10px; box-sizing: border-box; overflow: hidden; background: #fff; color: var(--ink);
        font-size: 14px; display: flex; flex-direction: column; height: 900px; width: 100%; max-width: 100%;
        min-width: 360px; min-height: 480px; resize: both; position: relative; }
  .ws * { box-sizing: border-box; color: inherit; }
  .ws.w-base { width: 1100px; } .ws.w-wide { width: 1400px; } .ws.w-full { width: 100%; }
  .ws button, .ws input, .ws select, .ws textarea { font-family: inherit; }
  .ws a { color: var(--blue); }
  .ws h2, .ws h3, .ws h4, .ws h5, .ws p, .ws li, .ws label, .ws td, .ws th, .ws dt, .ws dd, .ws summary { color: var(--ink); }
  .ws ::placeholder { color: #1e40af; opacity: .85; font-style: italic; }
  /* 상단 바 */
  .ws .topbar { display: flex; align-items: center; gap: 8px; padding: 6px 10px; background: var(--violet); color: #fff; font-size: 13px; flex-wrap: wrap; }
  .ws .topbar * { color: #fff; }
  .ws .topbar .name { font-weight: bold; margin: 0 4px; font-size: 14px; }
  .ws .topbar .spacer { flex: 1; }
  .ws .tb { background: transparent; border: 1px solid #b89bb9; border-radius: 6px; padding: 2px 9px; cursor: pointer; font-size: 13px; position: relative; }
  .ws .tb:hover { background: #5a2a5b; }
  .ws .tb.on { background: #fff; border-color: #fff; } .ws .tb.on, .ws .tb.on * { color: var(--violet); }
  .ws .tb .count { position: absolute; top: -6px; right: -8px; background: #e01e5a; border-radius: 9px; padding: 0 5px; font-size: 11px; display: none; }
  .ws.side-off .tb .count.has { display: inline; }
  .ws .tprog { display: flex; align-items: center; gap: 6px; font-size: 12px; }
  .ws .tprog .pb { width: 120px; height: 9px; background: #6b3d6c; border-radius: 6px; overflow: hidden; }
  .ws .tprog .pb i { display: block; height: 100%; background: #6ee7a8; width: 0; transition: width .4s; }
  .ws .me-chip { background: #5a2a5b; border: 1px solid #b89bb9; border-radius: 14px; padding: 2px 10px; cursor: pointer; font-size: 12.5px; }
  .ws .me-chip:hover { background: #6d3a6e; }
  /* 본문 */
  .ws .body { display: flex; flex: 1; min-height: 0; position: relative; }
  .ws .side { width: 236px; background: #3f0e40; color: #fff; padding: 8px 0 14px; overflow-y: auto; flex-shrink: 0; transition: width .2s, padding .2s; }
  .ws .side * { color: #fff; }
  .ws.side-off .side, .ws.onboard .side { width: 0; padding: 0; }
  .ws .side h4 { margin: 14px 14px 4px; font-size: 12.5px; font-weight: normal; line-height: 1.4; border-top: 1px solid #6d3a6e; padding-top: 10px; }
  .ws .side h4 b { font-weight: bold; }
  .ws .item { padding: 5px 14px; cursor: pointer; display: flex; align-items: center; gap: 6px; white-space: nowrap; font-size: 13.5px; }
  .ws .item span.t { overflow: hidden; text-overflow: ellipsis; flex: 1; }
  .ws .item:hover { background: #5a2a5b; }
  .ws .item.active { background: #1164a3; }
  .ws .item.unread { font-weight: bold; }
  .ws .item.soon span.t { font-style: italic; }
  .ws .item .ic { width: 18px; text-align: center; flex-shrink: 0; }
  .ws .item .mini { font-size: 11px; background: #1164a3; border-radius: 8px; padding: 0 6px; }
  .ws .badge { background: #e01e5a; color: #fff; border-radius: 9px; padding: 0 7px; font-size: 12px; }
  .ws .main { flex: 1; display: flex; flex-direction: column; min-width: 0; }
  .ws .title { padding: 10px 16px; font-weight: bold; border-bottom: 1px solid var(--line); font-size: 15px; }
  .ws .title small { font-weight: normal; margin-right: 6px; }
  .ws .scroll { flex: 1; overflow-y: auto; padding: 14px 18px 30px; min-height: 0; }
  .ws .footer { display: flex; gap: 10px; align-items: center; padding: 7px 14px; background: var(--soft); border-top: 1px solid var(--line); }
  .ws .note { flex: 1; font-size: 13px; } .ws .note.ok { color: var(--green); } .ws .note.bad { color: var(--red); font-weight: bold; }
  .ws.busy .note::before { content: '⏳ '; }
  /* 코드 보기 */
  .ws .codep { width: 0; overflow: hidden; border-left: 1px solid var(--line); background: #fbfcfe; display: flex; flex-direction: column; flex-shrink: 0; }
  .ws.code-on .codep { width: 410px; }
  .ws.compact.code-on .codep { position: absolute; right: 0; top: 0; bottom: 0; width: min(430px, 94%); z-index: 20; box-shadow: -8px 0 18px rgba(0,0,0,.22); }
  .ws.onboard .codep, .ws.onboard .code-tab, .ws.no-code .codep, .ws.no-code .code-tab, .ws.no-code #tb-code { display: none; }
  .ws .code-tab { position: absolute; right: 0; top: 30%; z-index: 10; writing-mode: vertical-rl; background: #1e293b; color: #fff; border: 0;
                  border-radius: 8px 0 0 8px; padding: 12px 6px; cursor: pointer; font-size: 13px; letter-spacing: 1px; box-shadow: -2px 2px 8px rgba(0,0,0,.2); }
  .ws .code-tab * { color: #fff; }
  .ws.code-on .code-tab { display: none; }
  .ws .codep .ch { display: flex; align-items: center; gap: 6px; padding: 9px 12px; background: #1e293b; }
  .ws .codep .ch, .ws .codep .ch * { color: #fff; }
  .ws .codep .ch b { flex: 1; font-size: 14px; }
  .ws .codep .ch button { background: transparent; border: 1px solid #94a3b8; border-radius: 6px; cursor: pointer; padding: 1px 8px; }
  .ws .codep .cs { flex: 1; overflow-y: auto; padding: 10px 12px 24px; }
  .ws .codep .intro { font-size: 12.5px; line-height: 1.55; background: var(--blue-bg); border-radius: 8px; padding: 8px 10px; margin-bottom: 10px; }
  .ws .codep h6 { font-size: 13px; margin: 12px 0 4px; display: flex; align-items: center; gap: 6px; }
  .ws .codep h6 .sp { flex: 1; }
  .ws .codep .desc { font-size: 12.5px; line-height: 1.5; margin: 0 0 6px; }
  .ws .codep details { border: 1px solid var(--line); border-radius: 8px; margin: 6px 0; background: #fff; }
  .ws .codep details > summary { padding: 6px 8px; cursor: pointer; font-size: 12.5px; color: var(--ink); }
  .ws .codep details > summary code { font-weight: bold; color: #6d28d9; }
  .ws .cb { display: flex; border-top: 1px solid #e5e9f0; font: 12px/18px Consolas, 'D2Coding', Menlo, monospace; }
  .ws .cb .ln { margin: 0; padding: 8px 6px; text-align: right; background: #eef2f7; user-select: none; min-width: 30px; }
  .ws .cb .cw { position: relative; overflow-x: auto; flex: 1; }
  .ws .cb pre { margin: 0; padding: 8px 10px; white-space: pre; position: relative; z-index: 1; font: inherit; }
  .ws .cb .hlb { position: absolute; left: 0; right: 0; height: 18px; background: #fff3b0; z-index: 0; }
  .ws .code .k { color: #7c3aed; font-weight: bold; } .ws .code .s { color: #b3261e; } .ws .code .c { color: #146c2e; font-style: italic; }
  .ws .code .n { color: #9a3412; } .ws .code .f { color: #0b57d0; }
  .ws .live pre { background: #0f172a; border-radius: 8px; padding: 10px; margin: 0; white-space: pre-wrap; word-break: break-all; font: 12px/1.6 Consolas, 'D2Coding', Menlo, monospace; }
  .ws .live pre, .ws .live pre * { color: #f8fafc; }
  .ws .live pre .c { color: #86efac; font-style: italic; } .ws .live pre .s { color: #fca5a5; } .ws .live pre .k { color: #c4b5fd; font-weight: bold; }
  .ws .live pre .n { color: #fdba74; } .ws .live pre .f { color: #93c5fd; }
  .ws .live .empty { font-size: 12.5px; padding: 8px 10px; border: 1px dashed var(--line); border-radius: 8px; }
  .ws .cpy { font-size: 11.5px; padding: 1px 8px; border-radius: 6px; border: 1px solid var(--line); background: #fff; cursor: pointer; }
  /* 공통 요소 */
  .ws .btn { padding: 7px 14px; border-radius: 7px; border: 1px solid #007a5a; background: #007a5a; color: #fff; cursor: pointer; font-size: 14px; font-weight: bold; }
  .ws .btn * { color: #fff; }
  .ws .btn:hover { filter: brightness(1.08); }
  .ws .btn.ghost { background: #fff; } .ws .btn.ghost, .ws .btn.ghost * { color: #006048; }
  .ws .btn.small { padding: 3px 10px; font-size: 12.5px; }
  .ws .btn.big { padding: 10px 20px; font-size: 15px; }
  .ws .btn.blue { background: var(--blue); border-color: var(--blue); }
  .ws .btn:disabled { background: #fff; border: 1px dashed #6b7280; cursor: not-allowed; font-weight: normal; }
  .ws .btn:disabled, .ws .btn:disabled * { color: var(--ink); }
  .ws .btn.pulse { animation: pulse 1.6s infinite; }
  @keyframes pulse { 0% { box-shadow: 0 0 0 0 rgba(11,87,208,.55); } 70% { box-shadow: 0 0 0 9px rgba(11,87,208,0); } 100% { box-shadow: 0 0 0 0 rgba(11,87,208,0); } }
  .ws .row { display: flex; flex-wrap: wrap; gap: 8px; align-items: center; margin: 8px 0; }
  .ws .card { border: 1px solid var(--line); border-radius: 10px; padding: 12px 14px; margin: 10px 0; background: #fff; }
  .ws .card.focus { border: 2px solid var(--blue); box-shadow: 0 0 0 3px var(--blue-bg); }
  .ws .card h4 { margin: 0 0 6px; font-size: 15px; }
  .ws .ok { color: var(--green); } .ws .bad { color: var(--red); }
  .ws .fb { margin-top: 8px; padding: 8px 10px; border-radius: 7px; font-size: 13.5px; line-height: 1.6; }
  .ws .fb.ok { background: var(--green-bg); color: #0b3d1a; border: 1px solid #9fd5ad; }
  .ws .fb.bad { background: var(--red-bg); color: #5c0f0a; border: 1px solid #efa9a3; }
  .ws .fb.info { background: var(--blue-bg); color: #0a2a66; border: 1px solid #a8c3f0; }
  .ws .fb.ok *, .ws .fb.bad *, .ws .fb.info * { color: inherit; }
  .ws .err { color: var(--red); font-size: 13px; font-weight: bold; margin-top: 4px; }
  .ws .err:empty { display: none; }
  .ws .opt-tag { display: inline-block; background: var(--blue-bg); border-radius: 4px; font-size: 11px; padding: 0 5px; margin-left: 4px; }
  .ws .missing { outline: 3px solid var(--red); outline-offset: 2px; border-radius: 6px; }
  .ws select, .ws input[type=text], .ws input[type=date], .ws input[type=time], .ws input[type=number], .ws textarea {
    padding: 6px 8px; font-size: 13.5px; background: #fff; color: var(--ink); border: 1px solid #8a94a6; border-radius: 6px; max-width: 100%; }
  .ws textarea { width: 100%; min-height: 56px; line-height: 1.5; }
  .ws select option, .ws select optgroup { color: var(--ink); background: #fff; }
  .ws .count-line { font-size: 12.5px; margin-top: 2px; }
  .ws .count-line.short { color: var(--red); font-weight: bold; } .ws .count-line.enough { color: var(--green); font-weight: bold; }
  .ws table.t { border-collapse: collapse; width: 100%; font-size: 13px; margin: 6px 0; }
  .ws table.t th, .ws table.t td { border-bottom: 1px solid #dfe4ea; padding: 6px 8px; text-align: left; vertical-align: middle; }
  .ws table.t th { font-weight: bold; background: var(--soft); }
  .ws table.t tr.split td { background: var(--amber-bg); }
  .ws .lbl-unk { font-style: italic; text-decoration: underline dotted; } .ws .lbl-wrong { color: var(--red); font-weight: bold; }
  .ws details > summary { cursor: pointer; color: var(--blue); margin: 6px 0; font-weight: bold; }
  .ws .hint { font-size: 13px; line-height: 1.55; }
  .ws .tip { background: var(--soft); border-left: 4px solid var(--blue); border-radius: 6px; padding: 8px 12px; font-size: 13.5px; line-height: 1.6; margin: 8px 0; }
  /* 할 일 상자 */
  /* 단계 안내 */
  .ws .sub { border-left: 4px solid #cbd5e1; padding: 6px 0 6px 12px; margin: 10px 0; }
  .ws .sub.now { border-left-color: var(--blue); background: linear-gradient(90deg, var(--blue-bg), #fff 70%); border-radius: 0 8px 8px 0; }
  .ws .sub.done { border-left-color: var(--green); }
  .ws .sub-h { font-weight: bold; margin-bottom: 6px; font-size: 14px; }
  .ws .look { background: #fff; border: 1px dashed var(--blue); border-radius: 8px; padding: 7px 10px; line-height: 1.6; }
  /* 선택지 */
  .ws .choices { display: flex; flex-wrap: wrap; gap: 6px; }
  .ws .choices.col { flex-direction: column; align-items: stretch; }
  .ws .choice { display: flex; align-items: flex-start; gap: 7px; border: 1px solid #8a94a6; border-radius: 8px; padding: 6px 10px; cursor: pointer; background: #fff; font-size: 13.5px; line-height: 1.45; }
  .ws .choice:hover { border-color: var(--blue); background: #f5f8ff; }
  .ws .choice.on { border: 2px solid var(--blue); background: var(--blue-bg); font-weight: bold; padding: 5px 9px; }
  .ws .choice input { margin: 2px 0 0; accent-color: var(--blue); }
  .ws .choice small { display: block; font-weight: normal; font-size: 12px; }
  .ws .choice.good { border-color: var(--green); background: var(--green-bg); } .ws .choice.wrong { border-color: var(--red); background: var(--red-bg); }
  .ws .checks { display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 6px; }
  .ws .fieldrow { margin: 10px 0; } .ws .fieldrow > .fl { font-weight: bold; margin-bottom: 4px; display: block; }
  /* 메신저 */
  .ws .msgs { flex: 1; overflow-y: auto; padding: 8px 16px; min-height: 0; }
  .ws .msg { display: flex; gap: 10px; padding: 7px 6px; border-radius: 6px; }
  .ws .msg:hover { background: var(--soft); }
  .ws .msg.mention { background: #fff4ce; border-left: 3px solid #f0b429; }
  .ws .avatar { width: 34px; height: 34px; border-radius: 6px; display: flex; align-items: center; justify-content: center; font-weight: bold; flex-shrink: 0; }
  .ws .avatar, .ws .avatar * { color: #fff; }
  .ws .meta { font-size: 13px; } .ws .meta b { margin-right: 4px; }
  .ws .time { font-size: 12px; margin-left: 6px; }
  .ws .text { white-space: pre-wrap; word-break: break-word; margin-top: 2px; line-height: 1.55; }
  .ws .tag { font-size: 11.5px; border-radius: 4px; padding: 1px 6px; margin-left: 4px; white-space: nowrap; background: #e9edf3; border: 1px solid var(--line); }
  .ws .tag.ext { background: #f3e8fd; border-color: #d7b8f5; }
  .ws .compose { display: flex; gap: 8px; padding: 10px 14px; border-top: 1px solid var(--line); }
  .ws .compose input { flex: 1; padding: 8px; font-size: 14px; }
  /* 첫 화면 */
  .ws .onb { max-width: 560px; margin: 26px auto; border: 2px solid var(--blue); border-radius: 14px; padding: 22px 24px; background: #fff; }
  .ws .onb h2 { margin: 0 0 6px; font-size: 21px; }
  .ws .onb .lead { line-height: 1.6; margin: 0 0 14px; }
  .ws .onb select, .ws .onb input { width: 100%; font-size: 15px; padding: 8px 10px; }
  .ws .preview-call { background: var(--blue-bg); border-radius: 8px; padding: 8px 12px; margin-top: 12px; }
  /* 홈 */
  .ws .hero { background: linear-gradient(135deg, #3f0e40, #1164a3); border-radius: 12px; padding: 18px 20px; }
  .ws .hero, .ws .hero * { color: #fff; }
  .ws .hero h2 { margin: 4px 0 8px; font-size: 21px; }
  .ws .hero p { margin: 0; line-height: 1.65; font-size: 13.5px; }
  .ws .greet { font-size: 16px; font-weight: bold; margin: 14px 0 6px; }
  .ws .bigprog { display: flex; align-items: center; gap: 10px; }
  .ws .bigprog .pb { flex: 1; height: 14px; background: #e3e8ef; border-radius: 8px; overflow: hidden; }
  .ws .bigprog .pb i { display: block; height: 100%; background: linear-gradient(90deg, #16a34a, #22c55e); }
  .ws .stages { display: grid; grid-template-columns: repeat(auto-fill, minmax(250px, 1fr)); gap: 12px; margin-top: 10px; }
  .ws .stage { border: 1px solid var(--line); border-radius: 10px; padding: 10px 12px; }
  .ws .stage h5 { margin: 0 0 8px; font-size: 13.5px; }
  .ws .mcard { display: flex; gap: 8px; align-items: flex-start; padding: 7px 8px; border-radius: 8px; cursor: pointer; border: 1px solid transparent; }
  .ws .mcard:hover { background: var(--soft); border-color: var(--line); }
  .ws .pill { font-size: 11.5px; padding: 1px 8px; border-radius: 10px; white-space: nowrap; border: 1px solid transparent; }
  .ws .pill.done { background: var(--green-bg); border-color: #9fd5ad; } .ws .pill.open { background: var(--blue-bg); border-color: #a8c3f0; }
  .ws .pill.locked { background: #fff; border: 1px dashed #6b7280; } .ws .pill.soon { background: var(--amber-bg); border-color: #e3c96b; }
  /* 미션 */
  .ws .stepper { display: flex; flex-wrap: wrap; gap: 6px; padding: 8px 16px; border-bottom: 1px solid var(--line); background: #fafbfd; }
  .ws .stp { font-size: 12.5px; padding: 4px 10px; border-radius: 14px; border: 1px solid #8a94a6; background: #fff; cursor: pointer; }
  .ws .stp.done { border-color: var(--green); background: var(--green-bg); }
  .ws .stp.cur, .ws .stp.cur.done { background: var(--blue); border-color: var(--blue); font-weight: bold; }
  .ws .stp.cur, .ws .stp.cur * { color: #fff; }
  .ws .stp:disabled { cursor: not-allowed; border-style: dashed; background: #fff; }
  .ws .brief h2 { margin: 6px 0 2px; font-size: 21px; }
  .ws .brief .sub-t { margin: 0 0 10px; font-size: 15px; }
  .ws .brief section { margin: 16px 0; }
  .ws .brief h4 { margin: 0 0 6px; font-size: 15px; }
  .ws .brief p, .ws .brief li { line-height: 1.7; }
  .ws .why { background: var(--soft); border-left: 4px solid #1164a3; border-radius: 6px; padding: 10px 14px; }
  .ws .terms { display: grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap: 12px; }
  .ws .term { border: 1px solid var(--line); border-radius: 10px; padding: 10px 12px; background: #fff; }
  .ws .term h5 { margin: 0 0 6px; font-size: 14.5px; } .ws .term h5 small { font-weight: normal; font-size: 11.5px; margin-left: 4px; }
  .ws .term dl { margin: 0; display: grid; grid-template-columns: 74px 1fr; gap: 4px 8px; font-size: 13px; line-height: 1.55; }
  .ws .term dt { font-weight: bold; } .ws .term dd { margin: 0; }
  .ws .term dt.k1 { color: var(--blue); } .ws .term dt.k2 { color: var(--green); } .ws .term dt.k3 { color: #9a3412; }
  .ws .fig { background: #f8fafc; border: 1px solid #e3e8ef; border-radius: 8px; margin: 4px 0 8px; padding: 4px; }
  .ws .fig svg { display: block; width: 100%; height: auto; max-height: 170px; }
  /* M1 실험실 */
  .ws .lab2 { display: grid; grid-template-columns: minmax(0, 1fr) minmax(0, 1fr); gap: 12px; }
  .ws .main.narrow .lab2 { grid-template-columns: minmax(0, 1fr); }
  .ws .labcol { border: 1px solid var(--line); border-radius: 10px; padding: 10px; min-width: 0; background: #fff; }
  .ws .labcol.focus { border: 2px solid var(--blue); box-shadow: 0 0 0 3px var(--blue-bg); }
  .ws .labcol h5 { margin: 0; font-size: 14px; } .ws .labcol .cap { font-size: 12.5px; margin: 2px 0 6px; }
  .ws .ctl { display: grid; grid-template-columns: 92px minmax(0, 1fr) 54px; align-items: center; gap: 6px; margin: 3px 0; font-size: 13px; border-radius: 6px; padding: 1px 3px; }
  .ws .ctl input[type=range], .ws .ctl select { width: 100%; min-width: 0; }
  .ws .ctl output { text-align: right; font-weight: bold; }
  .ws .ctl.flash { animation: flash 1.8s ease-out; }
  @keyframes flash { 0% { background: #fde047; } 100% { background: transparent; } }
  .ws .imgbox { position: relative; display: block; line-height: 0; margin-top: 8px; }
  .ws .imgbox img { width: 100%; display: block; border-radius: 6px; border: 1px solid var(--line); }
  .ws .imgbox.flash img { animation: imgflash 1.4s ease-out; }
  @keyframes imgflash { 0% { box-shadow: 0 0 0 4px #fde047; } 100% { box-shadow: 0 0 0 0 transparent; } }
  .ws .mk { position: absolute; transform: translate(-50%, -50%); background: rgba(255,255,255,.92); border: 1px solid #111827;
            border-radius: 10px; font-size: 11px; line-height: 16px; padding: 0 5px; pointer-events: none; font-weight: bold; }
  .ws .mk.cur { background: #fde047; border: 2px solid #b3261e; font-size: 13px; line-height: 18px; z-index: 2; }
  .ws .north { position: absolute; right: 6px; top: 6px; background: rgba(255,255,255,.92); border-radius: 4px; font-size: 11px; line-height: 15px; padding: 1px 4px; font-weight: bold; }
  .ws .info { margin-top: 8px; font-size: 13px; background: var(--soft); border-radius: 6px; padding: 6px 10px; line-height: 1.65; }
  .ws .legend { font-size: 12.5px; }
  .ws .qtabs { display: flex; flex-wrap: wrap; gap: 6px; margin: 6px 0 10px; }
  .ws .qhead { display: flex; align-items: center; gap: 8px; } .ws .qhead .sp { flex: 1; }
  .ws .qtext { font-size: 15px; line-height: 1.6; margin: 4px 0 8px; }
  .ws .split { display: grid; grid-template-columns: minmax(220px, 300px) minmax(0, 1fr); gap: 14px; align-items: start; }
  .ws .main.narrow .split { grid-template-columns: minmax(0, 1fr); }
  .ws .gsdlist .gr { display: flex; align-items: center; gap: 8px; padding: 7px 10px; border: 1px solid var(--line); border-radius: 8px; margin-bottom: 6px; cursor: pointer; background: #fff; }
  .ws .gsdlist .gr.cur { border: 2px solid var(--blue); background: var(--blue-bg); font-weight: bold; }
  .ws .gsdlist .gr.locked { cursor: not-allowed; border-style: dashed; }
  .ws .gsdlist .gr .sp { flex: 1; }
  .ws figure.ch { margin: 0; text-align: center; font-size: 12.5px; }
  .ws figure.ch img { width: 200px; max-width: 100%; border-radius: 6px; border: 1px solid var(--line); display: block; }
  .ws .ecard .imgs { display: flex; gap: 10px; flex-wrap: wrap; }
  .ws .ecard .imgs figure { margin: 0; text-align: center; font-size: 12.5px; }
  .ws .ecard .imgs img { width: 220px; max-width: 100%; border-radius: 6px; border: 1px solid var(--line); display: block; }
  /* M2 */
  .ws .strip { display: flex; flex-wrap: wrap; gap: 5px; margin: 6px 0 10px; }
  .ws .strip button { min-width: 38px; padding: 3px 7px; border-radius: 8px; border: 1px solid #8a94a6; background: #fff; cursor: pointer; font-size: 13px; }
  .ws .strip button.done { background: var(--green-bg); border-color: var(--green); }
  .ws .strip button.cur { background: var(--blue); border-color: var(--blue); font-weight: bold; } .ws .strip button.cur, .ws .strip button.cur * { color: #fff; }
  .ws .lab-l { display: grid; grid-template-columns: minmax(0, 1.3fr) minmax(260px, 1fr); gap: 14px; align-items: start; }
  .ws .main.narrow .lab-l { grid-template-columns: minmax(0, 1fr); }
  .ws .lab-l > .stick { position: sticky; top: 0; z-index: 3; background: #fff; padding-bottom: 4px; }
  .ws .main.narrow .lab-l > .stick { position: static; }
  .ws .ov { position: relative; display: block; line-height: 0; }
  .ws .ov img { width: 100%; border-radius: 8px; border: 1px solid var(--line); }
  .ws .chipbig img { width: 100%; max-width: 300px; border-radius: 8px; border: 1px solid var(--line); display: block; }
  .ws .tiles { display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr)); gap: 10px; }
  .ws .tile { border: 1px solid var(--line); border-radius: 8px; padding: 10px 12px; }
  .ws .tile .v { font-size: 24px; font-weight: bold; margin: 2px 0; }
  .ws .tile .d { font-size: 12.5px; line-height: 1.45; }
  .ws .acc { border: 1px solid var(--line); border-radius: 10px; margin: 8px 0; background: #fff; }
  .ws .acc .acc-h { display: flex; align-items: center; gap: 8px; padding: 9px 12px; font-weight: bold; }
  .ws .acc .acc-h .sp { flex: 1; }
  .ws .acc.locked .acc-h { border-style: dashed; }
  .ws .acc.open { border: 2px solid var(--blue); box-shadow: 0 0 0 3px var(--blue-bg); }
  .ws .acc .acc-b { padding: 0 12px 12px; }
  .ws .acc .acc-s { padding: 0 12px 10px; font-size: 13px; line-height: 1.55; }
  /* M3 · M4 */
  .ws .post { background: #fff; border: 1px solid var(--line); border-left: 4px solid #e01e5a; border-radius: 8px; padding: 9px 12px; font-size: 14.5px; line-height: 1.6; }
  .ws .kv { display: grid; grid-template-columns: auto 1fr; gap: 3px 12px; font-size: 13.5px; }
  .ws .kv b { white-space: nowrap; }
  .ws .raw { font: 12.5px/1.7 Consolas, 'D2Coding', Menlo, monospace; background: #0f172a; border-radius: 8px; padding: 10px 12px; overflow-x: auto; }
  .ws .raw, .ws .raw * { color: #e2e8f0; } .ws .raw .rk { color: #93c5fd; }
  .ws .jsonbox { font: 12px/1.6 Consolas, 'D2Coding', Menlo, monospace; background: #f8fafc; border: 1px solid var(--line); border-radius: 8px; padding: 10px; white-space: pre; overflow-x: auto; }
  .ws .sunres { background: #fff7ed; border: 1px solid #fdba74; border-radius: 8px; padding: 8px 12px; line-height: 1.7; }
  .ws .calc { display: flex; flex-wrap: wrap; gap: 8px; align-items: flex-end; }
  .ws .calc label { display: flex; flex-direction: column; font-size: 12.5px; font-weight: bold; gap: 2px; }
  .ws .calc input { width: 130px; }
  .ws .casebox { position: relative; line-height: 0; }
  .ws .casebox img { width: 100%; border-radius: 8px; border: 1px solid var(--line); }
  .ws .casebox svg { position: absolute; left: 0; top: 0; width: 100%; height: 100%; pointer-events: none; }
  .ws .casebox.small { width: 58%; flex-shrink: 0; } .ws .casebox.small .mk { font-size: 10px; line-height: 14px; }
  .ws .imgcap { font-size: 12.5px; font-weight: bold; margin: 4px 0; line-height: 1.4; }
  .ws .track { border: 1px solid var(--line); border-radius: 10px; padding: 10px; min-width: 0; }
  .ws .track h5 { margin: 0 0 6px; font-size: 14.5px; }
  .ws .stage-r { border: 1px solid #dfe4ea; border-radius: 8px; padding: 8px; margin: 6px 0; }
  .ws .stage-r.next { border: 2px solid var(--blue); background: #f7faff; }
  .ws .stage-r .sh { display: flex; align-items: center; gap: 8px; font-weight: bold; } .ws .stage-r .sh .sp { flex: 1; }
  .ws .stage-r img { width: 100%; max-height: 300px; object-fit: contain; background: #0f172a; border-radius: 6px; border: 1px solid var(--line); margin-top: 6px; display: block; }
  .ws .stage-r .st { font-size: 13px; line-height: 1.55; margin-top: 4px; }
  .ws .pipe { display: flex; flex-wrap: wrap; align-items: center; gap: 6px; font-size: 13px; margin: 6px 0; }
  .ws .pipe span.box { border: 1px solid var(--blue); background: var(--blue-bg); border-radius: 6px; padding: 3px 8px; font-weight: bold; }
  .ws .pipe span.x { border: 1px dashed var(--red); background: var(--red-bg); border-radius: 6px; padding: 3px 8px; text-decoration: line-through; }
  .ws .ordrow { display: grid; grid-template-columns: minmax(0, 1fr) 130px; gap: 8px; align-items: center; padding: 7px 0; border-bottom: 1px solid #e5e9f0; }
  .ws .ordrow b { display: block; } .ws .ordrow .d { font-size: 12.5px; }
  .ws .chk-chip { display: inline-block; font-size: 12px; border-radius: 10px; padding: 1px 7px; margin: 2px 3px 2px 0; border: 1px solid; }
  .ws .chk-chip.y { background: var(--green-bg); border-color: #9fd5ad; } .ws .chk-chip.n { background: var(--red-bg); border-color: #efa9a3; font-weight: bold; }
  .ws .tilerow { display: grid; grid-template-columns: 250px minmax(0, 1fr) 150px; gap: 12px; align-items: start; padding: 10px 0; border-bottom: 1px solid #e5e9f0; }
  .ws .main.narrow .tilerow { grid-template-columns: minmax(0, 1fr); }
  .ws .tilerow img { width: 100%; max-width: 250px; border-radius: 6px; border: 1px solid var(--line); display: block; cursor: zoom-in; image-rendering: pixelated; }
  .ws .file { border: 1px solid var(--line); border-radius: 8px; padding: 10px 12px; margin-bottom: 10px; }
  .ws .preview { width: 100%; height: 480px; border: 1px solid var(--line); border-radius: 8px; background: #fff; }
  /* 3단계 · 힌트 · 작업 진행 · 그래프 */
  .ws .jobbar { border: 1px solid var(--line); border-radius: 8px; padding: 8px 10px; margin: 8px 0; background: #f8fafc; }
  .ws .jobbar.err { border-color: #efa9a3; background: var(--red-bg); }
  .ws .jobbar .jb-top { display: flex; gap: 8px; align-items: center; font-size: 13px; flex-wrap: wrap; } .ws .jobbar .sp { flex: 1; }
  .ws .jobbar .pb { height: 10px; background: #e3e8ef; border-radius: 6px; overflow: hidden; margin-top: 6px; }
  .ws .jobbar .pb i { display: block; height: 100%; background: linear-gradient(90deg, #0b57d0, #38bdf8); transition: width .5s; }
  .ws .hintbox { border: 1px dashed #d4a72c; background: #fffbea; border-radius: 8px; padding: 6px 10px; margin: 6px 0; font-size: 13px; }
  .ws .hintbox .hb-h { display: flex; align-items: center; gap: 8px; } .ws .hintbox .sp { flex: 1; }
  .ws .hintbox .hb-i { margin-top: 5px; line-height: 1.55; }
  .ws .raw .rl { border-radius: 4px; padding: 0 3px; } .ws .raw .rl.hl { background: #a16207; } .ws .raw .rl.hl, .ws .raw .rl.hl * { color: #fffbea; }
  .ws .srcline { font-size: 12.5px; margin: 2px 0 4px; }
  .ws .srcchip { font: 12px Consolas, 'D2Coding', Menlo, monospace; border: 1px solid var(--blue); background: var(--blue-bg); border-radius: 5px; padding: 1px 6px; cursor: pointer; }
  .ws .convrow { border-top: 1px solid #eef2f7; padding-top: 8px; }
  .ws table.t.conv td, .ws table.t.conv th { font-size: 12.5px; padding: 4px 8px; }
  .ws .params { display: flex; flex-wrap: wrap; gap: 8px 14px; align-items: center; margin: 6px 0; font-size: 13px; }
  .ws .params label { display: flex; gap: 6px; align-items: center; flex-wrap: wrap; }
  .ws .metric { margin-top: 4px; font-size: 13px; background: #ecfdf5; border: 1px solid #a7f3d0; border-radius: 6px; padding: 4px 8px; line-height: 1.5; }
  .ws .chart { margin: 6px 0; flex: 1; min-width: 260px; max-width: 440px; } .ws .chart svg { width: 100%; height: auto; display: block; }
  .ws .card.warnc { border-color: #e3c96b; background: #fffdf3; }
  .ws .grid2 { display: grid; grid-template-columns: repeat(auto-fill, minmax(330px, 1fr)); gap: 12px; }
  .ws .bar { display: inline-block; height: 9px; background: var(--blue); border-radius: 4px; vertical-align: middle; }
  .ws .stress { border-top: 1px solid #e5e9f0; padding-top: 8px; margin-top: 10px; } .ws .stress h5 { margin: 0 0 4px; font-size: 14px; }
  .ws .zoombox { position: absolute; inset: 0; background: rgba(15, 23, 42, .84); z-index: 50; display: flex; align-items: center; justify-content: center; cursor: zoom-out; }
  .ws .zoombox .zb-in { max-width: 94%; max-height: 94%; text-align: center; }
  .ws .zoombox img { max-width: 100%; max-height: 78vh; image-rendering: pixelated; border-radius: 8px; background: #fff; }
  .ws .zoombox .zb-cap, .ws .zoombox .zb-cap * { color: #fff; font-size: 13px; margin-top: 6px; }
  .ws .gal { display: grid; grid-template-columns: repeat(auto-fill, minmax(300px, 1fr)); gap: 10px; margin-top: 6px; }
  .ws .gal-i { display: grid; grid-template-columns: 1fr 1fr; gap: 4px; align-content: start; } .ws .gal-i.one { grid-template-columns: 1fr; }
  .ws .gal-i img { width: 100%; border-radius: 6px; border: 1px solid var(--line); cursor: zoom-in; display: block; }
  .ws .gal-i .hint { grid-column: 1 / -1; }
  .ws .roles { display: grid; grid-template-columns: repeat(auto-fill, minmax(150px, 1fr)); gap: 8px; margin: 6px 0; }
  .ws .role { border-radius: 8px; padding: 8px 10px; border: 1px solid; font-size: 13px; }
  .ws .role.red { background: #fef2f2; border-color: #fca5a5; } .ws .role.blue { background: #eff6ff; border-color: #93c5fd; }
  .ws .cues { display: grid; grid-template-columns: repeat(auto-fill, minmax(260px, 1fr)); gap: 8px; margin: 6px 0; }
  .ws .cue { border: 1px solid var(--line); border-radius: 8px; padding: 8px; background: #fff; }
  .ws .cue .ctl { margin-top: 4px; }
  .ws .cardgrid { display: grid; grid-template-columns: repeat(auto-fill, minmax(210px, 1fr)); gap: 8px; margin-top: 8px; }
  .ws .ccard { border: 1px solid var(--line); border-radius: 8px; padding: 6px; background: #fff; } .ws .ccard.flag { border: 2px solid #dc2626; }
  .ws .ccard img { width: 96px; height: 96px; image-rendering: pixelated; border-radius: 4px; }
  .ws tr.locked-row td { opacity: .55; }
  .ws pre.jsonbox { white-space: pre; }
  /* 모델 들여다보기 · 4단계 */
  .ws .archwrap { overflow-x: auto; border: 1px solid var(--line); border-radius: 8px; background: #f8fafc; padding: 6px; margin: 6px 0; }
  .ws .archwrap svg { display: block; }
  .ws .vnav { position: sticky; top: 0; background: #fff; z-index: 2; padding: 4px 0; } .ws .vnav .sp { flex: 1; }
  .ws .vstep { border-top: 1px solid #e5e9f0; padding-top: 8px; } .ws .vstep h5 { font-size: 15px; margin: 4px 0; }
  .ws pre.formula { background: #0f172a; border-radius: 8px; padding: 8px 10px; white-space: pre-wrap; font: 12.5px/1.6 Consolas, 'D2Coding', Menlo, monospace; }
  .ws pre.formula, .ws pre.formula * { color: #e2e8f0; }
  .ws .example { background: #f0f9ff; border: 1px solid #bae6fd; border-radius: 8px; padding: 8px 10px; margin: 8px 0; }
  .ws table.num3 { border-collapse: collapse; font: 12px Consolas, 'D2Coding', Menlo, monospace; } .ws table.num3 td { border: 1px solid #cbd5e1; padding: 3px 6px; text-align: right; background: #fff; }
  .ws table.num3.w td { background: #fef3c7; }
  .ws .example .big { font-size: 18px; font-weight: bold; } .ws .calcline { margin-top: 6px; font: 13px/1.6 Consolas, 'D2Coding', Menlo, monospace; }
  .ws .fmaps { display: flex; flex-wrap: wrap; gap: 6px; } .ws .fmaps img { width: 64px; image-rendering: pixelated; border-radius: 4px; border: 1px solid var(--line); }
  .ws .legend7 { display: flex; flex-wrap: wrap; gap: 10px; font-size: 12.5px; margin: 6px 0; } .ws .legend7 i { display: inline-block; width: 14px; height: 10px; margin-right: 4px; border-radius: 2px; }
  .ws .maptile { display: grid; grid-template-columns: minmax(0, 300px) minmax(0, 1fr); gap: 10px; border-top: 1px solid #e5e9f0; padding: 8px 0; }
  .ws .main.narrow .maptile { grid-template-columns: minmax(0, 1fr); }
  .ws .maptile img { width: 100%; border-radius: 6px; cursor: zoom-in; } .ws .maptile li { margin: 3px 0; font-size: 13px; line-height: 1.5; }
  .ws .sw { display: inline-block; width: 10px; height: 10px; border-radius: 2px; margin-right: 4px; }
  .ws .adjcard { display: grid; grid-template-columns: 170px minmax(0, 1fr); gap: 10px; border-top: 1px solid #e5e9f0; padding: 10px 0; }
  .ws .main.narrow .adjcard { grid-template-columns: minmax(0, 1fr); }
  .ws .adjcard img, .ws .biascard img { width: 160px; max-width: 100%; border-radius: 6px; image-rendering: pixelated; cursor: zoom-in; }
  .ws .biasgrid { display: grid; grid-template-columns: repeat(auto-fill, minmax(170px, 1fr)); gap: 10px; }
  .ws .biascard { border: 1px solid var(--line); border-radius: 8px; padding: 6px; }
  .ws .flow { display: flex; flex-wrap: wrap; gap: 6px; align-items: center; margin: 6px 0; font-size: 13px; }
  .ws .flow .box { border: 1px solid var(--blue); background: var(--blue-bg); border-radius: 6px; padding: 3px 8px; font-weight: bold; }
  .ws .flow .box.auto { border-color: #9ca3af; background: #f3f4f6; }
  /* 무거운 작업 중 안내 · 가이드북 · 4·5단계 */
  .ws .stay { border: 2px solid #d97706; background: #fffbeb; border-radius: 10px; padding: 10px 12px; margin: 8px 0; font-size: 13.5px; line-height: 1.55; }
  .ws .stay b { color: #92400e; display: block; margin-bottom: 2px; }
  .ws .stay.shake { animation: wsShake .5s; } @keyframes wsShake { 0%, 100% { transform: translateX(0); } 25% { transform: translateX(-6px); } 75% { transform: translateX(6px); } }
  .ws .gtabs { display: flex; gap: 4px; padding: 6px 10px 0; background: #e2e8f0; }
  .ws .gtabs button { flex: 1; border: 1px solid var(--line); border-bottom: none; border-radius: 8px 8px 0 0; background: #f1f5f9; padding: 6px 4px; font-size: 13px; cursor: pointer; color: var(--ink); }
  .ws .gtabs button.on { background: #fff; font-weight: bold; color: var(--blue); }
  .ws .codep .term { margin: 8px 0; } .ws .term.now { border: 2px solid #f59e0b; background: #fffbeb; } .ws .term.flash { box-shadow: 0 0 0 3px #fde68a; }
  .ws .term dt.k4 { color: #6d28d9; } .ws .term dt.k5 { color: #0f766e; } .ws .term dd.fx { font: 12px/1.6 Consolas, 'D2Coding', Menlo, monospace; background: #f5f3ff; border-radius: 6px; padding: 2px 6px; }
  .ws .codep .term dl { grid-template-columns: 70px 1fr; font-size: 12.5px; }
  .ws tr.wrongrow td { background: #fef2f2; }
  .ws .heatwrap { display: grid; grid-template-columns: minmax(0, 1.25fr) minmax(0, 1fr); gap: 12px; align-items: start; margin-top: 8px; }
  .ws .main.narrow .heatwrap { grid-template-columns: minmax(0, 1fr); }
  .ws .heatimg { width: 100%; border-radius: 8px; border: 1px solid var(--line); cursor: zoom-in; }
  .ws details.case { border: 1px solid var(--line); border-radius: 8px; padding: 4px 10px; margin: 8px 0; background: #fbfcfe; }
  .ws details.case > summary { color: var(--ink); }
  .ws .casefig { margin: 4px 8px 4px 0; } .ws .casefig img { width: 200px; max-width: 100%; border-radius: 6px; border: 1px solid var(--line); cursor: zoom-in; image-rendering: pixelated; }
  .ws .casefig figcaption { font-size: 12px; }
  .ws .timeline { display: grid; gap: 3px; margin: 6px 0; font-size: 13px; } .ws .timeline > div { display: grid; grid-template-columns: 70px minmax(0, 1fr); gap: 6px; }
  .ws .voices { border-top: 1px dashed var(--line); margin-top: 6px; padding-top: 6px; }
  .ws .voice { display: flex; gap: 8px; align-items: flex-start; margin: 5px 0; font-size: 13px; line-height: 1.5; }
  .ws .voice .avatar { width: 24px; height: 24px; border-radius: 6px; color: #fff; font-size: 12px; display: flex; align-items: center; justify-content: center; flex-shrink: 0; }
  .ws .actgrp { border-top: 1px solid #e5e9f0; padding: 8px 0; } .ws .actgrp small, .ws .secgrid small { display: block; color: #475569; }
  .ws .flow .fbox { border: 1px solid var(--blue); background: var(--blue-bg); border-radius: 8px; padding: 5px 8px; max-width: 150px; }
  .ws .flow .fbox small { display: block; color: #475569; } .ws .flow .fbox .hint { font-size: 11.5px; } .ws .flow .arr { font-weight: bold; color: var(--blue); }
  .ws .orditem { display: flex; gap: 8px; align-items: center; border: 1px solid var(--line); border-radius: 8px; padding: 6px 8px; margin: 4px 0; background: #fff; }
  .ws .orditem small { color: #475569; } .ws .orditem.wrong { border: 2px solid var(--red); background: #fef2f2; }
  .ws .secgrid { display: grid; grid-template-columns: repeat(auto-fill, minmax(260px, 1fr)); gap: 4px 10px; }
  .ws .lint { display: grid; gap: 4px; margin: 6px 0; } .ws .qa { border-top: 1px dashed var(--line); padding-top: 6px; margin-top: 6px; }

  .ws .gstep { margin: 8px 0 4px; } .ws .gstep-h { font-size: 12px; font-weight: bold; color: #6b21a8; background: #f5f3ff; border-radius: 6px; padding: 3px 8px; margin: 6px 0 4px; }
  .ws .term .fig { margin: 4px 0 6px; } .ws .term .fig svg { width: 100%; max-width: 360px; height: auto; display: block; }
  /* 미션 블록 (명세로 만든 단계 화면) */
  .ws .stepfoot { margin: 14px 0 8px; }
  .ws .blkimg { max-width: 100%; border-radius: 8px; border: 1px solid var(--line); cursor: zoom-in; display: block; margin: 6px 0; }
  /* 글쓰기 블록: 제출한 뒤에 보이는 모범답안 (정답 판정 없음) */
  .ws .model { margin: 10px 0 2px; padding: 10px 12px; border: 1px solid #a8c3f0; border-radius: 8px; background: var(--blue-bg); }
  .ws .model-h { font-weight: bold; color: #0a2a66; margin-bottom: 2px; }
  .ws .model p { margin: 4px 0; line-height: 1.65; }
  /* 첫 화면 (입력 없이 인사만) */
  .ws .onb.welcome { text-align: center; max-width: 560px; margin: 8vh auto 0; }
  .ws .onb.welcome .wave { font-size: 44px; }
  .ws .onb.welcome h2 { font-size: 26px; margin: 8px 0 10px; }
  .ws .onb.welcome .lead { line-height: 1.7; }
  .ws .me-chip { cursor: default; }
  /* 해외 사례 토글 */
  .ws details.case { margin-top: 8px; border: 1px solid var(--line); border-radius: 8px; background: #fff; }
  .ws details.case > summary { cursor: pointer; padding: 7px 10px; font-weight: bold; font-size: 13px; }
  .ws details.case .case-b { padding: 2px 12px 10px; }
  .ws details.case .case-b p { margin: 6px 0; line-height: 1.65; font-size: 13.5px; }
  /* 산출물 단계 */
  .ws ul.todo { margin: 6px 0; padding-left: 20px; }
  .ws ul.todo li { margin: 4px 0; line-height: 1.55; }
  /* 바이브 코딩 실습 */
  .ws .vb-slot { margin: 10px 0; padding: 8px 10px; border: 1px solid var(--line); border-radius: 8px; background: var(--soft); }
  .ws .vb-label { margin: 8px 0 4px; font-size: 13.5px; }
  .ws .vb-slot .vb-label { margin-top: 0; }
  .ws .vb-prompt { background: #fff; border: 2px solid var(--blue); border-radius: 8px; padding: 9px 11px; white-space: pre-wrap; line-height: 1.6; font-family: inherit; font-size: 13.5px; margin: 4px 0 8px; }
  .ws .vb-prompt.empty { border-style: dashed; border-color: #8a94a6; }
  .ws .vb-replybox { scroll-margin-top: 8px; }
  .ws .vb-reply { margin-top: 12px; padding: 10px 12px; border: 1px solid #a8c3f0; border-left: 5px solid var(--blue); border-radius: 8px; background: #f8fbff; }
  .ws .vb-reply.ok { border-color: #9fd5ad; border-left-color: var(--green); background: #f6fbf7; }
  .ws .vb-who { font-weight: bold; margin-bottom: 4px; }
  .ws .vb-say { line-height: 1.6; margin-bottom: 6px; }
  .ws details.vb-codebox { margin: 6px 0; }
  .ws details.vb-codebox > summary { cursor: pointer; font-size: 13px; font-weight: bold; padding: 2px 0; }
  .ws .vb-code { background: #1e2430; color: #e6edf3; border-radius: 8px; padding: 9px 11px; margin: 4px 0; overflow-x: auto; font-family: Menlo, Consolas, monospace; font-size: 12.5px; line-height: 1.5; white-space: pre; }
  .ws .vb-code span { color: #e6edf3; }
  .ws .vb-code .c { color: #8b949e; } .ws .vb-code .s { color: #a5d6ff; } .ws .vb-code .k { color: #ff9e8e; } .ws .vb-code .n { color: #79c0ff; } .ws .vb-code .f { color: #d2a8ff; }
  .ws .vb-out { background: #fff; border: 1px solid var(--line); border-radius: 8px; padding: 8px 10px; margin: 4px 0; overflow-x: auto; font-family: Menlo, Consolas, monospace; font-size: 12.5px; white-space: pre-wrap; }
  .ws .vb-err { background: var(--red-bg); border: 1px solid #e6a8a2; border-radius: 8px; padding: 8px 10px; margin: 4px 0; overflow-x: auto; font-family: Menlo, Consolas, monospace; font-size: 12.5px; white-space: pre-wrap; }
  .ws .vb-tbl { overflow-x: auto; margin: 4px 0; }
  .ws .vb-tbl table { border-collapse: collapse; background: #fff; }
  .ws .vb-tbl th, .ws .vb-tbl td { border: 1px solid var(--line); padding: 3px 8px; font-size: 12.5px; text-align: left; white-space: nowrap; }
  .ws .vb-tbl th { background: #eef2f7; }
  .ws iframe.vb-doc { width: 100%; height: 300px; border: 1px solid var(--line); border-radius: 8px; background: #fff; }
  .ws .vb-problem { margin-top: 8px; padding: 7px 10px; border-radius: 8px; background: var(--red-bg); border: 1px solid #efa9a3; color: #5c0f0a; line-height: 1.6; }
  .ws .vb-problem * { color: inherit; }
  .ws .vb-reply.miss { border-color: #efa9a3; border-left-color: var(--red); background: #fffafa; }
  .ws details.vb-data { margin: 8px 0; }
  .ws details.vb-data > summary, .ws details.vb-log > summary { cursor: pointer; font-size: 13px; font-weight: bold; padding: 2px 0; }
  .ws details.vb-log { margin-top: 10px; }
  .ws details.vb-log th { white-space: nowrap; }
  .ws details.vb-log td:first-child, .ws details.vb-log td:nth-child(2) { white-space: nowrap; }
  .ws .model-t { white-space: pre-wrap; line-height: 1.6; }
  .ws .selfcheck { margin-top: 10px; }
  .ws .selfcheck .choice { display: flex; margin: 4px 0; }
  /* 틀린 선택: 주황색 대신 빨간색으로 (2026-11-05 피드백) */
  .ws .choice.wrong, .ws .choice.on.wrong { border: 2px solid var(--red); background: var(--red-bg); padding: 5px 9px; }
  .ws .choice.wrong input { accent-color: var(--red); }
  .ws .hintbox.wrongbox { border: 1px solid #efa9a3; background: var(--red-bg); color: #5c0f0a; }
  .ws .hintbox.wrongbox * { color: inherit; }
  .ws tr.wrongrow td { background: var(--red-bg); }
  .ws tr.wrongrow select { border: 2px solid var(--red); }
  /* 부제목 */
  .ws .stage-sub { margin: -4px 0 8px; }
  .ws .msub { margin-top: 2px; }
</style>

<div class="ws" id="ws-root">
  <div class="topbar">
    <button class="tb" id="tb-side" onclick="toggleSide()" title="왼쪽 메뉴 접기/펴기">☰<span class="count" id="tb-count"></span></button>
    <span class="name">__WORKSPACE__</span>
    <span class="tprog" id="tb-prog" title="전체 미션 진행률"></span>
    <span class="spacer"></span>
    <span class="me-chip" id="tb-me"></span>
    <button class="tb" id="tb-code" onclick="toggleCode()" title="핵심 용어와 지금 화면의 코드 보기">📘 가이드북</button>
    <span>화면 폭</span>
    <button class="tb" data-w="base" onclick="setWidth('base')">기본</button>
    <button class="tb" data-w="wide" onclick="setWidth('wide')">넓게</button>
    <button class="tb" data-w="full" onclick="setWidth('full')">가득</button>
    <span>높이</span>
    <button class="tb" onclick="setHeight(-120)" title="낮게">−</button>
    <button class="tb" onclick="setHeight(120)" title="높게">+</button>
  </div>
  <div class="body">
    <div class="side" id="side"></div>
    <div class="main" id="main"></div>
    <div class="codep" id="codep"></div>
    <button class="code-tab" id="code-tab" onclick="toggleCode(true)">📘 본 과정의 가이드북 ◀</button>
  </div>
  <div class="footer">
    <span class="note" id="note"></span>
    <button class="btn ghost small" onclick="resetAll()">↺ 처음부터</button>
  </div>
</div>

<script>
// ===================== 프로젝트 설정 (만드는 도구가 넣음) =====================
const PROJ = {"id": "pbl_isr_sat_a1", "team_name": "위성분석 TF", "ui_key": "pbl-ui-pbl_isr_sat_a1", "boot_note": "🔄 파이썬이 새로 시작돼 진행 상황이 처음으로 돌아갔어요.", "welcome_lead": "가상의 위성분석 TF에 바이브 코딩 담당으로 합류했어요. 팀원들의 요청을 프롬프트로 바꿔 AI와 함께 도구와 문서를 만들어 봐요. M1부터 차례로 진행해 보세요.", "use_label": "바이브 코딩에서는", "stay_jobs": {}};   // 팀 이름 · 저장 키 · 안내 문구 · 작업 중 잠글 작업
const PROJ_FIGS = {"f_loop": "<svg viewBox=\"0 0 320 156\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><ellipse cx=\"160.0\" cy=\"70\" rx=\"112\" ry=\"48\" fill=\"none\" stroke=\"#e5e7eb\" stroke-width=\"6\"/><path d=\"M198.6,24.9 L248.1,40.3\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><path d=\"M271.9,71.8 L253.0,96.7\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><path d=\"M190.6,116.2 L129.4,116.2\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><path d=\"M67.0,96.7 L48.1,71.8\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><path d=\"M71.9,40.3 L121.4,24.9\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"116.0\" y=\"10.0\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"25.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">요청<\/text><rect x=\"222.5\" y=\"43.2\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.5\" y=\"58.7\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">생성<\/text><rect x=\"181.8\" y=\"96.8\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"225.8\" y=\"112.3\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">실행<\/text><rect x=\"50.2\" y=\"96.8\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"94.2\" y=\"112.3\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">확인<\/text><rect x=\"9.5\" y=\"43.2\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"53.5\" y=\"58.7\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">수정 요청<\/text><text x=\"160.0\" y=\"74.0\" font-size=\"10.5\" text-anchor=\"middle\" fill=\"#475569\" font-weight=\"bold\">바이브 코딩<\/text><text x=\"160.0\" y=\"151.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">확인하고 다시 요청하는 일이 핵심이에요<\/text><\/svg>", "f_four": "<svg viewBox=\"0 0 320 120\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"83.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">칸<\/text><rect x=\"160.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"237.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">무엇을 적나<\/text><rect x=\"6.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">맥락<\/text><rect x=\"160.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">왜 필요한지<\/text><rect x=\"6.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">입력<\/text><rect x=\"160.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">파일·열·값<\/text><rect x=\"6.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">할 일<\/text><rect x=\"160.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">규칙을 숫자로<\/text><rect x=\"6.0\" y=\"80\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"92.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">출력 형식<\/text><rect x=\"160.0\" y=\"80\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"92.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">받고 싶은 모양<\/text><text x=\"160.0\" y=\"115.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">네 칸을 채우면 AI가 추측할 일이 줄어요<\/text><\/svg>", "f_gsd": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"57.3\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">GSD<\/text><rect x=\"108.7\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"160.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">전차 크기(약 5m)<\/text><rect x=\"211.3\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"262.7\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">보이는 정도<\/text><rect x=\"6.0\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">0.3m<\/text><rect x=\"108.7\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">약 17픽셀<\/text><rect x=\"211.3\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">식별<\/text><rect x=\"6.0\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">1m<\/text><rect x=\"108.7\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">약 5픽셀<\/text><rect x=\"211.3\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">탐지<\/text><rect x=\"6.0\" y=\"62\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">5m<\/text><rect x=\"108.7\" y=\"62\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">약 1픽셀<\/text><rect x=\"211.3\" y=\"62\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">어려움<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">숫자가 작을수록 자세히 보여요<\/text><\/svg>", "f_dri": "<svg viewBox=\"0 0 320 96\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"18\" width=\"99.3\" height=\"30\" rx=\"5\" fill=\"rgb(37,99,235)\" fill-opacity=\"0.35\"/><text x=\"57.7\" y=\"37.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\">탐지<\/text><text x=\"57.7\" y=\"62.0\" font-size=\"8\" text-anchor=\"middle\" fill=\"#475569\">1.5픽셀~<\/text><rect x=\"109.3\" y=\"18\" width=\"99.3\" height=\"30\" rx=\"5\" fill=\"rgb(128,68,136)\" fill-opacity=\"0.35\"/><text x=\"159.0\" y=\"37.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\">인식<\/text><text x=\"159.0\" y=\"62.0\" font-size=\"8\" text-anchor=\"middle\" fill=\"#475569\">6픽셀~<\/text><rect x=\"210.7\" y=\"18\" width=\"99.3\" height=\"30\" rx=\"5\" fill=\"rgb(220,38,38)\" fill-opacity=\"0.35\"/><text x=\"260.3\" y=\"37.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\">식별<\/text><text x=\"260.3\" y=\"62.0\" font-size=\"8\" text-anchor=\"middle\" fill=\"#475569\">12픽셀~<\/text><text x=\"160.0\" y=\"91.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">크게 보일수록 더 자세히 알 수 있어요<\/text><\/svg>", "f_agree": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"44.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">물체<\/text><rect x=\"83.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"121.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">A<\/text><rect x=\"160.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"198.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">B<\/text><rect x=\"237.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"275.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">같음<\/text><rect x=\"6.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"44.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">OBJ01<\/text><rect x=\"83.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"121.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">tank<\/text><rect x=\"160.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"198.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">tank<\/text><rect x=\"237.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"275.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">O<\/text><rect x=\"6.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"44.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">OBJ02<\/text><rect x=\"83.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"121.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">apc<\/text><rect x=\"160.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"198.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">tank<\/text><rect x=\"237.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fef3c7\" stroke=\"#e5e7eb\"/><text x=\"275.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">X<\/text><rect x=\"6.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"44.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">OBJ03<\/text><rect x=\"83.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"121.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">truck<\/text><rect x=\"160.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"198.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">truck<\/text><rect x=\"237.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"275.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">O<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">3개 중 2개가 같으면 일치율 67%<\/text><\/svg>", "f_clean": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"54.0\" y=\"34.0\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">Tank<\/text><text x=\"54.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">대문자<\/text><path d=\"M101.0,37.0 L113.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"114.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"34.0\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">tank <\/text><text x=\"160.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">빈칸<\/text><path d=\"M207.0,37.0 L219.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"220.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.0\" y=\"34.0\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">tank<\/text><text x=\"266.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">같은 값<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">제각각인 값을 하나로 맞춰요<\/text><\/svg>", "f_meta": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"83.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">항목<\/text><rect x=\"160.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"237.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">예<\/text><rect x=\"6.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">촬영 시각<\/text><rect x=\"160.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">UTC로 기록<\/text><rect x=\"6.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">GSD<\/text><rect x=\"160.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">0.5m<\/text><rect x=\"6.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">태양 방위각<\/text><rect x=\"160.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">150도<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">언제, 어디서, 어떤 조건인지 적어요<\/text><\/svg>", "f_shadow": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"54.0\" y=\"34.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">태양 방위각<\/text><text x=\"54.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">150도<\/text><path d=\"M101.0,37.0 L113.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"114.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"34.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">+ 180도<\/text><text x=\"160.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">반대쪽<\/text><path d=\"M207.0,37.0 L219.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"220.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.0\" y=\"34.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">예상 그림자<\/text><text x=\"266.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">330도<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">0도를 넘는 경우도 함께 생각해요<\/text><\/svg>", "f_debug": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"54.0\" y=\"40.5\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">오류 메시지<\/text><path d=\"M101.0,37.0 L113.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"114.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"40.5\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">원하는 동작<\/text><path d=\"M207.0,37.0 L219.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"220.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.0\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">고칠 범위<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">이 세 가지를 담아 다시 요청해요<\/text><\/svg>", "f_pre": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"54.0\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">정사보정<\/text><path d=\"M101.0,37.0 L113.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"114.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"40.5\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">GSD 표준화<\/text><path d=\"M207.0,37.0 L219.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"220.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.0\" y=\"40.5\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">구름 마스킹<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">같은 규격으로 맞춰요<\/text><\/svg>", "f_contract": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"83.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">항목<\/text><rect x=\"160.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"237.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">기준<\/text><rect x=\"6.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">GSD<\/text><rect x=\"160.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">0.5m ± 0.05<\/text><rect x=\"6.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">구름<\/text><rect x=\"160.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">20% 이하<\/text><rect x=\"6.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">오프나디르<\/text><rect x=\"160.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">30도 이하<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">기준을 숫자로 약속해요<\/text><\/svg>", "f_param": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"40.8\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">작은 함수<\/text><path d=\"M74.5,37.0 L86.5,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"87.5\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"120.2\" y=\"40.5\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">예시로 확인<\/text><path d=\"M154.0,37.0 L166.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"167.0\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"199.8\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">전체 적용<\/text><path d=\"M233.5,37.0 L245.5,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"246.5\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"279.2\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">비교표<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">작게 만들고 확인한 뒤 넓혀요<\/text><\/svg>", "f_verify": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"40.8\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">AI 답변<\/text><path d=\"M74.5,37.0 L86.5,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"87.5\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"120.2\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">근거 수치<\/text><path d=\"M154.0,37.0 L166.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"167.0\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"199.8\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">작은 예시<\/text><path d=\"M233.5,37.0 L245.5,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"246.5\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"279.2\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">판단<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">답변만 믿지 말고 근거를 함께 받아요<\/text><\/svg>", "f_leak": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"145.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"80.5\" y=\"34.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">검증 데이터<\/text><text x=\"80.5\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">임계값 고르기<\/text><path d=\"M154.0,37.0 L166.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"167.0\" y=\"14.0\" width=\"145.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"239.5\" y=\"34.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">테스트 데이터<\/text><text x=\"239.5\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">한 번 확인<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">고를 때 쓴 데이터로 성능을 재면 좋게 보여요<\/text><\/svg>", "f_halluc": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"54.0\" y=\"40.5\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">자료에 없음<\/text><path d=\"M101.0,37.0 L113.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"114.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"34.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">AI가 지어냄<\/text><text x=\"160.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">그럴듯함<\/text><path d=\"M207.0,37.0 L219.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"220.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.0\" y=\"34.0\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">사실 확인<\/text><text x=\"266.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">자료와 대조<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">없는 내용은 ‘자료 없음’으로 쓰게 해요<\/text><\/svg>", "f_card": "<svg viewBox=\"0 0 320 148\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><polygon points=\"20,118 90,104 180,104 110,118\" fill=\"#dbeafe\" fill-opacity=\"0.55\" stroke=\"#dbeafe\"/><text x=\"192.0\" y=\"115.0\" font-size=\"9.5\" text-anchor=\"start\" fill=\"#111827\" font-weight=\"bold\">사람이 판독할 조건<\/text><polygon points=\"20,92 90,78 180,78 110,92\" fill=\"#e0e7ff\" fill-opacity=\"0.55\" stroke=\"#e0e7ff\"/><text x=\"192.0\" y=\"89.0\" font-size=\"9.5\" text-anchor=\"start\" fill=\"#111827\" font-weight=\"bold\">조심할 조건<\/text><polygon points=\"20,66 90,52 180,52 110,66\" fill=\"#c7d2fe\" fill-opacity=\"0.55\" stroke=\"#c7d2fe\"/><text x=\"192.0\" y=\"63.0\" font-size=\"9.5\" text-anchor=\"start\" fill=\"#111827\" font-weight=\"bold\">조건별 성능<\/text><polygon points=\"20,40 90,26 180,26 110,40\" fill=\"#bfdbfe\" fill-opacity=\"0.55\" stroke=\"#bfdbfe\"/><text x=\"192.0\" y=\"37.0\" font-size=\"9.5\" text-anchor=\"start\" fill=\"#111827\" font-weight=\"bold\">결론 한 줄<\/text><text x=\"160.0\" y=\"143.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">언제 믿고 언제 조심할지 적어요<\/text><\/svg>", "f_decoy": "<svg viewBox=\"0 0 320 128\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><text x=\"200.0\" y=\"14.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#475569\" font-weight=\"bold\">단서 →<\/text><text x=\"28.0\" y=\"69.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#475569\" font-weight=\"bold\">실제 ↓<\/text><text x=\"152.0\" y=\"26.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">그림자·반사 있음<\/text><text x=\"248.0\" y=\"26.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">없거나 약함<\/text><text x=\"98.0\" y=\"51.0\" font-size=\"9.5\" text-anchor=\"end\" fill=\"#111827\" font-weight=\"bold\">장비<\/text><rect x=\"104\" y=\"30\" width=\"96\" height=\"36\" fill=\"#dcfce7\" stroke=\"#fff\" stroke-width=\"2\"/><text x=\"152.0\" y=\"52.0\" font-size=\"11\" text-anchor=\"middle\" fill=\"#16a34a\" font-weight=\"bold\">통과<\/text><rect x=\"200\" y=\"30\" width=\"96\" height=\"36\" fill=\"#fee2e2\" stroke=\"#fff\" stroke-width=\"2\"/><text x=\"248.0\" y=\"52.0\" font-size=\"11\" text-anchor=\"middle\" fill=\"#dc2626\" font-weight=\"bold\">재확인<\/text><text x=\"98.0\" y=\"87.0\" font-size=\"9.5\" text-anchor=\"end\" fill=\"#111827\" font-weight=\"bold\">기만체<\/text><rect x=\"104\" y=\"66\" width=\"96\" height=\"36\" fill=\"#fee2e2\" stroke=\"#fff\" stroke-width=\"2\"/><text x=\"152.0\" y=\"88.0\" font-size=\"11\" text-anchor=\"middle\" fill=\"#dc2626\" font-weight=\"bold\">드묾<\/text><rect x=\"200\" y=\"66\" width=\"96\" height=\"36\" fill=\"#dcfce7\" stroke=\"#fff\" stroke-width=\"2\"/><text x=\"248.0\" y=\"88.0\" font-size=\"11\" text-anchor=\"middle\" fill=\"#16a34a\" font-weight=\"bold\">의심<\/text><\/svg>", "f_edge": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"44.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">경우<\/text><rect x=\"83.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"121.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">그림자<\/text><rect x=\"160.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"198.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">SAR<\/text><rect x=\"237.0\" y=\"8\" width=\"77.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"275.5\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">판정<\/text><rect x=\"6.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"44.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">보통<\/text><rect x=\"83.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"121.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">있음<\/text><rect x=\"160.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"198.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">강함<\/text><rect x=\"237.0\" y=\"26\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"275.5\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">장비<\/text><rect x=\"6.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"44.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">기만<\/text><rect x=\"83.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"121.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">없음<\/text><rect x=\"160.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"198.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">약함<\/text><rect x=\"237.0\" y=\"44\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"275.5\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">의심<\/text><rect x=\"6.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"44.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">구름<\/text><rect x=\"83.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"121.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">빈칸<\/text><rect x=\"160.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"198.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">빈칸<\/text><rect x=\"237.0\" y=\"62\" width=\"77.0\" height=\"18\" fill=\"#fef3c7\" stroke=\"#e5e7eb\"/><text x=\"275.5\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">자료 없음<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">빈칸을 어떻게 할지 미리 알려 주세요<\/text><\/svg>", "f_count": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"83.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">구분<\/text><rect x=\"160.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"237.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">개수<\/text><rect x=\"6.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">모델이 센 수<\/text><rect x=\"160.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">12<\/text><rect x=\"6.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">정답: 진짜 장비<\/text><rect x=\"160.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">5<\/text><rect x=\"6.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">개수 오차<\/text><rect x=\"160.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fef3c7\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">7<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">개수만 세면 규모를 잘못 볼 수 있어요<\/text><\/svg>", "f_rel": "<svg viewBox=\"0 0 320 150\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"34\" y=\"12\" width=\"274\" height=\"108\" fill=\"#fff\" stroke=\"#94a3b8\"/><line x1=\"34.0\" y1=\"120.0\" x2=\"308.0\" y2=\"12.0\" stroke=\"#94a3b8\" stroke-dasharray=\"4 3\"/><text x=\"304.0\" y=\"24.0\" font-size=\"8.5\" text-anchor=\"end\" fill=\"#475569\">완벽<\/text><polyline points=\"61.4,115.7 116.2,107.0 171.0,90.8 225.8,66.0 280.6,31.4\" fill=\"none\" stroke=\"#2563eb\" stroke-width=\"2\"/><circle cx=\"61.4\" cy=\"115.7\" r=\"2.5\" fill=\"#2563eb\"/><circle cx=\"116.2\" cy=\"107.0\" r=\"2.5\" fill=\"#2563eb\"/><circle cx=\"171.0\" cy=\"90.8\" r=\"2.5\" fill=\"#2563eb\"/><circle cx=\"225.8\" cy=\"66.0\" r=\"2.5\" fill=\"#2563eb\"/><circle cx=\"280.6\" cy=\"31.4\" r=\"2.5\" fill=\"#2563eb\"/><text x=\"171.0\" y=\"142.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">mean score<\/text><text x=\"10\" y=\"66.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\" transform=\"rotate(-90 10 66.0)\">actual accuracy<\/text><text x=\"40.0\" y=\"24.0\" font-size=\"8.5\" text-anchor=\"start\" fill=\"#475569\">대각선 아래면 점수가 부풀어 있어요<\/text><\/svg>", "f_scale": "<svg viewBox=\"0 0 320 96\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"18\" width=\"74.0\" height=\"30\" rx=\"5\" fill=\"rgb(37,99,235)\" fill-opacity=\"0.35\"/><text x=\"45.0\" y=\"37.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\">낮음<\/text><text x=\"45.0\" y=\"62.0\" font-size=\"8\" text-anchor=\"middle\" fill=\"#475569\">20~45%<\/text><rect x=\"84.0\" y=\"18\" width=\"74.0\" height=\"30\" rx=\"5\" fill=\"rgb(98,78,169)\" fill-opacity=\"0.35\"/><text x=\"121.0\" y=\"37.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\">반반<\/text><text x=\"121.0\" y=\"62.0\" font-size=\"8\" text-anchor=\"middle\" fill=\"#475569\">45~55%<\/text><rect x=\"160.0\" y=\"18\" width=\"74.0\" height=\"30\" rx=\"5\" fill=\"rgb(159,58,103)\" fill-opacity=\"0.35\"/><text x=\"197.0\" y=\"37.0\" font-size=\"9.5\" text-anchor=\"middle\" fill=\"#111827\">높음<\/text><text x=\"197.0\" y=\"62.0\" font-size=\"8\" text-anchor=\"middle\" fill=\"#475569\">55~80%<\/text><rect x=\"236.0\" y=\"18\" width=\"74.0\" height=\"30\" rx=\"5\" fill=\"rgb(220,38,38)\" fill-opacity=\"0.35\"/><text x=\"273.0\" y=\"37.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#111827\">매우 높음<\/text><text x=\"273.0\" y=\"62.0\" font-size=\"8\" text-anchor=\"middle\" fill=\"#475569\">80~95%<\/text><text x=\"160.0\" y=\"91.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">낱말과 확률을 약속해 둬요 (일부)<\/text><\/svg>", "f_conf2": "<svg viewBox=\"0 0 320 84\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"57.3\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">구분<\/text><rect x=\"108.7\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"160.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">무엇을 말하나<\/text><rect x=\"211.3\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"262.7\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">예<\/text><rect x=\"6.0\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">가능성<\/text><rect x=\"108.7\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">얼마나 그럴듯한가<\/text><rect x=\"211.3\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">높음<\/text><rect x=\"6.0\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">확신도<\/text><rect x=\"108.7\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">근거가 얼마나 튼튼한가<\/text><rect x=\"211.3\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">낮음<\/text><text x=\"160.0\" y=\"79.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">두 가지를 나눠 써요<\/text><\/svg>", "f_hitl": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"54.0\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">AI 1차<\/text><path d=\"M101.0,37.0 L113.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"114.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">사람 판독<\/text><path d=\"M207.0,37.0 L219.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"220.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.0\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">승인<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">확인되지 않은 내용은 미확인으로 남겨요<\/text><\/svg>", "f_secret": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"83.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">넣어도 돼요<\/text><rect x=\"160.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"237.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">넣으면 안 돼요<\/text><rect x=\"6.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">열 이름<\/text><rect x=\"160.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">실제 위치<\/text><rect x=\"6.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">가상 샘플<\/text><rect x=\"160.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">실제 인물<\/text><rect x=\"6.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">오류 메시지<\/text><rect x=\"160.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">장비 정보<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">공개 AI에 넣기 전에 나눠 봐요<\/text><\/svg>", "f_review": "<svg viewBox=\"0 0 320 156\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><ellipse cx=\"160.0\" cy=\"70\" rx=\"112\" ry=\"48\" fill=\"none\" stroke=\"#e5e7eb\" stroke-width=\"6\"/><path d=\"M207.7,26.6 L261.3,49.6\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><path d=\"M261.3,90.4 L207.7,113.4\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><path d=\"M112.3,113.4 L58.7,90.4\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><path d=\"M58.7,49.6 L112.3,26.6\" stroke=\"#2563eb\" stroke-width=\"2\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"116.0\" y=\"10.0\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"25.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">설명<\/text><rect x=\"228.0\" y=\"58.0\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"272.0\" y=\"73.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">테스트<\/text><rect x=\"116.0\" y=\"106.0\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"121.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">수정<\/text><rect x=\"4.0\" y=\"58.0\" width=\"88.0\" height=\"24.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"48.0\" y=\"73.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">비교<\/text><text x=\"160.0\" y=\"74.0\" font-size=\"10.5\" text-anchor=\"middle\" fill=\"#475569\" font-weight=\"bold\">검토<\/text><text x=\"160.0\" y=\"151.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">고치기 전에 먼저 설명하게 해요<\/text><\/svg>", "f_scope": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"54.0\" y=\"34.0\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">고칠 곳<\/text><text x=\"54.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">이 함수만<\/text><path d=\"M101.0,37.0 L113.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"114.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"160.0\" y=\"34.0\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">그대로<\/text><text x=\"160.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">원본 파일<\/text><path d=\"M207.0,37.0 L219.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"220.0\" y=\"14.0\" width=\"92.0\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"266.0\" y=\"34.0\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">그대로<\/text><text x=\"266.0\" y=\"47.0\" font-size=\"8.5\" text-anchor=\"middle\" fill=\"#475569\">다른 코드<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">고칠 범위를 말해 주세요<\/text><\/svg>", "f_tool": "<svg viewBox=\"0 0 320 92\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"8.0\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"40.8\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">설정값<\/text><path d=\"M74.5,37.0 L86.5,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"87.5\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"120.2\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">계산 함수<\/text><path d=\"M154.0,37.0 L166.0,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"167.0\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"199.8\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">표·그림<\/text><path d=\"M233.5,37.0 L245.5,37.0\" stroke=\"#475569\" stroke-width=\"1.6\" marker-end=\"url(#tfa)\" fill=\"none\"/><rect x=\"246.5\" y=\"14.0\" width=\"65.5\" height=\"46.0\" rx=\"7\" fill=\"#dbeafe\" stroke=\"#94a3b8\"/><text x=\"279.2\" y=\"40.5\" font-size=\"10\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">보고서<\/text><text x=\"160.0\" y=\"87.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">한 번에 다시 돌릴 수 있게 묶어요<\/text><\/svg>", "f_lint": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"83.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">항목<\/text><rect x=\"160.0\" y=\"8\" width=\"154.0\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"237.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">있음?<\/text><rect x=\"6.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">표본 수<\/text><rect x=\"160.0\" y=\"26\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">O<\/text><rect x=\"6.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">한계<\/text><rect x=\"160.0\" y=\"44\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">O<\/text><rect x=\"6.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"83.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">AI 산출물 표기<\/text><rect x=\"160.0\" y=\"62\" width=\"154.0\" height=\"18\" fill=\"#fef3c7\" stroke=\"#e5e7eb\"/><text x=\"237.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">X<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">빠진 항목을 코드로 찾아요<\/text><\/svg>", "f_cond": "<svg viewBox=\"0 0 320 102\" xmlns=\"http://www.w3.org/2000/svg\" font-family=\"sans-serif\" font-size=\"10\" fill=\"#111827\"><defs><marker id=\"tfa\" viewBox=\"0 0 10 10\" refX=\"8\" refY=\"5\" markerWidth=\"6\" markerHeight=\"6\" orient=\"auto\"><path d=\"M0,0 L10,5 L0,10 z\" fill=\"#475569\"/><\/marker><\/defs><rect x=\"6.0\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"57.3\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">GSD<\/text><rect x=\"108.7\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"160.0\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">재현율<\/text><rect x=\"211.3\" y=\"8\" width=\"102.7\" height=\"18\" fill=\"#e5e7eb\" stroke=\"#fff\"/><text x=\"262.7\" y=\"20.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">표본<\/text><rect x=\"6.0\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">0.5m<\/text><rect x=\"108.7\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">높음<\/text><rect x=\"211.3\" y=\"26\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"38.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">많음<\/text><rect x=\"6.0\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">1m<\/text><rect x=\"108.7\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">중간<\/text><rect x=\"211.3\" y=\"44\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"56.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">많음<\/text><rect x=\"6.0\" y=\"62\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"57.3\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">2m<\/text><rect x=\"108.7\" y=\"62\" width=\"102.7\" height=\"18\" fill=\"#fff\" stroke=\"#e5e7eb\"/><text x=\"160.0\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\">낮음<\/text><rect x=\"211.3\" y=\"62\" width=\"102.7\" height=\"18\" fill=\"#fef3c7\" stroke=\"#e5e7eb\"/><text x=\"262.7\" y=\"74.5\" font-size=\"9\" text-anchor=\"middle\" fill=\"#111827\" font-weight=\"bold\">적음<\/text><text x=\"160.0\" y=\"97.0\" font-size=\"9\" text-anchor=\"middle\" fill=\"#475569\">수치에는 조건과 표본 수를 함께 적어요<\/text><\/svg>"};   // 핵심 용어 카드에 붙는 그림(SVG)
const MISSION_UI = {"m1": {"steps": [{"short": "네 칸 익히기", "title": "프롬프트의 네 칸", "lead": "AI는 전우님 업무도, 파일도 몰라요. **네 칸**을 채워 알려 주면 돼요.", "page": null, "blocks": [{"id": "s1_info", "type": "info", "title": "좋은 프롬프트의 네 칸", "icon": "🧩", "req": false, "text": ["같은 일을 시켜도 무엇을 알려 주느냐에 따라 결과가 달라져요. 처음에는 아래 네 칸만 기억하세요."], "table": {"head": ["칸", "무엇을 적나", "예"], "rows": [["맥락", "왜 필요한지, 어떤 일인지", "장비가 GSD별로 몇 픽셀로 보이는지 확인하려고 해"], ["입력", "파일 이름, 열 이름, 값의 뜻", "m1_equipment.csv, 열은 equipment·length_m·width_m"], ["할 일", "계산 규칙을 숫자로", "픽셀 수 = 대표 크기 ÷ GSD"], ["출력 형식", "받고 싶은 모양", "장비는 행, GSD는 열인 표"]]}, "figure": "f_four"}, {"id": "s1_q", "type": "quiz", "title": "어떤 프롬프트가 가장 좋을까요?", "req": true, "text": "같은 일을 시키는 세 프롬프트예요. 다시 묻는 일이 가장 적을 것은 무엇일까요?", "options": ["위성영상 분석하는 코드 좀 짜 줘.", "장비가 잘 보이는지 알려 줘. 잘 부탁해.", "장비가 GSD별로 몇 픽셀로 보이는지 확인하려고 해. m1_equipment.csv의 크기로 GSD마다 픽셀 수를 계산해 표로 보여 줘."]}]}, {"short": "AI에게 시키기", "title": "프롬프트를 만들어 AI에게 보내기", "lead": "칸마다 문장을 고르면 프롬프트가 만들어져요. 일부러 **아쉬운 문장**도 보내 보세요. 무엇이 달라지는지 보는 것이 이번 활동의 핵심이에요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "GSD별 판독 가능 수준표 만들기", "req": true, "ask": ["han", "장비 크기 파일을 드릴게요. GSD 0.3m~10m에서 각각 몇 픽셀로 보이는지 표로 받고 싶어요."], "has_start": false, "files": [{"name": "m1_equipment.csv", "head": ["equipment", "length_m", "width_m"], "rows": [["tank", "6.9", "3.6"], ["apc", "6.7", "3.2"], ["spg", "7.2", "3.4"], ["truck", "8.4", "2.5"]], "n": 4}], "slots": [{"id": "ctx", "label": "맥락", "why": "왜 필요한지 알려요.", "options": [{"i": 0, "text": "위성영상 판독 업무를 돕고 있어. 장비가 GSD별로 화면에서 몇 픽셀로 보이는지 확인하려고 해."}, {"i": 1, "text": "코드 좀 짜 줘."}, {"i": 2, "text": ""}]}, {"id": "input", "label": "입력", "why": "파일과 열을 알려요.", "options": [{"i": 0, "text": "데이터는 m1_equipment.csv이고, 열은 equipment(장비 종류), length_m(길이, 미터), width_m(폭, 미터)이야."}, {"i": 1, "text": "장비 데이터 파일이 있어."}, {"i": 2, "text": ""}]}, {"id": "task", "label": "할 일", "why": "계산 규칙을 숫자로 적어요.", "options": [{"i": 1, "text": "GSD별로 잘 보이는지 계산해 줘."}, {"i": 0, "text": "대표 크기 = √(length_m × width_m), 픽셀 수 = 대표 크기 ÷ GSD로 GSD 0.3, 0.5, 1, 2, 5, 10m마다 계산하고, 1.5픽셀 이상은 탐지, 6픽셀 이상은 인식, 12픽셀 이상은 식별로 표시해 줘."}, {"i": 2, "text": ""}]}, {"id": "out", "label": "출력 형식", "why": "받고 싶은 모양을 알려요.", "options": [{"i": 1, "text": "결과를 보여 줘."}, {"i": 0, "text": "장비는 행, GSD는 열인 표로 보여 주고, 같은 표를 넣은 HTML 보고서를 report 변수에 담아 줘. 전차 크기 물체가 GSD별로 어떻게 보이는지 그림도 그려 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "내 말로 쓰기", "title": "같은 일을 내 말로 요청하기", "lead": "고른 문장을 그대로 외울 필요는 없어요. **네 칸만 채우면** 말투는 자유예요.", "page": null, "blocks": [{"id": "s3_q", "type": "quiz", "title": "결과를 받은 뒤에는", "req": true, "text": "AI가 만든 코드가 오류 없이 돌았어요. 다음으로 무엇을 하면 좋을까요?", "options": ["오류가 없으니 그대로 보고한다.", "하나만 손으로 계산해 맞는지 확인한다.", "더 어려운 말로 다시 요청한다."]}, {"id": "s3_write", "type": "write", "title": "내 프롬프트 쓰기", "req": true, "prompt": "**새 요청**: 새 장비(길이 5.0m, 폭 2.0m)가 추가됐어요. 파일은 m1_equipment_new.csv이고 열은 같아요. GSD 0.3m, 1m에서의 픽셀 수를 보고 싶어요. 네 칸을 채워 내 말로 프롬프트를 써 보세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m2": {"steps": [{"short": "데이터 보여 주기", "title": "AI에게 데이터를 설명하는 법", "lead": "AI는 파일을 볼 수 없어요. **앞부분 몇 줄과 값의 종류**를 알려 주는 것이 가장 빠른 방법이에요.", "page": null, "blocks": [{"id": "s1_info", "type": "info", "title": "데이터를 설명할 때 넣을 것", "icon": "🗂", "req": false, "text": ["열 이름만으로는 부족할 때가 많아요. 값이 어떻게 적혀 있는지까지 알려 주세요."], "bullets": ["**파일 이름과 열 이름**: m2_reads.csv, object_id, reader_a, reader_b, reader_c", "**값의 종류**: tank, apc, truck, decoy, unknown 다섯 가지", "**주의할 점**: 대소문자와 뒤쪽 빈칸이 섞여 있음", "**샘플 몇 줄**: 실제 값 대신 가상 샘플로 보여 줘도 돼요"], "figure": "f_clean"}, {"id": "s1_q", "type": "quiz", "title": "먼저 알려 줄 것", "req": true, "text": "값에 ‘Tank’, ‘tank ’이 섞여 있어요. 이 사실을 알려 주지 않으면 어떻게 될까요?", "options": ["AI가 스스로 알아채 항상 정리해 준다.", "같은 값을 다른 값으로 세어 일치율이 실제보다 낮게 나올 수 있다.", "파일을 읽지 못해 반드시 오류가 난다."]}]}, {"short": "일치율 계산", "title": "일치율을 계산하고 갈린 물체 찾기", "lead": "**값 정리** 문장을 빼고도 보내 보세요. 숫자가 어떻게 달라지는지 비교해 보면 좋아요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "판독 일치율 계산", "req": true, "ask": ["park", "세 사람이 둘씩 얼마나 같게 판독했는지, 판독이 갈린 물체는 무엇인지 알고 싶어요."], "has_start": false, "files": [{"name": "m2_reads.csv", "head": ["object_id", "reader_a", "reader_b", "reader_c"], "rows": [["OBJ01", "tank", "tank", "decoy "], ["OBJ02", "tank", "tank ", "tank"], ["OBJ03", "decoy", "Decoy", "decoy "], ["OBJ04", "Unknown", "apc", "apc"], ["OBJ05", "decoy", "tank", "tank"]], "n": 24}], "slots": [{"id": "input", "label": "입력", "why": "값의 모양까지 알려요.", "options": [{"i": 1, "text": "m2_reads.csv에 세 사람의 판독이 있어."}, {"i": 0, "text": "m2_reads.csv의 열은 object_id, reader_a, reader_b, reader_c이고, 값은 tank, apc, truck, decoy, unknown이야. 대소문자와 빈칸이 섞여 있으니 먼저 소문자로 바꾸고 빈칸을 지워 줘."}, {"i": 2, "text": ""}]}, {"id": "task", "label": "할 일", "why": "무엇을 세는지 정해요.", "options": [{"i": 0, "text": "세 사람을 둘씩 묶어 같은 값으로 판독한 비율(%)을 계산하고, 세 사람이 모두 같지 않은 물체 목록도 뽑아 줘."}, {"i": 1, "text": "일치도를 분석해 줘."}, {"i": 2, "text": ""}]}, {"id": "out", "label": "출력 형식", "why": "회의에서 볼 모양으로 받아요.", "options": [{"i": 1, "text": "결과를 정리해 줘."}, {"i": 0, "text": "일치율 표와 갈린 물체 표를 보여 주고, 두 표를 넣은 HTML 문서를 report 변수에 담아 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "기준 합의", "title": "결과로 기준을 합의하고 내 말로 요청하기", "lead": "AI는 계산을 도와줄 뿐이에요. 어떤 기준을 쓸지는 **사람이 근거를 보고 정해요.**", "page": null, "blocks": [{"id": "s3_decide", "type": "decide", "title": "팀원의 제안 판단하기", "req": true, "text": "일치율 결과를 본 팀원들의 제안이에요. 받아들일지 정하고 이유를 적어 주세요.", "reason_min": 8, "items": [{"id": "a", "who": "han", "text": "애매한 물체는 일단 전부 tank로 적어요. 놓치는 것보다 낫잖아요."}, {"id": "b", "who": "park", "text": "값을 tank, apc, truck, decoy, unknown 다섯 가지 소문자로만 입력하게 규칙을 정해요."}]}, {"id": "s3_write", "type": "write", "title": "기준 문서를 요청하는 프롬프트 쓰기", "req": true, "prompt": "방금 합의한 내용으로 **판독 기준 문서**를 AI에게 요청해 보세요. 누가 읽는지, 어떤 항목이 들어가야 하는지, 근거 데이터를 적어 주세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m3": {"steps": [{"short": "점검표 만들기", "title": "그림자 점검표 만들기", "lead": "계산식을 프롬프트에 직접 적어 보세요. **0도를 넘는 경우**처럼 헷갈리기 쉬운 점도 알려 주면 좋아요.", "page": null, "blocks": [{"id": "s1_vibe", "type": "vibe", "title": "그림자 방향 점검표", "req": true, "ask": ["choi", "영상마다 그림자 방향이 태양 위치와 맞는지 표로 보고 싶어요."], "has_start": false, "files": [{"name": "m3_scenes.csv", "head": ["image_id", "captured_utc", "sun_azimuth_deg", "shadow_azimuth_deg"], "rows": [["S01", "2025-05-12T05:30:00Z", "150", "331"], ["S02", "2025-05-12T06:10:00Z", "165", "346"], ["S03", "2025-05-12T06:40:00Z", "178", "3"], ["S04", "2025-05-12T07:00:00Z", "190", "262"], ["S05", "2025-05-13T05:20:00Z", "148", "327"]], "n": 8}], "slots": [{"id": "input", "label": "입력", "why": "열의 뜻을 알려요.", "options": [{"i": 1, "text": "영상 메타데이터 CSV가 있어."}, {"i": 0, "text": "m3_scenes.csv의 열은 image_id, captured_utc(UTC 촬영 시각), sun_azimuth_deg(태양 방위각), shadow_azimuth_deg(영상에서 잰 그림자 방향)야."}, {"i": 2, "text": ""}]}, {"id": "task", "label": "할 일", "why": "규칙과 경계를 숫자로 적어요.", "options": [{"i": 1, "text": "그림자가 맞는지 확인해 줘."}, {"i": 0, "text": "그림자는 태양 반대쪽이니까 예상 방향 = (태양 방위각 + 180) % 360으로 구하고, 차이는 0도를 넘는 경우까지 생각해 min(|예상 − 실제|, 360 − |예상 − 실제|)로 계산해 줘. 차이가 15도 이하면 일치, 넘으면 재검토로 표시해 줘."}, {"i": 2, "text": ""}]}, {"id": "out", "label": "출력 형식", "why": "표의 모양을 정해요.", "options": [{"i": 1, "text": "보기 좋게 보여 줘."}, {"i": 0, "text": "image_id, 한국 시각(UTC+9), 예상 방향, 실제 방향, 차이, 판정 순서의 표로 보여 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "오류 고치기", "title": "오류 메시지로 수정 요청하기", "lead": "파일을 바꿨더니 **오류**가 났어요. 어떻게 다시 요청할까요?", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "오류 고쳐 달라고 하기", "req": true, "ask": ["jung", "다른 공급사 파일을 넣었더니 오류가 나요. 저는 원인을 잘 모르겠어요… 부탁해요!"], "has_start": true, "files": [{"name": "m3_scenes_new.csv", "head": ["image_id", "captured_utc", "sun_azimuth_deg", "shadow_azimuth_deg"], "rows": [["N01", "2025-05-20T05:30:00Z", "152", "330"], ["N02", "2025-05-20T06:00:00Z", "161", "343"], ["N03", "2025/05/20 06:30", "173", "355"], ["N04", "2025-05-20T07:00:00Z", "188", "7"], ["N05", "2025/05/20 07:30", "201", "95"]], "n": 5}], "slots": [{"id": "what", "label": "무슨 일이 있었는지", "why": "오류 메시지를 그대로 붙여 넣어요.", "options": [{"i": 1, "text": "안 돼. 다시 해 줘."}, {"i": 0, "text": "m3_scenes_new.csv를 읽다가 이 오류가 났어: ValueError: time data '2025/05/20 06:30' does not match format '%Y-%m-%dT%H:%M:%SZ'"}, {"i": 2, "text": ""}]}, {"id": "fix", "label": "어떻게 고칠지", "why": "원하는 동작과 범위를 정해요.", "options": [{"i": 0, "text": "공급사마다 날짜 형식이 달라. 두 형식을 모두 읽게 하고, 그 밖의 형식은 ‘자료 없음’으로 표시해 줘. 판정 계산은 그대로 두고, 표를 넣은 HTML 진위 점검 보고서를 report 변수에 담아 줘."}, {"i": 1, "text": "전부 새로 짜 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "내 말로 쓰기", "title": "다른 오류를 내 말로 요청하기", "lead": "오류를 만나면 **겁먹지 말고 그대로 붙여 넣으세요.** 코드를 다 읽지 못해도 수정을 요청할 수 있어요.", "page": null, "blocks": [{"id": "s3_write", "type": "write", "title": "수정 프롬프트 쓰기", "req": true, "prompt": "**새 상황**: 공급사 C의 파일을 넣었더니 `KeyError: 'sun_azimuth_deg'`가 났어요. 이 파일의 실제 열 이름은 sun_az예요. 잘 되던 판정 계산은 그대로 두고 싶어요. 어떻게 요청할까요?", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m4": {"steps": [{"short": "쪼개기", "title": "큰 일을 작게 나누기", "lead": "한 번에 다 시키는 대신, **확인할 수 있는 크기**로 나눠요.", "page": null, "blocks": [{"id": "s1_info", "type": "info", "title": "왜 쪼개야 할까요?", "icon": "✂️", "req": false, "text": ["한 번에 전부 만들어 달라고 하면 코드가 길어지고, 틀려도 어디인지 찾기 어려워요. 한 단계씩 시키고 확인하면 AI도 전우님도 덜 헤매요."], "bullets": ["**1단계**: 타일 하나만 검사하는 함수", "**2단계**: 작은 예시로 확인", "**3단계**: 전체 타일에 적용", "**4단계**: 계약별 비교표와 문서"], "figure": "f_param"}, {"id": "s1_order", "type": "order", "title": "요청 순서 정하기", "req": true, "text": "데이터 계약 검사기를 만들려고 해요. 어떤 순서로 요청하면 좋을까요?", "items": [{"id": "b", "text": "타일 하나를 검사하는 함수 만들기"}, {"id": "e", "text": "비교표를 넣은 계약 검사 문서 만들기"}, {"id": "c", "text": "예시 한 행으로 함수 확인하기"}, {"id": "d", "text": "전체 타일에 적용해 계약별 비교표 만들기"}, {"id": "a", "text": "데이터 모양을 알려 주고 파일을 읽게 하기"}]}]}, {"short": "함수로 받기", "title": "함수로 받아 계약 비교하기", "lead": "기준을 **하나로 정해 주지 말고**, 바꿔 돌릴 수 있게 요청해 보세요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "데이터 계약 검사기", "req": true, "ask": ["han", "엄격한 계약과 완화한 계약을 비교하고 싶어요. 앞으로 다른 기준도 볼 수 있으면 좋겠어요."], "has_start": false, "files": [{"name": "m4_tiles.csv", "head": ["tile_id", "sensor", "gsd_m", "cloud_pct", "off_nadir_deg", "crs"], "rows": [["T01", "optical", "0.5", "4", "12", "EPSG:32652"], ["T02", "optical", "0.5", "18", "28", "EPSG:32652"], ["T03", "optical", "0.5", "35", "15", "EPSG:32652"], ["T04", "optical", "1.0", "2", "10", "EPSG:32652"], ["T05", "sar", "0.5", "0", "33", "EPSG:32652"]], "n": 8}], "slots": [{"id": "split", "label": "나눠서 요청", "why": "작은 단위부터 시켜요.", "options": [{"i": 0, "text": "먼저 타일 한 행을 받아 어긴 기준을 목록으로 돌려주는 함수 check_tile을 만들고, 첫 번째 행으로 결과를 출력해 확인해 줘."}, {"i": 1, "text": "데이터 계약 검사를 한 번에 전부 만들어 줘."}, {"i": 2, "text": ""}]}, {"id": "param", "label": "바꿔 돌리기", "why": "값을 인자로 받게 해요.", "options": [{"i": 0, "text": "구름 기준 max_cloud와 오프나디르 기준 max_off는 인자로 받게 해서, 엄격(20, 30)과 완화(30, 35) 두 계약을 바꿔 돌릴 수 있게 해 줘."}, {"i": 1, "text": "구름 20%, 오프나디르 30도로 검사해 줘."}, {"i": 2, "text": ""}]}, {"id": "out", "label": "출력 형식", "why": "비교할 내용을 정해요.", "options": [{"i": 1, "text": "계약별로 결과를 보여 줘."}, {"i": 0, "text": "계약마다 통과·재처리 타일 수와 타일별 어긴 기준을 표로 보여 주고, 막대그림과 HTML 계약 검사 보고서를 report 변수에 담아 줘. GSD 기준은 0.5m, 허용 오차는 0.05m, 좌표계 기준은 EPSG:32652야."}, {"i": 2, "text": ""}]}]}]}, {"short": "합의하고 쓰기", "title": "결과로 합의하고 내 말로 요청하기", "lead": "표를 근거로 팀원의 제안을 판단하세요. 정답보다 **근거를 대는 것**이 중요해요.", "page": null, "blocks": [{"id": "s3_decide", "type": "decide", "title": "계약 제안 판단하기", "req": true, "text": "비교표를 근거로 제안을 판단해 주세요.", "reason_min": 8, "items": [{"id": "a", "who": "jung", "text": "구름 낀 타일은 생성형 AI로 구름 아래를 그려 넣어 통과시켜요. 타일 수가 늘잖아요."}, {"id": "b", "who": "han", "text": "기본은 엄격 계약으로 하고, 완화 계약으로 통과한 타일은 결과에 표시해 구분해요."}]}, {"id": "s3_write", "type": "write", "title": "쪼개서 요청하는 프롬프트 쓰기", "req": true, "prompt": "**새 요청**: SAR 타일에만 보정 여부(calibrated 열)를 추가로 검사하고 싶어요. 한 번에 시키지 말고, 첫 번째 요청만 써 보세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m5": {"steps": [{"short": "근거 요청하기", "title": "답만 받지 말고 근거도 받기", "lead": "AI에게 **답과 근거를 함께** 달라고 하면 틀린 답을 알아채기 쉬워져요.", "page": null, "blocks": [{"id": "s1_info", "type": "info", "title": "믿기 전에 받을 것", "icon": "🔍", "req": false, "text": ["AI가 자신 있게 말해도 사실과 다를 수 있어요. 아래 세 가지를 프롬프트에 넣으면 사람이 확인할 수 있어요."], "bullets": ["**계산에 쓴 수치**: TP, FN, FP처럼 손으로 다시 셀 수 있는 숫자", "**어떤 데이터로 계산했는지**: 고를 때 쓴 데이터와 확인에 쓴 데이터", "**한계**: 표본 수, 믿기 어려운 조건"], "figure": "f_leak"}, {"id": "s1_q", "type": "quiz", "title": "왜 공항 단위로 나눌까요?", "req": true, "text": "같은 공항의 영상을 학습·검증·테스트에 섞지 않고 공항 단위로 나눴어요. 이유는 무엇일까요?", "options": ["파일을 읽기 쉬워져서", "같은 공항의 비슷한 영상이 섞이면 성능이 실제보다 좋아 보일 수 있어서", "공항마다 파일 이름이 달라서"]}]}, {"short": "성능표와 모델 카드", "title": "검증까지 요청해 성능표와 모델 카드 받기", "lead": "**고를 때 쓴 데이터**와 **확인에 쓴 데이터**를 나눠 요청해 보세요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "조건별 성능표와 모델 카드", "req": true, "ask": ["jung", "임계값을 고르고, 조건별 성능과 모델 카드까지 받고 싶어요."], "has_start": false, "files": [{"name": "m5_detections.csv", "head": ["det_id", "airport_id", "split", "cloud_band", "off_nadir_band", "is_true", "score"], "rows": [["D001", "A1", "val", "clear", "high", "0", "0.153"], ["D002", "A1", "val", "cloudy", "low", "1", "0.774"], ["D003", "A1", "val", "cloudy", "low", "0", "0.49"], ["D004", "A1", "val", "clear", "low", "0", "0.348"], ["D005", "A1", "val", "clear", "low", "0", "0.096"]], "n": 360}], "slots": [{"id": "input", "label": "입력", "why": "데이터를 나눈 방법을 알려요.", "options": [{"i": 0, "text": "m5_detections.csv의 열은 det_id, airport_id, split(val 또는 test, 공항 단위로 나뉨), cloud_band, off_nadir_band, is_true(맞는 탐지면 1), score(가상 모델 점수)야."}, {"i": 1, "text": "모델 점수 파일이 있어."}, {"i": 2, "text": ""}]}, {"id": "task", "label": "할 일", "why": "어떤 데이터로 고를지 정해요.", "options": [{"i": 0, "text": "split이 val인 데이터에서 재현율 0.85 이상인 임계값 가운데 가장 높은 값을 고르고, 그 임계값을 test 데이터에서 한 번만 확인해 줘."}, {"i": 1, "text": "데이터 전체에서 가장 좋은 임계값을 찾아 줘."}, {"i": 2, "text": ""}]}, {"id": "verify", "label": "검증", "why": "다시 셀 수 있는 숫자를 받아요.", "options": [{"i": 0, "text": "val, test의 TP, FN, FP, 재현율, 정밀도를 표로 보여 주고, test는 cloud_band와 off_nadir_band 조건별로도 나눠 표본 수와 함께 보여 줘."}, {"i": 1, "text": "가장 좋은 임계값만 알려 줘."}, {"i": 2, "text": ""}]}, {"id": "out", "label": "문서", "why": "누가 읽을지 정해요.", "options": [{"i": 1, "text": "모델 카드도 만들어 줘."}, {"i": 0, "text": "요청 담당이 읽을 모델 카드를 HTML로 report 변수에 담아 줘. 순서는 결론 한 줄, 근거 표, 조건별 성능, 조심할 조건, 사람이 판독할 조건이야."}, {"i": 2, "text": ""}]}]}]}, {"short": "내 말로 쓰기", "title": "검증을 요청하는 프롬프트 쓰기", "lead": "**새 데이터가 와도** 같은 방식으로 요청할 수 있게 내 말로 써 봐요.", "page": null, "blocks": [{"id": "s3_write", "type": "write", "title": "검증 프롬프트 쓰기", "req": true, "prompt": "**새 요청**: 이번에는 요구 조건이 ‘정밀도 0.8 이상’이에요. 파일 m5_new.csv의 열은 같아요. 답을 믿을 수 있도록 근거까지 요청해 보세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m6": {"steps": [{"short": "숫자로 바꾸기", "title": "모호한 말을 숫자로", "lead": "‘진짜처럼 보이는 것’은 사람마다 달라요. **숫자로 바꾸면** AI와 팀이 같은 뜻으로 알아들어요.", "page": null, "blocks": [{"id": "s1_match", "type": "match", "title": "모호한 말을 숫자 규칙과 짝짓기", "req": true, "text": "왼쪽 말을 프롬프트에 쓸 수 있게 바꾼 문장을 골라 주세요.", "left_label": "모호한 말", "right_label": "프롬프트에 쓸 문장", "lefts": ["전파 반사가 약한 것", "단서를 알 수 없는 것", "그림자가 없는 것", "믿을 만한 탐지"], "rights": ["has_shadow가 0", "sar_db가 -15 미만", "has_shadow 또는 sar_db가 빈칸", "score가 0.6 이상"]}]}, {"short": "규칙 코드 받기", "title": "숫자 규칙과 예외를 담아 요청하기", "lead": "**예외 문장을 빼고도** 보내 보세요. 단서가 없는 O11이 어떻게 판정되는지 눈여겨보세요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "기만체 판별 표", "req": true, "ask": ["choi", "모델이 12개 모두 전차라고 해요. 교차 확인으로 다시 봐 주세요."], "has_start": false, "files": [{"name": "m6_objects.csv", "head": ["object_id", "model_label", "score", "has_shadow", "sar_db", "is_decoy"], "rows": [["O01", "tank", "0.93", "1", "-7.2", "0"], ["O02", "tank", "0.91", "0", "-18.5", "1"], ["O03", "tank", "0.88", "1", "-6.4", "0"], ["O04", "tank", "0.9", "0", "-17.1", "1"], ["O05", "tank", "0.86", "0", "-19.3", "1"]], "n": 12}], "slots": [{"id": "rule", "label": "규칙", "why": "숫자로 정해요.", "options": [{"i": 1, "text": "진짜 장비만 잘 골라 줘."}, {"i": 0, "text": "m6_objects.csv의 열은 object_id, model_label, score, has_shadow, sar_db, is_decoy(정답)야. has_shadow가 0이거나 sar_db가 -15 미만이면 기만 의심, 둘 다 아니면 장비로 판정해 줘."}, {"i": 2, "text": ""}]}, {"id": "edge", "label": "예외", "why": "단서가 빈칸인 경우를 알려요.", "options": [{"i": 1, "text": "빈칸은 알아서 처리해 줘."}, {"i": 0, "text": "has_shadow나 sar_db가 빈칸이면 판정하지 말고 ‘자료 없음, 사람 확인’으로 표시해 줘."}, {"i": 2, "text": ""}]}, {"id": "out", "label": "출력 형식", "why": "비교할 모양으로 받아요.", "options": [{"i": 0, "text": "object_id, 판정, 이유 표와, 모델이 센 수·판정별 수·정답 수를 비교한 표를 보여 주고, HTML 보고서를 report 변수에 담아 줘."}, {"i": 1, "text": "결과를 정리해 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "합의하고 쓰기", "title": "제안을 판단하고 내 말로 요청하기", "lead": "기준값을 바꾸는 일은 **사람이 정하고**, AI에게는 바꾼 값으로 다시 돌려 달라고 하면 돼요.", "page": null, "blocks": [{"id": "s3_decide", "type": "decide", "title": "규칙 조정 제안 판단하기", "req": true, "text": "판별 표를 본 팀원들의 제안이에요.", "reason_min": 8, "items": [{"id": "a", "who": "han", "text": "자료 없음은 번거로우니 전부 장비로 세어 보고해요."}, {"id": "b", "who": "park", "text": "레드팀이 그림자까지 칠한 기만체를 낼 수 있으니, SAR 기준을 -12로 바꾸면 표가 어떻게 달라지는지 돌려 보고 정해요."}]}, {"id": "s3_write", "type": "write", "title": "예외를 담은 프롬프트 쓰기", "req": true, "prompt": "**새 요청**: 두 시점을 비교한 changed 열(움직였으면 1, 아니면 0, 빈칸 있음)이 추가됐어요. 며칠 동안 전혀 움직이지 않은 물체도 기만 의심으로 보고 싶어요. 판별 표를 다시 요청해 보세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m7": {"steps": [{"short": "표현 규칙 익히기", "title": "점수와 확률은 달라요", "lead": "문서를 시키기 전에 **표현 규칙**을 정해 두면 AI가 단정하지 않아요.", "page": null, "blocks": [{"id": "s1_info", "type": "info", "title": "가능성 표현 7단계 (ICD 203)", "icon": "📏", "req": false, "text": ["모델 점수를 그대로 확률이라고 쓰지 않아요. 실제 정답률을 본 뒤 정해진 낱말로 나타내고, 분석 확신도는 근거의 양과 질로 따로 적어요."], "table": {"head": ["표현", "확률"], "rows": [["거의 없음", "1~5%"], ["매우 낮음", "5~20%"], ["낮음", "20~45%"], ["반반", "45~55%"], ["높음", "55~80%"], ["매우 높음", "80~95%"], ["거의 확실", "95~99%"]]}, "figure": "f_scale"}, {"id": "s1_q", "type": "quiz", "title": "어떤 문장이 좋을까요?", "req": true, "text": "수요 부서에 보낼 문장이에요. 어느 쪽이 좋을까요?", "options": ["이 물체는 80% 확실히 항공기입니다.", "이 물체가 항공기일 가능성은 높음(실제 정답률 기준)이고, 자료가 영상 한 장뿐이라 분석 확신도는 낮음입니다.", "이 물체는 틀림없이 항공기로 보입니다."]}]}, {"short": "점수표와 지도", "title": "점수표와 지도, 보고 문장 요청하기", "lead": "**표현 규칙 문장을 빼고** 보내 보세요. AI가 어떤 문장을 쓰는지 직접 보세요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "점수표와 식별 확률 지도", "req": true, "ask": ["lee", "점수를 믿어도 되는지, 항공기가 어디에 모여 있는지 보고 싶어요."], "has_start": false, "files": [{"name": "m7_scores.csv", "head": ["det_id", "x_km", "y_km", "score", "is_correct"], "rows": [["D001", "7.01", "9.57", "0.382", "0"], ["D002", "4.91", "0.71", "0.119", "0"], ["D003", "0.12", "9.74", "0.518", "0"], ["D004", "5.24", "0.99", "0.303", "0"], ["D005", "3.38", "7.19", "0.839", "1"]], "n": 240}], "slots": [{"id": "input", "label": "입력", "why": "열의 뜻을 알려요.", "options": [{"i": 1, "text": "점수 파일이 있어."}, {"i": 0, "text": "m7_scores.csv의 열은 det_id, x_km, y_km(위치), score(가상 모델 점수), is_correct(맞는 탐지면 1)야."}, {"i": 2, "text": ""}]}, {"id": "task", "label": "할 일", "why": "계산 방법을 정해요.", "options": [{"i": 0, "text": "score를 0.1 간격 구간으로 나눠 구간마다 평균 점수, 실제 정답률(is_correct 평균), 표본 수를 표로 만들고, 표본 수로 가중한 평균 차이(ECE)도 계산해 줘."}, {"i": 1, "text": "모델이 정확한지 알려 줘."}, {"i": 2, "text": ""}]}, {"id": "expr", "label": "표현 규칙", "why": "낱말의 뜻을 정해 줘요.", "options": [{"i": 1, "text": "점수를 퍼센트로 바꿔 ‘80% 확실’처럼 적어 줘."}, {"i": 0, "text": "구간별 실제 정답률을 ICD 203 7단계(거의 없음 1~5%, 매우 낮음 5~20%, 낮음 20~45%, 반반 45~55%, 높음 55~80%, 매우 높음 80~95%, 거의 확실 95~99%)로 바꿔 열로 넣어 줘. 모델 점수를 그대로 확률이라고 쓰지 마."}, {"i": 2, "text": ""}]}, {"id": "out", "label": "문서", "why": "독자와 사실 범위를 정해요.", "options": [{"i": 1, "text": "상황이 잘 드러나게 생생하게 보고서를 써 줘."}, {"i": 0, "text": "신뢰도 곡선과 x_km, y_km 위치를 점수로 칠한 식별 확률 지도를 그려 주고, HTML 보고서를 report 변수에 담아 줘. 보고 문장은 가능성 표현과 분석 확신도를 나눠 쓰고, 파일에 없는 내용은 쓰지 마."}, {"i": 2, "text": ""}]}]}]}, {"short": "내 말로 쓰기", "title": "표현 규칙을 담은 프롬프트 쓰기", "lead": "독자가 바뀌어도 쓸 수 있게 **표현 규칙과 사실 범위**를 내 말로 적어 보세요.", "page": null, "blocks": [{"id": "s3_write", "type": "write", "title": "문서 프롬프트 쓰기", "req": true, "prompt": "**새 요청**: 같은 결과로 팀장이 1분 안에 읽을 요약을 만들어 달라고 요청해 보세요. 가능성과 분석 확신도를 나눠 쓰고, 없는 내용을 지어내지 않게 해 주세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m8": {"steps": [{"short": "나눠 보기", "title": "AI에 넣어도 되는 것과 안 되는 것", "lead": "바이브 코딩에서는 **값을 보여 주지 않아도** 코드를 받을 수 있어요. 데이터의 모양만 알려 주면 돼요.", "page": null, "blocks": [{"id": "s1_cls", "type": "classify", "title": "공개 AI에 넣어도 될까요?", "req": true, "text": "조직이 승인한 환경이 아니라 일반 공개 AI라고 생각하고 나눠 보세요.", "primary_label": "판단", "item_label": "내용", "contrib": false, "categories": [["ok", "넣어도 돼요", ""], ["no", "넣으면 안 돼요", ""]], "items": [{"id": "a", "title": "열 이름: item_id, target_class, due_min"}, {"id": "b", "title": "판독 대기열 원본 전체"}, {"id": "c", "title": "직접 지어낸 가상 샘플 2줄"}, {"id": "d", "title": "실제 위치 코드"}]}]}, {"short": "대기열 만들기", "title": "민감한 값을 빼고 대기열 요청하기", "lead": "**파일 전체를 붙여 넣지 않아도** 돼요. 칸마다 문장을 골라 보내 보세요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "판독 대기열 만들기", "req": true, "ask": ["choi", "사람이 볼 대상이 빠지면 안 돼요. 위치 코드는 문서에 나오면 안 되고요."], "has_start": false, "files": [{"name": "m8_queue.csv", "head": ["item_id", "target_class", "calibrated_prob", "ai_label", "due_min", "location_code"], "rows": [["Q01", "normal", "0.93", "truck", "120", "SITE-A12"], ["Q02", "hvt", "0.95", "launcher", "45", "SITE-C03"], ["Q03", "normal", "0.62", "apc", "90", "SITE-A07"], ["Q04", "normal", "0.88", "truck", "180", "SITE-B01"], ["Q05", "hvt", "0.71", "radar", "30", "SITE-C09"]], "n": 10}], "slots": [{"id": "safe", "label": "넣을 정보", "why": "값 대신 모양만 알려요.", "options": [{"i": 1, "text": "대기열 파일 전체를 붙여 넣을게. 알아서 정리해 줘."}, {"i": 0, "text": "실제 데이터는 내 환경에서 돌릴 거라 값은 보여 주지 않을게. 파일은 m8_queue.csv이고 열은 item_id, target_class(hvt·normal), calibrated_prob, ai_label, due_min, location_code야. location_code는 결과에 절대 넣지 마."}, {"i": 2, "text": ""}]}, {"id": "rule", "label": "구분 규칙", "why": "사람이 볼 기준을 숫자로 정해요.", "options": [{"i": 0, "text": "target_class가 hvt이거나 calibrated_prob가 0.8 미만이면 사람 판독, 나머지는 자동 배포로 나누고, 자동 배포에는 ‘machine-generated’ 표시를 붙여 줘."}, {"i": 1, "text": "중요한 것만 사람이 보게 해 줘."}, {"i": 2, "text": ""}]}, {"id": "all", "label": "빠짐없이", "why": "요약하지 말아야 할 것을 정해요.", "options": [{"i": 0, "text": "사람 판독 대상은 하나도 빼지 말고 전부 넣고, hvt 먼저, 그 안에서는 due_min이 짧은 순서로 정렬하고 개수도 출력해 줘."}, {"i": 1, "text": "급한 것만 추려 줘."}, {"i": 2, "text": ""}]}, {"id": "form", "label": "양식", "why": "받는 사람이 할 일이 보이게 해요.", "options": [{"i": 0, "text": "HTML 문서를 report 변수에 담아 줘. 순서는 제목, 사람 판독 개수, 대기열 표, 자동 배포 표, 승인 기록 칸(판독자 A, 판독자 B, 승인자)이야."}, {"i": 1, "text": "보고서 형식으로 만들어 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "내 말로 쓰기", "title": "요청을 판단하고 내 말로 쓰기", "lead": "밖에서 온 요청도 같아요. **무엇을 줘도 되는지** 먼저 생각하세요.", "page": null, "blocks": [{"id": "s3_decide", "type": "decide", "title": "요청 판단하기", "req": true, "text": "다른 곳에서 온 요청과 팀 제안이에요. 어떻게 할까요?", "reason_min": 8, "items": [{"id": "a", "who": "lee", "text": "대기열 원본 전체를 메일로 보내 주세요. 저희 쪽 AI로 분석해 볼게요."}, {"id": "b", "who": "choi", "text": "hvt는 판독자 두 명이 따로 본 뒤 제가 승인하고, 기록을 남겨요."}]}, {"id": "s3_write", "type": "write", "title": "안전한 프롬프트 쓰기", "req": true, "prompt": "**새 요청**: 장비 점검 기록(파일 m8_equip.csv)에서 고장 난 장비 목록을 뽑는 코드가 필요해요. 파일에 담당자 이름과 설치 위치가 있어요. 값을 넘기지 않고 요청해 보세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m9": {"steps": [{"short": "먼저 설명 듣기", "title": "고치기 전에 설명을 듣는 이유", "lead": "AI 코드는 **고치기 전에 먼저 설명**하게 하면 엉뚱한 수정을 줄일 수 있어요.", "page": null, "blocks": [{"id": "s1_info", "type": "info", "title": "AI 코드를 고칠 때 순서", "icon": "🩺", "req": false, "text": ["AI는 빠르게 고치지만, 무엇을 왜 바꿨는지 묻지 않으면 알기 어려워요."], "bullets": ["**설명**: 지금 코드가 무엇을 하는지, 어디가 의심스러운지", "**테스트**: 답을 아는 작은 예시", "**범위**: 고칠 곳과 건드리면 안 되는 곳", "**비교**: 고치기 전후의 숫자"], "figure": "f_review"}, {"id": "s1_q", "type": "quiz", "title": "어떤 요청이 가장 안전할까요?", "req": true, "text": "숫자가 이상해 보이는 코드가 있어요.", "options": ["코드 전체를 네가 알아서 싹 고쳐 줘.", "고치지 말고 먼저 이 함수가 무엇을 계산하는지 설명하고, 의심되는 곳을 알려 줘.", "데이터가 이상한 것 같으니 이상한 행은 지워 줘."]}]}, {"short": "리뷰·테스트·수정", "title": "리뷰와 테스트를 시키고 범위를 정해 고치기", "lead": "**범위 문장을 빼고** 보내 보세요. AI가 기록을 어떻게 ‘정리’하는지 직접 보세요.", "page": null, "blocks": [{"id": "s2_vibe", "type": "vibe", "title": "집계 코드 고치기", "req": true, "ask": ["jung", "오판독률은 AI가 판정한 것(unknown 제외) 가운데 틀린 비율이어야 해요. 지금 숫자가 맞는지 봐 주세요."], "has_start": true, "files": [{"name": "m9_cases.csv", "head": ["case_id", "ai_label", "truth", "cause"], "rows": [["C001", "truck", "tank", "data"], ["C002", "truck", "truck", "none"], ["C003", "unknown", "tank", "none"], ["C004", "tank", "decoy", "communication"], ["C005", "truck", "truck", "none"]], "n": 60}], "slots": [{"id": "review", "label": "먼저 설명", "why": "고치기 전에 진단을 받아요.", "options": [{"i": 1, "text": "이 코드 고쳐 줘."}, {"i": 0, "text": "고치기 전에 misread_rate가 무엇을 나누고 있는지 설명하고, 내가 원하는 ‘unknown을 뺀 판정 가운데 틀린 비율’과 무엇이 다른지 알려 줘."}, {"i": 2, "text": ""}]}, {"id": "test", "label": "테스트", "why": "답을 아는 예시를 받아요.", "options": [{"i": 0, "text": "손으로 계산할 수 있는 작은 예시(unknown을 뺀 판정 4건 중 틀린 것 1건이면 0.25)로 확인하는 assert 테스트를 함께 만들어 줘."}, {"i": 1, "text": "잘 동작하는지 봐 줘."}, {"i": 2, "text": ""}]}, {"id": "scope", "label": "수정 범위", "why": "건드리면 안 되는 곳을 정해요.", "options": [{"i": 1, "text": "더 좋아 보이게 데이터도 정리해 줘."}, {"i": 0, "text": "고치는 건 misread_rate 함수만 해. m9_cases.csv 원본과 행은 지우거나 바꾸지 마. 전후 숫자와 cause별 건수를 표로 보여 주고, HTML 사후검토 문서를 report 변수에 담아 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "내 말로 쓰기", "title": "조치를 정하고 내 말로 요청하기", "lead": "숫자가 고쳐졌으면 이제 **무엇을 바꿀지** 팀이 정해요. AI에게는 안전하게 다음 작업을 요청해요.", "page": null, "blocks": [{"id": "s3_decide", "type": "decide", "title": "사후검토 후속 제안 판단하기", "req": true, "text": "고친 숫자를 본 뒤 나온 제안이에요.", "reason_min": 8, "items": [{"id": "a", "who": "han", "text": "AI가 원본 기록을 직접 고칠 수 있게 쓰기 권한을 줘요. 그러면 일이 빨라져요."}, {"id": "b", "who": "kim", "text": "건수가 가장 많은 원인 하나를 골라 기준을 바꾸고, 다음 판독 기록으로 다시 비교해요."}]}, {"id": "s3_write", "type": "write", "title": "리뷰 프롬프트 쓰기", "req": true, "prompt": "**새 요청**: 다른 팀에서 받은 평균 판독 시간 계산 코드가 이상해 보여요. AI에게 어떻게 요청할까요? 고치기 전 설명, 테스트, 범위를 담아 보세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 30}]}]}, "m10": {"steps": [{"short": "도구로 묶기", "title": "미니 도구와 보고서 요청하기", "lead": "지금까지 배운 것을 모두 써 보세요. **네 칸, 숫자 규칙, 근거 수치, 범위**가 모두 필요해요.", "page": null, "blocks": [{"id": "s1_vibe", "type": "vibe", "title": "보고서 자동화 도구", "req": true, "ask": ["kim", "GSD 구간별 재현율과 오경보 비율, 보고서와 자동 점검까지 한 번에 나오게 해 주세요."], "has_start": false, "files": [{"name": "m10_run.csv", "head": ["run_id", "gsd_band", "cloud_band", "target_type", "alert", "human_checked"], "rows": [["R001", "1m", "clear", "aircraft", "1", "1"], ["R002", "0.5m", "clear", "other", "0", "0"], ["R003", "0.5m", "cloudy", "other", "0", "0"], ["R004", "0.5m", "clear", "other", "0", "0"], ["R005", "1m", "cloudy", "other", "0", "0"]], "n": 200}], "slots": [{"id": "tool", "label": "구성", "why": "다시 돌릴 수 있게 해요.", "options": [{"i": 1, "text": "성능 분석 코드 짜 줘."}, {"i": 0, "text": "m10_run.csv의 열은 run_id, gsd_band(0.5m·1m·2m), cloud_band, target_type(aircraft·other), alert(탐지면 1), human_checked(사람이 확인했으면 1)야. 파일 이름과 표본 기준을 맨 위 설정값으로 두고, 계산은 함수로 나눠 줘. 다음 달 파일로도 바로 돌릴 수 있게 해 줘."}, {"i": 2, "text": ""}]}, {"id": "metric", "label": "지표", "why": "무엇을 어떻게 나눌지 정해요.", "options": [{"i": 0, "text": "gsd_band별로 aircraft의 재현율(alert 평균), other의 오경보 비율(alert 평균), 표본 수를 표로 만들어 줘. aircraft 표본이 30건 미만이면 ‘표본 적음’으로 표시하고, 사람이 확인한 비율도 알려 줘."}, {"i": 1, "text": "조건별 성능을 보여 줘."}, {"i": 2, "text": ""}]}, {"id": "report", "label": "보고서", "why": "독자와 조건 표시, 점검을 정해요.", "options": [{"i": 1, "text": "정확도가 높다고 보고서 써 줘."}, {"i": 0, "text": "요청 담당이 읽을 HTML 보고서를 report 변수에 담아 줘. 순서는 결론 한 줄, 조건별 표, 그림, 한계, AI 산출물 표기야. 수치에는 가상 데이터 몇 건 기준인지 적고, 표본 수와 AI 산출물 표기가 빠졌는지 검사하는 lint_report 함수도 만들어 줘."}, {"i": 2, "text": ""}]}]}]}, {"short": "피드백과 점검", "title": "수요자 피드백과 보고서 점검", "lead": "보고서를 올리기 전에 **사람이 마지막으로 확인**해요.", "page": null, "blocks": [{"id": "s2_decide", "type": "decide", "title": "수요자 피드백 판단하기", "req": true, "text": "보고서를 받은 사람들의 요청이에요. 받아들일지 정하고 이유를 적어 주세요.", "reason_min": 8, "items": [{"id": "a", "who": "lee", "text": "‘완전히 식별함’이라고 단정해 주세요. 읽는 사람이 헷갈려요."}, {"id": "b", "who": "lee", "text": "맨 위에 결론 한 줄을 두고, 표는 그 아래에 놓아 주세요."}]}, {"id": "s2_multi", "type": "multi", "title": "올리기 전에 확인할 것", "req": true, "text": "AI가 만든 보고서를 올리기 전에 꼭 확인할 것을 모두 골라 주세요.", "items": ["표의 숫자가 파일에서 다시 계산한 값과 같은지 한두 개 대조한다.", "수치마다 데이터와 표본 수, 조건이 적혀 있는지 본다.", "문장이 매끄러우면 내용도 맞으니 건너뛴다.", "파일에 없는 내용이 섞이지 않았는지 본다.", "AI가 만든 문서이니 작성 책임은 AI에게 있다고 적는다."]}]}, {"short": "프롬프트 모음", "title": "나만의 프롬프트 틀 만들기", "lead": "이번 프로젝트의 **가장 큰 산출물**이에요. 다음 업무에서 바로 쓸 나만의 틀을 남겨요.", "page": null, "blocks": [{"id": "s3_write", "type": "write", "title": "내 프롬프트 틀 쓰기", "req": true, "prompt": "앞으로 자주 쓸 **프롬프트 틀**을 써 보세요. 파일 이름이나 숫자처럼 바뀌는 자리는 [대괄호]로 비워 두세요. 네 칸에 더해 검증, 범위, 민감 정보 내용도 넣어 보세요.", "placeholder": "꼭 입력해 주세요 · 내 말로 프롬프트를 써 보세요", "model_label": "📝 모범 프롬프트", "min": 40}]}]}};   // 미션 단계와 블록 (정답·힌트·풀이는 빠져 있음)
// ===================== 상태 =====================
let A = null, view = 'welcome', seen = {}, mstep = {}, cache = {}, drafts = {};
let ui = { w: 'full', h: 900, side: true, code: null };
try { Object.assign(ui, JSON.parse(localStorage.getItem(PROJ.ui_key) || '{}')); } catch (e) {}
try { google.colab.output.setIframeHeight(0, true, { maxHeight: 5000 }); } catch (e) {}
const $ = id => document.getElementById(id);
const NUM = n => '①②③④⑤⑥⑦⑧⑨⑩⑪⑫'[n - 1] || n;

function h(tag, props, ...kids) {
  const e = document.createElement(tag);
  for (const [k, v] of Object.entries(props || {})) {
    if (v === undefined || v === null || v === false) continue;
    if (k === 'class') e.className = v;
    else if (k === 'html') e.innerHTML = v;            // 엔진·화면에 적힌 신뢰할 수 있는 문구에만 사용
    else if (k.startsWith('on')) e[k] = v;
    else if (k === 'style') e.style.cssText = v;
    else if (k === 'value') e.value = v;
    else e.setAttribute(k, v === true ? '' : v);
  }
  for (const c of kids.flat(9)) if (c !== null && c !== undefined && c !== false) e.append(c instanceof Node ? c : document.createTextNode(String(c)));
  return e;
}
const esc = s => String(s).replace(/&/g, '&amp;').replace(/</g, '&lt;').replace(/>/g, '&gt;');

let busy = 0;
async function invoke(name, ...args) {
  busy++; $('ws-root').classList.add('busy');
  try {
    const res = await google.colab.kernel.invokeFunction(name, args, {});
    const d = res.data['application/json'];
    if (d.app) refresh(d.app);
    if (d.msg) note(d.msg, d.ok === false ? 'bad' : 'ok');
    if (d.code) setLive(d.code, d.fn);
    return d;
  } catch (e) {
    note('⚠️ 파이썬과 연결되지 않았어요. 위의 🚀 [먼저 실행] 실습 준비하기 셀과 이 셀을 차례로 다시 실행해 주세요.', 'bad');
    throw e;
  } finally { busy--; if (!busy) $('ws-root').classList.remove('busy'); }
}
function note(t, kind) { const n = $('note'); n.textContent = t || ''; n.className = 'note' + (kind ? ' ' + kind : ''); }

// ===================== 화면 틀 =====================
function codeOn() { return ui.code === null ? $('ws-root').offsetWidth >= 1250 : ui.code; }
function applyUi() {
  const root = $('ws-root');
  root.classList.remove('w-base', 'w-wide', 'w-full');
  root.classList.add('w-' + ui.w);
  root.style.width = '';
  root.style.height = ui.h + 'px';
  root.classList.toggle('side-off', !ui.side);
  root.classList.toggle('code-on', codeOn());
  root.classList.toggle('compact', root.offsetWidth < 1000);
  document.querySelectorAll('.ws [data-w]').forEach(b => b.classList.toggle('on', b.dataset.w === ui.w));
  $('tb-side').classList.toggle('on', !ui.side);
  $('tb-code').classList.toggle('on', codeOn());
  try { localStorage.setItem(PROJ.ui_key, JSON.stringify(ui)); } catch (e) {}
}
function setWidth(w) { ui.w = w; applyUi(); }
function setHeight(d) { ui.h = Math.max(480, Math.min(2000, $('ws-root').offsetHeight + d)); applyUi(); }
function toggleSide() { ui.side = !ui.side; applyUi(); }
function toggleCode(force) { ui.code = force === true ? true : !codeOn(); applyUi(); if (codeOn()) renderCode(); }
new ResizeObserver(() => {
  const root = $('ws-root');
  root.classList.toggle('compact', root.offsetWidth < 1000);
  root.classList.toggle('code-on', codeOn());
  $('tb-code').classList.toggle('on', codeOn());
  const hh = root.offsetHeight;
  if (hh && Math.abs(hh - ui.h) > 2) { ui.h = hh; try { localStorage.setItem(PROJ.ui_key, JSON.stringify(ui)); } catch (e) {} }
}).observe($('ws-root'));
new ResizeObserver(() => { const m = $('main'); m.classList.toggle('narrow', m.offsetWidth < 760); }).observe($('main'));

// ===================== 상태 갱신 =====================
function refresh(app) {
  const first = !A;
  if (A && app.boot && A.boot && app.boot !== A.boot) {   // 파이썬이 새로 시작됨(런타임 재시작·셀 재실행) → 화면도 처음 상태로
    mstep = {}; cache = {}; drafts = {}; BUSY = null; for (const k in POLL) clearTimeout(POLL[k]);
    A = app; view = 'welcome'; renderTop(); renderSide(); renderMain();
    note(PROJ.boot_note, 'bad');
    return;
  }
  A = app;
  renderTop(); renderSide();
  if (first || !view.startsWith('mission:')) renderMain();
}
function openView(v) { if (v !== view && busyBlock()) return; view = v; renderSide(); renderMain(); }

// ===================== 상단: 진행률 · 내 정보 =====================
function renderTop() {
  const o = A.overall;
  $('tb-prog').innerHTML = '';
  $('tb-prog').append(h('span', { class: 'pb' }, h('i', { style: `width:${o.pct}%` })), h('b', {}, `전체 진행 ${o.pct}%`), h('span', {}, `· 미션 ${o.done}/${o.total} 완료`));
  $('tb-me').textContent = `👤 ${A.call}`;
}

// ===================== 첫 화면: 입력 없이 인사만 (로그인 없음) =====================
function renderWelcome(main) {
  main.append(h('div', { class: 'scroll' }, h('div', { class: 'onb welcome' },
    h('div', { class: 'wave' }, '👋'), h('h2', {}, `어서오세요, ${A.call}`),
    h('p', { class: 'lead' }, PROJ.welcome_lead),
    h('div', { class: 'row', style: 'justify-content:center;margin-top:18px' }, h('button', { class: 'btn big pulse', id: 'go-home', onclick: () => openView('home') }, '시작하기 ▶')))));
}

// ===================== 사이드바 =====================
const ICON = { done: '✅', open: '▶', soon: '⏳' };
function renderSide() {
  const side = $('side'); side.innerHTML = '';
  $('ws-root').classList.toggle('onboard', view === 'welcome');
  if (view === 'welcome') return;
  const item = (key, icon, label, extra, cls) => side.append(h('div', { class: 'item' + (view === key ? ' active' : '') + (cls ? ' ' + cls : ''), title: label, onclick: () => openView(key) },
    h('span', { class: 'ic' }, icon), h('span', { class: 't' }, label), extra));
  item('home', '🏠', '프로젝트 개요');
  side.append(h('h4', {}, h('b', {}, '채널')));
  let total = 0;
  for (const [id, name] of Object.entries(A.channels)) {
    const n = A.msgs.filter(m => m.ch === id && m.id >= (seen[id] || 0)).length;
    total += n;
    item(id, '', name, n ? h('span', { class: 'badge' }, n) : null, n ? 'unread' : '');
  }
  for (const s of A.project.stages) {
    side.append(h('h4', {}, h('b', {}, s.no + '단계'), ' · ' + s.name));
    for (const mid of s.missions) {
      const m = A.missions[mid];
      const comp = (A.views[mid] || {}).complete || [];
      const mini = m.status === 'open' && m.progress.started ? h('span', { class: 'mini' }, `${comp.filter(Boolean).length}/${comp.length}`) : null;
      item('mission:' + mid, ICON[m.status], `M${m.no} ${m.short}`, mini, m.status === 'soon' ? 'soon' : '');
    }
  }
  side.append(h('h4', {}, h('b', {}, '앱')));
  item('files', '📁', '파일', A.files.length ? h('span', { class: 'badge', style: 'background:#1164a3' }, A.files.length) : null);
  item('roster', '👥', '팀원');
  $('tb-count').textContent = total;
  $('tb-count').classList.toggle('has', total > 0);
}

// ===================== 안내 띠 =====================
function nextTodo() {
  for (const s of A.project.stages) for (const mid of s.missions) { const m = A.missions[mid]; if (m.status === 'open') return m; }
  return null;
}
// ===================== 본 과정의 코드 보기 =====================
const CODE = { topic: null, data: {}, live: null, liveFn: null, open: null };
function hl(code) {
  const re = /(#[^\n]*)|("{3}[\s\S]*?"{3}|'{3}[\s\S]*?'{3}|\b[rbfu]{0,2}"(?:\\.|[^"\\\n])*"|\b[rbfu]{0,2}'(?:\\.|[^'\\\n])*'|"(?:\\.|[^"\\\n])*"|'(?:\\.|[^'\\\n])*')|\b(def|return|if|elif|else|for|in|while|import|from|as|with|not|and|or|is|None|True|False|lambda|try|except|class|yield|global|continue|break|pass)\b|\b(\d+\.?\d*(?:e[-+]?\d+)?)\b|\b([A-Za-z_]\w*)(?=\()/g;
  let out = '', last = 0, m;
  while ((m = re.exec(code))) {
    out += esc(code.slice(last, m.index));
    const cls = m[1] ? 'c' : m[2] ? 's' : m[3] ? 'k' : m[4] ? 'n' : 'f';
    out += `<span class="${cls}">${esc(m[0])}</span>`;
    last = re.lastIndex;
  }
  return out + esc(code.slice(last));
}
function copyText(t) {
  const done = () => note('📋 코드를 복사했어요. 노트북에 새 코드 셀을 만들고 붙여 넣어 실행해 보세요.', 'ok');
  const fallback = () => {          // Colab 출력 창에서 클립보드 권한이 막혀 있을 때
    const ta = h('textarea', { style: 'position:fixed;left:-9999px;top:0' }); ta.value = t; document.body.append(ta); ta.select();
    let ok = false; try { ok = document.execCommand('copy'); } catch (e) {}
    ta.remove();
    if (ok) done(); else note('복사하지 못했어요. 코드를 드래그해서 복사해 주세요.', 'bad');
  };
  try { navigator.clipboard.writeText(t).then(done, fallback); } catch (e) { fallback(); }
}
async function setTopic(topic) {
  CODE.topic = topic;
  if (topic && !CODE.data[topic]) { try { CODE.data[topic] = await invoke('app.code', topic); } catch (e) {} }
  renderCode();
}
function setLive(code, fn) { CODE.live = code; CODE.liveFn = fn || null; if (fn) CODE.open = fn; renderCode(); }
function codeBlock(b, open) {
  const lines = b.code.split('\n');
  const cw = h('div', { class: 'cw' });
  for (const i of b.hl) cw.append(h('div', { class: 'hlb', style: `top:${8 + i * 18}px` }));
  cw.append(h('pre', { class: 'code', html: hl(b.code) }));
  return h('details', { open: open, 'data-fn': b.name }, h('summary', {}, h('code', {}, b.name + '()'), b.doc ? ': ' + b.doc : '', ' ',
      h('button', { class: 'cpy', onclick: e => { e.preventDefault(); copyText(b.code); } }, '📋 복사')),
    h('div', { class: 'cb' }, h('pre', { class: 'ln' }, lines.map((_, i) => i + 1).join('\n')), cw));
}
// 📘 본 과정의 가이드북: [핵심 용어] 탭(미션 소개의 용어 + 단계마다 나오는 어려운 말) · [코드 보기] 탭(지금 화면의 파이썬 코드)
const GUIDE = { tab: 'terms', focus: null };
function openGuide(tab, focus) { GUIDE.tab = tab || 'terms'; GUIDE.focus = focus || null; toggleCode(true); renderCode(); }
function guideTerms(sc) {
  const mid = view.startsWith('mission:') ? view.slice(8) : null, m = mid ? A.missions[mid] : null;
  if (!m || !m.brief) { sc.append(h('p', { class: 'desc' }, '미션을 열면 그 미션의 핵심 용어가 여기에 나와요.')); return; }
  const st = curStep(mid), g = m.brief.glossary || [];
  sc.append(h('div', { class: 'intro' }, '미션 소개에 나온 ', h('b', {}, '핵심 용어'), '와, 단계마다 나오는 어려운 말을 다시 볼 수 있어요. ',
    h('b', {}, '노란 테두리'), '는 지금 단계에 나오는 말이에요. 코드는 위의 ', h('b', {}, '💻 코드 보기'), ' 탭에 있어요.'));
  // 단계 이름은 머리글로 한 번만: 용어 이름과 같아도 헷갈리지 않게 ‘N단계 「이름」’으로 써요 (2026-10-20 제작자 피드백)
  const stepName = k => `${k}단계 「${m.brief.steps_short[k - 1] || ''}」`;
  const now = g.filter(t => t.step === st), rest = g.filter(t => t.step !== st);
  if (now.length) sc.append(h('h6', {}, `🔎 지금 ${stepName(st)}에 나오는 말`), ...now.map(t => termCard(t, true)));
  sc.append(h('h6', {}, `📖 M${m.no} 핵심 용어 (미션 소개)`), ...m.brief.terms.map(t => termCard(t)));
  if (rest.length) {
    const byStep = [...new Set(rest.map(t => t.step))].sort((a, b) => a - b);
    sc.append(h('details', { open: !!(GUIDE.focus && rest.some(t => t.t === GUIDE.focus)) }, h('summary', {}, `M${m.no}의 다른 단계에 나오는 말 (${rest.length}개)`),
      ...byStep.map(k => h('div', { class: 'gstep' }, h('div', { class: 'gstep-h' }, `🧭 ${stepName(k)}에 나오는 말`), ...rest.filter(t => t.step === k).map(t => termCard(t))))));
  }
  const others = Object.values(A.missions).filter(x => x.brief && x.id !== mid);
  if (others.length) sc.append(h('details', {}, h('summary', {}, '다른 미션의 핵심 용어'),
    ...others.map(x => h('div', {}, h('h6', {}, `M${x.no}. ${x.title}`), ...x.brief.terms.map(t => termCard(t))))));
  if (GUIDE.focus) {
    const el = [...sc.querySelectorAll('.term')].find(e => e.dataset.t === GUIDE.focus);
    if (el) { el.classList.add('flash'); setTimeout(() => el.scrollIntoView({ block: 'start', behavior: 'smooth' }), 30); }
    GUIDE.focus = null;
  }
}
function guideCode(sc) {
  const d = CODE.data[CODE.topic];
  if (!(d && d.ok) && !CODE.live) { sc.append(h('p', { class: 'desc' }, '이 화면에는 따로 보여 줄 코드가 없어요. 직접 계산하거나 파일을 읽는 활동에서 그 코드를 보여 줘요.')); return; }
  sc.append(h('div', { class: 'intro' }, '지금 화면이 파이썬 코드로 어떻게 동작하는지 보여 줘요. 고칠 필요는 없어요. ',
    h('b', {}, '📋 복사'), '를 눌러 노트북의 새 코드 셀에 붙여 넣고 실행하면 같은 결과를 직접 볼 수 있어요. ', h('b', {}, '노란 줄'), '은 이 단계의 핵심 부분이에요.'));
  sc.append(h('h6', {}, '▶ 방금 실행된 코드', h('span', { class: 'sp' }), CODE.live ? h('button', { class: 'cpy', onclick: () => copyText(CODE.live) }, '📋 복사') : null),
    h('div', { class: 'live' }, CODE.live ? h('pre', { html: hl(CODE.live) }) : h('div', { class: 'empty' }, '버튼을 누르거나 조건을 바꾸면, 그때 실행된 코드가 여기에 나타나요.')));
  if (d && d.ok) {
    sc.append(h('h6', {}, '📄 ' + d.title), h('p', { class: 'desc' }, d.desc));
    const openName = d.blocks.some(b => b.name === CODE.open) ? CODE.open : d.blocks[0].name;
    for (const b of d.blocks) sc.append(codeBlock(b, b.name === openName));
  } else sc.append(h('p', { class: 'desc' }, '이 화면에는 따로 보여 줄 코드가 없어요.'));
}
function renderCode() {
  const p = $('codep');
  if (!codeOn()) return;
  p.innerHTML = '';
  const sc = h('div', { class: 'cs' });
  const tabs = h('div', { class: 'gtabs' }, [['terms', '📖 핵심 용어'], ['code', '💻 코드 보기']].map(([k, t]) =>
    h('button', { class: GUIDE.tab === k ? 'on' : '', onclick: () => { GUIDE.tab = k; renderCode(); } }, t)));
  if (GUIDE.tab === 'terms') guideTerms(sc); else guideCode(sc);
  p.append(h('div', { class: 'ch' }, h('b', {}, '📘 본 과정의 가이드북'), h('button', { onclick: () => toggleCode() }, '닫기 ✕')), tabs, sc);
  if (GUIDE.tab === 'code') {
    const opened = sc.querySelector(`details[data-fn="${CODE.open}"]`);
    if (opened && CODE.liveFn) opened.scrollIntoView({ block: 'nearest' });
  }
}

// ===================== 공통 위젯 =====================
function taskBox() { return { el: document.createComment('task'), set() {} }; }   // 할 일 상자는 화면에서 뺐어요
function choiceGroup(name, options, value, onchange, opts) {
  opts = opts || {};
  const wrap = h('div', { class: 'choices' + (opts.col ? ' col' : '') });
  options.forEach((o, i) => {
    const [v, label, small] = Array.isArray(o) ? o : [i, o];
    const inp = h('input', { type: 'radio', name, value: v, checked: value !== null && value !== undefined && String(value) === String(v) });
    inp.onchange = () => { wrap.querySelectorAll('.choice').forEach(l => { l.classList.toggle('on', l.querySelector('input').checked); l.classList.remove('wrong'); }); if (onchange) onchange(String(v)); };
    wrap.append(h('label', { class: 'choice' + (inp.checked ? ' on' : '') }, inp, h('span', {}, label, small ? h('small', {}, small) : null)));
  });
  wrap.value = () => { const s = wrap.querySelector('input:checked'); return s ? s.value : ''; };
  return wrap;
}
function textCounter(ta, min) {
  const line = h('div', { class: 'count-line' });
  const upd = () => { const n = ta.value.trim().length; line.textContent = n >= min ? `✔ ${n}자 (충분해요)` : `지금 ${n}자 · ${min}자 이상 적어야 해요`; line.className = 'count-line ' + (n >= min ? 'enough' : 'short'); };
  ta.addEventListener('input', upd); upd();
  return line;
}
function flagMissing(el, msgEl, msg) {
  if (msgEl) msgEl.textContent = msg ? '⚠ ' + msg : '';
  if (el) el.classList.toggle('missing', !!msg);
}
function scrollTo(el) { if (el) el.scrollIntoView({ block: 'center', behavior: 'smooth' }); }
function nextBtn(mid, k, ready, why, label) {
  const msg = h('div', { class: 'err' });
  const btn = h('button', { class: 'btn', onclick: () => { const r = ready ? ready() : true; if (r) goStep(mid, k); else { msg.textContent = '⚠ ' + why(); } } }, label || '다음 단계 ▶');
  return { el: h('div', {}, h('div', { class: 'row' }, btn), msg), btn, msg };
}
// 틀린 선택 표시: 고른 보기를 빨간색으로 (다른 보기를 고르면 choiceGroup이 지움)
function markWrong(group) { const inp = group && group.querySelector('input:checked'); if (inp) inp.closest('.choice').classList.add('wrong'); }
function figure(key) { return FIG[key] ? h('div', { class: 'fig', html: FIG[key] }) : null; }

// ===================== 그림 (용어 설명) =====================
// 핵심 용어 카드에 붙는 작은 SVG 그림. 프로젝트 명세의 figures가 여기에 들어가요 (termCard의 fig 이름으로 찾음).
const T = 'font-family="sans-serif" font-size="10" fill="#111827"';
const FIG = Object.assign({}, PROJ_FIGS);

// ===================== 홈 · 채팅 · 팀원 · 파일 =====================
function renderMain() {
  if (!A) return;
  $('ws-root').classList.add('no-code');          // 코드 보기는 미션 실습 단계에서만 보여요
  const main = $('main'); main.innerHTML = '';
  if (view === 'welcome') return renderWelcome(main);
  if (view === 'home') { return renderHome(main); }
  if (view === 'files') { return renderFiles(main); }
  if (view === 'roster') { return renderTeam(main); }
  if (view.startsWith('mission:')) return renderMission(main, view.slice(8));
  if (A.channels[view]) { return renderChat(main, view); }
  view = 'home'; renderHome(main);
}
function titleBar(main, text, small) { main.append(h('div', { class: 'title' }, small ? h('small', {}, small) : null, text)); }

function renderHome(main) {
  titleBar(main, '프로젝트 개요', '🏠');
  const sc = h('div', { class: 'scroll' });
  const o = A.overall;
  sc.append(h('div', { class: 'hero' }, h('div', { style: 'font-size:12px' }, '프로젝트'), h('h2', {}, A.project.title), h('p', {}, A.project.goal)));
  if (A.teacher) sc.append(h('div', { class: 'tip', style: 'margin-top:12px' }, '🧑‍🏫 선생님용 파일이에요. 모든 활동의 값이 채워져 있고, 산출물도 📁 파일에 만들어져 있어요.'));
  if (A.assets && !A.assets.ok) sc.append(assetNotice());
  sc.append(h('div', { class: 'greet' }, `👋 어서오세요, ${A.call}.`),
    h('div', { class: 'card' }, h('div', { class: 'bigprog' }, h('b', {}, '전체 진행'), h('span', { class: 'pb' }, h('i', { style: `width:${o.pct}%` })), h('b', {}, o.pct + '%')),
      h('div', { class: 'hint', style: 'margin-top:6px' }, `미션 ${o.total}개 중 ${o.done}개 완료 · 끝낸 활동만큼 올라가요.`)));
  const nx = nextTodo();
  if (nx) sc.append(h('div', { class: 'row' }, h('button', { class: 'btn big pulse', onclick: () => openView('mission:' + nx.id) },
    `▶ ${nx.progress.started ? '이어서 하기' : '시작하기'}: M${nx.no} ${nx.title}`)));
  const grid = h('div', { class: 'stages' });
  for (const s of A.project.stages) {
    const box = h('div', { class: 'stage' }, h('h5', {}, `${s.no}단계 · ${s.name}`), s.subtitle ? h('div', { class: 'hint stage-sub' }, s.subtitle) : null);
    for (const mid of s.missions) {
      const m = A.missions[mid];
      const label = { done: '완료', open: '진행 가능', soon: '준비 중' }[m.status];
      box.append(h('div', { class: 'mcard', onclick: () => openView('mission:' + mid) }, h('span', {}, ICON[m.status]),
        h('div', { style: 'flex:1' }, h('div', {}, h('b', {}, `M${m.no} `), m.title), m.brief && m.brief.subtitle ? h('div', { class: 'hint msub' }, m.brief.subtitle) : null, h('div', { class: 'hint' }, m.kind === '협업' ? '👥 협업' : '👤 개인')),
        h('span', { class: 'pill ' + m.status }, label)));
    }
    grid.append(box);
  }
  sc.append(h('h4', { style: 'margin:18px 0 0' }, '단계와 미션'), grid);
  main.append(sc);
}

function renderChat(main, ch) {
  seen[ch] = A.msgs.length;
  titleBar(main, A.channels[ch]);
  const box = h('div', { class: 'msgs' });
  for (const m of A.msgs.filter(m => m.ch === ch)) {
    const who = A.members[m.who];
    box.append(h('div', { class: 'msg' + (m.who !== 'me' && m.text.includes(A.call) ? ' mention' : '') },
      h('div', { class: 'avatar', style: 'background:' + who.color }, who.name[0]),
      h('div', {}, h('div', { class: 'meta' }, h('b', {}, who.name), who.role, h('span', { class: 'tag' + (who.ext ? ' ext' : '') }, who.org), h('span', { class: 'time' }, m.time)),
        h('div', { class: 'text' }, m.text))));
  }
  const input = h('input', { type: 'text', placeholder: '메시지 보내기 (Enter) · 가상 팀원은 미션 진행에 맞춰 메시지를 보내요' });
  const send = () => { if (input.value.trim()) { invoke('app.send', ch, input.value); input.value = ''; } };
  input.addEventListener('keydown', e => { if (e.key === 'Enter' && !e.isComposing) send(); });
  main.append(box, h('div', { class: 'compose' }, input, h('button', { class: 'btn', onclick: send }, '보내기')));
  box.scrollTop = box.scrollHeight;
  renderSide();
}

function renderTeam(main) {
  titleBar(main, '팀원', '👥');
  const sc = h('div', { class: 'scroll' });
  for (const m of Object.values(A.members)) sc.append(h('div', { class: 'msg' }, h('div', { class: 'avatar', style: 'background:' + m.color }, m.name[0]),
    h('div', {}, h('b', {}, m.name), ' ', m.role, h('span', { class: 'tag' + (m.ext ? ' ext' : '') }, m.org))));
  main.append(sc);
}

function renderFiles(main, openId) {
  titleBar(main, '파일', '📁');
  const sc = h('div', { class: 'scroll' });
  main.append(sc);
  if (!A.files.length) { sc.append(h('div', { class: 'tip' }, '아직 산출물이 없어요. 미션의 [산출물] 단계에서 받으면 여기에 저장돼요.')); return; }
  sc.append(h('div', { class: 'tip' }, '💾 산출물은 노트북이 켜져 있는 동안만 보관돼요. 받을 때마다 [⬇ 내려받기]로 내 컴퓨터에 저장해 두세요.'));
  const prev = h('div');
  for (const f of A.files) sc.append(h('div', { class: 'file' }, h('b', {}, '📄 ' + f.title), h('span', { class: 'hint' }, `  ${f.name} · M${A.missions[f.mission].no}`), f.blanks ? h('span', { class: 'pill soon', style: 'margin-left:6px' }, `안 한 활동 ${f.blanks}개`) : null,
    h('div', { class: 'row' }, h('button', { class: 'btn small ghost', onclick: () => showFile(f, prev) }, '미리보기'), h('button', { class: 'btn small', onclick: () => downloadFile(f) }, '⬇ 내려받기'))));
  sc.append(prev);
  showFile(A.files.find(f => f.id === openId) || A.files[A.files.length - 1], prev);
}
async function showFile(f, box) { const d = await invoke('app.file', f.id); box.innerHTML = ''; box.append(h('h4', {}, '미리보기: ' + f.title), h('iframe', { class: 'preview', srcdoc: d.html })); }
async function downloadFile(f) {
  const d = await invoke('app.file', f.id);
  const a = h('a', { href: URL.createObjectURL(new Blob([d.html], { type: 'text/html' })), download: d.name });
  document.body.append(a); a.click(); a.remove();
}
function openFile(id) { if (busyBlock()) return; view = 'files'; renderSide(); const main = $('main'); main.innerHTML = ''; renderFiles(main, id); }

// ===================== 미션 공통 =====================
function nSteps(m) { return m.brief ? m.brief.steps_short.length : 0; }
function curStep(mid) { return mstep[mid] !== undefined ? mstep[mid] : 0; }
function goStep(mid, k) { if (busyBlock()) return; mstep[mid] = k; renderSide(); renderMain(); }
function renderMission(main, mid) {
  const m = A.missions[mid];
  if (m.status === 'soon' || !m.brief) {
    setTopic(null);
    main.append(h('div', { class: 'scroll brief' }, h('span', { class: 'pill soon' }, '준비 중'), h('h2', {}, `M${m.no}. ${m.title}`),
      h('p', { class: 'sub-t' }, (m.kind === '협업' ? '👥 협업' : '👤 개인') + ' 미션'),
      h('section', {}, h('h4', {}, '🎯 미션 소개'), h('p', {}, m.outline)), h('div', { class: 'tip' }, '이 미션은 아직 준비 중이에요. 열려 있는 미션을 먼저 진행해 주세요.')));
    return;
  }
  const n = nSteps(m), st = curStep(mid);
  const bar = h('div', { class: 'stepper' });
  const comp = (A.views[mid] || {}).complete || [];
  // 잠그지 않음: 뒷부분을 미리 볼 수 있게 모든 단계를 열어 둠. 처음 들어갈 때 시작 메시지만 보냄
  ['미션 소개', ...m.brief.steps_short.map((s, i) => `${i + 1} ${s}`), '📦 산출물'].forEach((name, i) => {
    const ok = i >= 1 && i <= n && comp[i - 1];
    bar.append(h('button', { class: 'stp' + (i === st ? ' cur' : '') + (ok ? ' done' : ''), onclick: async () => {
      if (i >= 1 && !m.progress.started) await invoke('app.start', mid);
      goStep(mid, i);
    } }, (ok ? '✅ ' : '') + name));
  });
  const body = h('div', { class: 'scroll' });
  main.append(bar, body);
  const practice = st >= 1 && st <= n;
  $('ws-root').classList.toggle('no-code', !practice);
  setTopic(practice ? TOPICS[mid][st] : null);
  STEPS[mid][Math.min(st, n + 1)](body, m);
}

function termCard(t, now) {   // 핵심 용어 카드: 미션 소개와 📘 가이드북이 함께 씀 (가이드북 용어에는 수식·더 알아보기가 붙음)
  const rows = [['k1', '한 줄 정의', t.d], ['k2', '쉽게 말하면', t.easy], ['k3', PROJ.use_label, t.use], ['k4', '수식', t.formula], ['k5', '더 알아보기', t.more]].filter(r => r[2]);
  return h('div', { class: 'term' + (now ? ' now' : ''), 'data-t': t.t }, h('h5', {}, '📖 ' + t.t, t.en ? h('small', {}, ' ' + t.en) : null), figure(t.fig),
    h('dl', {}, rows.map(([k, a, b]) => [h('dt', { class: k }, a), h('dd', { class: k === 'k4' ? 'fx' : '' }, b)])));
}
function startButtons(m) {
  const go = async () => { if (!m.progress.started) await invoke('app.start', m.id); goStep(m.id, firstTodoStep(m)); };
  return h('button', { class: 'btn big' + (m.progress.started ? '' : ' pulse'), onclick: go }, m.progress.started ? '이어서 하기 ▶' : '미션 시작 ▶');
}
function firstTodoStep(m) { const c = (A.views[m.id] || {}).complete || []; const k = c.findIndex(x => !x); return k < 0 ? 1 : k + 1; }
// 해외 사례: 기사 주소를 달지 않고, 기사의 주요 내용을 눌러서 열어 보는 토글 (2026-11-05 피드백)
function caseToggle(cs) {
  if (!cs) return null;
  return h('details', { class: 'case' }, h('summary', {}, '📰 ' + cs.title),
    h('div', { class: 'case-b' }, (cs.points || []).map(t => h('p', { html: md(t) })), h('div', { class: 'hint' }, '출처: ' + cs.source)));
}
function assetNotice(extra) {
  const S = A.assets || {};
  return h('div', { class: 'card focus' }, h('h4', {}, '📷 참고자료(사진)를 넣어 주세요'),
    h('p', {}, '사진이 필요한 활동이 있어요. 왼쪽 📁 파일 패널에서 아래 폴더에 받은 사진(또는 zip)을 넣은 뒤 [다시 확인]을 눌러 주세요.'),
    h('pre', { class: 'vb-out' }, S.folder || ''), extra || null,
    S.missing && S.missing.length ? h('div', { class: 'hint' }, `아직 없는 사진 ${S.missing.length}개: ${S.missing.slice(0, 8).join(', ')}${S.missing.length > 8 ? ' …' : ''}`) : null,
    h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => { await invoke('app.assets_check'); cache.asset = {}; renderMain(); } }, '다시 확인')));
}
function renderBrief(body, m) {
  const b = m.brief;
  const sec = (title, ...kids) => h('section', {}, h('h4', {}, title), ...kids);
  body.append(h('div', { class: 'brief' },
    h('h2', {}, `M${m.no}. ${m.title}`), h('p', { class: 'sub-t' }, b.subtitle),
    sec('🎯 미션 목표', h('p', {}, b.goal)),
    h('div', { class: 'row' }, startButtons(m)),
    h('section', { class: 'why' }, h('h4', {}, '📰 왜 중요할까요?'), h('p', { style: 'margin:0' }, b.why),
      caseToggle(b.case)),
    sec('🧭 수행 단계', h('ol', {}, b.steps.map(s => h('li', {}, s)))),
    sec('📦 산출물', h('ul', {}, b.outputs.map(s => h('li', {}, s)))),
    b.comps && b.comps.length ? sec('🧩 이 미션에서 기르는 역량', h('ul', {}, b.comps.map(c => h('li', {}, h('b', {}, c.name), c.criteria ? ': ' + c.criteria : '')))) : null,
    b.team ? sec('👥 함께하는 사람', h('p', {}, b.team)) : null,
    sec('📖 핵심 용어', h('p', { class: 'hint', style: 'margin:0 0 8px' }, `용어마다 ‘한 줄 정의 → 쉽게 말하면 → ${PROJ.use_label}’ 순서로 설명했어요. 미션 중에도 이 화면으로 돌아와 다시 볼 수 있어요.`),
      h('div', { class: 'terms' }, b.terms.map(termCard))),
    h('div', { class: 'row', style: 'margin-top:16px' }, startButtons(m))));
}
// 산출물 단계: 언제든 받을 수 있음. 안 한 활동이 있으면 먼저 알려 주고 물어본 뒤, 그래도 받으면 그 부분은 빈칸으로 만듦 (2026-11-05 피드백)
function todoList(m, todo) {
  return h('ul', { class: 'todo' }, todo.map(t => h('li', {}, h('a', { href: '#', onclick: e => { e.preventDefault(); goStep(m.id, t.step); } }, `${t.step}단계 「${t.short}」`), ': ', t.blocks.join(', '))));
}
function doneView(body, m, files, nextMid) {
  const area = h('div'), box = h('div', { class: 'brief' }, h('h2', {}, `📦 M${m.no} 산출물`), h('p', { class: 'sub-t' }, m.title), area);
  body.append(box);
  const draw = async () => {
    const r = await invoke('blk.todo', m.id), todo = r.todo || [];
    area.innerHTML = '';
    const mine = A.files.filter(f => f.mission === m.id);
    area.append(h('section', {}, h('h4', {}, '📋 이 미션의 진행 상황'),
      todo.length ? todoList(m, todo) : h('div', { class: 'fb ok' }, '✅ 필수 활동을 모두 마쳤어요.')));
    const ask = h('div');
    area.append(h('section', {}, h('h4', {}, '📄 받을 문서'), h('ul', {}, files.map(([, t]) => h('li', {}, t))),
      h('div', { class: 'row' }, h('button', { class: 'btn big' + (todo.length ? '' : ' pulse'), id: 'get-output', onclick: async () => {
        const d = await invoke('blk.finish', m.id);
        if (!d.need_confirm) return draw();
        ask.innerHTML = '';
        ask.append(h('div', { class: 'card focus', id: 'confirm-box' }, h('h4', {}, '⚠ 아직 하지 않은 활동이 있어요'), todoList(m, d.todo),
          h('p', {}, '그래도 산출물을 받을까요? 하지 않은 부분은 빈칸으로 나와요.'),
          h('div', { class: 'row' }, h('button', { class: 'btn ghost', onclick: () => goStep(m.id, d.todo[0].step) }, '마저 하러 가기'),
            h('button', { class: 'btn', id: 'force-output', onclick: async () => { await invoke('blk.finish', m.id, true); draw(); } }, '그래도 받기'))));
      } }, mine.length ? '📦 산출물 다시 받기' : '📦 산출물 받기')), ask));
    if (mine.length) area.append(h('section', {}, h('h4', {}, '📁 저장된 산출물'), mine.map(f => h('div', { class: 'row' }, h('span', {}, '📄 ' + f.title),
      f.blanks ? h('span', { class: 'pill soon' }, `안 한 활동 ${f.blanks}개`) : null, h('button', { class: 'btn small ghost', onclick: () => openFile(f.id) }, '열어 보기'))),
      h('div', { class: 'tip' }, '💾 산출물은 노트북이 켜져 있는 동안만 보관돼요. 📁 파일에서 [⬇ 내려받기]로 저장해 두세요.')));
    area.append(h('div', { class: 'row', style: 'margin-top:16px' }, nextMid && A.missions[nextMid].brief
      ? h('button', { class: 'btn ghost', onclick: () => openView('mission:' + nextMid) }, `다음 미션: M${A.missions[nextMid].no} ${A.missions[nextMid].title} ▶`)
      : h('button', { class: 'btn ghost', onclick: () => openView('home') }, '🏠 프로젝트 개요로')));
  };
  draw();
}

// ===================== 공통 위젯: 작업 진행 · 그래프 · 힌트 · 크게 보기 · 작업 중 화면 잠금 · 확인 질문 =====================
const POLL = {};
const pct = v => v === null || v === undefined ? '없음' : (v * 100).toFixed(1) + '%';
// 받침에 맞는 조사: josa('판정', '을', '를') → '을'
function josa(w, a, b) { const c = String(w || '').trim().slice(-1).charCodeAt(0); return c >= 0xAC00 && c <= 0xD7A3 && (c - 0xAC00) % 28 ? a : b; }
const ci = c => c ? `[${Math.round(c[0] * 100)}~${Math.round(c[1] * 100)}%]` : '';
function jobBar(d) {
  const p = d.state === 'done' ? 100 : d.pct || 0;
  return h('div', { class: 'jobbar' + (d.state === 'error' ? ' err' : '') },
    h('div', { class: 'jb-top' }, h('b', {}, d.state === 'error' ? '⚠ 문제가 생겼어요' : d.state === 'done' ? '✅ 끝났어요' : `⏳ ${p}%`), h('span', {}, d.msg || ''),
      h('span', { class: 'sp' }), d.sec ? h('span', { class: 'hint' }, `${Math.floor(d.sec / 60)}분 ${d.sec % 60}초 지남`) : null),
    h('div', { class: 'pb' }, h('i', { style: `width:${p}%` })));
}
async function jobState(name) {
  try { const r = await google.colab.kernel.invokeFunction('app.job', [name], {}); return r.data['application/json']; } catch (e) { return null; }
}
// 뒤에서 도는 작업(내려받기·학습·평가)을 1.5초마다 확인해 진행 막대를 그림. 끝나면 onDone (이 화면에서 진행을 본 경우만)
function watchJob(name, el, onDone, onTick) {
  clearTimeout(POLL[name]);
  let seen = false;
  const tick = async () => {
    const d = await jobState(name);
    if (d && d.state !== 'running' && BUSY === name) BUSY = null;
    if (!d || !el.isConnected) return;
    if (d.state === 'running') {
      if (STAY_JOBS[name]) { BUSY = name; BUSY_T = Date.now(); }
      seen = true; el.innerHTML = ''; el.append(jobBar(d)); if (onTick) onTick(d, el);
      POLL[name] = setTimeout(tick, 1500); return;
    }
    if (d.app) refresh(d.app);
    if (d.state === 'error') {
      if (!seen) return;                 // 예전에 끝난 작업의 오류는 다시 보여 주지 않음 (버튼을 다시 누르면 새로 시작)
      el.innerHTML = '';
      el.append(jobBar(d), h('div', { class: 'fb bad' }, '오류 내용: ' + (d.error || '') + '. 버튼을 다시 눌러 보세요. 계속되면 런타임 → 모두 실행으로 다시 시작해 주세요.'));
      return;
    }
    if (d.state === 'done' && seen) { el.innerHTML = ''; if (onDone) onDone(d); }
  };
  tick();
}
async function startJob(fn, args, name, el, onDone, onTick) {
  const d = await invoke(fn, ...args);
  if (d && d.ok !== false) watchJob(name, el, onDone, onTick);
  return d;
}
// 무거운 준비 작업이 도는 동안에는 다른 화면으로 못 가게 막음: 작업 중 다른 화면을 열면 Colab 런타임이 끊기는 일이 있었음
let BUSY = null, BUSY_T = 0;
const STAY_JOBS = PROJ.stay_jobs;          // 작업 이름 → 화면에 보일 이름 (프로젝트 명세의 stay_jobs)
function stayCard(what, extra) {
  return h('div', { class: 'stay', id: 'stay-card' }, h('b', {}, '⚠ 이 화면에서 기다려 주세요'),
    h('div', {}, `${what} 작업이 도는 동안(약 1~3분)에는 왼쪽 메뉴나 위쪽 단계 버튼 등 다른 화면을 누르지 마세요. 다른 화면을 열면 Colab 런타임이 끊길 수 있어요.`),
    extra ? h('div', { class: 'hint' }, extra) : null);
}
function busyBlock() {
  if (!BUSY) return false;
  if (Date.now() - BUSY_T > 15 * 60 * 1000) { BUSY = null; return false; }     // 15분이 지나도 소식이 없으면 풀어 줌
  note(`⏳ ${STAY_JOBS[BUSY] || '준비'} 작업이 끝날 때까지 이 화면에서 기다려 주세요. 다른 화면을 열면 Colab 런타임이 끊길 수 있어요.`, 'bad');
  const c = document.getElementById('stay-card');
  if (c) { c.classList.remove('shake'); void c.offsetWidth; c.classList.add('shake'); c.scrollIntoView({ block: 'center', behavior: 'smooth' }); }
  return true;
}
// 확인 질문 카드 (4·5단계 공통): 틀리면 힌트가 한 단계씩 자세해지고, 맞혀도 화면이 저절로 넘어가지 않음 → 풀이를 읽고 [다음 단계]를 직접 누름
// (이 동작은 화면 글로 따로 알리지 않음: 2026-10-19 피드백)
function quizCard(mid, qid, fnName, onSolved, title) {
  const box = h('div', { class: 'card' });
  const draw = () => {
    const q = A.views[mid].quiz[qid];
    box.className = 'card' + (q.ok ? '' : ' focus');
    box.innerHTML = '';
    box.append(h('h4', {}, (q.ok ? '✅ ' : '❓ ') + (title || '확인 질문')), h('div', { class: 'qtext' }, q.text));
    if (q.ok) {
      box.append(h('div', { class: 'choices col' }, q.options.map((o, i) => h('div', { class: 'choice' + (i === q.pick ? ' on' : '') }, h('span', {}, (i === q.pick ? '✔ ' : '') + o)))),
        h('div', { class: 'fb ok' }, '✅ 맞아요. ' + q.explain));
      return;
    }
    const ch = choiceGroup(`${mid}-${qid}`, q.options, null, null, { col: true }), fb = h('div');
    box.append(ch, h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
      const v = ch.value();
      if (v === '') { fb.className = 'fb bad'; fb.textContent = '⚠ 보기를 하나 골라 주세요.'; return; }
      const r = await invoke(fnName, qid, Number(v));
      if (r.correct) { draw(); if (onSolved) onSolved(); } else { fb.className = 'fb bad'; fb.textContent = r.text; }
    } }, '확인')), fb);
  };
  draw();
  return box;
}
// 📘 가이드북(오른쪽)의 핵심 용어 탭을 열고 그 용어로 이동
function guideBtn(label, focus) { return h('button', { class: 'btn small ghost', onclick: () => openGuide('terms', focus) }, label || '📘 가이드북에서 용어 자세히 보기'); }
// 단계별 힌트: 버튼을 누를 때마다 한 줄씩 더 보여 줌 (스스로 생각할 기회를 먼저 줌)
function hintBox(key, hints, title) {
  const st = cache.hints || (cache.hints = {});
  const box = h('div', { class: 'hintbox' });
  const draw = () => {
    box.innerHTML = '';
    const n = st[key] || 0;
    box.append(h('div', { class: 'hb-h' }, h('b', {}, '💡 ' + (title || '힌트')), h('span', { class: 'sp' }),
      n < hints.length ? h('button', { class: 'btn small ghost', onclick: () => { st[key] = n + 1; draw(); } }, n ? `힌트 더 보기 (${n + 1}/${hints.length})` : `힌트 보기 (1/${hints.length})`)
        : h('span', { class: 'hint' }, '힌트를 모두 봤어요')));
    hints.slice(0, n).forEach((t, i) => box.append(h('div', { class: 'hb-i' }, h('b', {}, `${i + 1}. `), t)));
  };
  draw();
  return box;
}
// 영상 크게 보기
function zoomImg(src, cap) {
  const root = $('ws-root');
  const box = h('div', { class: 'zoombox', onclick: () => box.remove() }, h('div', { class: 'zb-in' }, h('img', { src }), h('div', { class: 'zb-cap' }, (cap || '') + '  · 아무 곳이나 누르면 닫혀요')));
  root.append(box);
}
// 작은 SVG 그래프: o = {x:[최소,최대], y:[최소,최대], xl, yl, series:[{pts, color, dash, dots, label}], marks:[{x,y,color,label}], vlines, hlines, w, h}
function chart(o) {
  const W = o.w || 380, H = o.h || 230, L = 44, R = 14, T = 14, B = 38;
  const sx = v => L + (v - o.x[0]) / (o.x[1] - o.x[0]) * (W - L - R), sy = v => H - B - (v - o.y[0]) / (o.y[1] - o.y[0]) * (H - T - B);
  const f = v => Math.abs(v) >= 10 || Number.isInteger(v) ? String(Math.round(v * 10) / 10) : v.toFixed(2);
  let s = `<svg viewBox="0 0 ${W} ${H}" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" fill="#111827">`;
  s += `<rect x="${L}" y="${T}" width="${W - L - R}" height="${H - T - B}" fill="#fff" stroke="#c3cad5"/>`;
  for (let i = 0; i <= 4; i++) {
    const xv = o.x[0] + (o.x[1] - o.x[0]) * i / 4, yv = o.y[0] + (o.y[1] - o.y[0]) * i / 4;
    s += `<line x1="${sx(xv)}" y1="${T}" x2="${sx(xv)}" y2="${H - B}" stroke="#eef2f7"/><text x="${sx(xv)}" y="${H - B + 13}" text-anchor="middle">${f(xv)}</text>`;
    s += `<line x1="${L}" y1="${sy(yv)}" x2="${W - R}" y2="${sy(yv)}" stroke="#eef2f7"/><text x="${L - 4}" y="${sy(yv) + 3}" text-anchor="end">${f(yv)}</text>`;
  }
  s += `<text x="${(L + W - R) / 2}" y="${H - 6}" text-anchor="middle" font-weight="bold">${esc(o.xl || '')}</text>`;
  s += `<text x="12" y="${(T + H - B) / 2}" text-anchor="middle" font-weight="bold" transform="rotate(-90 12 ${(T + H - B) / 2})">${esc(o.yl || '')}</text>`;
  for (const v of o.vlines || []) s += `<line x1="${sx(v.x ?? v)}" y1="${T}" x2="${sx(v.x ?? v)}" y2="${H - B}" stroke="${v.color || '#8a4b00'}" stroke-dasharray="4 3"/>` + (v.label ? `<text x="${sx(v.x) + 3}" y="${T + 10}" fill="${v.color || '#8a4b00'}">${esc(v.label)}</text>` : '');
  for (const v of o.hlines || []) s += `<line x1="${L}" y1="${sy(v.y ?? v)}" x2="${W - R}" y2="${sy(v.y ?? v)}" stroke="${v.color || '#8a4b00'}" stroke-dasharray="4 3"/>` + (v.label ? `<text x="${W - R - 3}" y="${sy(v.y) - 3}" text-anchor="end" fill="${v.color || '#8a4b00'}">${esc(v.label)}</text>` : '');
  let ly = T + 12;
  for (const se of o.series || []) {
    const pts = se.pts.filter(p => p[1] !== null && p[1] !== undefined);
    if (se.line !== false && pts.length > 1) s += `<polyline fill="none" stroke="${se.color}" stroke-width="2" ${se.dash ? 'stroke-dasharray="5 3"' : ''} points="${pts.map(p => sx(p[0]) + ',' + sy(p[1])).join(' ')}"/>`;
    if (se.dots) for (const p of pts) s += `<circle cx="${sx(p[0])}" cy="${sy(p[1])}" r="${se.r || 2.6}" fill="${p[2] || se.color}" ${p[3] ? `stroke="${p[3]}" stroke-width="1.5"` : ''}/>`;
    if (se.label) { s += `<rect x="${W - R - 118}" y="${ly - 8}" width="10" height="10" fill="${se.color}"/><text x="${W - R - 104}" y="${ly + 1}">${esc(se.label)}</text>`; ly += 14; }
  }
  for (const mk of o.marks || []) s += `<circle cx="${sx(mk.x)}" cy="${sy(mk.y)}" r="5.5" fill="none" stroke="${mk.color || '#dc2626'}" stroke-width="2.5"/>` + (mk.label ? `<text x="${sx(mk.x) + 8}" y="${sy(mk.y) - 6}" font-weight="bold" fill="${mk.color || '#dc2626'}">${esc(mk.label)}</text>` : '');
  return h('div', { class: 'chart', html: s + '</svg>' });
}
function reqChip(text, ok) { return h('span', { class: 'chk-chip ' + (ok ? 'y' : 'n') }, (ok ? '✅ ' : '⚠ ') + text); }
function statTile(label, value, desc) { return h('div', { class: 'tile' }, h('div', { class: 'd' }, label), h('div', { class: 'v' }, value), desc ? h('div', { class: 'd' }, desc) : null); }
function selectBox(opts, value, onchange) {
  const s = h('select', {}, opts.map(([v, t]) => h('option', { value: v, selected: String(value) === String(v) }, t)));
  s.onchange = () => onchange(s.value);
  return s;
}
function textBox(value, min, placeholder, oninput) {
  const ta = h('textarea', { placeholder });
  ta.value = value || '';
  ta.addEventListener('input', () => oninput(ta.value));
  return { ta, line: textCounter(ta, min) };
}

// ===================== 결과 보여 주기 도우미 (2026-10-20 제작자 피드백) =====================
// put: el.append는 null·undefined를 "null" 글자로 그려요(‘null’이 보이는 문제). 조건부 자식이 있으면 늘 put으로 붙여요.
function put(el, ...kids) { el.append(...kids.flat(9).filter(k => k !== null && k !== undefined && k !== false && k !== '')); return el; }
// 혼동행렬 표: o = { rows: [실제 이름…], cols: [예측 이름…], cells: [[개수…]…], good: (i, j) => 맞은 칸인가, rowHead, colHead }
// 행(실제)마다 비율로 칸을 칠해요: 맞은 칸은 초록, 틀린 칸은 빨강.
function confusionTable(o) {
  const good = o.good || ((i, j) => i === j);
  const tot = o.cells.map(r => r.reduce((a, b) => a + b, 0));
  const head = h('tr', {}, h('th', { colspan: 2 }, ''), h('th', { colspan: o.cols.length, style: 'text-align:center' }, (o.colHead || '예측') + ' →'));
  const head2 = h('tr', {}, h('th', {}, ''), h('th', {}, ''), o.cols.map(c => h('th', { style: 'text-align:center' }, c)));
  const rows = o.rows.map((r, i) => h('tr', {},
    i === 0 ? h('th', { rowspan: o.rows.length, style: 'text-align:center;vertical-align:middle' }, (o.rowHead || '실제') + ' ↓') : null, h('th', {}, r),
    o.cells[i].map((v, j) => {
      const f = tot[i] ? v / tot[i] : 0, ok = good(i, j);
      return h('td', { style: `text-align:center;font-weight:bold;min-width:72px;background:${ok ? `rgba(22,163,74,${0.08 + 0.55 * f})` : `rgba(220,38,38,${0.06 + 0.55 * f})`}` },
        String(v), h('div', { class: 'hint' }, tot[i] ? Math.round(f * 100) + '%' : '0%'), o.tags && o.tags[i] && o.tags[i][j] ? h('div', { class: 'hint' }, o.tags[i][j]) : null);
    })));
  return h('table', { class: 't cm', style: 'width:auto' }, head, head2, rows);
}
// 지표 표: rows = [[지표, 값, 뜻], …]
function metricTable(rows) {
  return h('table', { class: 't', style: 'width:auto' }, h('tr', {}, h('th', {}, '지표'), h('th', {}, '값'), h('th', {}, '뜻')),
    rows.map(r => h('tr', {}, h('td', {}, h('b', {}, r[0])), h('td', { style: 'text-align:right;font-weight:bold' }, r[1]), h('td', { class: 'hint' }, r[2] || ''))));
}
// 가로 막대: items = [[이름, 값, 색?], …], o = { max, unit, signed(음수 허용), fmt }
function barChart(items, o) {
  o = o || {};
  const max = o.max || Math.max(1e-9, ...items.map(x => Math.abs(x[1])));
  const fmt = o.fmt || (v => (Math.abs(v) >= 10 ? Math.round(v) : Math.round(v * 100) / 100) + (o.unit || ''));
  return h('div', { class: 'barchart', style: 'display:grid;grid-template-columns:max-content 1fr max-content;gap:4px 8px;align-items:center;font-size:12.5px;margin:6px 0' },
    items.map(([name, v, color]) => {
      const w = Math.min(100, Math.abs(v) / max * (o.signed ? 50 : 100));
      const bar = o.signed
        ? h('div', { style: 'position:relative;height:14px;background:#f1f5f9;border-radius:3px' }, h('div', { style: 'position:absolute;left:50%;top:0;bottom:0;width:1px;background:#94a3b8' }),
          h('div', { style: `position:absolute;top:0;bottom:0;${v >= 0 ? 'left:50%' : `left:${50 - w}%`};width:${w}%;background:${color || (v >= 0 ? '#dc2626' : '#2563eb')};border-radius:3px` }))
        : h('div', { style: 'height:14px;background:#f1f5f9;border-radius:3px' }, h('div', { style: `height:100%;width:${w}%;background:${color || '#2563eb'};border-radius:3px` }));
      return [h('span', {}, name), bar, h('b', {}, fmt(v))];
    }));
}
// 분포(히스토그램, 겹쳐 쌓기): o = { edges: [경계…], series: [{ data: [개수…], color, label }], vline: 값, vlabel, xl, yl }
function histChart(o) {
  const W = 520, H = 220, L = 42, R = 12, T = 12, B = 36, n = o.edges.length - 1;
  const tot = Array.from({ length: n }, (_, i) => o.series.reduce((a, s) => a + s.data[i], 0)), max = Math.max(1, ...tot);
  const x0 = o.edges[0], x1 = o.edges[n], sx = v => L + (v - x0) / (x1 - x0) * (W - L - R), sy = v => H - B - v / max * (H - T - B);
  let s = `<svg viewBox="0 0 ${W} ${H}" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" fill="#111827"><rect x="${L}" y="${T}" width="${W - L - R}" height="${H - T - B}" fill="#fff" stroke="#c3cad5"/>`;
  for (let i = 0; i <= 4; i++) { const v = max * i / 4; s += `<text x="${L - 4}" y="${sy(v) + 3}" text-anchor="end">${Math.round(v)}</text><line x1="${L}" x2="${W - R}" y1="${sy(v)}" y2="${sy(v)}" stroke="#eef2f7"/>`; }
  for (let i = 0; i <= 4; i++) { const v = x0 + (x1 - x0) * i / 4; s += `<text x="${sx(v)}" y="${H - B + 13}" text-anchor="middle">${Math.round(v * 100) / 100}</text>`; }
  for (let i = 0; i < n; i++) {
    let acc = 0;
    for (const se of o.series) { const v = se.data[i]; if (v) s += `<rect x="${sx(o.edges[i]) + 1}" y="${sy(acc + v)}" width="${sx(o.edges[i + 1]) - sx(o.edges[i]) - 2}" height="${sy(acc) - sy(acc + v)}" fill="${se.color}" fill-opacity="0.85"/>`; acc += v; }
  }
  if (o.vline !== undefined && o.vline !== null) s += `<line x1="${sx(o.vline)}" x2="${sx(o.vline)}" y1="${T}" y2="${H - B}" stroke="#111827" stroke-width="2" stroke-dasharray="5 3"/><text x="${sx(o.vline) + 4}" y="${T + 11}" font-weight="bold">${esc(o.vlabel || '')}</text>`;
  let ly = T + 12;
  for (const se of o.series) { s += `<rect x="${W - R - 120}" y="${ly - 8}" width="10" height="10" fill="${se.color}"/><text x="${W - R - 106}" y="${ly + 1}">${esc(se.label)}</text>`; ly += 14; }
  s += `<text x="${(L + W - R) / 2}" y="${H - 6}" text-anchor="middle" font-weight="bold">${esc(o.xl || '')}</text><text x="11" y="${(T + H - B) / 2}" text-anchor="middle" font-weight="bold" transform="rotate(-90 11 ${(T + H - B) / 2})">${esc(o.yl || '')}</text>`;
  return h('div', { class: 'chart', html: s + '</svg>' });
}
// 결정 영역 지도: o = { x: [최소, 최대], y: [최소, 최대], n, p: [[점수…]…] (아래쪽 줄부터), pts: [[x, y, 종류]…], thr, xl, yl, kinds: { 종류: [색, 모양('o'|'^'|'s'|'d'), 이름] } }
// 칸 색: 임계값 이상이면 빨강 쪽(양성), 아래면 파랑 쪽. 경계가 굵은 선으로 보여요.
function regionMap(o) {
  const W = 460, H = 380, L = 46, R = 12, T = 12, B = 40, n = o.n;
  const cw = (W - L - R) / n, ch = (H - T - B) / n;
  const sx = v => L + (v - o.x[0]) / (o.x[1] - o.x[0]) * (W - L - R), sy = v => H - B - (v - o.y[0]) / (o.y[1] - o.y[0]) * (H - T - B);
  let s = `<svg viewBox="0 0 ${W} ${H}" xmlns="http://www.w3.org/2000/svg" font-family="sans-serif" font-size="10" fill="#111827">`;
  for (let r = 0; r < n; r++) for (let c = 0; c < n; c++) {
    const p = o.p[r][c], pos = p >= o.thr, a = pos ? 0.15 + 0.55 * (p - o.thr) / Math.max(1e-6, 1 - o.thr) : 0.15 + 0.55 * (o.thr - p) / Math.max(1e-6, o.thr);
    const base = pos ? [220, 38, 38] : [37, 99, 235], mix = v => Math.round(255 - (255 - v) * a);   // 흰 바탕에 섞은 불투명 색 (칸 사이 틈이 비치지 않게)
    s += `<rect x="${(L + c * cw).toFixed(2)}" y="${(H - B - (r + 1) * ch).toFixed(2)}" width="${(cw + 1).toFixed(2)}" height="${(ch + 1).toFixed(2)}" fill="rgb(${base.map(mix).join(',')})" shape-rendering="crispEdges"/>`;
  }
  for (let r = 0; r < n; r++) for (let c = 0; c < n; c++) {          // 경계선: 이웃 칸과 판정이 다르면 선을 그음
    const pos = o.p[r][c] >= o.thr, x = L + c * cw, y = H - B - (r + 1) * ch;
    if (c + 1 < n && (o.p[r][c + 1] >= o.thr) !== pos) s += `<line x1="${(x + cw).toFixed(1)}" x2="${(x + cw).toFixed(1)}" y1="${y.toFixed(1)}" y2="${(y + ch).toFixed(1)}" stroke="#111827" stroke-width="1.6"/>`;
    if (r + 1 < n && (o.p[r + 1][c] >= o.thr) !== pos) s += `<line x1="${x.toFixed(1)}" x2="${(x + cw).toFixed(1)}" y1="${y.toFixed(1)}" y2="${y.toFixed(1)}" stroke="#111827" stroke-width="1.6"/>`;
  }
  const mark = (x, y, [color, shape]) => shape === '^' ? `<path d="M${x},${y - 4.5} L${x + 4},${y + 3} L${x - 4},${y + 3} Z" fill="${color}" stroke="#fff" stroke-width="0.8"/>`
    : shape === 's' ? `<rect x="${x - 3.5}" y="${y - 3.5}" width="7" height="7" fill="${color}" stroke="#fff" stroke-width="0.8"/>`
      : shape === 'd' ? `<path d="M${x},${y - 4.5} L${x + 4.5},${y} L${x},${y + 4.5} L${x - 4.5},${y} Z" fill="${color}" stroke="#fff" stroke-width="0.8"/>`
        : `<circle cx="${x}" cy="${y}" r="3.6" fill="${color}" stroke="#fff" stroke-width="0.8"/>`;
  for (const [x, y, k] of o.pts) if (x >= o.x[0] && x <= o.x[1] && y >= o.y[0] && y <= o.y[1]) s += mark(sx(x).toFixed(1), sy(y).toFixed(1), o.kinds[k] || ['#111827', 'o']);
  s += `<rect x="${L}" y="${T}" width="${W - L - R}" height="${H - T - B}" fill="none" stroke="#64748b"/>`;
  for (let i = 0; i <= 4; i++) {
    const xv = o.x[0] + (o.x[1] - o.x[0]) * i / 4, yv = o.y[0] + (o.y[1] - o.y[0]) * i / 4;
    s += `<text x="${sx(xv)}" y="${H - B + 13}" text-anchor="middle">${Math.round(xv * 100) / 100}</text><text x="${L - 4}" y="${sy(yv) + 3}" text-anchor="end">${Math.round(yv * 100) / 100}</text>`;
  }
  s += `<text x="${(L + W - R) / 2}" y="${H - 6}" text-anchor="middle" font-weight="bold">${esc(o.xl || '')}</text><text x="11" y="${(T + H - B) / 2}" text-anchor="middle" font-weight="bold" transform="rotate(-90 11 ${(T + H - B) / 2})">${esc(o.yl || '')}</text>`;
  const legend = h('div', { class: 'row', style: 'gap:10px;flex-wrap:wrap;font-size:12px' }, Object.values(o.kinds).map(([color, shape, name]) =>
    h('span', { html: `<svg width="12" height="12" viewBox="-6 -6 12 12">${mark(0, 0, [color, shape])}</svg> ${esc(name)}` })),
    h('span', { class: 'hint' }, `빨간 영역: 점수 ${o.thr} 이상 (양성으로 판정) · 파란 영역: 그 아래 · 검은 선: 결정 경계`));
  return h('div', {}, h('div', { class: 'chart', html: s + '</svg>' }), legend);
}

// ===================== 미션 단계: 명세(MISSION_UI)의 블록으로 화면 그리기 =====================
// 블록 종류마다 그리는 함수(BLOCK_VIEW)가 있고, 단계 화면(stepPage)이 블록을 차례로 그린 뒤 맨 아래에 [다음 단계]나 [완료하기]를 둬요.
// 원칙: 맞혀도 저절로 넘어가지 않음 · 틀리면 힌트가 한 단계씩 자세해짐 · 정답은 파이썬에만 있음(화면에는 맞힌 뒤의 풀이만 옴)
// 이런 동작 원칙은 화면 글로 따로 알리지 않아요 ("틀리면 힌트가 나와요" 같은 안내는 과한 정보: 2026-10-19 피드백)
function bst(mid, bid) { return ((A.views[mid] || {}).blk || {})[bid] || { ok: false, tries: 0, value: null }; }
const md = t => esc(t === undefined || t === null ? '' : t).replace(/\*\*(.+?)\*\*/g, '<b>$1</b>');
function mdp(t, tag) { return h(tag || 'p', { html: md(t) }); }
function stepPage(mid, si) { return (body, m) => renderStep(body, m, si); }
function renderStep(body, m, si) {
  const st = MISSION_UI[m.id].steps[si - 1], n = MISSION_UI[m.id].steps.length;
  const foot = h('div', { class: 'stepfoot' });
  const drawFoot = () => {      // 잠그지 않음: 뒷부분을 미리 볼 수 있게 [다음 단계]를 늘 보여 줌
    foot.innerHTML = '';
    const req = st.blocks.filter(b => b.req), left = req.filter(b => !bst(m.id, b.id).ok).length;
    if (left) foot.append(h('div', { class: 'hint' }, `이 단계의 필수 활동 ${req.length}개 중 ${req.length - left}개를 마쳤어요.`));
    foot.append(si < n ? nextBtn(m.id, si + 1).el : nextBtn(m.id, n + 1, null, null, '📦 산출물 받으러 가기 ▶').el);
  };
  if (st.lead) body.append(h('div', { class: 'tip', html: md(st.lead) }));
  for (const b of st.blocks) body.append(blockView(m, si, b, drawFoot));
  body.append(foot);
  drawFoot();
}
// 블록 하나: ctx.changed()를 부르면 그 블록과 단계 아래쪽 버튼을 다시 그려요 (직접 만든 활동도 같은 방식)
function blockView(m, si, b, changed) {
  const wrap = h('div');
  const ctx = { m, si, b, wrap, st: () => bst(m.id, b.id), redraw: null, changed: () => { ctx.redraw(); changed(); } };
  const fn = BLOCK_VIEW[b.type];
  ctx.redraw = () => { wrap.innerHTML = ''; fn ? fn(wrap, ctx) : wrap.append(h('div', { class: 'fb bad' }, `알 수 없는 활동 종류: ${b.type}`)); };
  ctx.redraw();
  return wrap;
}
function voicesView(vs, title) {
  return h('div', { class: 'voices' }, title ? h('div', { class: 'hint' }, title) : null, vs.map(([who, text]) => {
    const p = A.members[who] || { name: who, color: '#64748b' };
    return h('div', { class: 'voice' }, h('span', { class: 'avatar', style: `background:${p.color}` }, p.name[0]), h('div', {}, h('b', {}, p.name), ' ', h('span', { html: md(text) })));
  }));
}
// 실습 자료 이미지: "asset:번호"는 파이썬에서 받아 오고(한 번 받으면 캐시), 그 밖의 주소는 그대로 씀. 누르면 크게 보기
function assetImg(ref, cap, cls) {
  const img = h('img', { class: cls || '', alt: cap || '', title: '눌러서 크게 보기' });
  if (typeof ref === 'string' && ref.startsWith('asset:')) {
    const id = ref.slice(6), c = cache.asset || (cache.asset = {});
    const fail = r => {
      if (!img.isConnected) return;
      img.replaceWith(r && r.missing ? assetNotice(h('div', { class: 'fb bad' }, r.msg)) : h('div', { class: 'fb bad' }, (r && r.msg) || '사진을 불러오지 못했어요.'));
    };
    if (c[id]) img.src = c[id];
    else invoke('app.asset', id, 960).then(r => { if (r && r.ok) { c[id] = r.url; img.src = r.url; } else fail(r); }).catch(fail);
    img.onclick = () => invoke('app.asset', id, 1800).then(r => { if (r && r.ok) zoomImg(r.url, cap || ''); });
  } else {
    img.src = ref;
    img.onclick = () => zoomImg(ref, cap || '');
  }
  return img;
}
function assetCaption(ref) { return null; }   // 이미지마다 같은 안내를 반복하지 않아요 (출처는 미션을 마친 뒤 산출물에 정리)
function caseBody(it) {
  return h('div', {}, (it.body || []).map(t => mdp(t)),
    it.facts ? h('table', { class: 't' }, it.facts.map(([k, v]) => h('tr', {}, h('th', { style: 'width:170px', html: md(k) }), h('td', { html: md(v) })))) : null,
    it.images ? h('div', { class: 'row', style: 'align-items:flex-start' }, it.images.map(([cap, src]) => h('figure', { class: 'casefig' },
      assetImg(src, cap), h('figcaption', {}, cap)))) : null,
    it.timeline ? h('div', { class: 'timeline' }, it.timeline.map(([t, x]) => h('div', {}, h('b', {}, t), h('span', { html: md(x) })))) : null,
    it.voices ? voicesView(it.voices, '나온 말') : null);
}
function hintList(hints, tries) {
  return hints && hints.length ? h('div', { class: 'hintbox wrongbox' }, h('div', { class: 'hb-h' }, h('b', {}, `❌ 힌트 (틀린 횟수 ${tries}번)`)),
    hints.map((t, i) => h('div', { class: 'hb-i' }, h('b', {}, `${i + 1}. `), t))) : null;
}
const guideRow = b => b.guide ? h('div', { class: 'row' }, guideBtn(b.guide_label || '📘 가이드북에서 용어 자세히 보기', b.guide)) : null;

const BLOCK_VIEW = {
  info(w, c) {
    const b = c.b, ok = c.st().ok;
    w.append(h('div', { class: 'card' + (b.confirm && !ok ? ' focus' : '') }, b.title ? h('h4', {}, (b.icon || '📄') + ' ' + b.title) : null,
      (b.text || []).map(t => mdp(t)), b.figure ? figure(b.figure) : null,
      b.image ? assetImg(b.image, b.title || '', 'blkimg') : null, b.image ? assetCaption(b.image) : null,
      b.bullets ? h('ul', {}, b.bullets.map(x => mdp(x, 'li'))) : null,
      b.table ? h('table', { class: 't' }, h('tr', {}, b.table.head.map(x => h('th', { html: md(x) }))), b.table.rows.map(r => h('tr', {}, r.map(x => h('td', { html: md(x) }))))) : null,
      b.voices ? voicesView(b.voices) : null, b.tip ? h('div', { class: 'tip', html: md(b.tip) }) : null,
      b.case ? caseToggle(b.case) : null,
      b.sources ? h('div', { class: 'hint' }, '출처: ' + b.sources.join(' · ')) : null,
      guideRow(b),
      b.confirm ? (ok ? h('div', { class: 'fb ok' }, '✅ 확인했어요.') : h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
        const r = await invoke('blk.check', c.m.id, b.id, true); if (r.correct) c.changed(); } }, b.confirm === true ? '읽었어요 ✔' : b.confirm))) : null));
  },
  quiz(w, c) {
    const b = c.b, s = c.st();
    const box = h('div', { class: 'card' + (s.ok ? '' : ' focus') }, h('h4', {}, (s.ok ? '✅ ' : '❓ ') + (b.title || '확인 질문')), h('div', { class: 'qtext', html: md(b.text) }));
    if (s.ok) {
      const pick = (s.value || {}).pick;
      box.append(h('div', { class: 'choices col' }, b.options.map((o, i) => h('div', { class: 'choice' + (i === pick ? ' on' : '') }, h('span', {}, (i === pick ? '✔ ' : '') + o)))),
        h('div', { class: 'fb ok', html: '✅ 맞아요. ' + md(s.explain) }));
    } else {
      const ch = choiceGroup(`${c.m.id}-${b.id}`, b.options, null, null, { col: true }), fb = h('div');
      box.append(ch, h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
        const v = ch.value();
        if (v === '') { fb.className = 'fb bad'; fb.textContent = '⚠ 보기를 하나 골라 주세요.'; return; }
        const r = await invoke('blk.check', c.m.id, b.id, Number(v));
        if (r.correct) c.changed(); else { fb.className = 'fb bad'; fb.textContent = r.text; markWrong(ch); }
      } }, '확인')), fb);
    }
    box.append(guideRow(b) || '');
    w.append(box);
  },
  multi(w, c) {
    const b = c.b, s = c.st(), key = 'mp' + c.m.id + b.id;
    const pick = drafts[key] || (drafts[key] = new Set((s.value || {}).picked || []));
    const res = h('div');
    const box = h('div', { class: 'card' + (s.ok ? '' : ' focus') }, h('h4', {}, (s.ok ? '✅ ' : '☑ ') + (b.title || '해당하는 것 모두 고르기')), h('div', { class: 'qtext', html: md(b.text) }));
    b.items.forEach((it, i) => {
      const cb = h('input', { type: 'checkbox', checked: pick.has(i), disabled: s.ok, onchange: () => { cb.checked ? pick.add(i) : pick.delete(i); cb.closest('.choice').classList.remove('wrong'); } });
      box.append(h('label', { class: 'choice', 'data-i': i, style: 'margin:5px 0' }, cb, h('span', { html: md(it) })));
      if (s.ok && (s.explains || [])[i]) box.append(h('div', { class: 'hint', style: 'margin:-2px 0 6px 28px', html: md(s.explains[i]) }));
    });
    if (s.ok) box.append(h('div', { class: 'fb ok' }, '✅ 모두 맞게 골랐어요.'));
    else box.append(h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
      const r = await invoke('blk.check', c.m.id, b.id, [...pick]);
      if (r.correct) { c.changed(); return; }
      res.innerHTML = '';
      box.querySelectorAll('.choice').forEach(l => l.classList.remove('wrong'));
      (r.items || []).filter(x => !x.ok).forEach(x => { const l = box.querySelector(`.choice[data-i="${x.i}"]`); if (l) l.classList.add('wrong'); });
      res.append(h('div', { class: 'fb bad' }, r.text), ...(r.items || []).filter(x => !x.ok).map(x => h('div', { class: 'fb bad' }, `${x.i + 1}번: ${x.text}`)));
    } }, '확인')), res);
    box.append(guideRow(b) || '');
    w.append(box);
  },
  match(w, c) {
    const b = c.b, key = 'mt' + c.m.id + b.id;
    const mt = drafts[key] || (drafts[key] = Object.assign({}, c.st().value || {}));
    const box = h('div');
    const draw = (res, hints, msg) => {
      box.innerHTML = '';
      const ok = c.st().ok;
      const opts = [['', '고르세요'], ...b.rights.map(r => [r, r])];
      const tbl = h('table', { class: 't' }, h('tr', {}, h('th', {}, b.left_label || '항목'), h('th', {}, b.right_label || '짝'), h('th', {}, '채점')),
        b.lefts.map(l => {
          const r = res ? res[l] : (ok ? true : null);
          return h('tr', { class: r === false ? 'wrongrow' : '' }, h('td', { html: '<b>' + md(l) + '</b>' }),
            h('td', {}, ok ? (mt[l] || '') : selectBox(opts, mt[l] || '', v => { mt[l] = v; })),
            h('td', { class: r === null ? '' : r ? 'ok' : 'bad' }, r === null ? '' : r ? '✅' : '❌ 다시'));
        }));
      box.append(h('div', { class: 'card' + (ok ? '' : ' focus') }, h('h4', {}, (ok ? '✅ ' : '🔗 ') + (b.title || '짝짓기')), b.text ? mdp(b.text) : null, tbl,
        ok ? h('div', { class: 'fb ok', html: '✅ 모두 맞게 짝지었어요.' + (c.st().explain ? ' ' + md(c.st().explain) : '') }) : h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
          const r = await invoke('blk.check', c.m.id, b.id, mt);
          if (r.correct) c.changed(); else draw(r.result, r.hints, r.text);
        } }, '확인')),
        msg ? h('div', { class: 'fb bad' }, msg) : null, hintList(hints, c.st().tries), guideRow(b)));
    };
    draw(null, null);
    w.append(box);
  },
  order(w, c) {
    const b = c.b, s = c.st(), key = 'od' + c.m.id + b.id;
    const ord = drafts[key] || (drafts[key] = (s.value || {}).order ? [...s.value.order] : b.items.map(x => x.id));
    const nm = Object.fromEntries(b.items.map(x => [x.id, x]));
    const list = h('div'), fb = h('div');
    const draw = wrong => {
      list.innerHTML = '';
      const ok = c.st().ok;
      ord.forEach((id, i) => list.append(h('div', { class: 'orditem' + (wrong === i ? ' wrong' : '') }, h('b', {}, `${i + 1}.`),
        h('span', { style: 'flex:1', html: md(nm[id].text) + (nm[id].desc ? ' <small>(' + md(nm[id].desc) + ')</small>' : '') }),
        ok ? null : h('button', { class: 'btn small ghost', disabled: i === 0, onclick: () => { [ord[i - 1], ord[i]] = [ord[i], ord[i - 1]]; draw(); } }, '▲'),
        ok ? null : h('button', { class: 'btn small ghost', disabled: i === ord.length - 1, onclick: () => { [ord[i + 1], ord[i]] = [ord[i], ord[i + 1]]; draw(); } }, '▼'))));
    };
    draw();
    w.append(h('div', { class: 'card' + (s.ok ? '' : ' focus') }, h('h4', {}, (s.ok ? '✅ ' : '🧩 ') + (b.title || '순서 맞추기')), b.text ? mdp(b.text) : null, list,
      s.ok ? h('div', { class: 'fb ok', html: '✅ 순서가 맞아요.' + (s.explain ? ' ' + md(s.explain) : '') }) : h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
        const r = await invoke('blk.check', c.m.id, b.id, ord);
        if (r.correct) c.changed(); else { fb.className = 'fb bad'; fb.textContent = r.text; draw(r.wrong); }
      } }, '순서 확인'), h('span', { class: 'hint' }, '▲▼ 버튼으로 순서를 바꿔요.')), fb, guideRow(b)));
  },
  classify(w, c) {
    const b = c.b, C = b.categories, PL = b.primary_label || '직접 원인';
    w.append(h('div', { class: 'card' }, h('h4', {}, (c.st().ok ? '✅ ' : '🔎 ') + (b.title || '분류하기')), b.text ? mdp(b.text) : null,
      h('div', {}, C.map(([k, n, d]) => h('span', { class: 'chk-chip', title: d || '' }, n))), guideRow(b)));
    const name = k => (C.find(x => x[0] === k) || [k, k])[1];
    for (const it of b.items) {
      const box = h('div');
      const draw = () => {
        box.innerHTML = '';
        const st = c.st(), z = (st.value || {})[it.id], ex = (st.explains || {})[it.id];
        const card = h('div', { class: 'card' + (z ? '' : ' focus') }, h('h4', {}, (z ? '✅ ' : '📂 ') + it.title), caseBody(it));
        if (z) {
          const cs = b.contrib === false ? '' : (z.contrib.length ? ` · 함께 작용한 것: ${z.contrib.map(name).join(', ')}` : '');
          card.append(h('div', { class: 'fb ok' }, `✅ ${PL}: ${name(z.primary)}${cs}`));
          if (ex) card.append(h('div', { class: 'hint', html: md(ex) }));
          box.append(card);
          return;
        }
        const key = 'cl' + c.m.id + b.id + it.id, d = drafts[key] || (drafts[key] = { p: null, k: new Set() });
        const pr = choiceGroup(key, C.map(([k, n, ds]) => [k, n, ds]), d.p, v => { d.p = v; });
        const ks = b.contrib === false ? null : h('div', { class: 'row' }, C.map(([k, n]) => {
          const cb = h('input', { type: 'checkbox', checked: d.k.has(k), onchange: () => { cb.checked ? d.k.add(k) : d.k.delete(k); } });
          return h('label', { class: 'choice' }, cb, h('span', {}, n));
        }));
        const fb = h('div');
        card.append(...[h('div', { class: 'qtext' }, `${PL}${josa(PL, '을', '를')} ${b.contrib === false ? '' : '하나 '}고르세요`), pr,
          ks ? h('div', { class: 'qtext' }, '함께 작용한 것도 있으면 골라 주세요 (여러 개 골라도 돼요)') : null, ks,
          h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
            if (!d.p) { fb.className = 'fb bad'; fb.textContent = '⚠ 하나를 골라 주세요.'; return; }
            const r = await invoke('blk.check', c.m.id, b.id, { item: it.id, primary: d.p, contrib: [...d.k] });
            if (r.correct) c.changed(); else if (r.item_ok) draw(); else { fb.className = 'fb bad'; fb.textContent = r.text || r.msg || '다시 확인해 주세요.'; markWrong(pr); }
          } }, '확인')), fb].filter(Boolean));
        box.append(card);
      };
      draw();
      w.append(box);
    }
  },
  // 글쓰기: 정답을 판정하지 않아요. 최소 글자 수만 채우면 제출되고, 제출한 뒤에 모범답안이 보여요 (2026-10-19 피드백)
  write(w, c) {
    const b = c.b, s = c.st(), key = 'wr' + c.m.id + b.id;
    const d = drafts[key] || (drafts[key] = { t: s.value || '' });
    const tb = textBox(d.t, b.min || 10, b.placeholder || '꼭 입력해 주세요', v => { d.t = v; });
    const fb = h('div');
    if (s.ok) fb.append(h('div', { class: 'fb ok' }, '✅ 제출했어요.'));
    w.append(h('div', { class: 'card' + (s.ok ? '' : ' focus') }, h('h4', {}, (s.ok ? '✅ ' : '✍ ') + (b.title || '글쓰기')), b.prompt ? mdp(b.prompt) : null,
      h('div', { class: 'fieldrow' }, tb.ta, tb.line),
      h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
        const r = await invoke('blk.check', c.m.id, b.id, tb.ta.value);
        if (r.correct) c.changed(); else { fb.className = 'fb bad'; fb.textContent = r.text; }
      } }, s.ok ? '고쳐서 다시 제출' : '제출하기'), b.guide ? guideBtn(b.guide_label || '📘 가이드북', b.guide) : null), fb,
      s.ok && s.model ? h('div', { class: 'model' }, h('div', { class: 'model-h' }, b.model_label || '📝 모범답안'), h('div', { class: 'model-t' }, s.model),
        s.explain ? h('div', { class: 'hint', html: md(s.explain) }) : null) : null,
      s.ok && s.checklist && s.checklist.length ? h('div', { class: 'selfcheck' }, h('div', { class: 'model-h' }, '🔍 내 글 점검'),
        s.checklist.map(t => h('label', { class: 'choice' }, h('input', { type: 'checkbox' }), h('span', {}, t)))) : null));
  },
  decide(w, c) {
    const b = c.b, s = c.st(), key = 'dc' + c.m.id + b.id;
    const d = drafts[key] || (drafts[key] = { ch: {}, rs: {} });
    if (s.value) for (const k in s.value) { d.ch[k] = d.ch[k] || s.value[k].choice; d.rs[k] = d.rs[k] || s.value[k].reason; }
    const opts = b.options || [['accept', '받아들임'], ['reject', '거절하고 대안 제시']];
    const res = {}, box = h('div'), fbAll = h('div');
    const draw = () => {
      box.innerHTML = '';
      const ok = c.st().ok;
      for (const it of b.items) {
        const p = A.members[it.who];
        const ch = choiceGroup(key + it.id, opts, d.ch[it.id] || null, v => { d.ch[it.id] = v; });
        const tb = textBox(d.rs[it.id] || '', b.reason_min || 10, '꼭 입력해 주세요 · 결정한 이유 또는 대안', v => { d.rs[it.id] = v; });
        const r = res[it.id], ex = ok ? (c.st().explains || {})[it.id] : null;
        if (r && !r.ok) markWrong(ch);
        box.append(h('div', { class: 'card' + (r && !r.ok ? ' focus' : '') },
          h('div', { class: 'voice' }, p ? h('span', { class: 'avatar', style: `background:${p.color}` }, p.name[0]) : null,
            h('div', {}, p ? h('b', {}, `${p.name} (${p.role})`) : null, ' ', h('span', { html: '“' + md(it.text) + '”' }))),
          ch, tb.ta, tb.line, r && !r.ok ? h('div', { class: 'fb bad' }, r.text) : null, ex ? h('div', { class: 'hint', html: md(ex) }) : null));
      }
    };
    draw();
    w.append(h('div', { class: 'card' + (s.ok ? '' : ' focus') }, h('h4', {}, (s.ok ? '✅ ' : '🗣 ') + (b.title || '결정하기')), b.text ? mdp(b.text) : null, box,
      h('div', { class: 'row' }, h('button', { class: 'btn', onclick: async () => {
        const r = await invoke('blk.check', c.m.id, b.id, { choices: d.ch, reasons: d.rs });
        for (const k in res) delete res[k];
        (r.items || []).forEach(x => { res[x.id] = x; });
        if (r.correct) c.changed(); else { fbAll.className = 'fb bad'; fbAll.textContent = r.text; draw(); }
      } }, s.ok ? '다시 저장' : '결정 저장')), fbAll, guideRow(b)));
  },
  // 바이브 코딩 실습: 칸마다 문장을 골라 프롬프트를 만들고, 가상 AI에게 보내고, AI가 만든 코드의 실제 실행 결과를 보고 고쳐 다시 보냄
  vibe(w, c) {
    const b = c.b, s = c.st(), key = 'vb' + c.m.id + b.id;
    const d = drafts[key] || (drafts[key] = { picks: {}, reply: null, attempts: [], loaded: false });
    const replyBox = h('div', { class: 'vb-replybox' }), fb = h('div'), promptBox = h('pre', { class: 'vb-prompt' });
    const promptText = () => b.slots.map(sl => { const o = sl.options.find(x => String(x.i) === String(d.picks[sl.id])); return o && o.text ? o.text : ''; }).filter(Boolean).join('\n');
    const drawPrompt = () => { const t = promptText(); promptBox.textContent = t || '칸마다 문장을 고르면 여기에 프롬프트가 만들어져요.'; promptBox.classList.toggle('empty', !t); };
    const drawReply = () => {
      replyBox.innerHTML = '';
      const r = d.reply;
      if (r) replyBox.append(vibeReply(r));
      if (d.attempts.length > 1 || (d.attempts.length && !d.attempts[0].ok)) replyBox.append(h('details', { class: 'vb-log', open: !(r && r.ok) }, h('summary', {}, `📝 프롬프트 개선 기록 (${d.attempts.length}번)`),
        h('table', { class: 't' }, h('tr', {}, h('th', {}, '시도'), h('th', {}, '부족했던 칸'), h('th', {}, '결과')),
          d.attempts.map(a => h('tr', {}, h('td', {}, a.no), h('td', {}, a.ok ? '✅ 성공' : (a.slot || '')), h('td', {}, a.result))))));
    };
    if (!d.loaded && (s.ok || s.tries || b.has_start)) {      // 다시 그릴 때 파이썬에 남은 마지막 응답과 ‘지금까지의 코드’를 받아 옴 (선생님용은 처음부터 채워져 있음)
      d.loaded = true;
      invoke('app.vibe', c.m.id, b.id).then(r => {
        d.reply = r.reply; d.attempts = r.attempts || []; d.start = r.start;
        for (const k in r.picks || {}) d.picks[k] = String(r.picks[k]);
        c.redraw();
      });
    }
    const send = async () => {
      if (b.slots.some(sl => d.picks[sl.id] === undefined)) { fb.className = 'fb bad'; fb.textContent = '⚠ 칸마다 문장을 하나씩 골라 주세요.'; return; }
      fb.className = 'fb'; fb.textContent = '⏳ 가상 AI가 코드를 만들고 실행하는 중이에요…';
      const r = await invoke('blk.check', c.m.id, b.id, { picks: d.picks });
      fb.className = ''; fb.textContent = '';
      d.loaded = true;
      if (r.reply) { d.reply = r.reply; d.attempts = r.attempts || []; }
      if (r.correct) c.changed(); else drawReply();
      w.querySelectorAll('.vb-slot .choice').forEach(l => l.classList.remove('wrong'));
      if (!r.correct && r.reply && r.reply.slot_id) markWrong(w.querySelector(`.vb-slot[data-slot="${r.reply.slot_id}"]`));
      const el = wrapEl().querySelector('.vb-replybox'), sc = el && el.closest('.scroll');      // 바깥 페이지는 움직이지 않고 미션 화면만 응답 위치로
      if (sc) sc.scrollTop += el.getBoundingClientRect().top - sc.getBoundingClientRect().top - 8;
    };
    const wrapEl = () => w;
    w.append(h('div', { class: 'card' + (s.ok ? '' : ' focus') }, h('h4', {}, (s.ok ? '✅ ' : '🤖 ') + (b.title || '바이브 코딩 실습')),
      b.ask ? voicesView([b.ask]) : null, b.text ? mdp(b.text) : null,
      d.start ? h('div', { class: 'vb-start' }, h('div', { class: 'vb-label' }, h('b', {}, '📌 지금까지의 코드와 결과')), vibeReply(Object.assign({}, d.start, { open_code: true }))) : null,
      (b.files || []).map(f => h('details', { class: 'vb-data', open: true }, h('summary', {}, `📄 ${f.name} (${f.n}행, 앞부분만 보여요)`),
        h('div', { class: 'vb-tbl' }, h('table', { class: 't' }, h('tr', {}, f.head.map(x => h('th', {}, x))), f.rows.map(r => h('tr', {}, r.map(x => h('td', {}, x)))))))),
      b.slots.map(sl => h('div', { class: 'vb-slot', 'data-slot': sl.id }, h('div', { class: 'vb-label' }, h('b', {}, sl.label), sl.why ? h('span', { class: 'hint' }, ' ' + sl.why) : null),
        choiceGroup(key + sl.id, sl.options.map(o => [o.i, o.text || '(이 칸은 비워 두기)']), d.picks[sl.id] === undefined ? null : d.picks[sl.id], v => { d.picks[sl.id] = v; drawPrompt(); }, { col: true }))),
      h('div', { class: 'vb-label' }, h('b', {}, '✍ 내 프롬프트')), promptBox,
      h('div', { class: 'row' }, h('button', { class: 'btn', onclick: send }, s.ok || d.attempts.length ? '🤖 다시 보내기' : '🤖 가상 AI에게 보내기'), guideRow(b)), fb, replyBox));
    drawPrompt();
    drawReply();
  },
  custom(w, c) {
    const fn = window[c.b.renderer];
    if (typeof fn !== 'function') { w.append(h('div', { class: 'fb bad' }, `화면 함수 ${c.b.renderer}를 찾지 못했어요.`)); return; }
    fn(w, c);
  },
};
// 가상 AI의 응답 한 건: 말 → 코드 → 실제 실행 결과(출력·표·그림·오류) → 만들어진 문서 → 짚어 볼 점
function vibeReply(r) {
  const out = h('div', { class: 'vb-reply' + (r.ok ? ' ok' : '') + (r.slot ? ' miss' : '') }, h('div', { class: 'vb-who' }, '🤖 가상 AI'), r.say ? h('div', { class: 'vb-say', html: md(r.say) }) : null);
  if (r.code) out.append(h('details', { class: 'vb-codebox', open: !!r.open_code }, h('summary', {}, '💻 AI가 만든 코드 (눌러서 보기) ', h('button', { class: 'cpy', onclick: e => { e.preventDefault(); copyText(r.code); } }, '📋 복사')),
    h('pre', { class: 'vb-code', html: hl(r.code) })));
  if (r.code) out.append(h('div', { class: 'vb-label' }, h('b', {}, '▶ 실행 결과')));
  if (r.stdout) out.append(h('pre', { class: 'vb-out' }, r.stdout));
  (r.tables || []).forEach(t => out.append(h('div', { class: 'vb-tbl', html: t })));
  (r.figs || []).forEach(u => out.append(h('img', { class: 'blkimg', src: u, title: '눌러서 크게 보기', onclick: () => zoomImg(u, '') })));
  if (r.error) out.append(h('pre', { class: 'vb-err' }, r.error));
  if (r.report) out.append(h('div', { class: 'vb-label' }, h('b', {}, '📄 AI가 만든 문서')), h('iframe', { class: 'preview vb-doc', srcdoc: '<meta charset="utf-8"><style>' + VB_DOC_CSS + '</style>' + r.report }));
  if (r.problem) out.append(h('div', { class: 'vb-problem' }, h('b', {}, '🔎 결과를 보니: '), h('span', { html: md(r.problem) })));
  if (r.hint) out.append(h('div', { class: 'vb-problem' }, h('b', {}, `❌ ${r.slot ? '‘' + r.slot + '’ 칸의 ' : ''}현재 선택을 다시 고민해 보세요: `), h('span', { html: md(r.hint) })));
  if (r.lesson) out.append(h('div', { class: 'fb ok', html: '✅ ' + md(r.lesson) }));
  return out;
}
const VB_DOC_CSS = 'body{font-family:sans-serif;color:#111827;background:#fff;line-height:1.6;font-size:14px;padding:6px 10px}table{border-collapse:collapse;margin:6px 0}td,th{border:1px solid #c8ced6;padding:4px 8px;font-size:13px;text-align:left}th{background:#eef2f7}h1,h2,h3{margin:8px 0 6px}';

// ===================== 단계 등록 =====================
const STEPS = {
  m1: [renderBrief, stepPage('m1', 1), stepPage('m1', 2), stepPage('m1', 3), (b, m) => doneView(b, m, [["m1_gsd_table", "M1 GSD별 판독 가능 수준표"]], 'm2')],
  m2: [renderBrief, stepPage('m2', 1), stepPage('m2', 2), stepPage('m2', 3), (b, m) => doneView(b, m, [["m2_reader_doc", "M2 판독 일치율과 판독 기준 요청"]], 'm3')],
  m3: [renderBrief, stepPage('m3', 1), stepPage('m3', 2), stepPage('m3', 3), (b, m) => doneView(b, m, [["m3_authenticity", "M3 영상 진위 점검표와 수정 기록"]], 'm4')],
  m4: [renderBrief, stepPage('m4', 1), stepPage('m4', 2), stepPage('m4', 3), (b, m) => doneView(b, m, [["m4_contract_doc", "M4 데이터 계약 검사와 합의"]], 'm5')],
  m5: [renderBrief, stepPage('m5', 1), stepPage('m5', 2), stepPage('m5', 3), (b, m) => doneView(b, m, [["m5_model_card", "M5 조건별 성능표와 모델 카드"]], 'm6')],
  m6: [renderBrief, stepPage('m6', 1), stepPage('m6', 2), stepPage('m6', 3), (b, m) => doneView(b, m, [["m6_decoy_table", "M6 기만체 판별 표"]], 'm7')],
  m7: [renderBrief, stepPage('m7', 1), stepPage('m7', 2), stepPage('m7', 3), (b, m) => doneView(b, m, [["m7_reliability_doc", "M7 점수 신뢰도와 보고 문장"]], 'm8')],
  m8: [renderBrief, stepPage('m8', 1), stepPage('m8', 2), stepPage('m8', 3), (b, m) => doneView(b, m, [["m8_queue_doc", "M8 판독 대기열과 승인 기록"]], 'm9')],
  m9: [renderBrief, stepPage('m9', 1), stepPage('m9', 2), stepPage('m9', 3), (b, m) => doneView(b, m, [["m9_review_doc", "M9 사후검토 문서와 수정 기록"]], 'm10')],
  m10: [renderBrief, stepPage('m10', 1), stepPage('m10', 2), stepPage('m10', 3), (b, m) => doneView(b, m, [["m10_final_report", "M10 조건별 성능 보고서와 프롬프트 틀"]], null)],
};
const TOPICS = {
  m1: [null, null, null, null, null],
  m2: [null, null, null, null, null],
  m3: [null, null, null, null, null],
  m4: [null, null, null, null, null],
  m5: [null, null, null, null, null],
  m6: [null, null, null, null, null],
  m7: [null, null, null, null, null],
  m8: [null, null, null, null, null],
  m9: [null, null, null, null, null],
  m10: [null, null, null, null, null],
};

async function resetAll() {
  if (!confirm('모든 진행 상황을 지우고 처음부터 할까요?')) return;
  mstep = {}; seen = {}; cache = {}; drafts = {}; CODE.live = null; BUSY = null;
  for (const k in POLL) clearTimeout(POLL[k]);
  await invoke('app.reset');
  openView('home');
}

applyUi();
invoke('app.state');
</script>
"""

if "app_state" not in globals():
    display(HTML("<div style='padding:14px 16px;border:2px solid #b3261e;border-radius:10px;background:#fce8e6;color:#111827;font-size:15px'>"
                 "⚠️ 먼저 위의 <b>🚀 [먼저 실행] 실습 준비하기</b> 셀을 실행해 주세요. 진행 막대가 100%가 된 뒤 이 셀을 다시 실행하면 메신저가 열려요.</div>"))
else:
    display(HTML(APP_HTML.replace("__WORKSPACE__", WORKSPACE_NAME)))